## Evaluación de modelos de aprendizaje profundo para la predicción de sinergia entre fármacos

#### Abstract: 
La predicción de sinergia entre fármacos es clave para el desarrollo de terapias combinadas frente a enfermedades complejas como el cáncer. Dado que el espacio de combinaciones posibles es demasiado amplio para su evaluación experimental, las técnicas computacionales constituyen una alternativa eficaz para priorizar las combinaciones más prometedoras.
Se analizarán los conjuntos de datos disponibles y se implementarán distintos modelos, incluyendo baselines clásicos y arquitecturas de aprendizaje profundo diseñadas para capturar interacciones entre fármacos y contexto celular. El rendimiento se medirá mediante métricas adecuadas para datos desbalanceados, y se hará especial énfasis en la capacidad de generalización de los modelos, estudiando el escenario en el que un fármaco aparece únicamente en el conjunto de test.

# Índice

1. [Importacion librerias](#Importamos-las-librerías-necesarias)
2. [Carga de datos](#Carga-de-datos)

### LDO CV PARCIAL: CON SOLAPAMIENTO  // STRATIFIED KFOLD
3. [RF LDO](#RANDOM-FOREST:-LDO-CV=10)
4. [RF STRKFOLD](#RANDOM-FOREST:-STRATIFIED-KFOLD)
5. [XGBOOST LDO](#XGBOOST-:-Leave-Drug-Out-CV=10)
6. [XGBOOST STRKFOLD](#XGBOOST:-STRATIFIED-K-FOLD-CV=10)
7. [MULTIMODAL MLP LDO](#MLP:-Leave-Drug-Out-CV=10-(Siamese-Multimodal-Fusion-MLP))
8. [MULTIMODAL MLP STRKFOLD](#MLP-TRIPLE:-STRATIFIED-K-FOLD-CV=10-(Siamese-Multimodal-Fusion-MLP))
9. [MULTIMODAL MLP INTERACT LDO](#MLP-TRIPLE-:-VARAICIÓN-CON-INTERACCIÓN-DE-DROGAS-(Interaction-Aware-Triple-Branch-MLP)-Leave-Drug-Out-CV=10)
10. [MULTIMODAL MLP INTERACT STRKFOLD](#MLP-TRIPLE-:-VARIACIÓN-CON-INTERACCIÓN-DE-DROGAS-(Interaction-Aware-Triple-Branch-MLP)-Stratified-K-Fold-CV=10)
11. [MLP LDO](#MLP-:-Leave-Drug-Out-CV=10)
12. [MLP STRKFOLD](#MLP-:-Stratified-K-Fold-CV=10)
13. [TRANSFORMER LDO](#TRANSFORMER-+-MLP-Leave-Drug-Out-CV=10)
14. [TRANSFORMER STRKFOLD](#TRANSFORMER-+-MLP-Stratified-K-Fold-CV=10)
15. [GNNs LDO](#GNNs-:-Leave-Drug-Out-CV=10)
16. [GNNs STRKFOLD](#GNN:-Stratified-K-Fold-CV=10)

### HOLD OUT CV : SIN SOLAPAMIENTO
17. [LDO SIN SOLAPAMIENTO 1 FOLD](#LDO-sin-solapamiento.-1-FOLD)
18. [RANDOM FOREST HOLD OUT](#REPEATED-HOLD-OUT)
19. [XGBOOST HOLD OUT](#XGBOOST-LDO-DISJUNTO-A-NIVEL-DE-MUESTRA)
20. [TRIPLE MLP HOLD OUT](#TRIPLEMLP-LDO-DISJUNTO-A-NIVEL-DE-MUESTRA)
21. [MLP HOLD OUT](#MLP-LDO-DISJUNTO-A-NIVEL-DE-MUESTRA)
22. [TRANSFORMER HOLD OUT](#TRANSFORMER-LDO-DISJUNTO-A-NIVEL-DE-MUESTRA)
23. [GNN LDO HOLD OUT](#GNN-LDO-DISJUNTO-A-NIVEL-DE-MUESTRA)

### Salidas-Resultados
24. [ANALISIS FINAL](#Estudios-postresultados)



### Importamos las librerías necesarias

In [ ]:
import os
import json
import shutil
import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from torch.utils.tensorboard import SummaryWriter
from PIL import Image
if not hasattr(Image, "Resampling"):
    class Resampling:
        LANCZOS = Image.ANTIALIAS

    Image.Resampling = Resampling

import collections            
import matplotlib.pyplot as plt 
import rdkit
import json
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem import Draw
from rdkit import DataStructs 

import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

### Carga de datos

In [ ]:
##### DRUGS SMILES  ---------------------------------------------------------------------------------------------------------------------
data_smiles = pd.read_pickle('../data/preprocessed/drugs_info.pkl')
# Shape: (4350,5)
# COLUMNAS 'dname', 'cid', 'isomeric_smiles', 'canonical_smiles', 'drugbank_id'
smiles = data_smiles['isomeric_smiles'].values



##### READ CELL_EXPR_DEPMAP  ------------------------------------------------------------------------------------------------------------
data_expr = pd.read_pickle('../data/preprocessed/cell_expr_depmap.pkl') 
# Shape: (171, 19215)
# COLUMNAS: Genes, FILAS: ModelID (Index) Cell lines



##### SYNERGY_EXPR ---------------------------------------------------------------------------------------------------------------------
data_synergy = pd.read_pickle('../data/preprocessed/df_synergy_expr.pkl')
# Shape: (370381, 8)
# COLUMNAS  'drug_row_id', 'drug_col_id', 'cell_line_name', 'study_name', 'synergy_zip', 'synergy_loewe', 'synergy_hsa', 'synergy_bliss'



##### JSON TEJIDOS  ---------------------------------------------------------------------------------------------------------------------
with open('../data/preprocessed/cellline2tissue.json', 'r') as f:
    cellline2tissue = json.load(f)
    
df_tissues = pd.DataFrame(cellline2tissue.items(), columns=['cell_line_name', 'tissue'])
# Diccionario {'A-673': 'bone', 'EW-8': 'bone', 'TC-32': 'bone', 'TC-71': 'bone', ...}

In [ ]:
def synergy_th(val, th=10):
    if abs(val) >= th:
        return 1 if val > 0 else -1
    return 0

### Preparación de Datos

In [ ]:
# VARIABLE BIN : Umbral sinergia (>10 sinergia, <-10 antagonismo, entre -10 y 10 sinergia neutra : 1, -1, 0 respectivamente)
data_synergy['synergy_loewe_bin'] = data_synergy['synergy_loewe'].apply(synergy_th)
data_loewe = data_synergy[
    [
        'drug_row_id',
        'drug_col_id',
        'cell_line_name',
        'study_name',
        'synergy_loewe',
        'synergy_loewe_bin'
    ]
].copy()

# UMBRAL  200 > |x|
threshold1 = 200
threshold2 = -200
print(f"data_loewe len :  {len(data_loewe)} antes del filtrado |200|")
data_loewe_filtered = data_loewe[(data_loewe['synergy_loewe'] <= threshold1) & (data_loewe['synergy_loewe'] >= threshold2)]
print(f"data_loewe len :  {len(data_loewe_filtered)} despues del filtrado ||200||")




# Filtrado por estudios
print("Filtrado antes de dividir estudios: ", data_loewe_filtered.shape)
dloewe_all = data_loewe_filtered[data_loewe_filtered["study_name"].isin(['ALMANAC', 'FRIEDMAN', 'ONEIL', 'ASTRAZENECA'])] # (361486, 6)
dloewe_extra = data_loewe_filtered[~data_loewe_filtered["study_name"].isin(['ALMANAC', 'FRIEDMAN', 'ONEIL', 'ASTRAZENECA'])] #(8801, 6)
print("Filtrado después de dividir estudios: ",dloewe_all.shape)
print("Tamaño dloewe extra: ", dloewe_extra.shape)




# Eliminar el repetido
counts = collections.Counter(smiles)
repetidos = {s: c for s, c in counts.items() if c > 1}
print(f"Número de estructuras que se repiten: {len(repetidos)}")
df_repetidos = data_smiles[data_smiles['isomeric_smiles'].isin(repetidos.keys())]
df_repetidos_ordenado = df_repetidos.sort_values(by='isomeric_smiles')
print(df_repetidos_ordenado.loc[:, ['dname','isomeric_smiles']])
data_smiles_reduced = data_smiles.drop_duplicates(subset=['isomeric_smiles']) 
print(f" \n len data_smiles_reduced: {len(data_smiles_reduced)}, len data_smiles original: {len(data_smiles)}")



# Variable "binaria": sinergia (1) o antagónica (-1), eliminamos los ejemplos con valor de sinergia 0
print("Filtrado antes de eliminar sinergias 0: ", dloewe_all.shape)
dloewe_all = dloewe_all[(dloewe_all['synergy_loewe_bin'] == 1) | (dloewe_all['synergy_loewe_bin'] == -1)]
print("Filtrado después de eliminar sinergias 0: ", dloewe_all.shape)


# Variable tissue (tejido) a partir de cell_line_name
dloewe_all['tissue'] = dloewe_all['cell_line_name'].map(cellline2tissue)
# Eliminamos entradas de tejidos que proporcionan ruido en el análisis (stomach)
print("Filtrado antes de eliminar sinergias stomach: ", dloewe_all.shape)
dloewe_all = dloewe_all[dloewe_all['tissue'] != 'stomach']
print("Filtrado después de eliminar sinergias stomach: ", dloewe_all.shape)

In [ ]:
drug_appearances = pd.concat([
    dloewe_all["drug_row_id"],
    dloewe_all["drug_col_id"]
], ignore_index=True).astype(str)

drug_counts = (
    drug_appearances
    .value_counts()
    .rename_axis("drug_id")
    .reset_index(name="n_apariciones")
)

drug_counts["rank"] = np.arange(1, len(drug_counts) + 1)

x = drug_counts["rank"].to_numpy()
y = drug_counts["n_apariciones"].to_numpy()

plt.figure(figsize=(8.5, 5))

plt.plot(
    x,
    y,
    marker="o",
    markersize=3,
    linewidth=1.2,
    color="#4C78A8"
)

plt.yscale("log")

plt.xlabel("Fármacos ordenados por frecuencia")
plt.ylabel("Número de apariciones, escala logarítmica")
plt.title("Distribución de apariciones por fármaco")
plt.xticks(range(0, 300, 25))
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
print(dloewe_all['synergy_loewe_bin'].value_counts())
print(dloewe_all['study_name'].value_counts())
print(dloewe_all['tissue'].value_counts())
print(dloewe_all['tissue'].isna().sum())


Existe una fuerte dominancia de los estudios 'ALMANAC' y 'FRIEDMAN'. Revisamos si la proporcion de clases cambia mucho por estudio. Es importante porque el modelo podría aprender sesgos asociados al estudio

In [ ]:
pd.crosstab(dloewe_all['study_name'], dloewe_all['synergy_loewe_bin'], normalize='index')

Vemos también la distribución de clase por tejido:

In [ ]:
pd.crosstab(dloewe_all['tissue'], dloewe_all['synergy_loewe_bin'], normalize='index')


Hemos determinado que skin tiene un 24.4% de sinergia, mientras que brain,kidney... apenas un 7%. Debemos tener cuidado, ya que si un modelo predice mejor skin, podria parecer que generaliza bien cuando en realidad esta arpovechando una region del dataset con mas positivos

Numero de fármacos unicos que tenemos: 

In [ ]:
unique_drugs = pd.unique(
    dloewe_all[['drug_row_id', 'drug_col_id']].values.ravel()
)
len(unique_drugs)


Numero de combinaciones unicas

In [ ]:
dloewe_all[['drug_row_id', 'drug_col_id']].drop_duplicates().shape

Repeticiones de registros para la misma combinacion, linea celular y estudio

In [ ]:
dloewe_all.duplicated(
    subset=['drug_row_id', 'drug_col_id', 'cell_line_name', 'study_name']
).sum()


Comprobar que todos los farmacos tienen SMILES disponibles

In [ ]:
synergy_drugs = set(
    pd.unique(dloewe_all[['drug_row_id', 'drug_col_id']].values.ravel())
)

cid_set = set(data_smiles['cid'].dropna().astype(str))
drugbank_set = set(data_smiles['drugbank_id'].dropna().astype(str))

smiles_ids = cid_set | drugbank_set

missing_drugs = {str(d) for d in synergy_drugs} - smiles_ids

len(missing_drugs), missing_drugs


Comprobamos que todas las lineas celulares tienen expresión génica

In [ ]:
synergy_cells = set(dloewe_all['cell_line_name'])
expr_cells = set(data_expr.index)
len(synergy_cells - expr_cells)

Distribucion original de synergy_loewe

In [ ]:
dloewe_all['synergy_loewe'].describe()

Problema desbalanceado

In [ ]:
data_loewe_filtered['synergy_loewe'].describe()


Comprobar proporción de tejidos por study_name:

In [ ]:
print(pd.crosstab(dloewe_all['study_name'], dloewe_all['tissue']))

## En proporción de cada estudio, qué proporción de ejemplos corresponden a cada tejido
pd.crosstab(
    dloewe_all['study_name'],
    dloewe_all['tissue'],
    normalize='index'
)


Tabla que nos determina si la clase positiva depende de combinaciones concretas de estudio y tejido:

In [ ]:
pd.crosstab(
    [dloewe_all['study_name'], dloewe_all['tissue']],
    dloewe_all['synergy_loewe_bin'],
    normalize='index'
)

Revision sobre si las parejas A,B B,A aparecen como combianciones distintas

In [ ]:
dloewe_all.dtypes

In [ ]:
def normalize_drug_id(x):
    if isinstance(x, np.ndarray):
        if x.size == 1:
            return str(x.item())
        return str(tuple(x.tolist()))
    return str(x)

pairs = dloewe_all[['drug_row_id', 'drug_col_id']].copy()
pairs['drug_row_id'] = pairs['drug_row_id'].apply(normalize_drug_id)
pairs['drug_col_id'] = pairs['drug_col_id'].apply(normalize_drug_id)

pairs_ordered = pairs.drop_duplicates()

pairs_unordered = (
    pairs
    .apply(lambda row: tuple(sorted([row['drug_row_id'], row['drug_col_id']])), axis=1)
    .drop_duplicates()
)

print("Combinaciones ordenadas:", len(pairs_ordered))
print("Combinaciones no ordenadas:", len(pairs_unordered))
print("Diferencia:", len(pairs_ordered) - len(pairs_unordered))



In [ ]:
drug_counts = pd.concat([
    dloewe_all[['drug_row_id', 'synergy_loewe_bin']].rename(columns={'drug_row_id': 'drug_id'}),
    dloewe_all[['drug_col_id', 'synergy_loewe_bin']].rename(columns={'drug_col_id': 'drug_id'})
])

pd.crosstab(drug_counts['drug_id'], drug_counts['synergy_loewe_bin'])


Se observó una asociación importante entre el estudio de origen, el tejido y la proporción de clases. En particular, FRIEDMAN contiene exclusivamente muestras de tejido skin y presenta una proporción de sinergia superior a ALMANAC. Por ello, el rendimiento de los modelos se evaluó teniendo en cuenta posibles sesgos derivados del origen experimental de los datos.


Distribucion de clase por fármaco

In [ ]:
drug_labels = pd.concat([
    dloewe_all[['drug_row_id', 'synergy_loewe_bin']]
        .rename(columns={'drug_row_id': 'drug_id'}),
    dloewe_all[['drug_col_id', 'synergy_loewe_bin']]
        .rename(columns={'drug_col_id': 'drug_id'})
])

drug_labels['drug_id'] = drug_labels['drug_id'].apply(normalize_drug_id)

drug_class_counts = pd.crosstab(
    drug_labels['drug_id'],
    drug_labels['synergy_loewe_bin']
)

drug_class_counts['total'] = drug_class_counts.sum(axis=1)
drug_class_counts['pos_rate'] = drug_class_counts.get(1, 0) / drug_class_counts['total']

drug_class_counts.sort_values('total', ascending=False).head(20)


Esto significa que algunos fármacos están muy asociados a una clase concreta dentro del dataset.

In [ ]:
print((drug_class_counts['total'] <50).sum())
print((drug_class_counts['total'] >50).sum())


## Esto te dirá si el split por fármaco unseen puede quedar inestable.

drug_study = pd.concat([
    dloewe_all[['drug_row_id', 'study_name']]
        .rename(columns={'drug_row_id': 'drug_id'}),
    dloewe_all[['drug_col_id', 'study_name']]
        .rename(columns={'drug_col_id': 'drug_id'})
])

drug_study['drug_id'] = drug_study['drug_id'].apply(normalize_drug_id)

drug_study_counts = pd.crosstab(
    drug_study['drug_id'],
    drug_study['study_name']
)

drug_study_counts['n_studies'] = (drug_study_counts > 0).sum(axis=1)

drug_study_counts['n_studies'].value_counts().sort_index()


In [ ]:
drug_class_counts['total'].hist(bins=50)

#drug_class_counts.sort_values('total', ascending=False)['total'].head(20).plot(kind='bar')


In [ ]:
# Lineas celulares que solo aparecen en un estudio:

cell_study_counts = pd.crosstab(
    dloewe_all['cell_line_name'],
    dloewe_all['study_name']
)

cell_study_counts['n_studies'] = (cell_study_counts > 0).sum(axis=1)

cell_study_counts['n_studies'].value_counts().sort_index()


La mayoría de líneas celulares aparecen en un único estudio, lo que sugiere una asociación fuerte entre contexto celular y origen experimental. Por ello, los resultados deben interpretarse teniendo en cuenta que parte de la variabilidad celular puede estar ligada al estudio de procedencia.


Conclusiones generales: El dataset es utilizable, pero se han identificado tres sesgos importantes:

- Fuerte desbalance de clases;
- Asociación entre estudio, tejido y clase;
- Variabilidad alta entre fármacos.
Lo siguiente será el merge y, sobre todo, cómo se define el split.

Ultimas comprobaciones: Farmacos/Cell lines con una sola clase

In [ ]:
only_one_class = (drug_class_counts[[-1, 1]] > 0).sum(axis=1) == 1
only_one_class.sum()

only_antagonism = (drug_class_counts[-1] > 0) & (drug_class_counts[1] == 0)
only_synergy = (drug_class_counts[-1] == 0) & (drug_class_counts[1] > 0)

print("Solo antagonismo:", only_antagonism.sum())
print("Solo sinergia:", only_synergy.sum())

cell_class_counts = pd.crosstab(
    dloewe_all['cell_line_name'],
    dloewe_all['synergy_loewe_bin']
)

only_one_class_cells = (cell_class_counts[[-1, 1]] > 0).sum(axis=1) == 1

only_one_class_cells.sum()


Aunque se identificaron 7 fármacos y 5 líneas celulares asociados únicamente a una clase tras el filtrado binario, no se eliminaron del conjunto de datos. A diferencia del tejido stomach, que constituía una categoría completa con solo 14 muestras y ausencia total de variabilidad de clase, estos fármacos y líneas celulares representan entidades individuales dentro de un conjunto mucho más amplio y pueden reflejar patrones reales presentes en los datos. Su eliminación podría introducir un sesgo adicional al forzar artificialmente la presencia de ambas clases para cada entidad.


No obstante, esta característica se tuvo en cuenta en el diseño de los splits y en la interpretación de resultados, especialmente en el escenario Leave-Drug-Out.

Se eliminan categorías muy pequeñas, completas y poco informativas que pueden distorsionar análisis estratificados.
No se eliminan automáticamente entidades individuales por presentar una única clase

In [ ]:
print(dloewe_all['cell_line_name'].value_counts().describe())
dloewe_all['cell_line_name'].value_counts().head(20)

La distribución de muestras por línea celular es muy heterogénea, con una mediana de 100 muestras y algunas líneas celulares con varios miles de observaciones. Esto indica que el conjunto de datos no está uniformemente distribuido entre contextos celulares, por lo que los resultados agregados pueden estar influidos por las líneas celulares más representadas.

### Merge de los datasets:

Los Morgan fingerprints se utilizaron como representación molecular estructural de los fármacos, ya que codifican subestructuras químicas locales en vectores binarios de tamaño fijo y son ampliamente empleados en tareas de modelado QSAR y predicción de propiedades farmacológicas.

La expresión génica de la línea celular se incorporó para representar el contexto biológico en el que se evalúa la combinación farmacológica, dado que la respuesta a fármacos puede depender fuertemente del estado molecular de la célula.

Dado el elevado número de genes disponibles, se seleccionaron los 500 genes con mayor varianza en el conjunto de entrenamiento. Esta reducción dimensional conserva genes con mayor variabilidad entre líneas celulares, reduce ruido y complejidad computacional, y evita introducir miles de variables en relación con el número limitado de líneas celulares.


In [ ]:
def smiles_to_fp(smiles):
    if pd.isna(smiles):
        return None

    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        return None

    return np.array(
        AllChem.GetMorganFingerprintAsBitVect(mol, radius=2, nBits=1024)
    )


# --- 1. PREPARACIÓN Y REORDENAMIENTO ---
dloewe_all_clean = dloewe_all.copy()

# --- 2. LOOKUPS (prioridad: drugbank_id, fallback: cid) ---
lookup_smiles = data_smiles_reduced[['drugbank_id', 'cid', 'isomeric_smiles']].copy()

drugbank_to_smiles = (
    lookup_smiles.dropna(subset=['drugbank_id'])
    .drop_duplicates(subset=['drugbank_id'])
    .set_index('drugbank_id')['isomeric_smiles']
)

cid_to_smiles = (
    lookup_smiles.dropna(subset=['cid'])
    .assign(cid=lambda d: d['cid'].astype(str).str.strip())
    .drop_duplicates(subset=['cid'])
    .set_index('cid')['isomeric_smiles']
)

# --- 3. MAPEAR SMILES (primero por drugbank_id, si falla por cid) ---
df_final = dloewe_all_clean.copy()

# Row drug
df_final['smiles_row'] = df_final['drug_row_id'].map(drugbank_to_smiles)
mask_row_nan = df_final['smiles_row'].isna()
df_final.loc[mask_row_nan, 'smiles_row'] = (
    df_final.loc[mask_row_nan, 'drug_row_id']
    .astype(str)
    .str.strip()
    .map(cid_to_smiles)
)

# Col drug
df_final['smiles_col'] = df_final['drug_col_id'].map(drugbank_to_smiles)
mask_col_nan = df_final['smiles_col'].isna()
df_final.loc[mask_col_nan, 'smiles_col'] = (
    df_final.loc[mask_col_nan, 'drug_col_id']
    .astype(str)
    .str.strip()
    .map(cid_to_smiles)
)

# ORDENAMIENTO LEXICOGRÁFICO (Garantiza A+B == B+A)
# Aseguramos que los IDs sean strings y no haya nulos antes de comparar
df_final['drug_row_id'] = df_final['drug_row_id'].astype(str)
df_final['drug_col_id'] = df_final['drug_col_id'].astype(str)

# Ahora la comparación funcionará perfectamente (letras contra letras)
mask = df_final['drug_row_id'] > df_final['drug_col_id']

# El intercambio sigue igual
df_final.loc[mask, ['drug_row_id', 'drug_col_id', 'smiles_row', 'smiles_col']] = \
    df_final.loc[mask, ['drug_col_id', 'drug_row_id', 'smiles_col', 'smiles_row']].values

# Quitamos filas sin SMILES tras aplicar ambos cruces (drugbank_id/cid)
df_final = df_final.dropna(subset=['smiles_row', 'smiles_col'])

# Creamos un diccionario de SMILES a fingerprints para evitar cálculos repetidos
unique_smiles = pd.concat([df_final['smiles_row'], df_final['smiles_col']]).unique()
smiles_dict = {s: smiles_to_fp(s) for s in unique_smiles}

# Mapeamos los fingerprints
df_final['fp_row'] = df_final['smiles_row'].map(smiles_dict)
df_final['fp_col'] = df_final['smiles_col'].map(smiles_dict)
df_final = df_final.dropna(subset=['fp_row', 'fp_col'])


# Creamos matrices de bits
fps_row_matrix = np.stack(df_final['fp_row'].values)
fps_col_matrix = np.stack(df_final['fp_col'].values)

row_fp_cols = [f'drug_row_bit_{i}' for i in range(1024)]
col_fp_cols = [f'drug_col_bit_{i}' for i in range(1024)]

df_fps_row = pd.DataFrame(fps_row_matrix, columns=row_fp_cols, index=df_final.index)
df_fps_col = pd.DataFrame(fps_col_matrix, columns=col_fp_cols, index=df_final.index)

# Metadata y labels (sin genes todavía)
df_meta = df_final[['drug_row_id', 'drug_col_id', 'cell_line_name', 'study_name', 'tissue', 'synergy_loewe_bin']]

# Dataset que usaremos para los Splits
# Nota: Los genes los pegaremos justo antes de entrenar cada Fold
dataset_pre_split = pd.concat([df_meta, df_fps_row, df_fps_col], axis=1)

In [ ]:
df_final.duplicated(
    subset=['drug_row_id', 'drug_col_id', 'cell_line_name', 'study_name']
).sum()


In [ ]:
dup = df_final[df_final.duplicated(
    subset=['drug_row_id', 'drug_col_id', 'cell_line_name', 'study_name'],
    keep=False
)]

dup.shape


In [ ]:
dup.groupby(
    ['drug_row_id', 'drug_col_id', 'cell_line_name', 'study_name']
)['synergy_loewe_bin'].nunique().value_counts()


Para aquellas entradas donde un mismo grupo tenga la misma etiqueta binaria, se agrega el grupo y conservamos una unica fila. Si el grupo tiene etiqeutas contradictorias, eliminariamos el grupo por completo.

Son contradicciones directas para la misma combinación, línea celular y estudio. Si una misma muestra experimental equivalente aparece como sinérgica y antagónica, no quieres imponer una decisión artificial.

Además, si son solo 43 grupos, el impacto sobre el tamaño del dataset será mínimo y la justificación es limpia.

Tras normalizar el orden de los fármacos para considerar equivalentes las combinaciones A+B y B+A, se identificaron duplicados para la misma combinación, línea celular y estudio. Cuando los duplicados presentaban etiquetas consistentes, se conservaron como una única observación. Los casos con etiquetas contradictorias se eliminaron al considerarse ambiguos, ya que no proporcionaban una señal fiable para el aprendizaje supervisado.


In [ ]:
def normalize_drug_id(x):
    if isinstance(x, np.ndarray):
        if x.size == 1:
            return str(x.item()).strip()
        return str(tuple(x.tolist())).strip()
    return str(x).strip()


def smiles_to_fp(smiles):
    if pd.isna(smiles):
        return None

    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        return None

    return np.array(
        AllChem.GetMorganFingerprintAsBitVect(mol, radius=2, nBits=1024)
    )


# --- 1. PREPARACION ---
dloewe_all_clean = dloewe_all.copy()
print("Filas iniciales:", len(dloewe_all_clean))

dloewe_all_clean['drug_row_id'] = dloewe_all_clean['drug_row_id'].apply(normalize_drug_id)
dloewe_all_clean['drug_col_id'] = dloewe_all_clean['drug_col_id'].apply(normalize_drug_id)


# --- 2. LOOKUPS: prioridad drugbank_id, fallback cid ---
lookup_smiles = data_smiles_reduced[['drugbank_id', 'cid', 'isomeric_smiles']].copy()

lookup_smiles['drugbank_id'] = lookup_smiles['drugbank_id'].apply(
    lambda x: normalize_drug_id(x) if pd.notna(x) else np.nan
)
lookup_smiles['cid'] = lookup_smiles['cid'].apply(
    lambda x: normalize_drug_id(x) if pd.notna(x) else np.nan
)

drugbank_to_smiles = (
    lookup_smiles
    .dropna(subset=['drugbank_id'])
    .drop_duplicates(subset=['drugbank_id'])
    .set_index('drugbank_id')['isomeric_smiles']
)

cid_to_smiles = (
    lookup_smiles
    .dropna(subset=['cid'])
    .drop_duplicates(subset=['cid'])
    .set_index('cid')['isomeric_smiles']
)


# --- 3. MAPEAR SMILES ---
df_final = dloewe_all_clean.copy()

df_final['smiles_row'] = df_final['drug_row_id'].map(drugbank_to_smiles)
mask_row_nan = df_final['smiles_row'].isna()
df_final.loc[mask_row_nan, 'smiles_row'] = (
    df_final.loc[mask_row_nan, 'drug_row_id'].map(cid_to_smiles)
)

df_final['smiles_col'] = df_final['drug_col_id'].map(drugbank_to_smiles)
mask_col_nan = df_final['smiles_col'].isna()
df_final.loc[mask_col_nan, 'smiles_col'] = (
    df_final.loc[mask_col_nan, 'drug_col_id'].map(cid_to_smiles)
)

print("Filas sin smiles_row:", df_final['smiles_row'].isna().sum())
print("Filas sin smiles_col:", df_final['smiles_col'].isna().sum())

before_smiles_drop = len(df_final)
df_final = df_final.dropna(subset=['smiles_row', 'smiles_col']).reset_index(drop=True)
print("Filas eliminadas por falta de SMILES:", before_smiles_drop - len(df_final))


# --- 4. ORDENAMIENTO LEXICOGRAFICO: A+B == B+A ---
mask = df_final['drug_row_id'] > df_final['drug_col_id']

df_final.loc[mask, ['drug_row_id', 'drug_col_id', 'smiles_row', 'smiles_col']] = (
    df_final.loc[mask, ['drug_col_id', 'drug_row_id', 'smiles_col', 'smiles_row']].values
)


# --- 5. RESOLVER DUPLICADOS TRAS NORMALIZAR A+B / B+A ---
group_cols = ['drug_row_id', 'drug_col_id', 'cell_line_name', 'study_name']

duplicated_rows = df_final.duplicated(subset=group_cols, keep=False).sum()
print("Filas en grupos duplicados tras ordenar farmacos:", duplicated_rows)

label_nunique = (
    df_final
    .groupby(group_cols)['synergy_loewe_bin']
    .nunique()
    .reset_index(name='n_labels')
)

contradictory_groups = label_nunique[label_nunique['n_labels'] > 1][group_cols]
print("Grupos contradictorios:", len(contradictory_groups))

df_final = df_final.merge(
    contradictory_groups.assign(is_contradictory=True),
    on=group_cols,
    how='left'
)

print("Filas en grupos contradictorios:", df_final['is_contradictory'].sum())

df_final = (
    df_final[df_final['is_contradictory'].isna()]
    .drop(columns='is_contradictory')
    .reset_index(drop=True)
)

before_agg = len(df_final)

df_final = (
    df_final
    .groupby(group_cols, as_index=False)
    .agg({
        'synergy_loewe': 'mean',
        'synergy_loewe_bin': 'first',
        'tissue': 'first',
        'smiles_row': 'first',
        'smiles_col': 'first'
    })
)

print("Filas eliminadas/agregadas por duplicados consistentes:", before_agg - len(df_final))
print("Tamaño tras resolver duplicados:", df_final.shape)

remaining_duplicates = df_final.duplicated(subset=group_cols).sum()
print("Duplicados restantes:", remaining_duplicates)


# --- 6. FINGERPRINTS ---
unique_smiles = pd.concat([df_final['smiles_row'], df_final['smiles_col']]).unique()
smiles_dict = {s: smiles_to_fp(s) for s in unique_smiles}

df_final['fp_row'] = df_final['smiles_row'].map(smiles_dict)
df_final['fp_col'] = df_final['smiles_col'].map(smiles_dict)

invalid_fp_rows = df_final['fp_row'].isna().sum()
invalid_fp_cols = df_final['fp_col'].isna().sum()

print("Fingerprints invalidos row:", invalid_fp_rows)
print("Fingerprints invalidos col:", invalid_fp_cols)

before_fp_drop = len(df_final)
df_final = df_final.dropna(subset=['fp_row', 'fp_col']).reset_index(drop=True)
print("Filas eliminadas por fingerprints invalidos:", before_fp_drop - len(df_final))


# --- 7. MATRICES DE BITS ---
fps_row_matrix = np.stack(df_final['fp_row'].values)
fps_col_matrix = np.stack(df_final['fp_col'].values)

row_fp_cols = [f'drug_row_bit_{i}' for i in range(1024)]
col_fp_cols = [f'drug_col_bit_{i}' for i in range(1024)]

df_fps_row = pd.DataFrame(fps_row_matrix, columns=row_fp_cols, index=df_final.index)
df_fps_col = pd.DataFrame(fps_col_matrix, columns=col_fp_cols, index=df_final.index)


# --- 8. DATASET PRE-SPLIT ---
df_meta = df_final[
    [
        'drug_row_id',
        'drug_col_id',
        'cell_line_name',
        'study_name',
        'tissue',
        'synergy_loewe',
        'synergy_loewe_bin'
    ]
]

dataset_pre_split = pd.concat([df_meta, df_fps_row, df_fps_col], axis=1)

print("Distribucion final de clases antes de mapear a 0/1:")
print(dataset_pre_split['synergy_loewe_bin'].value_counts())
print(dataset_pre_split['synergy_loewe_bin'].value_counts(normalize=True))


# --- 9. VARIABLE OBJETIVO 0/1 ---
dataset_pre_split['synergy_loewe_bin'] = dataset_pre_split['synergy_loewe_bin'].replace(-1, 0)

X_meta = dataset_pre_split[
    [
        'drug_row_id',
        'drug_col_id',
        'cell_line_name',
        'study_name',
        'tissue',
        'synergy_loewe',
        'synergy_loewe_bin'
    ]
].reset_index(drop=True)

X_bits = dataset_pre_split[row_fp_cols + col_fp_cols].reset_index(drop=True)

X_final_df = pd.concat([X_meta, X_bits], axis=1)

names_drug1 = [f'drug1_bit_{i}' for i in range(1024)]
names_drug2 = [f'drug2_bit_{i}' for i in range(1024)]

X_final_df.columns = (
    [
        'drug_row_id',
        'drug_col_id',
        'cell_line_name',
        'study_name',
        'tissue',
        'synergy_loewe',
        'synergy_loewe_bin'
    ]
    + names_drug1
    + names_drug2
)

print(f"Dataset con IDs listo: {X_final_df.shape}")
print("Distribucion final de clases 0/1:")
print(X_final_df['synergy_loewe_bin'].value_counts())
print(X_final_df['synergy_loewe_bin'].value_counts(normalize=True))


In [ ]:
X_final_df["study_name"].value_counts()

El split aleatorio se utiliza como escenario de referencia, mientras que Leave-Drug-Out evalúa un escenario de generalización más exigente en el que el modelo debe predecir combinaciones que incluyen fármacos no observados durante el entrenamiento.


Selección de modelos:
Baselines clásicos -> redes densas simples -> arquitecturas multimodales -> modelos moleculares más expresivos

- RandomForestClassifier: baseline clásico robusto.
- XGBClassifier: baseline tabular fuerte.
- MLP: primera red neuronal sobre features concatenadas.
- Red profunda separando MFP y genes: arquitectura más justificada, porque trata modalidades distintas por ramas.
- Transformer inspirado en paper: modelo más experimental.
- GNN: potencialmente muy interesante, si representas los fármacos como grafos moleculares en vez de fingerprints.

Para los modelos clásicos, cuidado con tener 2048 bits + 500 genes. No es absurdo, pero hay alta dimensionalidad y desbalance. Por eso métricas como accuracy pueden engañar mucho.

Se prioriza reportar: 

AUPRC
ROC-AUC
F1
balanced accuracy
precision
recall
confusion matrix


No se aplicaron técnicas de sobremuestreo sintético, ya que el desbalance refleja una característica real del problema: las combinaciones sinérgicas son mucho menos frecuentes que las no sinérgicas. En su lugar, se priorizó el uso de métricas robustas frente al desbalance y, cuando proceda, pesos de clase en el entrenamiento.



Se va a hacer una estratificación aproximada por fármacos, intentando equilibrar el numero de muestras test por fold y el porcentaje de positivos por fold
Hay una limitación importante: si queremos que los tests no se solapen, entonces una muestra con fármacos asignados a folds distintos solo puede ir a uno de ellos. Por tanto, en cada fold puede haber algunas muestras con fármacos held-out que no se usan ni en train ni en test de ese fold. Es el precio de evitar que la misma muestra se evalúe varias veces.


In [ ]:
def build_balanced_lodo_folds(df, n_splits=10, random_state=42, alpha=1.0):
    df = df.copy()

    df['drug_row_id'] = df['drug_row_id'].astype(str)
    df['drug_col_id'] = df['drug_col_id'].astype(str)

    rng = np.random.default_rng(random_state)

    all_drugs = pd.Index(
        pd.concat([df['drug_row_id'], df['drug_col_id']], ignore_index=True)
        .dropna()
        .unique()
    ).to_numpy()

    # Estadisticas por farmaco: apariciones y positivos
    drug_rows = pd.concat([
        df[['drug_row_id', 'synergy_loewe_bin']]
            .rename(columns={'drug_row_id': 'drug_id'}),
        df[['drug_col_id', 'synergy_loewe_bin']]
            .rename(columns={'drug_col_id': 'drug_id'})
    ], ignore_index=True)

    drug_stats = (
        drug_rows
        .groupby('drug_id')['synergy_loewe_bin']
        .agg(total='count', positives='sum')
        .reset_index()
    )

    drug_stats['drug_id'] = drug_stats['drug_id'].astype(str)

    # Mezcla ligera para desempates reproducibles
    drug_stats = drug_stats.sample(frac=1, random_state=random_state)
    drug_stats = drug_stats.sort_values(
        ['total', 'positives'],
        ascending=False
    ).reset_index(drop=True)

    folds = [set() for _ in range(n_splits)]
    fold_total = np.zeros(n_splits, dtype=float)
    fold_pos = np.zeros(n_splits, dtype=float)

    target_total = drug_stats['total'].sum() / n_splits
    target_pos = drug_stats['positives'].sum() / n_splits

    # Asignacion greedy de farmacos a folds
    for _, row in drug_stats.iterrows():
        drug = row['drug_id']
        total = row['total']
        positives = row['positives']

        scores = []
        for k in range(n_splits):
            new_total = fold_total[k] + total
            new_pos = fold_pos[k] + positives

            total_score = ((new_total - target_total) / target_total) ** 2

            if target_pos > 0:
                pos_score = ((new_pos - target_pos) / target_pos) ** 2
            else:
                pos_score = 0

            scores.append(total_score + alpha * pos_score)

        best_fold = int(np.argmin(scores))

        folds[best_fold].add(drug)
        fold_total[best_fold] += total
        fold_pos[best_fold] += positives

    drug_to_fold = {
        drug: fold_id
        for fold_id, fold_drugs in enumerate(folds)
        for drug in fold_drugs
    }

    # Asignacion de cada muestra a un unico fold de test.
    # Si sus dos farmacos pertenecen a folds distintos, se asigna al fold
    # que deje mejor balance final de muestras y positivos.
    sample_fold = []
    assigned_total = np.zeros(n_splits, dtype=float)
    assigned_pos = np.zeros(n_splits, dtype=float)

    target_sample_total = len(df) / n_splits
    target_sample_pos = df['synergy_loewe_bin'].sum() / n_splits

    row_order = np.arange(len(df))
    rng.shuffle(row_order)

    assigned = np.full(len(df), -1, dtype=int)

    for idx in row_order:
        row = df.iloc[idx]
        y = row['synergy_loewe_bin']

        f1 = drug_to_fold[str(row['drug_row_id'])]
        f2 = drug_to_fold[str(row['drug_col_id'])]

        candidate_folds = sorted(set([f1, f2]))

        scores = []
        for k in candidate_folds:
            new_total = assigned_total[k] + 1
            new_pos = assigned_pos[k] + y

            total_score = ((new_total - target_sample_total) / target_sample_total) ** 2

            if target_sample_pos > 0:
                pos_score = ((new_pos - target_sample_pos) / target_sample_pos) ** 2
            else:
                pos_score = 0

            scores.append(total_score + alpha * pos_score)

        best_fold = candidate_folds[int(np.argmin(scores))]

        assigned[idx] = best_fold
        assigned_total[best_fold] += 1
        assigned_pos[best_fold] += y

    df['lodo_fold'] = assigned

    fold_sets = [set(fold) for fold in folds]

    # Verificacion: cada farmaco pertenece a un unico fold
    total_unique = len(set().union(*fold_sets))
    total_sum = sum(len(s) for s in fold_sets)
    assert total_unique == total_sum, 'Hay farmacos repetidos entre folds LODO'

    for a in range(len(fold_sets)):
        for b in range(a + 1, len(fold_sets)):
            assert fold_sets[a].isdisjoint(fold_sets[b]), f'Solapamiento entre folds {a + 1} y {b + 1}'

    # Verificacion: cada muestra pertenece a un unico fold de test
    assert df['lodo_fold'].between(0, n_splits - 1).all()

    return df, fold_sets


In [ ]:
# Conteo de apariciones por farmaco en drug_row_id y drug_col_id
drug_counts = pd.concat([
    X_final_df['drug_row_id'].astype(str),
    X_final_df['drug_col_id'].astype(str)
]).value_counts()

print("Numero de farmacos unicos:", drug_counts.shape[0])
print(drug_counts.describe())

plt.figure(figsize=(8, 5))
plt.hist(drug_counts.values, bins=40, edgecolor='black', alpha=0.75)
plt.xlabel('Numero de apariciones por farmaco')
plt.ylabel('Numero de farmacos')
plt.title('Distribucion de apariciones por farmaco')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(10, 5))
drug_counts.head(280).sort_values().plot(kind='barh')
plt.xlabel('Numero de apariciones')
plt.ylabel('Farmaco')
plt.title('Top 280 farmacos por numero de apariciones')
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
global_pos_pct = 100 * X_final_df['synergy_loewe_bin'].mean()
print(f"% positivo global: {global_pos_pct:.2f}%")

drug_counts = pd.concat([
    X_final_df['drug_row_id'].astype(str),
    X_final_df['drug_col_id'].astype(str)
]).value_counts()

print("Farmacos unicos:", len(drug_counts))
print(drug_counts.describe())

print("\nTop 20 farmacos:")
print(drug_counts.head(-20))


In [ ]:
def evaluate_lodo_partition(df, fold_sets, target_pos_rate=None):
    if target_pos_rate is None:
        target_pos_rate = df['synergy_loewe_bin'].mean()

    sizes = []
    pos_rates = []

    for fold_set in fold_sets:
        mask = (
            df['drug_row_id'].astype(str).isin(fold_set)
            | df['drug_col_id'].astype(str).isin(fold_set)
        )
        fold_df = df[mask]

        n = len(fold_df)
        pos_rate = fold_df['synergy_loewe_bin'].mean() if n > 0 else 0

        sizes.append(n)
        pos_rates.append(pos_rate)

    sizes = np.array(sizes)
    pos_rates = np.array(pos_rates)

    target_size = len(df) / len(fold_sets)

    size_error = np.mean(((sizes - target_size) / target_size) ** 2)
    pos_error = np.mean((pos_rates - target_pos_rate) ** 2)

    empty_penalty = np.sum(sizes == 0) * 10

    score = size_error + 20 * pos_error + empty_penalty

    return score, sizes, pos_rates


In [ ]:
def build_lodo_folds_random_search(df, n_splits=10, n_iter=5000, random_state=42):
    rng = np.random.default_rng(random_state)

    drugs = pd.Index(
        pd.concat([df['drug_row_id'], df['drug_col_id']], ignore_index=True)
        .astype(str)
        .unique()
    ).to_numpy()

    best_score = np.inf
    best_folds = None
    best_sizes = None
    best_pos_rates = None

    for _ in range(n_iter):
        shuffled = drugs.copy()
        rng.shuffle(shuffled)

        fold_arrays = np.array_split(shuffled, n_splits)
        fold_sets = [set(arr.tolist()) for arr in fold_arrays]

        score, sizes, pos_rates = evaluate_lodo_partition(df, fold_sets)

        if score < best_score:
            best_score = score
            best_folds = fold_sets
            best_sizes = sizes
            best_pos_rates = pos_rates

    return best_folds, best_score, best_sizes, best_pos_rates


In [ ]:
import time

start = time.time()

fold_sets, score, sizes, pos_rates = build_lodo_folds_random_search(
    X_final_df,
    n_splits=10,
    n_iter=10,
    random_state=42
)

print(f"Tiempo: {(time.time() - start):.2f} segundos")

summary = pd.DataFrame({
    'fold': np.arange(1, 11),
    'n_examples': sizes,
    'positive_pct': pos_rates * 100,
    'n_heldout_drugs': [len(s) for s in fold_sets]
})

print(summary)
print("Score:", score)


In [ ]:
test_indices_by_fold = []

for fold_set in fold_sets:
    mask = (
        X_final_df['drug_row_id'].astype(str).isin(fold_set)
        | X_final_df['drug_col_id'].astype(str).isin(fold_set)
    )
    test_indices_by_fold.append(set(X_final_df.index[mask]))

all_test_indices = []
for s in test_indices_by_fold:
    all_test_indices.extend(list(s))

overlap_count = len(all_test_indices) - len(set(all_test_indices))
overlap_pct = 100 * overlap_count / len(all_test_indices)

print("Apariciones test totales:", len(all_test_indices))
print("Muestras test unicas:", len(set(all_test_indices)))
print("Muestras repetidas en test:", overlap_count)
print(f"% solapamiento test: {overlap_pct:.2f}%")


In [ ]:
from collections import Counter

test_indices_by_fold = []

for fold_set in fold_sets:
    mask = (
        X_final_df['drug_row_id'].astype(str).isin(fold_set)
        | X_final_df['drug_col_id'].astype(str).isin(fold_set)
    )
    test_indices_by_fold.extend(X_final_df.index[mask].tolist())

test_count_per_sample = pd.Series(Counter(test_indices_by_fold))

print(test_count_per_sample.value_counts().sort_index())
print(test_count_per_sample.describe())


Debido a que cada observación contiene dos fármacos, los conjuntos de test pueden solaparse entre folds cuando ambos fármacos de una combinación pertenecen a grupos held-out distintos. Este solapamiento se cuantificó y los resultados se interpretan como evaluación de generalización por grupos de fármacos no vistos, no como validación cruzada disjunta a nivel de observaciones.


Random split: StratifiedKFold 10 folds.
LODO parcial: 10 folds de fármacos, optimizados por búsqueda aleatoria, permitiendo posible solapamiento de muestras test y reportándolo.


Para evaluar la capacidad de generalización de los modelos a fármacos no observados durante el entrenamiento, se empleó una variante parcial de Leave-Drug-Out. En cada fold se seleccionó un subconjunto disjunto de fármacos held-out, que fue excluido completamente del conjunto de entrenamiento. El conjunto de test de cada fold se formó con todas aquellas combinaciones que contenían al menos uno de los fármacos held-out. Por tanto, cada muestra de test incluye al menos un fármaco no visto durante el entrenamiento, aunque el segundo fármaco de la combinación puede haber aparecido en train.

Dado que cada observación está definida por una combinación de dos fármacos, esta estrategia puede producir solapamiento entre los conjuntos de test. Por ejemplo, si una combinación contiene dos fármacos asignados a folds held-out distintos, dicha combinación puede aparecer en el test de ambos folds, evaluando en cada caso la generalización respecto a un fármaco no visto diferente. Por este motivo, esta estrategia no debe interpretarse como una validación cruzada disjunta a nivel de muestras, sino como una evaluación repetida por grupos de fármacos no vistos.

Para reducir la variabilidad entre folds, la asignación de fármacos a folds no se realizó de forma puramente aleatoria. Se generaron múltiples particiones candidatas y se seleccionó aquella que producía conjuntos de test más equilibrados en términos de número de muestras y proporción de clase positiva. Esta decisión es especialmente importante en este problema, ya que el dataset presenta una distribución muy heterogénea de apariciones por fármaco y un desbalance global de clases. Como resultado, los folds obtenidos mantienen tamaños de test similares y una proporción de sinergias próxima a la proporción global del dataset.

El principal riesgo de esta estrategia es que las métricas obtenidas en los distintos folds no son completamente independientes, debido al solapamiento entre conjuntos de test. Por ello, la desviación estándar entre folds debe interpretarse con cautela, no como una estimación estricta de variabilidad sobre particiones independientes. Para caracterizar esta limitación, se cuantificó explícitamente el solapamiento entre conjuntos de test.

### EVALUACION DE MODELOS:
#### IMPORTS NECESARIOS Y FUNCIONES GLOBALES


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    confusion_matrix,
    balanced_accuracy_score,
    matthews_corrcoef,
    log_loss,
    brier_score_loss,
    precision_recall_curve,
    roc_curve,
)
from torch.utils.tensorboard import SummaryWriter



In [ ]:
def prepare_fold_data_no_leakage(df_train_raw, df_test_raw, data_expr, n_genes=500, random_state=42):
    df_train_base, df_val_base = train_test_split(
        df_train_raw,
        test_size=0.15,
        stratify=df_train_raw['synergy_loewe_bin'],
        random_state=random_state
    )

    train_cells = df_train_base['cell_line_name'].unique()

    top_genes = (
        data_expr
        .loc[train_cells]
        .var()
        .sort_values(ascending=False)
        .head(n_genes)
        .index
        .tolist()
    )

    df_train = df_train_base.merge(
        data_expr[top_genes],
        left_on='cell_line_name',
        right_index=True,
        how='inner'
    )

    df_val = df_val_base.merge(
        data_expr[top_genes],
        left_on='cell_line_name',
        right_index=True,
        how='inner'
    )

    df_test = df_test_raw.merge(
        data_expr[top_genes],
        left_on='cell_line_name',
        right_index=True,
        how='inner'
    )

    scaler = StandardScaler()

    df_train.loc[:, top_genes] = scaler.fit_transform(df_train[top_genes])
    df_val.loc[:, top_genes] = scaler.transform(df_val[top_genes])
    df_test.loc[:, top_genes] = scaler.transform(df_test[top_genes])

    return df_train.copy(), df_val.copy(), df_test.copy(), top_genes


def evaluate_lodo_partition(df, fold_sets, target_pos_rate=None):
    if target_pos_rate is None:
        target_pos_rate = df['synergy_loewe_bin'].mean()

    sizes = []
    pos_rates = []

    for fold_set in fold_sets:
        mask = (
            df['drug_row_id'].astype(str).isin(fold_set)
            | df['drug_col_id'].astype(str).isin(fold_set)
        )
        fold_df = df[mask]

        n = len(fold_df)
        pos_rate = fold_df['synergy_loewe_bin'].mean() if n > 0 else 0

        sizes.append(n)
        pos_rates.append(pos_rate)

    sizes = np.array(sizes)
    pos_rates = np.array(pos_rates)

    target_size = len(df) / len(fold_sets)

    size_error = np.mean(((sizes - target_size) / target_size) ** 2)
    pos_error = np.mean((pos_rates - target_pos_rate) ** 2)
    empty_penalty = np.sum(sizes == 0) * 10

    score = size_error + 20 * pos_error + empty_penalty

    return score, sizes, pos_rates


def build_lodo_folds_random_search(df, n_splits=10, n_iter=10000, random_state=42):
    rng = np.random.default_rng(random_state)

    drugs = pd.Index(
        pd.concat([df['drug_row_id'], df['drug_col_id']], ignore_index=True)
        .astype(str)
        .unique()
    ).to_numpy()

    best_score = np.inf
    best_folds = None
    best_sizes = None
    best_pos_rates = None

    for _ in range(n_iter):
        shuffled = drugs.copy()
        rng.shuffle(shuffled)

        fold_arrays = np.array_split(shuffled, n_splits)
        fold_sets = [set(arr.tolist()) for arr in fold_arrays]

        score, sizes, pos_rates = evaluate_lodo_partition(df, fold_sets)

        if score < best_score:
            best_score = score
            best_folds = fold_sets
            best_sizes = sizes
            best_pos_rates = pos_rates

    return best_folds, best_score, best_sizes, best_pos_rates


def compute_metrics_at_threshold(y_true, y_prob, threshold):
    y_true = np.asarray(y_true).astype(int).ravel()
    y_prob = np.asarray(y_prob).astype(float).ravel()
    y_pred = (y_prob >= threshold).astype(int)

    out = {}

    if len(np.unique(y_true)) > 1:
        out['auc'] = roc_auc_score(y_true, y_prob)
    else:
        out['auc'] = np.nan

    out['auprc'] = average_precision_score(y_true, y_prob)
    out['baseline_auprc'] = float(np.mean(y_true))

    p = np.clip(y_prob, 1e-7, 1 - 1e-7)
    out['log_loss'] = log_loss(y_true, np.c_[1 - p, p], labels=[0, 1])
    out['brier'] = brier_score_loss(y_true, y_prob)

    out['f1'] = f1_score(y_true, y_pred, zero_division=0)
    out['precision'] = precision_score(y_true, y_pred, zero_division=0)
    out['recall'] = recall_score(y_true, y_pred, zero_division=0)
    out['balanced_accuracy'] = balanced_accuracy_score(y_true, y_pred)
    out['mcc'] = matthews_corrcoef(y_true, y_pred)

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

    out['tn'] = int(tn)
    out['fp'] = int(fp)
    out['fn'] = int(fn)
    out['tp'] = int(tp)
    out['specificity'] = tn / (tn + fp) if (tn + fp) > 0 else np.nan
    out['threshold'] = float(threshold)

    return out


def find_best_thresholds(y_true, y_prob, thresholds):
    rows = []

    for thr in thresholds:
        m = compute_metrics_at_threshold(y_true, y_prob, thr)
        rows.append(m)

    df_thr = pd.DataFrame(rows)

    best_f1_row = df_thr.sort_values(
        ['f1', 'precision', 'recall'],
        ascending=False
    ).iloc[0]

    valid_precision = df_thr[df_thr['recall'] > 0].copy()

    if len(valid_precision) > 0:
        best_precision_row = valid_precision.sort_values(
            ['precision', 'recall', 'f1'],
            ascending=False
        ).iloc[0]
    else:
        best_precision_row = best_f1_row

    return df_thr, float(best_f1_row['threshold']), float(best_precision_row['threshold'])


def log_curves_and_metrics(y_true, y_prob, metrics, fold_id, writer, tag):
    for key, value in metrics.items():
        if isinstance(value, (int, float, np.integer, np.floating)):
            writer.add_scalar(f'{tag}/{key}', value, fold_id)

    cm = confusion_matrix(y_true, (np.asarray(y_prob) >= metrics['threshold']).astype(int), labels=[0, 1])

    fig_cm, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax)
    ax.set_title(f'{tag} Confusion Matrix - Fold {fold_id}')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    writer.add_figure(f'{tag}/confusion_matrix', fig_cm, fold_id)
    plt.close(fig_cm)

    precision, recall, _ = precision_recall_curve(y_true, y_prob)
    fig_pr, ax = plt.subplots(figsize=(5, 4))
    ax.plot(recall, precision)
    ax.axhline(metrics['baseline_auprc'], linestyle='--', color='gray')
    ax.set_title(f'{tag} PR Curve - Fold {fold_id}')
    ax.set_xlabel('Recall')
    ax.set_ylabel('Precision')
    ax.grid(alpha=0.3)
    writer.add_figure(f'{tag}/pr_curve', fig_pr, fold_id)
    plt.close(fig_pr)

    if len(np.unique(y_true)) > 1:
        fpr, tpr, _ = roc_curve(y_true, y_prob)
        fig_roc, ax = plt.subplots(figsize=(5, 4))
        ax.plot(fpr, tpr)
        ax.plot([0, 1], [0, 1], linestyle='--', color='gray')
        ax.set_title(f'{tag} ROC Curve - Fold {fold_id}')
        ax.set_xlabel('FPR')
        ax.set_ylabel('TPR')
        ax.grid(alpha=0.3)
        writer.add_figure(f'{tag}/roc_curve', fig_roc, fold_id)
        plt.close(fig_roc)



def build_tabular_matrices(df_train, df_val, df_test, target_col='synergy_loewe_bin'):
    cols_to_exclude = {
        target_col,
        'synergy_loewe',
        'drug_row_id',
        'drug_col_id',
        'drug_row',
        'drug_col',
        'smiles_row',
        'smiles_col',
        'cell_line_name',
        'study_name',
        'tissue',
        'lodo_fold',
    }

    feature_cols = [
        c for c in df_train.columns
        if c not in cols_to_exclude and pd.api.types.is_numeric_dtype(df_train[c])
    ]

    if len(feature_cols) == 0:
        raise ValueError('No se encontraron columnas numericas para entrenar el modelo.')

    X_train = df_train[feature_cols].fillna(0.0).to_numpy(dtype=np.float32)
    X_val = df_val.reindex(columns=feature_cols).fillna(0.0).to_numpy(dtype=np.float32)
    X_test = df_test.reindex(columns=feature_cols).fillna(0.0).to_numpy(dtype=np.float32)

    y_train = df_train[target_col].astype(int).to_numpy()
    y_val = df_val[target_col].astype(int).to_numpy()
    y_test = df_test[target_col].astype(int).to_numpy()

    return X_train, y_train, X_val, y_val, X_test, y_test, feature_cols



Todos los modelos fueron evaluados sobre las mismas particiones, garantizando una comparación directa bajo idénticas condiciones experimentales.


### RANDOM FOREST: LDO CV=10


In [ ]:
def build_rf_matrices(df_train, df_val, df_test):
    target_col = 'synergy_loewe_bin'

    cols_to_exclude = {
        target_col,
        'synergy_loewe',
        'drug_row_id',
        'drug_col_id',
        'drug_row',
        'drug_col',
        'smiles_row',
        'smiles_col',
        'cell_line_name',
        'study_name',
        'tissue',
    }

    feature_cols = [
        c for c in df_train.columns
        if c not in cols_to_exclude and pd.api.types.is_numeric_dtype(df_train[c])
    ]

    if len(feature_cols) == 0:
        raise ValueError('No se encontraron columnas numericas para entrenar RandomForest.')

    X_train = df_train[feature_cols].fillna(0.0).to_numpy(dtype=np.float32)
    X_val = df_val.reindex(columns=feature_cols).fillna(0.0).to_numpy(dtype=np.float32)
    X_test = df_test.reindex(columns=feature_cols).fillna(0.0).to_numpy(dtype=np.float32)

    y_train = df_train[target_col].astype(int).to_numpy()
    y_val = df_val[target_col].astype(int).to_numpy()
    y_test = df_test[target_col].astype(int).to_numpy()

    return X_train, y_train, X_val, y_val, X_test, y_test, feature_cols


def summarize_test_overlap(df, fold_sets, output_path=None):
    test_indices_by_fold = []

    for fold_id, fold_set in enumerate(fold_sets, start=1):
        mask = (
            df['drug_row_id'].astype(str).isin(fold_set)
            | df['drug_col_id'].astype(str).isin(fold_set)
        )

        fold_indices = df.index[mask].tolist()

        for idx in fold_indices:
            test_indices_by_fold.append({
                'sample_index': idx,
                'fold': fold_id
            })

    overlap_df = pd.DataFrame(test_indices_by_fold)

    sample_counts = (
        overlap_df['sample_index']
        .value_counts()
        .sort_index()
    )

    total_test_appearances = int(len(overlap_df))
    unique_test_samples = int(sample_counts.shape[0])
    repeated_test_samples = int(total_test_appearances - unique_test_samples)
    overlap_pct = (
        100.0 * repeated_test_samples / total_test_appearances
        if total_test_appearances > 0 else 0.0
    )

    summary = {
        'total_test_appearances': total_test_appearances,
        'unique_test_samples': unique_test_samples,
        'repeated_test_samples': repeated_test_samples,
        'overlap_pct': overlap_pct,
        'min_test_appearances_per_sample': int(sample_counts.min()) if len(sample_counts) > 0 else 0,
        'max_test_appearances_per_sample': int(sample_counts.max()) if len(sample_counts) > 0 else 0,
        'mean_test_appearances_per_sample': float(sample_counts.mean()) if len(sample_counts) > 0 else 0.0,
    }

    counts_distribution = (
        sample_counts
        .value_counts()
        .sort_index()
        .rename_axis('n_test_appearances')
        .reset_index(name='n_samples')
    )

    if output_path is not None:
        os.makedirs(output_path, exist_ok=True)

        with open(os.path.join(output_path, 'test_overlap_summary.json'), 'w') as f:
            json.dump(summary, f, indent=4)

        overlap_df.to_csv(
            os.path.join(output_path, 'test_overlap_by_sample_fold.csv'),
            index=False
        )

        counts_distribution.to_csv(
            os.path.join(output_path, 'test_overlap_counts_distribution.csv'),
            index=False
        )

    return summary, overlap_df, counts_distribution



def save_experiment_config(config, output_dir, filename='config.json'):
    os.makedirs(output_dir, exist_ok=True)

    config_clean = {}

    for key, value in config.items():
        if isinstance(value, (np.integer, np.int64, np.int32)):
            config_clean[key] = int(value)
        elif isinstance(value, (np.floating, np.float64, np.float32)):
            config_clean[key] = float(value)
        elif isinstance(value, np.ndarray):
            config_clean[key] = value.tolist()
        elif isinstance(value, set):
            config_clean[key] = sorted(list(value))
        else:
            config_clean[key] = value

    path = os.path.join(output_dir, filename)

    with open(path, 'w') as f:
        json.dump(config_clean, f, indent=4)

    return path

def save_split_predictions(
    df_split,
    y_true,
    y_prob,
    best_thr_f1,
    best_thr_precision,
    output_dir,
    fold_idx,
    split_name,
    filename_prefix=None
):
    if filename_prefix is None:
        filename_prefix = f'{split_name}_predictions'

    pred_cols = [
        'drug_row_id',
        'drug_col_id',
        'cell_line_name',
        'study_name',
        'tissue',
        'synergy_loewe',
        'synergy_loewe_bin'
    ]

    predictions = df_split[pred_cols].copy()

    predictions['fold'] = fold_idx
    predictions['split'] = split_name
    predictions['y_true'] = np.asarray(y_true).astype(int)
    predictions['y_prob'] = np.asarray(y_prob).astype(float)
    predictions['y_pred_best_f1'] = (predictions['y_prob'] >= best_thr_f1).astype(int)
    predictions['y_pred_best_precision'] = (predictions['y_prob'] >= best_thr_precision).astype(int)
    predictions['threshold_best_f1'] = best_thr_f1
    predictions['threshold_best_precision'] = best_thr_precision

    predictions.to_csv(
        os.path.join(
            output_dir,
            'predictions',
            f'{filename_prefix}_fold_{fold_idx:02d}.csv'
        ),
        index=False
    )

    return predictions


In [ ]:
import os
import json
import shutil
import joblib
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from torch.utils.tensorboard import SummaryWriter
from PIL import Image

if not hasattr(Image, "Resampling"):
    class Resampling:
        LANCZOS = Image.ANTIALIAS

    Image.Resampling = Resampling

# CONFIGURACION


SAVE_MODELS = False

output_dir = 'results/random_forest_lodo'
tb_dir = os.path.join(output_dir, 'tensorboard')

if os.path.exists(output_dir):
    shutil.rmtree(output_dir)

os.makedirs(output_dir, exist_ok=True)
os.makedirs(os.path.join(output_dir, 'predictions'), exist_ok=True)
os.makedirs(os.path.join(output_dir, 'figures'), exist_ok=True)
os.makedirs(tb_dir, exist_ok=True)

if SAVE_MODELS:
    os.makedirs(os.path.join(output_dir, 'models'), exist_ok=True)

rf_config = {
    'n_estimators': 600,
    'max_depth': None,
    'min_samples_split': 2,
    'min_samples_leaf': 1,
    'max_features': 'sqrt',
    'class_weight': 'balanced_subsample',
    'random_state': 42,
    'n_jobs': 4,
}

experiment_config = {
    'model': 'RandomForestClassifier',
    'split_type': 'partial_lodo_optimized',
    'n_splits': 10,
    'n_genes': 500,
    'random_state': 42,
    'threshold_grid_min': 0.05,
    'threshold_grid_max': 0.95,
    'threshold_grid_step': 0.01,
    'rf_config': rf_config,
    'save_models': SAVE_MODELS,
    'lodo_definition': (
        'Partial Leave-Drug-Out: each test sample contains at least one held-out drug. '
        'The second drug may appear in train.'
    )
}

save_experiment_config(experiment_config, output_dir)


# DATASET BASE


base_df = X_final_df.copy()
base_df['drug_row_id'] = base_df['drug_row_id'].astype(str)
base_df['drug_col_id'] = base_df['drug_col_id'].astype(str)
base_df['synergy_loewe_bin'] = base_df['synergy_loewe_bin'].replace(-1, 0).astype(int)

thresholds = np.round(np.arange(0.05, 0.951, 0.01), 2)


# CONSTRUCCION LODO PARCIAL OPTIMIZADO

fold_sets, lodo_score, lodo_sizes, lodo_pos_rates = build_lodo_folds_random_search(
    base_df,
    n_splits=experiment_config['n_splits'],
    n_iter=20,
    random_state=experiment_config['random_state']
)

fold_partition_summary = pd.DataFrame({
    'fold': np.arange(1, experiment_config['n_splits'] + 1),
    'n_examples_induced_test': lodo_sizes,
    'positive_pct_induced_test': lodo_pos_rates * 100,
    'baseline_auprc_induced_test': lodo_pos_rates,
    'n_heldout_drugs': [len(s) for s in fold_sets]
})

fold_partition_summary.to_csv(
    os.path.join(output_dir, 'lodo_partition_summary.csv'),
    index=False
)

print('\nResumen particion LODO:')
print(fold_partition_summary)
print('LODO partition score:', lodo_score)



# SOLAPAMIENTO ENTRE TESTS

test_overlap_info, overlap_df, overlap_counts = summarize_test_overlap(
    base_df,
    fold_sets,
    output_path=output_dir
)

print('\nSolapamiento entre conjuntos test:')
print(test_overlap_info)
print('\nDistribucion de apariciones por muestra en test:')
print(overlap_counts)


# BUCLE CV

cv_val_results_f1 = []
cv_val_results_precision = []
cv_test_results_f1 = []
cv_test_results_precision = []
cv_split_info = []
cv_val_threshold_metrics = []
cv_test_threshold_metrics = []
cv_best_thresholds = []

for fold_idx, test_set in enumerate(fold_sets, start=1):
    writer = SummaryWriter(log_dir=os.path.join(tb_dir, f'Fold_{fold_idx}'))

    print(f'\nPROCESANDO FOLD {fold_idx}/{experiment_config["n_splits"]} - RANDOM FOREST LODO')

    mask_test = (
        base_df['drug_row_id'].isin(test_set)
        | base_df['drug_col_id'].isin(test_set)
    )

    df_test_raw = base_df[mask_test].reset_index(drop=True)
    df_train_raw = base_df[~mask_test].reset_index(drop=True)

    if len(df_train_raw) == 0 or len(df_test_raw) == 0:
        print(f'Fold {fold_idx}: sin datos suficientes, se omite.')
        writer.close()
        continue

    assert (
        df_test_raw['drug_row_id'].isin(test_set)
        | df_test_raw['drug_col_id'].isin(test_set)
    ).all(), f'Fold {fold_idx}: hay muestras test sin droga held-out.'

    train_drugs = set(pd.concat([
        df_train_raw['drug_row_id'],
        df_train_raw['drug_col_id']
    ]).astype(str))

    heldout_overlap_train = len(train_drugs & test_set)
    assert heldout_overlap_train == 0, f'Fold {fold_idx}: drogas held-out aparecen en train.'

    df_train, df_val, df_test, top_genes = prepare_fold_data_no_leakage(
        df_train_raw,
        df_test_raw,
        data_expr,
        n_genes=experiment_config['n_genes'],
        random_state=experiment_config['random_state']
    )

    X_train, y_train, X_val, y_val, X_test, y_test, feature_cols = build_tabular_matrices(
        df_train,
        df_val,
        df_test
    )

    if len(np.unique(y_train)) < 2:
        print(f'Fold {fold_idx}: train con una sola clase, se omite.')
        writer.close()
        continue

    rf = RandomForestClassifier(**rf_config)

    print(f'Entrenando RandomForest fold {fold_idx}...')
    rf.fit(X_train, y_train)
    print(f'Entrenamiento terminado fold {fold_idx}.')

    y_prob_val = rf.predict_proba(X_val)[:, 1]
    y_prob_test = rf.predict_proba(X_test)[:, 1]

    df_val_thr, best_thr_f1, best_thr_precision = find_best_thresholds(
        y_val,
        y_prob_val,
        thresholds
    )

    df_val_thr.insert(0, 'fold', fold_idx)
    df_val_thr.insert(1, 'split', 'val')
    df_val_thr.insert(2, 'threshold_type', 'grid')
    cv_val_threshold_metrics.append(df_val_thr)

    test_thr_rows = []
    for thr in thresholds:
        m_test_thr = compute_metrics_at_threshold(y_test, y_prob_test, thr)
        m_test_thr.update({
            'fold': fold_idx,
            'split': 'test',
            'threshold_type': 'grid',
        })
        test_thr_rows.append(m_test_thr)

    df_test_thr = pd.DataFrame(test_thr_rows)
    cv_test_threshold_metrics.append(df_test_thr)

    val_metrics_f1 = compute_metrics_at_threshold(y_val, y_prob_val, best_thr_f1)
    val_metrics_precision = compute_metrics_at_threshold(y_val, y_prob_val, best_thr_precision)

    test_metrics_f1 = compute_metrics_at_threshold(y_test, y_prob_test, best_thr_f1)
    test_metrics_precision = compute_metrics_at_threshold(y_test, y_prob_test, best_thr_precision)

    val_metrics_f1.update({'fold': fold_idx, 'split': 'val', 'threshold_type': 'best_f1'})
    val_metrics_precision.update({'fold': fold_idx, 'split': 'val', 'threshold_type': 'best_precision'})
    test_metrics_f1.update({'fold': fold_idx, 'split': 'test', 'threshold_type': 'best_f1'})
    test_metrics_precision.update({'fold': fold_idx, 'split': 'test', 'threshold_type': 'best_precision'})

    cv_val_results_f1.append(val_metrics_f1)
    cv_val_results_precision.append(val_metrics_precision)
    cv_test_results_f1.append(test_metrics_f1)
    cv_test_results_precision.append(test_metrics_precision)

    cv_best_thresholds.append({
        'fold': fold_idx,
        'best_threshold_f1': best_thr_f1,
        'best_threshold_precision': best_thr_precision,
        'val_f1_at_best_f1': val_metrics_f1['f1'],
        'val_precision_at_best_f1': val_metrics_f1['precision'],
        'val_recall_at_best_f1': val_metrics_f1['recall'],
        'val_precision_at_best_precision': val_metrics_precision['precision'],
        'val_recall_at_best_precision': val_metrics_precision['recall'],
        'val_f1_at_best_precision': val_metrics_precision['f1'],
    })

    log_curves_and_metrics(y_val, y_prob_val, val_metrics_f1, fold_idx, writer, tag='Val_BestF1')
    log_curves_and_metrics(y_val, y_prob_val, val_metrics_precision, fold_idx, writer, tag='Val_BestPrecision')
    log_curves_and_metrics(y_test, y_prob_test, test_metrics_f1, fold_idx, writer, tag='Test_BestF1')
    log_curves_and_metrics(y_test, y_prob_test, test_metrics_precision, fold_idx, writer, tag='Test_BestPrecision')

    predictions = df_test[
        [
            'drug_row_id',
            'drug_col_id',
            'cell_line_name',
            'study_name',
            'tissue',
            'synergy_loewe',
            'synergy_loewe_bin'
        ]
    ].copy()

    predictions['y_true'] = y_test
    predictions['y_prob'] = y_prob_test
    predictions['y_pred_best_f1'] = (y_prob_test >= best_thr_f1).astype(int)
    predictions['y_pred_best_precision'] = (y_prob_test >= best_thr_precision).astype(int)
    predictions['threshold_best_f1'] = best_thr_f1
    predictions['threshold_best_precision'] = best_thr_precision

    predictions.to_csv(
        os.path.join(output_dir, 'predictions', f'predictions_fold_{fold_idx:02d}.csv'),
        index=False
    )

    if SAVE_MODELS:
        joblib.dump(
            rf,
            os.path.join(output_dir, 'models', f'random_forest_fold_{fold_idx:02d}.joblib')
        )

    n_train = len(y_train)
    n_val = len(y_val)
    n_test = len(y_test)

    split_info = {
        'fold': fold_idx,
        'n_train': n_train,
        'n_val': n_val,
        'n_test': n_test,
        'train_pos': int(np.sum(y_train)),
        'val_pos': int(np.sum(y_val)),
        'test_pos': int(np.sum(y_test)),
        'train_pos_pct': 100 * np.mean(y_train),
        'val_pos_pct': 100 * np.mean(y_val),
        'test_pos_pct': 100 * np.mean(y_test),
        'baseline_auprc_test': float(np.mean(y_test)),
        'n_heldout_drugs': len(test_set),
        'heldout_drugs': ';'.join(sorted(test_set)),
        'heldout_overlap_train': heldout_overlap_train,
        'best_threshold_f1': best_thr_f1,
        'best_threshold_precision': best_thr_precision,
        'n_features': len(feature_cols),
        'n_top_genes': len(top_genes),
    }

    cv_split_info.append(split_info)

    writer.add_text('Split/Info', json.dumps(split_info, indent=2))
    writer.add_text('Genes/TopGenes_First100', ', '.join(top_genes[:100]))

    importances = pd.Series(rf.feature_importances_, index=feature_cols)
    top_imp = importances.sort_values(ascending=False).head(30)
    writer.add_text('Model/Top30_Feature_Importance', top_imp.to_string())

    print(
        f"Fold {fold_idx} | "
        f"Train={n_train} ({100*np.mean(y_train):.2f}% pos), "
        f"Val={n_val} ({100*np.mean(y_val):.2f}% pos), "
        f"Test={n_test} ({100*np.mean(y_test):.2f}% pos) | "
        f"ThrF1={best_thr_f1:.2f}, ThrPrec={best_thr_precision:.2f} | "
        f"AUPRC={test_metrics_f1['auprc']:.4f} "
        f"(base={test_metrics_f1['baseline_auprc']:.4f}) | "
        f"F1={test_metrics_f1['f1']:.4f} | "
        f"Prec={test_metrics_f1['precision']:.4f} | "
        f"Rec={test_metrics_f1['recall']:.4f} | "
        f"MCC={test_metrics_f1['mcc']:.4f}"
    )

    writer.close()



# GUARDADO DE RESULTADOS


df_val_f1 = pd.DataFrame(cv_val_results_f1)
df_val_precision = pd.DataFrame(cv_val_results_precision)
df_test_f1 = pd.DataFrame(cv_test_results_f1)
df_test_precision = pd.DataFrame(cv_test_results_precision)
df_split_info = pd.DataFrame(cv_split_info)
df_best_thresholds = pd.DataFrame(cv_best_thresholds)

df_val_thresholds = pd.concat(cv_val_threshold_metrics, ignore_index=True)
df_test_thresholds = pd.concat(cv_test_threshold_metrics, ignore_index=True)

df_val_f1.to_csv(os.path.join(output_dir, 'val_metrics_best_f1.csv'), index=False)
df_val_precision.to_csv(os.path.join(output_dir, 'val_metrics_best_precision.csv'), index=False)
df_test_f1.to_csv(os.path.join(output_dir, 'test_metrics_best_f1.csv'), index=False)
df_test_precision.to_csv(os.path.join(output_dir, 'test_metrics_best_precision.csv'), index=False)
df_split_info.to_csv(os.path.join(output_dir, 'split_info.csv'), index=False)
df_best_thresholds.to_csv(os.path.join(output_dir, 'best_thresholds.csv'), index=False)
df_val_thresholds.to_csv(os.path.join(output_dir, 'val_metrics_by_threshold.csv'), index=False)
df_test_thresholds.to_csv(os.path.join(output_dir, 'test_metrics_by_threshold.csv'), index=False)



# RESUMEN FINAL


metric_cols = [
    'auc',
    'auprc',
    'baseline_auprc',
    'f1',
    'precision',
    'recall',
    'specificity',
    'balanced_accuracy',
    'mcc',
    'log_loss',
    'brier',
    'tp',
    'tn',
    'fp',
    'fn',
]

summary_f1 = (
    df_test_f1[metric_cols]
    .agg(['mean', 'std'])
    .T
    .reset_index()
    .rename(columns={'index': 'metric'})
)

summary_precision = (
    df_test_precision[metric_cols]
    .agg(['mean', 'std'])
    .T
    .reset_index()
    .rename(columns={'index': 'metric'})
)

summary_f1.to_csv(os.path.join(output_dir, 'summary_test_best_f1.csv'), index=False)
summary_precision.to_csv(os.path.join(output_dir, 'summary_test_best_precision.csv'), index=False)

print('\n' + '=' * 70)
print('RESULTADOS RANDOM FOREST - LODO PARCIAL CV=10')
print('=' * 70)

print('\n[TEST - threshold elegido por mejor F1 en validacion]')
print(summary_f1)

print('\n[TEST - threshold conservador por mejor precision en validacion]')
print(summary_precision)

print('\nArchivos guardados en:', output_dir)



In [ ]:
import os
import json
import shutil
import joblib
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from torch.utils.tensorboard import SummaryWriter
from PIL import Image

if not hasattr(Image, "Resampling"):
    class Resampling:
        LANCZOS = Image.ANTIALIAS

    Image.Resampling = Resampling

# ============================================================
# CONFIGURACION
# ============================================================

SAVE_MODELS = False

output_dir = 'results/aaatrain_random_forest_lodo'
tb_dir = os.path.join(output_dir, 'tensorboard')

if os.path.exists(output_dir):
    shutil.rmtree(output_dir)

os.makedirs(output_dir, exist_ok=True)
os.makedirs(os.path.join(output_dir, 'predictions'), exist_ok=True)
os.makedirs(os.path.join(output_dir, 'figures'), exist_ok=True)
os.makedirs(tb_dir, exist_ok=True)

if SAVE_MODELS:
    os.makedirs(os.path.join(output_dir, 'models'), exist_ok=True)

rf_config = {
    'n_estimators': 600,
    'max_depth': None,
    'min_samples_split': 2,
    'min_samples_leaf': 1,
    'max_features': 'sqrt',
    'class_weight': 'balanced_subsample',
    'random_state': 42,
    'n_jobs': 4,
}

experiment_config = {
    'model': 'RandomForestClassifier',
    'split_type': 'partial_lodo_optimized',
    'n_splits': 10,
    'n_genes': 500,
    'random_state': 42,
    'threshold_grid_min': 0.05,
    'threshold_grid_max': 0.95,
    'threshold_grid_step': 0.01,
    'rf_config': rf_config,
    'save_models': SAVE_MODELS,
    'lodo_definition': (
        'Partial Leave-Drug-Out: each test sample contains at least one held-out drug. '
        'The second drug may appear in train.'
    )
}

save_experiment_config(experiment_config, output_dir)


# ============================================================
# DATASET BASE
# ============================================================

base_df = X_final_df.copy()
base_df['drug_row_id'] = base_df['drug_row_id'].astype(str)
base_df['drug_col_id'] = base_df['drug_col_id'].astype(str)
base_df['synergy_loewe_bin'] = base_df['synergy_loewe_bin'].replace(-1, 0).astype(int)

thresholds = np.round(np.arange(0.05, 0.951, 0.01), 2)


# ============================================================
# CONSTRUCCION LODO PARCIAL OPTIMIZADO
# ============================================================

fold_sets, lodo_score, lodo_sizes, lodo_pos_rates = build_lodo_folds_random_search(
    base_df,
    n_splits=experiment_config['n_splits'],
    n_iter=20,
    random_state=experiment_config['random_state']
)

fold_partition_summary = pd.DataFrame({
    'fold': np.arange(1, experiment_config['n_splits'] + 1),
    'n_examples_induced_test': lodo_sizes,
    'positive_pct_induced_test': lodo_pos_rates * 100,
    'baseline_auprc_induced_test': lodo_pos_rates,
    'n_heldout_drugs': [len(s) for s in fold_sets]
})

fold_partition_summary.to_csv(
    os.path.join(output_dir, 'lodo_partition_summary.csv'),
    index=False
)

print('\nResumen particion LODO:')
print(fold_partition_summary)
print('LODO partition score:', lodo_score)


# ============================================================
# SOLAPAMIENTO ENTRE TESTS
# ============================================================

test_overlap_info, overlap_df, overlap_counts = summarize_test_overlap(
    base_df,
    fold_sets,
    output_path=output_dir
)

print('\nSolapamiento entre conjuntos test:')
print(test_overlap_info)
print('\nDistribucion de apariciones por muestra en test:')
print(overlap_counts)


# ============================================================
# BUCLE CV
# ============================================================

cv_train_results_f1 = []
cv_train_results_precision = []
cv_val_results_f1 = []
cv_val_results_precision = []
cv_test_results_f1 = []
cv_test_results_precision = []
cv_split_info = []
cv_val_threshold_metrics = []
cv_test_threshold_metrics = []
cv_best_thresholds = []

for fold_idx, test_set in enumerate(fold_sets, start=1):
    writer = SummaryWriter(log_dir=os.path.join(tb_dir, f'Fold_{fold_idx}'))

    print(f'\nPROCESANDO FOLD {fold_idx}/{experiment_config["n_splits"]} - RANDOM FOREST LODO')

    mask_test = (
        base_df['drug_row_id'].isin(test_set)
        | base_df['drug_col_id'].isin(test_set)
    )

    df_test_raw = base_df[mask_test].reset_index(drop=True)
    df_train_raw = base_df[~mask_test].reset_index(drop=True)

    if len(df_train_raw) == 0 or len(df_test_raw) == 0:
        print(f'Fold {fold_idx}: sin datos suficientes, se omite.')
        writer.close()
        continue

    assert (
        df_test_raw['drug_row_id'].isin(test_set)
        | df_test_raw['drug_col_id'].isin(test_set)
    ).all(), f'Fold {fold_idx}: hay muestras test sin droga held-out.'

    train_drugs = set(pd.concat([
        df_train_raw['drug_row_id'],
        df_train_raw['drug_col_id']
    ]).astype(str))

    heldout_overlap_train = len(train_drugs & test_set)
    assert heldout_overlap_train == 0, f'Fold {fold_idx}: drogas held-out aparecen en train.'

    df_train, df_val, df_test, top_genes = prepare_fold_data_no_leakage(
        df_train_raw,
        df_test_raw,
        data_expr,
        n_genes=experiment_config['n_genes'],
        random_state=experiment_config['random_state']
    )

    X_train, y_train, X_val, y_val, X_test, y_test, feature_cols = build_tabular_matrices(
        df_train,
        df_val,
        df_test
    )

    if len(np.unique(y_train)) < 2:
        print(f'Fold {fold_idx}: train con una sola clase, se omite.')
        writer.close()
        continue

    rf = RandomForestClassifier(**rf_config)

    print(f'Entrenando RandomForest fold {fold_idx}...')
    rf.fit(X_train, y_train)
    print(f'Entrenamiento terminado fold {fold_idx}.')

    y_prob_train = rf.predict_proba(X_train)[:, 1]
    y_prob_val = rf.predict_proba(X_val)[:, 1]
    y_prob_test = rf.predict_proba(X_test)[:, 1]

    df_val_thr, best_thr_f1, best_thr_precision = find_best_thresholds(
        y_val,
        y_prob_val,
        thresholds
    )

    df_val_thr.insert(0, 'fold', fold_idx)
    df_val_thr.insert(1, 'split', 'val')
    df_val_thr.insert(2, 'threshold_type', 'grid')
    cv_val_threshold_metrics.append(df_val_thr)

    test_thr_rows = []
    for thr in thresholds:
        m_test_thr = compute_metrics_at_threshold(y_test, y_prob_test, thr)
        m_test_thr.update({
            'fold': fold_idx,
            'split': 'test',
            'threshold_type': 'grid',
        })
        test_thr_rows.append(m_test_thr)

    df_test_thr = pd.DataFrame(test_thr_rows)
    cv_test_threshold_metrics.append(df_test_thr)

    train_metrics_f1 = compute_metrics_at_threshold(y_train, y_prob_train, best_thr_f1)
    train_metrics_precision = compute_metrics_at_threshold(y_train, y_prob_train, best_thr_precision)

    val_metrics_f1 = compute_metrics_at_threshold(y_val, y_prob_val, best_thr_f1)
    val_metrics_precision = compute_metrics_at_threshold(y_val, y_prob_val, best_thr_precision)

    test_metrics_f1 = compute_metrics_at_threshold(y_test, y_prob_test, best_thr_f1)
    test_metrics_precision = compute_metrics_at_threshold(y_test, y_prob_test, best_thr_precision)



    train_metrics_f1.update({'fold': fold_idx, 'split': 'train', 'threshold_type': 'best_f1'})
    train_metrics_precision.update({'fold': fold_idx, 'split': 'train', 'threshold_type': 'best_precision'})
    val_metrics_f1.update({'fold': fold_idx, 'split': 'val', 'threshold_type': 'best_f1'})
    val_metrics_precision.update({'fold': fold_idx, 'split': 'val', 'threshold_type': 'best_precision'})
    test_metrics_f1.update({'fold': fold_idx, 'split': 'test', 'threshold_type': 'best_f1'})
    test_metrics_precision.update({'fold': fold_idx, 'split': 'test', 'threshold_type': 'best_precision'})

    cv_train_results_f1.append(train_metrics_f1)
    cv_train_results_precision.append(train_metrics_precision)
    cv_val_results_f1.append(val_metrics_f1)
    cv_val_results_precision.append(val_metrics_precision)
    cv_test_results_f1.append(test_metrics_f1)
    cv_test_results_precision.append(test_metrics_precision)

    cv_best_thresholds.append({
        'fold': fold_idx,
        'best_threshold_f1': best_thr_f1,
        'best_threshold_precision': best_thr_precision,
        'val_f1_at_best_f1': val_metrics_f1['f1'],
        'val_precision_at_best_f1': val_metrics_f1['precision'],
        'val_recall_at_best_f1': val_metrics_f1['recall'],
        'val_precision_at_best_precision': val_metrics_precision['precision'],
        'val_recall_at_best_precision': val_metrics_precision['recall'],
        'val_f1_at_best_precision': val_metrics_precision['f1'],
    })

    log_curves_and_metrics(y_val, y_prob_val, val_metrics_f1, fold_idx, writer, tag='Val_BestF1')
    log_curves_and_metrics(y_val, y_prob_val, val_metrics_precision, fold_idx, writer, tag='Val_BestPrecision')
    log_curves_and_metrics(y_test, y_prob_test, test_metrics_f1, fold_idx, writer, tag='Test_BestF1')
    log_curves_and_metrics(y_test, y_prob_test, test_metrics_precision, fold_idx, writer, tag='Test_BestPrecision')

    train_predictions = save_split_predictions(
        df_split=df_train,
        y_true=y_train,
        y_prob=y_prob_train,
        best_thr_f1=best_thr_f1,
        best_thr_precision=best_thr_precision,
        output_dir=output_dir,
        fold_idx=fold_idx,
        split_name='train'
    )

    val_predictions = save_split_predictions(
        df_split=df_val,
        y_true=y_val,
        y_prob=y_prob_val,
        best_thr_f1=best_thr_f1,
        best_thr_precision=best_thr_precision,
        output_dir=output_dir,
        fold_idx=fold_idx,
        split_name='val'
    )

    test_predictions = save_split_predictions(
        df_split=df_test,
        y_true=y_test,
        y_prob=y_prob_test,
        best_thr_f1=best_thr_f1,
        best_thr_precision=best_thr_precision,
        output_dir=output_dir,
        fold_idx=fold_idx,
        split_name='test',
        filename_prefix='predictions'
    )


    if SAVE_MODELS:
        joblib.dump(
            rf,
            os.path.join(output_dir, 'models', f'random_forest_fold_{fold_idx:02d}.joblib')
        )

    n_train = len(y_train)
    n_val = len(y_val)
    n_test = len(y_test)

    split_info = {
        'fold': fold_idx,
        'n_train': n_train,
        'n_val': n_val,
        'n_test': n_test,
        'train_pos': int(np.sum(y_train)),
        'val_pos': int(np.sum(y_val)),
        'test_pos': int(np.sum(y_test)),
        'train_pos_pct': 100 * np.mean(y_train),
        'val_pos_pct': 100 * np.mean(y_val),
        'test_pos_pct': 100 * np.mean(y_test),
        'baseline_auprc_test': float(np.mean(y_test)),
        'n_heldout_drugs': len(test_set),
        'heldout_drugs': ';'.join(sorted(test_set)),
        'heldout_overlap_train': heldout_overlap_train,
        'best_threshold_f1': best_thr_f1,
        'best_threshold_precision': best_thr_precision,
        'n_features': len(feature_cols),
        'n_top_genes': len(top_genes),
    }

    cv_split_info.append(split_info)

    writer.add_text('Split/Info', json.dumps(split_info, indent=2))
    writer.add_text('Genes/TopGenes_First100', ', '.join(top_genes[:100]))

    importances = pd.Series(rf.feature_importances_, index=feature_cols)
    top_imp = importances.sort_values(ascending=False).head(30)
    writer.add_text('Model/Top30_Feature_Importance', top_imp.to_string())

    print(
        f"Fold {fold_idx} | "
        f"Train={n_train} ({100*np.mean(y_train):.2f}% pos), "
        f"Val={n_val} ({100*np.mean(y_val):.2f}% pos), "
        f"Test={n_test} ({100*np.mean(y_test):.2f}% pos) | "
        f"ThrF1={best_thr_f1:.2f}, ThrPrec={best_thr_precision:.2f} | "
        f"AUPRC={test_metrics_f1['auprc']:.4f} "
        f"(base={test_metrics_f1['baseline_auprc']:.4f}) | "
        f"F1={test_metrics_f1['f1']:.4f} | "
        f"Prec={test_metrics_f1['precision']:.4f} | "
        f"Rec={test_metrics_f1['recall']:.4f} | "
        f"MCC={test_metrics_f1['mcc']:.4f}"
    )

    writer.close()


# ============================================================
# GUARDADO DE RESULTADOS
# ============================================================

df_train_f1 = pd.DataFrame(cv_train_results_f1)
df_train_precision = pd.DataFrame(cv_train_results_precision)
df_val_f1 = pd.DataFrame(cv_val_results_f1)
df_val_precision = pd.DataFrame(cv_val_results_precision)
df_test_f1 = pd.DataFrame(cv_test_results_f1)
df_test_precision = pd.DataFrame(cv_test_results_precision)
df_split_info = pd.DataFrame(cv_split_info)
df_best_thresholds = pd.DataFrame(cv_best_thresholds)

df_val_thresholds = pd.concat(cv_val_threshold_metrics, ignore_index=True)
df_test_thresholds = pd.concat(cv_test_threshold_metrics, ignore_index=True)


df_train_f1.to_csv(os.path.join(output_dir, 'train_metrics_best_f1.csv'), index=False)
df_train_precision.to_csv(os.path.join(output_dir, 'train_metrics_best_precision.csv'), index=False)
df_val_f1.to_csv(os.path.join(output_dir, 'val_metrics_best_f1.csv'), index=False)
df_val_precision.to_csv(os.path.join(output_dir, 'val_metrics_best_precision.csv'), index=False)
df_test_f1.to_csv(os.path.join(output_dir, 'test_metrics_best_f1.csv'), index=False)
df_test_precision.to_csv(os.path.join(output_dir, 'test_metrics_best_precision.csv'), index=False)
df_split_info.to_csv(os.path.join(output_dir, 'split_info.csv'), index=False)
df_best_thresholds.to_csv(os.path.join(output_dir, 'best_thresholds.csv'), index=False)
df_val_thresholds.to_csv(os.path.join(output_dir, 'val_metrics_by_threshold.csv'), index=False)
df_test_thresholds.to_csv(os.path.join(output_dir, 'test_metrics_by_threshold.csv'), index=False)


# ============================================================
# RESUMEN FINAL
# ============================================================

metric_cols = [
    'auc',
    'auprc',
    'baseline_auprc',
    'f1',
    'precision',
    'recall',
    'specificity',
    'balanced_accuracy',
    'mcc',
    'log_loss',
    'brier',
    'tp',
    'tn',
    'fp',
    'fn',
]

summary_f1 = (
    df_test_f1[metric_cols]
    .agg(['mean', 'std'])
    .T
    .reset_index()
    .rename(columns={'index': 'metric'})
)

summary_precision = (
    df_test_precision[metric_cols]
    .agg(['mean', 'std'])
    .T
    .reset_index()
    .rename(columns={'index': 'metric'})
)

summary_train_f1 = (
    df_train_f1[metric_cols]
    .agg(['mean', 'std'])
    .T
    .reset_index()
    .rename(columns={'index': 'metric'})
)

summary_train_precision = (
    df_train_precision[metric_cols]
    .agg(['mean', 'std'])
    .T
    .reset_index()
    .rename(columns={'index': 'metric'})
)

summary_val_f1 = (
    df_val_f1[metric_cols]
    .agg(['mean', 'std'])
    .T
    .reset_index()
    .rename(columns={'index': 'metric'})
)

summary_val_precision = (
    df_val_precision[metric_cols]
    .agg(['mean', 'std'])
    .T
    .reset_index()
    .rename(columns={'index': 'metric'})
)

summary_train_f1.to_csv(os.path.join(output_dir, 'summary_train_best_f1.csv'), index=False)
summary_train_precision.to_csv(os.path.join(output_dir, 'summary_train_best_precision.csv'), index=False)
summary_val_f1.to_csv(os.path.join(output_dir, 'summary_val_best_f1.csv'), index=False)
summary_val_precision.to_csv(os.path.join(output_dir, 'summary_val_best_precision.csv'), index=False)
summary_f1.to_csv(os.path.join(output_dir, 'summary_test_best_f1.csv'), index=False)
summary_precision.to_csv(os.path.join(output_dir, 'summary_test_best_precision.csv'), index=False)

print('\n' + '=' * 70)
print('RESULTADOS RANDOM FOREST - LODO PARCIAL CV=10')
print('=' * 70)

print('\n[TEST - threshold elegido por mejor F1 en validacion]')
print(summary_f1)

print('\n[TEST - threshold conservador por mejor precision en validacion]')
print(summary_precision)

print('\nArchivos guardados en:', output_dir)



## RANDOM FOREST: STRATIFIED KFOLD

In [ ]:
import os
import json
import shutil
import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from torch.utils.tensorboard import SummaryWriter
from PIL import Image

if not hasattr(Image, "Resampling"):
    class Resampling:
        LANCZOS = Image.ANTIALIAS

    Image.Resampling = Resampling


# ============================================================
# CONFIGURACION
# ============================================================

SAVE_MODELS = False

output_dir = 'results/random_forest_stratified_cv'
tb_dir = os.path.join(output_dir, 'tensorboard')

if os.path.exists(output_dir):
    shutil.rmtree(output_dir)

os.makedirs(output_dir, exist_ok=True)
os.makedirs(os.path.join(output_dir, 'predictions'), exist_ok=True)
os.makedirs(os.path.join(output_dir, 'figures'), exist_ok=True)
os.makedirs(tb_dir, exist_ok=True)

if SAVE_MODELS:
    os.makedirs(os.path.join(output_dir, 'models'), exist_ok=True)

rf_config = {
    'n_estimators': 600,
    'max_depth': None,
    'min_samples_split': 2,
    'min_samples_leaf': 1,
    'max_features': 'sqrt',
    'class_weight': 'balanced_subsample',
    'random_state': 42,
    'n_jobs': 4,
}

experiment_config = {
    'model': 'RandomForestClassifier',
    'split_type': 'stratified_random_cv',
    'n_splits': 10,
    'n_genes': 500,
    'random_state': 42,
    'threshold_grid_min': 0.05,
    'threshold_grid_max': 0.95,
    'threshold_grid_step': 0.01,
    'rf_config': rf_config,
    'save_models': SAVE_MODELS,
}

save_experiment_config(experiment_config, output_dir)


# ============================================================
# DATASET BASE
# ============================================================

base_df = X_final_df.copy()
base_df['drug_row_id'] = base_df['drug_row_id'].astype(str)
base_df['drug_col_id'] = base_df['drug_col_id'].astype(str)
base_df['synergy_loewe_bin'] = base_df['synergy_loewe_bin'].replace(-1, 0).astype(int)

thresholds = np.round(np.arange(0.05, 0.951, 0.01), 2)

# ============================================================
# CREAR/CARGAR SPLITS STRATIFIED K-FOLD COMPARTIDOS
# ============================================================

shared_splits_dir = 'results/shared_splits'
os.makedirs(shared_splits_dir, exist_ok=True)

skf_splits_path = os.path.join(shared_splits_dir, 'stratified_kfold_indices.json')

if os.path.exists(skf_splits_path):
    with open(skf_splits_path, 'r') as f:
        split_indices = json.load(f)

    print('Splits StratifiedKFold cargados desde:', skf_splits_path)

else:
    skf = StratifiedKFold(
        n_splits=experiment_config['n_splits'],
        shuffle=True,
        random_state=experiment_config['random_state']
    )

    split_indices = []

    for fold_idx, (train_idx, test_idx) in enumerate(
        skf.split(base_df, base_df['synergy_loewe_bin']),
        start=1
    ):
        split_indices.append({
            'fold': fold_idx,
            'train_idx': train_idx.tolist(),
            'test_idx': test_idx.tolist()
        })

    with open(skf_splits_path, 'w') as f:
        json.dump(split_indices, f, indent=4)

    print('Splits StratifiedKFold creados y guardados en:', skf_splits_path)



# ============================================================
# BUCLE CV
# ============================================================

cv_val_results_f1 = []
cv_val_results_precision = []
cv_test_results_f1 = []
cv_test_results_precision = []
cv_split_info = []
cv_val_threshold_metrics = []
cv_test_threshold_metrics = []
cv_best_thresholds = []

for split in split_indices:

    fold_idx = split['fold']
    train_idx = np.array(split['train_idx'])
    test_idx = np.array(split['test_idx'])
    writer = SummaryWriter(log_dir=os.path.join(tb_dir, f'Fold_{fold_idx}'))

    print(f'\nPROCESANDO FOLD {fold_idx}/{experiment_config["n_splits"]} - RANDOM FOREST STRATIFIED CV')

    df_train_raw = base_df.iloc[train_idx].reset_index(drop=True)
    df_test_raw = base_df.iloc[test_idx].reset_index(drop=True)

    if len(df_train_raw) == 0 or len(df_test_raw) == 0:
        print(f'Fold {fold_idx}: sin datos suficientes, se omite.')
        writer.close()
        continue

    train_drugs = set(pd.concat([
        df_train_raw['drug_row_id'],
        df_train_raw['drug_col_id']
    ]).astype(str))

    test_drugs = set(pd.concat([
        df_test_raw['drug_row_id'],
        df_test_raw['drug_col_id']
    ]).astype(str))

    drug_overlap_train_test = len(train_drugs & test_drugs)
    drug_overlap_train_test_pct = (
        100 * drug_overlap_train_test / len(test_drugs)
        if len(test_drugs) > 0 else 0.0
    )

    df_train, df_val, df_test, top_genes = prepare_fold_data_no_leakage(
        df_train_raw,
        df_test_raw,
        data_expr,
        n_genes=experiment_config['n_genes'],
        random_state=experiment_config['random_state']
    )

    X_train, y_train, X_val, y_val, X_test, y_test, feature_cols = build_tabular_matrices(
        df_train,
        df_val,
        df_test
    )

    if len(np.unique(y_train)) < 2:
        print(f'Fold {fold_idx}: train con una sola clase, se omite.')
        writer.close()
        continue

    rf = RandomForestClassifier(**rf_config)

    print(f'Entrenando RandomForest fold {fold_idx}...')
    rf.fit(X_train, y_train)
    print(f'Entrenamiento terminado fold {fold_idx}.')

    y_prob_val = rf.predict_proba(X_val)[:, 1]
    y_prob_test = rf.predict_proba(X_test)[:, 1]

    df_val_thr, best_thr_f1, best_thr_precision = find_best_thresholds(
        y_val,
        y_prob_val,
        thresholds
    )

    df_val_thr.insert(0, 'fold', fold_idx)
    df_val_thr.insert(1, 'split', 'val')
    df_val_thr.insert(2, 'threshold_type', 'grid')
    cv_val_threshold_metrics.append(df_val_thr)

    test_thr_rows = []
    for thr in thresholds:
        m_test_thr = compute_metrics_at_threshold(y_test, y_prob_test, thr)
        m_test_thr.update({
            'fold': fold_idx,
            'split': 'test',
            'threshold_type': 'grid',
        })
        test_thr_rows.append(m_test_thr)

    df_test_thr = pd.DataFrame(test_thr_rows)
    cv_test_threshold_metrics.append(df_test_thr)

    val_metrics_f1 = compute_metrics_at_threshold(y_val, y_prob_val, best_thr_f1)
    val_metrics_precision = compute_metrics_at_threshold(y_val, y_prob_val, best_thr_precision)

    test_metrics_f1 = compute_metrics_at_threshold(y_test, y_prob_test, best_thr_f1)
    test_metrics_precision = compute_metrics_at_threshold(y_test, y_prob_test, best_thr_precision)

    val_metrics_f1.update({'fold': fold_idx, 'split': 'val', 'threshold_type': 'best_f1'})
    val_metrics_precision.update({'fold': fold_idx, 'split': 'val', 'threshold_type': 'best_precision'})
    test_metrics_f1.update({'fold': fold_idx, 'split': 'test', 'threshold_type': 'best_f1'})
    test_metrics_precision.update({'fold': fold_idx, 'split': 'test', 'threshold_type': 'best_precision'})

    cv_val_results_f1.append(val_metrics_f1)
    cv_val_results_precision.append(val_metrics_precision)
    cv_test_results_f1.append(test_metrics_f1)
    cv_test_results_precision.append(test_metrics_precision)

    cv_best_thresholds.append({
        'fold': fold_idx,
        'best_threshold_f1': best_thr_f1,
        'best_threshold_precision': best_thr_precision,
        'val_f1_at_best_f1': val_metrics_f1['f1'],
        'val_precision_at_best_f1': val_metrics_f1['precision'],
        'val_recall_at_best_f1': val_metrics_f1['recall'],
        'val_precision_at_best_precision': val_metrics_precision['precision'],
        'val_recall_at_best_precision': val_metrics_precision['recall'],
        'val_f1_at_best_precision': val_metrics_precision['f1'],
    })

    log_curves_and_metrics(y_val, y_prob_val, val_metrics_f1, fold_idx, writer, tag='Val_BestF1')
    log_curves_and_metrics(y_val, y_prob_val, val_metrics_precision, fold_idx, writer, tag='Val_BestPrecision')
    log_curves_and_metrics(y_test, y_prob_test, test_metrics_f1, fold_idx, writer, tag='Test_BestF1')
    log_curves_and_metrics(y_test, y_prob_test, test_metrics_precision, fold_idx, writer, tag='Test_BestPrecision')

    predictions = df_test[
        [
            'drug_row_id',
            'drug_col_id',
            'cell_line_name',
            'study_name',
            'tissue',
            'synergy_loewe',
            'synergy_loewe_bin'
        ]
    ].copy()

    predictions['y_true'] = y_test
    predictions['y_prob'] = y_prob_test
    predictions['y_pred_best_f1'] = (y_prob_test >= best_thr_f1).astype(int)
    predictions['y_pred_best_precision'] = (y_prob_test >= best_thr_precision).astype(int)
    predictions['threshold_best_f1'] = best_thr_f1
    predictions['threshold_best_precision'] = best_thr_precision

    predictions.to_csv(
        os.path.join(output_dir, 'predictions', f'predictions_fold_{fold_idx:02d}.csv'),
        index=False
    )

    if SAVE_MODELS:
        joblib.dump(
            rf,
            os.path.join(output_dir, 'models', f'random_forest_fold_{fold_idx:02d}.joblib')
        )

    n_train = len(y_train)
    n_val = len(y_val)
    n_test = len(y_test)

    split_info = {
        'fold': fold_idx,
        'n_train': n_train,
        'n_val': n_val,
        'n_test': n_test,
        'train_pos': int(np.sum(y_train)),
        'val_pos': int(np.sum(y_val)),
        'test_pos': int(np.sum(y_test)),
        'train_pos_pct': 100 * np.mean(y_train),
        'val_pos_pct': 100 * np.mean(y_val),
        'test_pos_pct': 100 * np.mean(y_test),
        'baseline_auprc_test': float(np.mean(y_test)),
        'n_train_drugs': len(train_drugs),
        'n_test_drugs': len(test_drugs),
        'drug_overlap_train_test': drug_overlap_train_test,
        'drug_overlap_train_test_pct': drug_overlap_train_test_pct,
        'best_threshold_f1': best_thr_f1,
        'best_threshold_precision': best_thr_precision,
        'n_features': len(feature_cols),
        'n_top_genes': len(top_genes),
        'split_indices_path': skf_splits_path
    }

    cv_split_info.append(split_info)

    writer.add_text('Split/Info', json.dumps(split_info, indent=2))
    writer.add_text('Genes/TopGenes_First100', ', '.join(top_genes[:100]))

    importances = pd.Series(rf.feature_importances_, index=feature_cols)
    top_imp = importances.sort_values(ascending=False).head(30)
    writer.add_text('Model/Top30_Feature_Importance', top_imp.to_string())

    print(
        f"Fold {fold_idx} | "
        f"Train={n_train} ({100*np.mean(y_train):.2f}% pos), "
        f"Val={n_val} ({100*np.mean(y_val):.2f}% pos), "
        f"Test={n_test} ({100*np.mean(y_test):.2f}% pos) | "
        f"DrugOverlap={drug_overlap_train_test_pct:.2f}% | "
        f"ThrF1={best_thr_f1:.2f}, ThrPrec={best_thr_precision:.2f} | "
        f"AUPRC={test_metrics_f1['auprc']:.4f} "
        f"(base={test_metrics_f1['baseline_auprc']:.4f}) | "
        f"F1={test_metrics_f1['f1']:.4f} | "
        f"Prec={test_metrics_f1['precision']:.4f} | "
        f"Rec={test_metrics_f1['recall']:.4f} | "
        f"MCC={test_metrics_f1['mcc']:.4f}"
    )

    writer.close()


# ============================================================
# GUARDADO DE RESULTADOS
# ============================================================

df_val_f1 = pd.DataFrame(cv_val_results_f1)
df_val_precision = pd.DataFrame(cv_val_results_precision)
df_test_f1 = pd.DataFrame(cv_test_results_f1)
df_test_precision = pd.DataFrame(cv_test_results_precision)
df_split_info = pd.DataFrame(cv_split_info)
df_best_thresholds = pd.DataFrame(cv_best_thresholds)

df_val_thresholds = pd.concat(cv_val_threshold_metrics, ignore_index=True)
df_test_thresholds = pd.concat(cv_test_threshold_metrics, ignore_index=True)

df_val_f1.to_csv(os.path.join(output_dir, 'val_metrics_best_f1.csv'), index=False)
df_val_precision.to_csv(os.path.join(output_dir, 'val_metrics_best_precision.csv'), index=False)
df_test_f1.to_csv(os.path.join(output_dir, 'test_metrics_best_f1.csv'), index=False)
df_test_precision.to_csv(os.path.join(output_dir, 'test_metrics_best_precision.csv'), index=False)
df_split_info.to_csv(os.path.join(output_dir, 'split_info.csv'), index=False)
df_best_thresholds.to_csv(os.path.join(output_dir, 'best_thresholds.csv'), index=False)
df_val_thresholds.to_csv(os.path.join(output_dir, 'val_metrics_by_threshold.csv'), index=False)
df_test_thresholds.to_csv(os.path.join(output_dir, 'test_metrics_by_threshold.csv'), index=False)


# ============================================================
# RESUMEN FINAL
# ============================================================

metric_cols = [
    'auc',
    'auprc',
    'baseline_auprc',
    'f1',
    'precision',
    'recall',
    'specificity',
    'balanced_accuracy',
    'mcc',
    'log_loss',
    'brier',
    'tp',
    'tn',
    'fp',
    'fn',
]

summary_f1 = (
    df_test_f1[metric_cols]
    .agg(['mean', 'std'])
    .T
    .reset_index()
    .rename(columns={'index': 'metric'})
)

summary_precision = (
    df_test_precision[metric_cols]
    .agg(['mean', 'std'])
    .T
    .reset_index()
    .rename(columns={'index': 'metric'})
)

summary_f1.to_csv(os.path.join(output_dir, 'summary_test_best_f1.csv'), index=False)
summary_precision.to_csv(os.path.join(output_dir, 'summary_test_best_precision.csv'), index=False)

print('\n' + '=' * 70)
print('RESULTADOS RANDOM FOREST - STRATIFIED RANDOM CV=10')
print('=' * 70)

print('\n[TEST - threshold elegido por mejor F1 en validacion]')
print(summary_f1)

print('\n[TEST - threshold conservador por mejor precision en validacion]')
print(summary_precision)

print('\nArchivos guardados en:', output_dir)


In [ ]:
import os
import json
import shutil
import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from torch.utils.tensorboard import SummaryWriter
from PIL import Image

if not hasattr(Image, "Resampling"):
    class Resampling:
        LANCZOS = Image.ANTIALIAS

    Image.Resampling = Resampling


# ============================================================
# CONFIGURACION
# ============================================================

SAVE_MODELS = False

output_dir = 'results/aaatrain_random_forest_stratified_cv'
tb_dir = os.path.join(output_dir, 'tensorboard')

if os.path.exists(output_dir):
    shutil.rmtree(output_dir)

os.makedirs(output_dir, exist_ok=True)
os.makedirs(os.path.join(output_dir, 'predictions'), exist_ok=True)
os.makedirs(os.path.join(output_dir, 'figures'), exist_ok=True)
os.makedirs(tb_dir, exist_ok=True)

if SAVE_MODELS:
    os.makedirs(os.path.join(output_dir, 'models'), exist_ok=True)

rf_config = {
    'n_estimators': 600,
    'max_depth': None,
    'min_samples_split': 2,
    'min_samples_leaf': 1,
    'max_features': 'sqrt',
    'class_weight': 'balanced_subsample',
    'random_state': 42,
    'n_jobs': 4,
}

experiment_config = {
    'model': 'RandomForestClassifier',
    'split_type': 'stratified_random_cv',
    'n_splits': 10,
    'n_genes': 500,
    'random_state': 42,
    'threshold_grid_min': 0.05,
    'threshold_grid_max': 0.95,
    'threshold_grid_step': 0.01,
    'rf_config': rf_config,
    'save_models': SAVE_MODELS,
}

save_experiment_config(experiment_config, output_dir)


# ============================================================
# DATASET BASE
# ============================================================

base_df = X_final_df.copy()
base_df['drug_row_id'] = base_df['drug_row_id'].astype(str)
base_df['drug_col_id'] = base_df['drug_col_id'].astype(str)
base_df['synergy_loewe_bin'] = base_df['synergy_loewe_bin'].replace(-1, 0).astype(int)

thresholds = np.round(np.arange(0.05, 0.951, 0.01), 2)

# ============================================================
# CREAR/CARGAR SPLITS STRATIFIED K-FOLD COMPARTIDOS
# ============================================================

shared_splits_dir = 'results/shared_splits'
os.makedirs(shared_splits_dir, exist_ok=True)

skf_splits_path = os.path.join(shared_splits_dir, 'stratified_kfold_indices.json')

if os.path.exists(skf_splits_path):
    with open(skf_splits_path, 'r') as f:
        split_indices = json.load(f)

    print('Splits StratifiedKFold cargados desde:', skf_splits_path)

else:
    skf = StratifiedKFold(
        n_splits=experiment_config['n_splits'],
        shuffle=True,
        random_state=experiment_config['random_state']
    )

    split_indices = []

    for fold_idx, (train_idx, test_idx) in enumerate(
        skf.split(base_df, base_df['synergy_loewe_bin']),
        start=1
    ):
        split_indices.append({
            'fold': fold_idx,
            'train_idx': train_idx.tolist(),
            'test_idx': test_idx.tolist()
        })

    with open(skf_splits_path, 'w') as f:
        json.dump(split_indices, f, indent=4)

    print('Splits StratifiedKFold creados y guardados en:', skf_splits_path)



# ============================================================
# BUCLE CV
# ============================================================

cv_train_results_f1 = []
cv_train_results_precision = []
cv_val_results_f1 = []
cv_val_results_precision = []
cv_test_results_f1 = []
cv_test_results_precision = []
cv_split_info = []
cv_val_threshold_metrics = []
cv_test_threshold_metrics = []
cv_best_thresholds = []

for split in split_indices:

    fold_idx = split['fold']
    train_idx = np.array(split['train_idx'])
    test_idx = np.array(split['test_idx'])
    writer = SummaryWriter(log_dir=os.path.join(tb_dir, f'Fold_{fold_idx}'))

    print(f'\nPROCESANDO FOLD {fold_idx}/{experiment_config["n_splits"]} - RANDOM FOREST STRATIFIED CV')

    df_train_raw = base_df.iloc[train_idx].reset_index(drop=True)
    df_test_raw = base_df.iloc[test_idx].reset_index(drop=True)

    if len(df_train_raw) == 0 or len(df_test_raw) == 0:
        print(f'Fold {fold_idx}: sin datos suficientes, se omite.')
        writer.close()
        continue

    train_drugs = set(pd.concat([
        df_train_raw['drug_row_id'],
        df_train_raw['drug_col_id']
    ]).astype(str))

    test_drugs = set(pd.concat([
        df_test_raw['drug_row_id'],
        df_test_raw['drug_col_id']
    ]).astype(str))

    drug_overlap_train_test = len(train_drugs & test_drugs)
    drug_overlap_train_test_pct = (
        100 * drug_overlap_train_test / len(test_drugs)
        if len(test_drugs) > 0 else 0.0
    )

    df_train, df_val, df_test, top_genes = prepare_fold_data_no_leakage(
        df_train_raw,
        df_test_raw,
        data_expr,
        n_genes=experiment_config['n_genes'],
        random_state=experiment_config['random_state']
    )

    X_train, y_train, X_val, y_val, X_test, y_test, feature_cols = build_tabular_matrices(
        df_train,
        df_val,
        df_test
    )

    if len(np.unique(y_train)) < 2:
        print(f'Fold {fold_idx}: train con una sola clase, se omite.')
        writer.close()
        continue

    rf = RandomForestClassifier(**rf_config)

    print(f'Entrenando RandomForest fold {fold_idx}...')
    rf.fit(X_train, y_train)
    print(f'Entrenamiento terminado fold {fold_idx}.')

    y_prob_train = rf.predict_proba(X_train)[:, 1]
    y_prob_val = rf.predict_proba(X_val)[:, 1]
    y_prob_test = rf.predict_proba(X_test)[:, 1]

    df_val_thr, best_thr_f1, best_thr_precision = find_best_thresholds(
        y_val,
        y_prob_val,
        thresholds
    )

    df_val_thr.insert(0, 'fold', fold_idx)
    df_val_thr.insert(1, 'split', 'val')
    df_val_thr.insert(2, 'threshold_type', 'grid')
    cv_val_threshold_metrics.append(df_val_thr)

    test_thr_rows = []
    for thr in thresholds:
        m_test_thr = compute_metrics_at_threshold(y_test, y_prob_test, thr)
        m_test_thr.update({
            'fold': fold_idx,
            'split': 'test',
            'threshold_type': 'grid',
        })
        test_thr_rows.append(m_test_thr)

    df_test_thr = pd.DataFrame(test_thr_rows)
    cv_test_threshold_metrics.append(df_test_thr)

    train_metrics_f1 = compute_metrics_at_threshold(y_train, y_prob_train, best_thr_f1)
    train_metrics_precision = compute_metrics_at_threshold(y_train, y_prob_train, best_thr_precision)

    val_metrics_f1 = compute_metrics_at_threshold(y_val, y_prob_val, best_thr_f1)
    val_metrics_precision = compute_metrics_at_threshold(y_val, y_prob_val, best_thr_precision)

    test_metrics_f1 = compute_metrics_at_threshold(y_test, y_prob_test, best_thr_f1)
    test_metrics_precision = compute_metrics_at_threshold(y_test, y_prob_test, best_thr_precision)

    train_metrics_f1.update({'fold': fold_idx, 'split': 'train', 'threshold_type': 'best_f1'})
    train_metrics_precision.update({'fold': fold_idx, 'split': 'train', 'threshold_type': 'best_precision'})
    val_metrics_f1.update({'fold': fold_idx, 'split': 'val', 'threshold_type': 'best_f1'})
    val_metrics_precision.update({'fold': fold_idx, 'split': 'val', 'threshold_type': 'best_precision'})
    test_metrics_f1.update({'fold': fold_idx, 'split': 'test', 'threshold_type': 'best_f1'})
    test_metrics_precision.update({'fold': fold_idx, 'split': 'test', 'threshold_type': 'best_precision'})

    cv_val_results_f1.append(val_metrics_f1)
    cv_val_results_precision.append(val_metrics_precision)
    cv_test_results_f1.append(test_metrics_f1)
    cv_test_results_precision.append(test_metrics_precision)
    cv_train_results_f1.append(train_metrics_f1)
    cv_train_results_precision.append(train_metrics_precision)

    cv_best_thresholds.append({
        'fold': fold_idx,
        'best_threshold_f1': best_thr_f1,
        'best_threshold_precision': best_thr_precision,
        'val_f1_at_best_f1': val_metrics_f1['f1'],
        'val_precision_at_best_f1': val_metrics_f1['precision'],
        'val_recall_at_best_f1': val_metrics_f1['recall'],
        'val_precision_at_best_precision': val_metrics_precision['precision'],
        'val_recall_at_best_precision': val_metrics_precision['recall'],
        'val_f1_at_best_precision': val_metrics_precision['f1'],
    })

    log_curves_and_metrics(y_val, y_prob_val, val_metrics_f1, fold_idx, writer, tag='Val_BestF1')
    log_curves_and_metrics(y_val, y_prob_val, val_metrics_precision, fold_idx, writer, tag='Val_BestPrecision')
    log_curves_and_metrics(y_test, y_prob_test, test_metrics_f1, fold_idx, writer, tag='Test_BestF1')
    log_curves_and_metrics(y_test, y_prob_test, test_metrics_precision, fold_idx, writer, tag='Test_BestPrecision')

    train_predictions = save_split_predictions(
        df_split=df_train,
        y_true=y_train,
        y_prob=y_prob_train,
        best_thr_f1=best_thr_f1,
        best_thr_precision=best_thr_precision,
        output_dir=output_dir,
        fold_idx=fold_idx,
        split_name='train'
    )

    val_predictions = save_split_predictions(
        df_split=df_val,
        y_true=y_val,
        y_prob=y_prob_val,
        best_thr_f1=best_thr_f1,
        best_thr_precision=best_thr_precision,
        output_dir=output_dir,
        fold_idx=fold_idx,
        split_name='val'
    )

    test_predictions = save_split_predictions(
        df_split=df_test,
        y_true=y_test,
        y_prob=y_prob_test,
        best_thr_f1=best_thr_f1,
        best_thr_precision=best_thr_precision,
        output_dir=output_dir,
        fold_idx=fold_idx,
        split_name='test',
        filename_prefix='predictions'
    )


    if SAVE_MODELS:
        joblib.dump(
            rf,
            os.path.join(output_dir, 'models', f'random_forest_fold_{fold_idx:02d}.joblib')
        )

    n_train = len(y_train)
    n_val = len(y_val)
    n_test = len(y_test)

    split_info = {
        'fold': fold_idx,
        'n_train': n_train,
        'n_val': n_val,
        'n_test': n_test,
        'train_pos': int(np.sum(y_train)),
        'val_pos': int(np.sum(y_val)),
        'test_pos': int(np.sum(y_test)),
        'train_pos_pct': 100 * np.mean(y_train),
        'val_pos_pct': 100 * np.mean(y_val),
        'test_pos_pct': 100 * np.mean(y_test),
        'baseline_auprc_test': float(np.mean(y_test)),
        'n_train_drugs': len(train_drugs),
        'n_test_drugs': len(test_drugs),
        'drug_overlap_train_test': drug_overlap_train_test,
        'drug_overlap_train_test_pct': drug_overlap_train_test_pct,
        'best_threshold_f1': best_thr_f1,
        'best_threshold_precision': best_thr_precision,
        'n_features': len(feature_cols),
        'n_top_genes': len(top_genes),
        'split_indices_path': skf_splits_path
    }

    cv_split_info.append(split_info)

    writer.add_text('Split/Info', json.dumps(split_info, indent=2))
    writer.add_text('Genes/TopGenes_First100', ', '.join(top_genes[:100]))

    importances = pd.Series(rf.feature_importances_, index=feature_cols)
    top_imp = importances.sort_values(ascending=False).head(30)
    writer.add_text('Model/Top30_Feature_Importance', top_imp.to_string())

    print(
        f"Fold {fold_idx} | "
        f"Train={n_train} ({100*np.mean(y_train):.2f}% pos), "
        f"Val={n_val} ({100*np.mean(y_val):.2f}% pos), "
        f"Test={n_test} ({100*np.mean(y_test):.2f}% pos) | "
        f"DrugOverlap={drug_overlap_train_test_pct:.2f}% | "
        f"ThrF1={best_thr_f1:.2f}, ThrPrec={best_thr_precision:.2f} | "
        f"AUPRC={test_metrics_f1['auprc']:.4f} "
        f"(base={test_metrics_f1['baseline_auprc']:.4f}) | "
        f"F1={test_metrics_f1['f1']:.4f} | "
        f"Prec={test_metrics_f1['precision']:.4f} | "
        f"Rec={test_metrics_f1['recall']:.4f} | "
        f"MCC={test_metrics_f1['mcc']:.4f}"
    )

    writer.close()


# ============================================================
# GUARDADO DE RESULTADOS
# ============================================================

df_train_f1 = pd.DataFrame(cv_train_results_f1)
df_train_precision = pd.DataFrame(cv_train_results_precision)
df_val_f1 = pd.DataFrame(cv_val_results_f1)
df_val_precision = pd.DataFrame(cv_val_results_precision)
df_test_f1 = pd.DataFrame(cv_test_results_f1)
df_test_precision = pd.DataFrame(cv_test_results_precision)
df_split_info = pd.DataFrame(cv_split_info)
df_best_thresholds = pd.DataFrame(cv_best_thresholds)

df_val_thresholds = pd.concat(cv_val_threshold_metrics, ignore_index=True)
df_test_thresholds = pd.concat(cv_test_threshold_metrics, ignore_index=True)

df_train_f1.to_csv(os.path.join(output_dir, 'train_metrics_best_f1.csv'), index=False)
df_train_precision.to_csv(os.path.join(output_dir, 'train_metrics_best_precision.csv'), index=False)
df_val_f1.to_csv(os.path.join(output_dir, 'val_metrics_best_f1.csv'), index=False)
df_val_precision.to_csv(os.path.join(output_dir, 'val_metrics_best_precision.csv'), index=False)
df_test_f1.to_csv(os.path.join(output_dir, 'test_metrics_best_f1.csv'), index=False)
df_test_precision.to_csv(os.path.join(output_dir, 'test_metrics_best_precision.csv'), index=False)
df_split_info.to_csv(os.path.join(output_dir, 'split_info.csv'), index=False)
df_best_thresholds.to_csv(os.path.join(output_dir, 'best_thresholds.csv'), index=False)
df_val_thresholds.to_csv(os.path.join(output_dir, 'val_metrics_by_threshold.csv'), index=False)
df_test_thresholds.to_csv(os.path.join(output_dir, 'test_metrics_by_threshold.csv'), index=False)


# ============================================================
# RESUMEN FINAL
# ============================================================

metric_cols = [
    'auc',
    'auprc',
    'baseline_auprc',
    'f1',
    'precision',
    'recall',
    'specificity',
    'balanced_accuracy',
    'mcc',
    'log_loss',
    'brier',
    'tp',
    'tn',
    'fp',
    'fn',
]

summary_f1 = (
    df_test_f1[metric_cols]
    .agg(['mean', 'std'])
    .T
    .reset_index()
    .rename(columns={'index': 'metric'})
)

summary_precision = (
    df_test_precision[metric_cols]
    .agg(['mean', 'std'])
    .T
    .reset_index()
    .rename(columns={'index': 'metric'})
)

summary_train_f1 = (
    df_train_f1[metric_cols]
    .agg(['mean', 'std'])
    .T
    .reset_index()
    .rename(columns={'index': 'metric'})
)

summary_train_precision = (
    df_train_precision[metric_cols]
    .agg(['mean', 'std'])
    .T
    .reset_index()
    .rename(columns={'index': 'metric'})
)

summary_val_f1 = (
    df_val_f1[metric_cols]
    .agg(['mean', 'std'])
    .T
    .reset_index()
    .rename(columns={'index': 'metric'})
)

summary_val_precision = (
    df_val_precision[metric_cols]
    .agg(['mean', 'std'])
    .T
    .reset_index()
    .rename(columns={'index': 'metric'})
)

summary_train_f1.to_csv(os.path.join(output_dir, 'summary_train_best_f1.csv'), index=False)
summary_train_precision.to_csv(os.path.join(output_dir, 'summary_train_best_precision.csv'), index=False)
summary_val_f1.to_csv(os.path.join(output_dir, 'summary_val_best_f1.csv'), index=False)
summary_val_precision.to_csv(os.path.join(output_dir, 'summary_val_best_precision.csv'), index=False)
summary_f1.to_csv(os.path.join(output_dir, 'summary_test_best_f1.csv'), index=False)
summary_precision.to_csv(os.path.join(output_dir, 'summary_test_best_precision.csv'), index=False)

print('\n' + '=' * 70)
print('RESULTADOS RANDOM FOREST - STRATIFIED RANDOM CV=10')
print('=' * 70)

print('\n[TEST - threshold elegido por mejor F1 en validacion]')
print(summary_f1)

print('\n[TEST - threshold conservador por mejor precision en validacion]')
print(summary_precision)

print('\nArchivos guardados en:', output_dir)


## XGBOOST : Leave-Drug-Out CV=10

In [ ]:
import os
import json
import shutil
import joblib
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from torch.utils.tensorboard import SummaryWriter
from PIL import Image
from xgboost import XGBClassifier

if not hasattr(Image, "Resampling"):
    class Resampling:
        LANCZOS = Image.ANTIALIAS

    Image.Resampling = Resampling

# ============================================================
# CONFIGURACION
# ============================================================

SAVE_MODELS = False

output_dir = 'results/xgboost_lodo'
tb_dir = os.path.join(output_dir, 'tensorboard')

if os.path.exists(output_dir):
    shutil.rmtree(output_dir)

os.makedirs(output_dir, exist_ok=True)
os.makedirs(os.path.join(output_dir, 'predictions'), exist_ok=True)
os.makedirs(os.path.join(output_dir, 'figures'), exist_ok=True)
os.makedirs(tb_dir, exist_ok=True)

if SAVE_MODELS:
    os.makedirs(os.path.join(output_dir, 'models'), exist_ok=True)

xgb_config = {
    'n_estimators': 600,
    'max_depth': 8,
    'learning_rate': 0.03,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'objective': 'binary:logistic',
    'eval_metric': 'aucpr',
    'tree_method': 'hist',
    'random_state': 42,
    'n_jobs': 4,
}


experiment_config = {
    'model': 'XGBClassifier',
    'split_type': 'partial_lodo_optimized',
    'n_splits': 10,
    'n_genes': 500,
    'random_state': 42,
    'threshold_grid_min': 0.05,
    'threshold_grid_max': 0.95,
    'threshold_grid_step': 0.01,
    'xgb_config': xgb_config,
    'save_models': SAVE_MODELS,
    'lodo_definition': (
        'Partial Leave-Drug-Out: each test sample contains at least one held-out drug. '
        'The second drug may appear in train.'
    ),
    'external_lodo_splits_path': 'results/shared_splits/lodo_fold_sets.json',
}

save_experiment_config(experiment_config, output_dir)


# ============================================================
# DATASET BASE
# ============================================================

base_df = X_final_df.copy()
base_df['drug_row_id'] = base_df['drug_row_id'].astype(str)
base_df['drug_col_id'] = base_df['drug_col_id'].astype(str)
base_df['synergy_loewe_bin'] = base_df['synergy_loewe_bin'].replace(-1, 0).astype(int)

thresholds = np.round(np.arange(0.05, 0.951, 0.01), 2)


# ============================================================
# CONSTRUCCION LODO PARCIAL OPTIMIZADO : CARGAR SPLITS LODO COMPARTIDOS
# ============================================================

shared_splits_path = 'results/shared_splits/lodo_fold_sets.json'

with open(shared_splits_path, 'r') as f:
    fold_sets_dict = json.load(f)

fold_sets = [
    set(fold_sets_dict[str(i + 1)])
    for i in range(len(fold_sets_dict))
]

assert len(fold_sets) == experiment_config['n_splits'], (
    f"Se esperaban {experiment_config['n_splits']} folds, "
    f"pero se cargaron {len(fold_sets)}."
)

# Recalcular resumen de la particion cargada
partition_rows = []

for fold_idx, fold_set in enumerate(fold_sets, start=1):
    mask_test = (
        base_df['drug_row_id'].isin(fold_set)
        | base_df['drug_col_id'].isin(fold_set)
    )

    fold_df = base_df[mask_test]

    n_examples = len(fold_df)
    pos_rate = fold_df['synergy_loewe_bin'].mean() if n_examples > 0 else np.nan

    partition_rows.append({
        'fold': fold_idx,
        'n_examples_induced_test': n_examples,
        'positive_pct_induced_test': pos_rate * 100,
        'baseline_auprc_induced_test': pos_rate,
        'n_heldout_drugs': len(fold_set)
    })

fold_partition_summary = pd.DataFrame(partition_rows)

fold_partition_summary.to_csv(
    os.path.join(output_dir, 'lodo_partition_summary.csv'),
    index=False
)

print('\nSplits LODO compartidos cargados desde:', shared_splits_path)
print('\nResumen particion LODO:')
print(fold_partition_summary)


# ============================================================
# SOLAPAMIENTO ENTRE TESTS
# ============================================================

test_overlap_info, overlap_df, overlap_counts = summarize_test_overlap(
    base_df,
    fold_sets,
    output_path=output_dir
)

print('\nSolapamiento entre conjuntos test:')
print(test_overlap_info)
print('\nDistribucion de apariciones por muestra en test:')
print(overlap_counts)


# ============================================================
# BUCLE CV
# ============================================================

cv_val_results_f1 = []
cv_val_results_precision = []
cv_test_results_f1 = []
cv_test_results_precision = []
cv_split_info = []
cv_val_threshold_metrics = []
cv_test_threshold_metrics = []
cv_best_thresholds = []

for fold_idx, test_set in enumerate(fold_sets, start=1):
    writer = SummaryWriter(log_dir=os.path.join(tb_dir, f'Fold_{fold_idx}'))

    print(f'\nPROCESANDO FOLD {fold_idx}/{experiment_config["n_splits"]} - XGBOOST LODO')

    mask_test = (
        base_df['drug_row_id'].isin(test_set)
        | base_df['drug_col_id'].isin(test_set)
    )

    df_test_raw = base_df[mask_test].reset_index(drop=True)
    df_train_raw = base_df[~mask_test].reset_index(drop=True)

    if len(df_train_raw) == 0 or len(df_test_raw) == 0:
        print(f'Fold {fold_idx}: sin datos suficientes, se omite.')
        writer.close()
        continue

    assert (
        df_test_raw['drug_row_id'].isin(test_set)
        | df_test_raw['drug_col_id'].isin(test_set)
    ).all(), f'Fold {fold_idx}: hay muestras test sin droga held-out.'

    train_drugs = set(pd.concat([
        df_train_raw['drug_row_id'],
        df_train_raw['drug_col_id']
    ]).astype(str))

    heldout_overlap_train = len(train_drugs & test_set)
    assert heldout_overlap_train == 0, f'Fold {fold_idx}: drogas held-out aparecen en train.'

    df_train, df_val, df_test, top_genes = prepare_fold_data_no_leakage(
        df_train_raw,
        df_test_raw,
        data_expr,
        n_genes=experiment_config['n_genes'],
        random_state=experiment_config['random_state']
    )

    X_train, y_train, X_val, y_val, X_test, y_test, feature_cols = build_tabular_matrices(
        df_train,
        df_val,
        df_test
    )

    if len(np.unique(y_train)) < 2:
        print(f'Fold {fold_idx}: train con una sola clase, se omite.')
        writer.close()
        continue

    n_pos = np.sum(y_train)
    n_neg = len(y_train) - n_pos

    xgb_config_fold = xgb_config.copy()
    xgb_config_fold['scale_pos_weight'] = n_neg / n_pos

    xgb = XGBClassifier(**xgb_config_fold)

    print(f'Entrenando XGBoost fold {fold_idx}...')
    xgb.fit(X_train, y_train)
    print(f'Entrenamiento terminado fold {fold_idx}.')

    y_prob_val = xgb.predict_proba(X_val)[:, 1]
    y_prob_test = xgb.predict_proba(X_test)[:, 1]

    df_val_thr, best_thr_f1, best_thr_precision = find_best_thresholds(
        y_val,
        y_prob_val,
        thresholds
    )

    df_val_thr.insert(0, 'fold', fold_idx)
    df_val_thr.insert(1, 'split', 'val')
    df_val_thr.insert(2, 'threshold_type', 'grid')
    cv_val_threshold_metrics.append(df_val_thr)

    test_thr_rows = []
    for thr in thresholds:
        m_test_thr = compute_metrics_at_threshold(y_test, y_prob_test, thr)
        m_test_thr.update({
            'fold': fold_idx,
            'split': 'test',
            'threshold_type': 'grid',
        })
        test_thr_rows.append(m_test_thr)

    df_test_thr = pd.DataFrame(test_thr_rows)
    cv_test_threshold_metrics.append(df_test_thr)

    val_metrics_f1 = compute_metrics_at_threshold(y_val, y_prob_val, best_thr_f1)
    val_metrics_precision = compute_metrics_at_threshold(y_val, y_prob_val, best_thr_precision)

    test_metrics_f1 = compute_metrics_at_threshold(y_test, y_prob_test, best_thr_f1)
    test_metrics_precision = compute_metrics_at_threshold(y_test, y_prob_test, best_thr_precision)

    val_metrics_f1.update({'fold': fold_idx, 'split': 'val', 'threshold_type': 'best_f1'})
    val_metrics_precision.update({'fold': fold_idx, 'split': 'val', 'threshold_type': 'best_precision'})
    test_metrics_f1.update({'fold': fold_idx, 'split': 'test', 'threshold_type': 'best_f1'})
    test_metrics_precision.update({'fold': fold_idx, 'split': 'test', 'threshold_type': 'best_precision'})

    cv_val_results_f1.append(val_metrics_f1)
    cv_val_results_precision.append(val_metrics_precision)
    cv_test_results_f1.append(test_metrics_f1)
    cv_test_results_precision.append(test_metrics_precision)

    cv_best_thresholds.append({
        'fold': fold_idx,
        'best_threshold_f1': best_thr_f1,
        'best_threshold_precision': best_thr_precision,
        'val_f1_at_best_f1': val_metrics_f1['f1'],
        'val_precision_at_best_f1': val_metrics_f1['precision'],
        'val_recall_at_best_f1': val_metrics_f1['recall'],
        'val_precision_at_best_precision': val_metrics_precision['precision'],
        'val_recall_at_best_precision': val_metrics_precision['recall'],
        'val_f1_at_best_precision': val_metrics_precision['f1'],
    })

    log_curves_and_metrics(y_val, y_prob_val, val_metrics_f1, fold_idx, writer, tag='Val_BestF1')
    log_curves_and_metrics(y_val, y_prob_val, val_metrics_precision, fold_idx, writer, tag='Val_BestPrecision')
    log_curves_and_metrics(y_test, y_prob_test, test_metrics_f1, fold_idx, writer, tag='Test_BestF1')
    log_curves_and_metrics(y_test, y_prob_test, test_metrics_precision, fold_idx, writer, tag='Test_BestPrecision')

    predictions = df_test[
        [
            'drug_row_id',
            'drug_col_id',
            'cell_line_name',
            'study_name',
            'tissue',
            'synergy_loewe',
            'synergy_loewe_bin'
        ]
    ].copy()

    predictions['y_true'] = y_test
    predictions['y_prob'] = y_prob_test
    predictions['y_pred_best_f1'] = (y_prob_test >= best_thr_f1).astype(int)
    predictions['y_pred_best_precision'] = (y_prob_test >= best_thr_precision).astype(int)
    predictions['threshold_best_f1'] = best_thr_f1
    predictions['threshold_best_precision'] = best_thr_precision

    predictions.to_csv(
        os.path.join(output_dir, 'predictions', f'predictions_fold_{fold_idx:02d}.csv'),
        index=False
    )

    if SAVE_MODELS:
        joblib.dump(
            xgb,
            os.path.join(output_dir, 'models', f'xgboost_fold_{fold_idx:02d}.joblib')
        )

    n_train = len(y_train)
    n_val = len(y_val)
    n_test = len(y_test)

    split_info = {
        'fold': fold_idx,
        'n_train': n_train,
        'n_val': n_val,
        'n_test': n_test,
        'train_pos': int(np.sum(y_train)),
        'val_pos': int(np.sum(y_val)),
        'test_pos': int(np.sum(y_test)),
        'train_pos_pct': 100 * np.mean(y_train),
        'val_pos_pct': 100 * np.mean(y_val),
        'test_pos_pct': 100 * np.mean(y_test),
        'baseline_auprc_test': float(np.mean(y_test)),
        'n_heldout_drugs': len(test_set),
        'heldout_drugs': ';'.join(sorted(test_set)),
        'heldout_overlap_train': heldout_overlap_train,
        'best_threshold_f1': best_thr_f1,
        'best_threshold_precision': best_thr_precision,
        'n_features': len(feature_cols),
        'n_top_genes': len(top_genes),
    }

    cv_split_info.append(split_info)

    writer.add_text('Split/Info', json.dumps(split_info, indent=2))
    writer.add_text('Genes/TopGenes_First100', ', '.join(top_genes[:100]))

    importances = pd.Series(xgb.feature_importances_, index=feature_cols)
    top_imp = importances.sort_values(ascending=False).head(30)
    writer.add_text('Model/Top30_Feature_Importance', top_imp.to_string())

    print(
        f"Fold {fold_idx} | "
        f"Train={n_train} ({100*np.mean(y_train):.2f}% pos), "
        f"Val={n_val} ({100*np.mean(y_val):.2f}% pos), "
        f"Test={n_test} ({100*np.mean(y_test):.2f}% pos) | "
        f"ThrF1={best_thr_f1:.2f}, ThrPrec={best_thr_precision:.2f} | "
        f"AUPRC={test_metrics_f1['auprc']:.4f} "
        f"(base={test_metrics_f1['baseline_auprc']:.4f}) | "
        f"F1={test_metrics_f1['f1']:.4f} | "
        f"Prec={test_metrics_f1['precision']:.4f} | "
        f"Rec={test_metrics_f1['recall']:.4f} | "
        f"MCC={test_metrics_f1['mcc']:.4f}"
    )

    writer.close()


# ============================================================
# GUARDADO DE RESULTADOS
# ============================================================

df_val_f1 = pd.DataFrame(cv_val_results_f1)
df_val_precision = pd.DataFrame(cv_val_results_precision)
df_test_f1 = pd.DataFrame(cv_test_results_f1)
df_test_precision = pd.DataFrame(cv_test_results_precision)
df_split_info = pd.DataFrame(cv_split_info)
df_best_thresholds = pd.DataFrame(cv_best_thresholds)

df_val_thresholds = pd.concat(cv_val_threshold_metrics, ignore_index=True)
df_test_thresholds = pd.concat(cv_test_threshold_metrics, ignore_index=True)

df_val_f1.to_csv(os.path.join(output_dir, 'val_metrics_best_f1.csv'), index=False)
df_val_precision.to_csv(os.path.join(output_dir, 'val_metrics_best_precision.csv'), index=False)
df_test_f1.to_csv(os.path.join(output_dir, 'test_metrics_best_f1.csv'), index=False)
df_test_precision.to_csv(os.path.join(output_dir, 'test_metrics_best_precision.csv'), index=False)
df_split_info.to_csv(os.path.join(output_dir, 'split_info.csv'), index=False)
df_best_thresholds.to_csv(os.path.join(output_dir, 'best_thresholds.csv'), index=False)
df_val_thresholds.to_csv(os.path.join(output_dir, 'val_metrics_by_threshold.csv'), index=False)
df_test_thresholds.to_csv(os.path.join(output_dir, 'test_metrics_by_threshold.csv'), index=False)


# ============================================================
# RESUMEN FINAL
# ============================================================

metric_cols = [
    'auc',
    'auprc',
    'baseline_auprc',
    'f1',
    'precision',
    'recall',
    'specificity',
    'balanced_accuracy',
    'mcc',
    'log_loss',
    'brier',
    'tp',
    'tn',
    'fp',
    'fn',
]

summary_f1 = (
    df_test_f1[metric_cols]
    .agg(['mean', 'std'])
    .T
    .reset_index()
    .rename(columns={'index': 'metric'})
)

summary_precision = (
    df_test_precision[metric_cols]
    .agg(['mean', 'std'])
    .T
    .reset_index()
    .rename(columns={'index': 'metric'})
)

summary_f1.to_csv(os.path.join(output_dir, 'summary_test_best_f1.csv'), index=False)
summary_precision.to_csv(os.path.join(output_dir, 'summary_test_best_precision.csv'), index=False)

print('\n' + '=' * 70)
print('RESULTADOS XGBOOST - LODO PARCIAL CV=10')
print('=' * 70)

print('\n[TEST - threshold elegido por mejor F1 en validacion]')
print(summary_f1)

print('\n[TEST - threshold conservador por mejor precision en validacion]')
print(summary_precision)

print('\nArchivos guardados en:', output_dir)



## XGBOOST: STRATIFIED K-FOLD CV=10

In [ ]:
import os
import json
import shutil
import joblib
import numpy as np
import pandas as pd
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from torch.utils.tensorboard import SummaryWriter
from PIL import Image

if not hasattr(Image, "Resampling"):
    class Resampling:
        LANCZOS = Image.ANTIALIAS

    Image.Resampling = Resampling


# ============================================================
# CONFIGURACION
# ============================================================

SAVE_MODELS = False

output_dir = 'results/xgboost_stratified_cv'
tb_dir = os.path.join(output_dir, 'tensorboard')

if os.path.exists(output_dir):
    shutil.rmtree(output_dir)

os.makedirs(output_dir, exist_ok=True)
os.makedirs(os.path.join(output_dir, 'predictions'), exist_ok=True)
os.makedirs(os.path.join(output_dir, 'figures'), exist_ok=True)
os.makedirs(tb_dir, exist_ok=True)

if SAVE_MODELS:
    os.makedirs(os.path.join(output_dir, 'models'), exist_ok=True)


xgb_config = {
    'n_estimators': 600,
    'max_depth': 8,
    'learning_rate': 0.03,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'objective': 'binary:logistic',
    'eval_metric': 'aucpr',
    'tree_method': 'hist',
    'random_state': 42,
    'n_jobs': 4,
}


experiment_config = {
    'model': 'XGBClassifier',
    'split_type': 'partial_lodo_optimized',
    'n_splits': 10,
    'n_genes': 500,
    'random_state': 42,
    'threshold_grid_min': 0.05,
    'threshold_grid_max': 0.95,
    'threshold_grid_step': 0.01,
    'xgb_config': xgb_config,
    'save_models': SAVE_MODELS,
    'stratified_definition': (
        'Random Stratified CV: each fold is a random sample stratified by the target variable, '
    ),
    'external_lodo_splits_path': 'results/shared_splits/stratified_kfold_indices.json',
}
save_experiment_config(experiment_config, output_dir)


# ============================================================
# DATASET BASE
# ============================================================

base_df = X_final_df.copy()
base_df['drug_row_id'] = base_df['drug_row_id'].astype(str)
base_df['drug_col_id'] = base_df['drug_col_id'].astype(str)
base_df['synergy_loewe_bin'] = base_df['synergy_loewe_bin'].replace(-1, 0).astype(int)

thresholds = np.round(np.arange(0.05, 0.951, 0.01), 2)

# ============================================================
# CREAR/CARGAR SPLITS STRATIFIED K-FOLD COMPARTIDOS
# ============================================================

shared_splits_dir = 'results/shared_splits'
os.makedirs(shared_splits_dir, exist_ok=True)

skf_splits_path = os.path.join(shared_splits_dir, 'stratified_kfold_indices.json')

if os.path.exists(skf_splits_path):
    with open(skf_splits_path, 'r') as f:
        split_indices = json.load(f)

    print('Splits StratifiedKFold cargados desde:', skf_splits_path)

else:
    skf = StratifiedKFold(
        n_splits=experiment_config['n_splits'],
        shuffle=True,
        random_state=experiment_config['random_state']
    )

    split_indices = []

    for fold_idx, (train_idx, test_idx) in enumerate(
        skf.split(base_df, base_df['synergy_loewe_bin']),
        start=1
    ):
        split_indices.append({
            'fold': fold_idx,
            'train_idx': train_idx.tolist(),
            'test_idx': test_idx.tolist()
        })

    with open(skf_splits_path, 'w') as f:
        json.dump(split_indices, f, indent=4)

    print('Splits StratifiedKFold creados y guardados en:', skf_splits_path)



# ============================================================
# BUCLE CV
# ============================================================

cv_val_results_f1 = []
cv_val_results_precision = []
cv_test_results_f1 = []
cv_test_results_precision = []
cv_split_info = []
cv_val_threshold_metrics = []
cv_test_threshold_metrics = []
cv_best_thresholds = []

for split in split_indices:

    fold_idx = split['fold']
    train_idx = np.array(split['train_idx'])
    test_idx = np.array(split['test_idx'])
    writer = SummaryWriter(log_dir=os.path.join(tb_dir, f'Fold_{fold_idx}'))

    print(f'\nPROCESANDO FOLD {fold_idx}/{experiment_config["n_splits"]} - XGBOOST STRATIFIED CV')

    df_train_raw = base_df.iloc[train_idx].reset_index(drop=True)
    df_test_raw = base_df.iloc[test_idx].reset_index(drop=True)

    if len(df_train_raw) == 0 or len(df_test_raw) == 0:
        print(f'Fold {fold_idx}: sin datos suficientes, se omite.')
        writer.close()
        continue

    train_drugs = set(pd.concat([
        df_train_raw['drug_row_id'],
        df_train_raw['drug_col_id']
    ]).astype(str))

    test_drugs = set(pd.concat([
        df_test_raw['drug_row_id'],
        df_test_raw['drug_col_id']
    ]).astype(str))

    drug_overlap_train_test = len(train_drugs & test_drugs)
    drug_overlap_train_test_pct = (
        100 * drug_overlap_train_test / len(test_drugs)
        if len(test_drugs) > 0 else 0.0
    )

    df_train, df_val, df_test, top_genes = prepare_fold_data_no_leakage(
        df_train_raw,
        df_test_raw,
        data_expr,
        n_genes=experiment_config['n_genes'],
        random_state=experiment_config['random_state']
    )

    X_train, y_train, X_val, y_val, X_test, y_test, feature_cols = build_tabular_matrices(
        df_train,
        df_val,
        df_test
    )

    if len(np.unique(y_train)) < 2:
        print(f'Fold {fold_idx}: train con una sola clase, se omite.')
        writer.close()
        continue

    n_pos = np.sum(y_train)
    n_neg = len(y_train) - n_pos

    xgb_config_fold = xgb_config.copy()
    xgb_config_fold['scale_pos_weight'] = n_neg / n_pos

    xgb = XGBClassifier(**xgb_config_fold)

    print(f'Entrenando XGBoost fold {fold_idx}...')
    xgb.fit(X_train, y_train)
    print(f'Entrenamiento terminado fold {fold_idx}.')

    y_prob_val = xgb.predict_proba(X_val)[:, 1]
    y_prob_test = xgb.predict_proba(X_test)[:, 1]

    df_val_thr, best_thr_f1, best_thr_precision = find_best_thresholds(
        y_val,
        y_prob_val,
        thresholds
    )

    df_val_thr.insert(0, 'fold', fold_idx)
    df_val_thr.insert(1, 'split', 'val')
    df_val_thr.insert(2, 'threshold_type', 'grid')
    cv_val_threshold_metrics.append(df_val_thr)

    test_thr_rows = []
    for thr in thresholds:
        m_test_thr = compute_metrics_at_threshold(y_test, y_prob_test, thr)
        m_test_thr.update({
            'fold': fold_idx,
            'split': 'test',
            'threshold_type': 'grid',
        })
        test_thr_rows.append(m_test_thr)

    df_test_thr = pd.DataFrame(test_thr_rows)
    cv_test_threshold_metrics.append(df_test_thr)

    val_metrics_f1 = compute_metrics_at_threshold(y_val, y_prob_val, best_thr_f1)
    val_metrics_precision = compute_metrics_at_threshold(y_val, y_prob_val, best_thr_precision)

    test_metrics_f1 = compute_metrics_at_threshold(y_test, y_prob_test, best_thr_f1)
    test_metrics_precision = compute_metrics_at_threshold(y_test, y_prob_test, best_thr_precision)

    val_metrics_f1.update({'fold': fold_idx, 'split': 'val', 'threshold_type': 'best_f1'})
    val_metrics_precision.update({'fold': fold_idx, 'split': 'val', 'threshold_type': 'best_precision'})
    test_metrics_f1.update({'fold': fold_idx, 'split': 'test', 'threshold_type': 'best_f1'})
    test_metrics_precision.update({'fold': fold_idx, 'split': 'test', 'threshold_type': 'best_precision'})

    cv_val_results_f1.append(val_metrics_f1)
    cv_val_results_precision.append(val_metrics_precision)
    cv_test_results_f1.append(test_metrics_f1)
    cv_test_results_precision.append(test_metrics_precision)

    cv_best_thresholds.append({
        'fold': fold_idx,
        'best_threshold_f1': best_thr_f1,
        'best_threshold_precision': best_thr_precision,
        'val_f1_at_best_f1': val_metrics_f1['f1'],
        'val_precision_at_best_f1': val_metrics_f1['precision'],
        'val_recall_at_best_f1': val_metrics_f1['recall'],
        'val_precision_at_best_precision': val_metrics_precision['precision'],
        'val_recall_at_best_precision': val_metrics_precision['recall'],
        'val_f1_at_best_precision': val_metrics_precision['f1'],
    })

    log_curves_and_metrics(y_val, y_prob_val, val_metrics_f1, fold_idx, writer, tag='Val_BestF1')
    log_curves_and_metrics(y_val, y_prob_val, val_metrics_precision, fold_idx, writer, tag='Val_BestPrecision')
    log_curves_and_metrics(y_test, y_prob_test, test_metrics_f1, fold_idx, writer, tag='Test_BestF1')
    log_curves_and_metrics(y_test, y_prob_test, test_metrics_precision, fold_idx, writer, tag='Test_BestPrecision')

    predictions = df_test[
        [
            'drug_row_id',
            'drug_col_id',
            'cell_line_name',
            'study_name',
            'tissue',
            'synergy_loewe',
            'synergy_loewe_bin'
        ]
    ].copy()

    predictions['y_true'] = y_test
    predictions['y_prob'] = y_prob_test
    predictions['y_pred_best_f1'] = (y_prob_test >= best_thr_f1).astype(int)
    predictions['y_pred_best_precision'] = (y_prob_test >= best_thr_precision).astype(int)
    predictions['threshold_best_f1'] = best_thr_f1
    predictions['threshold_best_precision'] = best_thr_precision

    predictions.to_csv(
        os.path.join(output_dir, 'predictions', f'predictions_fold_{fold_idx:02d}.csv'),
        index=False
    )

    if SAVE_MODELS:
        joblib.dump(
            xgb,
            os.path.join(output_dir, 'models', f'xgboost_fold_{fold_idx:02d}.joblib')
        )

    n_train = len(y_train)
    n_val = len(y_val)
    n_test = len(y_test)

    split_info = {
        'fold': fold_idx,
        'n_train': n_train,
        'n_val': n_val,
        'n_test': n_test,
        'train_pos': int(np.sum(y_train)),
        'val_pos': int(np.sum(y_val)),
        'test_pos': int(np.sum(y_test)),
        'train_pos_pct': 100 * np.mean(y_train),
        'val_pos_pct': 100 * np.mean(y_val),
        'test_pos_pct': 100 * np.mean(y_test),
        'baseline_auprc_test': float(np.mean(y_test)),
        'n_train_drugs': len(train_drugs),
        'n_test_drugs': len(test_drugs),
        'drug_overlap_train_test': drug_overlap_train_test,
        'drug_overlap_train_test_pct': drug_overlap_train_test_pct,
        'best_threshold_f1': best_thr_f1,
        'best_threshold_precision': best_thr_precision,
        'n_features': len(feature_cols),
        'n_top_genes': len(top_genes),
        'split_indices_path': skf_splits_path
    }

    cv_split_info.append(split_info)

    writer.add_text('Split/Info', json.dumps(split_info, indent=2))
    writer.add_text('Genes/TopGenes_First100', ', '.join(top_genes[:100]))

    importances = pd.Series(xgb.feature_importances_, index=feature_cols)
    top_imp = importances.sort_values(ascending=False).head(30)
    writer.add_text('Model/Top30_Feature_Importance', top_imp.to_string())

    print(
        f"Fold {fold_idx} | "
        f"Train={n_train} ({100*np.mean(y_train):.2f}% pos), "
        f"Val={n_val} ({100*np.mean(y_val):.2f}% pos), "
        f"Test={n_test} ({100*np.mean(y_test):.2f}% pos) | "
        f"DrugOverlap={drug_overlap_train_test_pct:.2f}% | "
        f"ThrF1={best_thr_f1:.2f}, ThrPrec={best_thr_precision:.2f} | "
        f"AUPRC={test_metrics_f1['auprc']:.4f} "
        f"(base={test_metrics_f1['baseline_auprc']:.4f}) | "
        f"F1={test_metrics_f1['f1']:.4f} | "
        f"Prec={test_metrics_f1['precision']:.4f} | "
        f"Rec={test_metrics_f1['recall']:.4f} | "
        f"MCC={test_metrics_f1['mcc']:.4f}"
    )

    writer.close()


# ============================================================
# GUARDADO DE RESULTADOS
# ============================================================

df_val_f1 = pd.DataFrame(cv_val_results_f1)
df_val_precision = pd.DataFrame(cv_val_results_precision)
df_test_f1 = pd.DataFrame(cv_test_results_f1)
df_test_precision = pd.DataFrame(cv_test_results_precision)
df_split_info = pd.DataFrame(cv_split_info)
df_best_thresholds = pd.DataFrame(cv_best_thresholds)

df_val_thresholds = pd.concat(cv_val_threshold_metrics, ignore_index=True)
df_test_thresholds = pd.concat(cv_test_threshold_metrics, ignore_index=True)

df_val_f1.to_csv(os.path.join(output_dir, 'val_metrics_best_f1.csv'), index=False)
df_val_precision.to_csv(os.path.join(output_dir, 'val_metrics_best_precision.csv'), index=False)
df_test_f1.to_csv(os.path.join(output_dir, 'test_metrics_best_f1.csv'), index=False)
df_test_precision.to_csv(os.path.join(output_dir, 'test_metrics_best_precision.csv'), index=False)
df_split_info.to_csv(os.path.join(output_dir, 'split_info.csv'), index=False)
df_best_thresholds.to_csv(os.path.join(output_dir, 'best_thresholds.csv'), index=False)
df_val_thresholds.to_csv(os.path.join(output_dir, 'val_metrics_by_threshold.csv'), index=False)
df_test_thresholds.to_csv(os.path.join(output_dir, 'test_metrics_by_threshold.csv'), index=False)


# ============================================================
# RESUMEN FINAL
# ============================================================

metric_cols = [
    'auc',
    'auprc',
    'baseline_auprc',
    'f1',
    'precision',
    'recall',
    'specificity',
    'balanced_accuracy',
    'mcc',
    'log_loss',
    'brier',
    'tp',
    'tn',
    'fp',
    'fn',
]

summary_f1 = (
    df_test_f1[metric_cols]
    .agg(['mean', 'std'])
    .T
    .reset_index()
    .rename(columns={'index': 'metric'})
)

summary_precision = (
    df_test_precision[metric_cols]
    .agg(['mean', 'std'])
    .T
    .reset_index()
    .rename(columns={'index': 'metric'})
)

summary_f1.to_csv(os.path.join(output_dir, 'summary_test_best_f1.csv'), index=False)
summary_precision.to_csv(os.path.join(output_dir, 'summary_test_best_precision.csv'), index=False)

print('\n' + '=' * 70)
print('RESULTADOS XGBOOST - STRATIFIED RANDOM CV=10')
print('=' * 70)

print('\n[TEST - threshold elegido por mejor F1 en validacion]')
print(summary_f1)

print('\n[TEST - threshold conservador por mejor precision en validacion]')
print(summary_precision)

print('\nArchivos guardados en:', output_dir)


## MLP: Leave-Drug-Out CV=10 (Siamese Multimodal Fusion MLP)

In [ ]:
# import random
# import numpy as np
# import torch

# def set_global_seed(seed=42):
#     random.seed(seed)
#     np.random.seed(seed)
#     torch.manual_seed(seed)
#     torch.cuda.manual_seed_all(seed)

#     torch.backends.cudnn.deterministic = True
#     torch.backends.cudnn.benchmark = False

#     try:
#         torch.use_deterministic_algorithms(True)
#     except Exception:
#         pass

In [ ]:
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import io
import random
from torch.utils.data import Dataset, DataLoader
from torch.utils.tensorboard import SummaryWriter
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (roc_auc_score, f1_score, precision_score, 
                             recall_score, average_precision_score, confusion_matrix)

In [ ]:
class SynergyDataset(Dataset):
    def __init__(self, df, gene_cols):
        d1_cols = [f'drug1_bit_{i}' for i in range(1024)]
        d2_cols = [f'drug2_bit_{i}' for i in range(1024)]
        self.d1 = torch.tensor(df[d1_cols].values, dtype=torch.float32)
        self.d2 = torch.tensor(df[d2_cols].values, dtype=torch.float32)
        self.genes = torch.tensor(df[gene_cols].values, dtype=torch.float32)
        self.y = torch.tensor(df['synergy_loewe_bin'].values, dtype=torch.float32).view(-1, 1)

    def __len__(self): return len(self.y)
    def __getitem__(self, idx): return self.d1[idx], self.d2[idx], self.genes[idx], self.y[idx]

class TripleBranchMLP(nn.Module):
    def __init__(self, drug_dim=1024, gene_dim=500):
        super(TripleBranchMLP, self).__init__()
        # Rama Siamesa para Drogas
        self.drug_branch = nn.Sequential(
            nn.Linear(drug_dim, 512),
            #nn.BatchNorm1d(512),
            nn.ReLU(), 
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.ReLU()
        )
        # Rama para Genes
        self.gene_branch = nn.Sequential(
            nn.Linear(gene_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.3), # el dropout anterior con 0.3
            nn.Linear(512, 256), 
            nn.ReLU()
        )
        # Fusión
        self.classifier = nn.Sequential(
            nn.Linear(256*3, 256),
            nn.ReLU(), 
            nn.Dropout(0.3),
            nn.Linear(256, 64), 
            nn.ReLU(), 
            nn.Linear(64, 1)
        )

    def forward(self, d1, d2, g):
        return self.classifier(torch.cat((self.drug_branch(d1), self.drug_branch(d2), self.gene_branch(g)), dim=1))

In [ ]:
import os
import json
import copy
import shutil
import numpy as np
import pandas as pd
import torch

from torch import nn
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
from PIL import Image

if not hasattr(Image, "Resampling"):
    class Resampling:
        LANCZOS = Image.ANTIALIAS

    Image.Resampling = Resampling


# ============================================================
# CONFIGURACION
# ============================================================

SAVE_MODELS = False

output_dir = 'results/triple_branch_mlp_lodo'
tb_dir = os.path.join(output_dir, 'tensorboard')

if os.path.exists(output_dir):
    shutil.rmtree(output_dir)

os.makedirs(output_dir, exist_ok=True)
os.makedirs(os.path.join(output_dir, 'predictions'), exist_ok=True)
os.makedirs(os.path.join(output_dir, 'figures'), exist_ok=True)
os.makedirs(tb_dir, exist_ok=True)

if SAVE_MODELS:
    os.makedirs(os.path.join(output_dir, 'models'), exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

mlp_config = {
    'lr': 1e-3,
    'batch_size': 256,
    'epochs': 80,
    'patience': 12,
    'weight_decay': 1e-4,
    'drug_dim': 1024,
    'n_genes': 500,
    'random_state': 42,
    'use_pos_weight': True,
}

experiment_config = {
    'model': 'TripleBranchMLP',
    'split_type': 'partial_lodo_optimized',
    'n_splits': 10,
    'n_genes': mlp_config['n_genes'],
    'random_state': mlp_config['random_state'],
    'threshold_grid_min': 0.05,
    'threshold_grid_max': 0.95,
    'threshold_grid_step': 0.01,
    'mlp_config': mlp_config,
    'save_models': SAVE_MODELS,
    'device': str(device),
    'external_lodo_splits_path': 'results/shared_splits/lodo_fold_sets.json',
    'lodo_definition': (
        'Partial Leave-Drug-Out: each test sample contains at least one held-out drug. '
        'The second drug may appear in train.'
    )
}

save_experiment_config(experiment_config, output_dir)




In [ ]:

# ============================================================
# FUNCIONES TORCH
# ============================================================

def run_torch_epoch(model, loader, criterion, optimizer=None, device='cpu'):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss = 0.0
    y_true_all = []
    y_prob_all = []

    context = torch.enable_grad() if is_train else torch.no_grad()

    with context:
        for d1, d2, g, y in loader:
            d1 = d1.to(device)
            d2 = d2.to(device)
            g = g.to(device)
            y = y.to(device)

            if is_train:
                optimizer.zero_grad()

            logits = model(d1, d2, g)
            loss = criterion(logits, y)

            if is_train:
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * y.size(0)

            probs = torch.sigmoid(logits).detach().cpu().numpy().ravel()
            y_true = y.detach().cpu().numpy().ravel()

            y_prob_all.extend(probs)
            y_true_all.extend(y_true)

    avg_loss = total_loss / len(loader.dataset)

    return avg_loss, np.asarray(y_true_all), np.asarray(y_prob_all)


def train_torch_model_fold(model, train_loader, val_loader, y_train, config, device, writer, fold_idx):
    n_pos = float(np.sum(y_train))
    n_neg = float(len(y_train) - n_pos)

    if config.get('use_pos_weight', True) and n_pos > 0:
        pos_weight = torch.tensor([n_neg / n_pos], dtype=torch.float32).to(device)
        criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    else:
        criterion = nn.BCEWithLogitsLoss()

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config['lr'],
        weight_decay=config['weight_decay']
    )

    best_val_loss = np.inf
    best_model_state = copy.deepcopy(model.state_dict())
    best_epoch = 0
    epochs_no_improve = 0

    history = []

    for epoch in range(1, config['epochs'] + 1):
        train_loss, y_true_train, y_prob_train = run_torch_epoch(
            model,
            train_loader,
            criterion,
            optimizer=optimizer,
            device=device
        )

        val_loss, y_true_val, y_prob_val = run_torch_epoch(
            model,
            val_loader,
            criterion,
            optimizer=None,
            device=device
        )

        val_aucpr = compute_metrics_at_threshold(
            y_true_val,
            y_prob_val,
            threshold=0.5
        )['auprc']

        writer.add_scalar('Loss/Train', train_loss, epoch)
        writer.add_scalar('Loss/Val', val_loss, epoch)
        writer.add_scalar('Metrics/Val_AUPRC_thr05', val_aucpr, epoch)

        history.append({
            'fold': fold_idx,
            'epoch': epoch,
            'train_loss': train_loss,
            'val_loss': val_loss,
            'val_auprc_thr05': val_aucpr,
        })

        print(
            f"Fold {fold_idx} | Epoch {epoch:03d}/{config['epochs']} | "
            f"TrainLoss={train_loss:.4f} | ValLoss={val_loss:.4f} | "
            f"ValAUPRC={val_aucpr:.4f}",
            end='\r'
        )

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model_state = copy.deepcopy(model.state_dict())
            best_epoch = epoch
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1

            if epochs_no_improve >= config['patience']:
                break

    print()

    model.load_state_dict(best_model_state)

    return model, pd.DataFrame(history), best_epoch, best_val_loss


def predict_torch_model(model, loader, device):
    model.eval()

    y_true_all = []
    y_prob_all = []

    with torch.no_grad():
        for d1, d2, g, y in loader:
            d1 = d1.to(device)
            d2 = d2.to(device)
            g = g.to(device)

            logits = model(d1, d2, g)
            probs = torch.sigmoid(logits).cpu().numpy().ravel()

            y_prob_all.extend(probs)
            y_true_all.extend(y.numpy().ravel())

    return np.asarray(y_true_all), np.asarray(y_prob_all)

In [ ]:
# ============================================================
# DATASET BASE
# ============================================================

base_df = X_final_df.copy()
base_df['drug_row_id'] = base_df['drug_row_id'].astype(str)
base_df['drug_col_id'] = base_df['drug_col_id'].astype(str)
base_df['synergy_loewe_bin'] = base_df['synergy_loewe_bin'].replace(-1, 0).astype(int)

thresholds = np.round(np.arange(0.05, 0.951, 0.01), 2)


# ============================================================
# CARGAR SPLITS LODO COMPARTIDOS
# ============================================================

shared_splits_path = experiment_config['external_lodo_splits_path']

with open(shared_splits_path, 'r') as f:
    fold_sets_dict = json.load(f)

fold_sets = [
    set(fold_sets_dict[str(i + 1)])
    for i in range(len(fold_sets_dict))
]

assert len(fold_sets) == experiment_config['n_splits']

partition_rows = []

for fold_idx, fold_set in enumerate(fold_sets, start=1):
    mask_test = (
        base_df['drug_row_id'].isin(fold_set)
        | base_df['drug_col_id'].isin(fold_set)
    )

    fold_df = base_df[mask_test]

    n_examples = len(fold_df)
    pos_rate = fold_df['synergy_loewe_bin'].mean() if n_examples > 0 else np.nan

    partition_rows.append({
        'fold': fold_idx,
        'n_examples_induced_test': n_examples,
        'positive_pct_induced_test': pos_rate * 100,
        'baseline_auprc_induced_test': pos_rate,
        'n_heldout_drugs': len(fold_set)
    })

fold_partition_summary = pd.DataFrame(partition_rows)
fold_partition_summary.to_csv(
    os.path.join(output_dir, 'lodo_partition_summary.csv'),
    index=False
)

print('\nSplits LODO compartidos cargados desde:', shared_splits_path)
print(fold_partition_summary)


# ============================================================
# SOLAPAMIENTO TEST
# ============================================================

test_overlap_info, overlap_df, overlap_counts = summarize_test_overlap(
    base_df,
    fold_sets,
    output_path=output_dir
)

print('\nSolapamiento entre conjuntos test:')
print(test_overlap_info)
print('\nDistribucion de apariciones por muestra en test:')
print(overlap_counts)


In [ ]:
# ============================================================
# BUCLE CV
# ============================================================

cv_val_results_f1 = []
cv_val_results_precision = []
cv_test_results_f1 = []
cv_test_results_precision = []
cv_split_info = []
cv_val_threshold_metrics = []
cv_test_threshold_metrics = []
cv_best_thresholds = []
cv_histories = []



for fold_idx, test_set in enumerate(fold_sets, start=1):
    set_global_seed(experiment_config['random_state'] + fold_idx)    
    writer = SummaryWriter(log_dir=os.path.join(tb_dir, f'Fold_{fold_idx}'))

    print(f'\nPROCESANDO FOLD {fold_idx}/{experiment_config["n_splits"]} - TRIPLE BRANCH MLP LODO')

    mask_test = (
        base_df['drug_row_id'].isin(test_set)
        | base_df['drug_col_id'].isin(test_set)
    )

    df_test_raw = base_df[mask_test].reset_index(drop=True)
    df_train_raw = base_df[~mask_test].reset_index(drop=True)

    if len(df_train_raw) == 0 or len(df_test_raw) == 0:
        print(f'Fold {fold_idx}: sin datos suficientes, se omite.')
        writer.close()
        continue

    assert (
        df_test_raw['drug_row_id'].isin(test_set)
        | df_test_raw['drug_col_id'].isin(test_set)
    ).all(), f'Fold {fold_idx}: hay muestras test sin droga held-out.'

    train_drugs = set(pd.concat([
        df_train_raw['drug_row_id'],
        df_train_raw['drug_col_id']
    ]).astype(str))

    heldout_overlap_train = len(train_drugs & test_set)
    assert heldout_overlap_train == 0, f'Fold {fold_idx}: drogas held-out aparecen en train.'

    df_train, df_val, df_test, top_genes = prepare_fold_data_no_leakage(
        df_train_raw,
        df_test_raw,
        data_expr,
        n_genes=experiment_config['n_genes'],
        random_state=experiment_config['random_state']
    )

    y_train_np = df_train['synergy_loewe_bin'].astype(int).to_numpy()

    if len(np.unique(y_train_np)) < 2:
        print(f'Fold {fold_idx}: train con una sola clase, se omite.')
        writer.close()
        continue

    train_loader = DataLoader(
        SynergyDataset(df_train, top_genes),
        batch_size=mlp_config['batch_size'],
        shuffle=True,

    )

    val_loader = DataLoader(
        SynergyDataset(df_val, top_genes),
        batch_size=mlp_config['batch_size'],
        shuffle=False
    )

    test_loader = DataLoader(
        SynergyDataset(df_test, top_genes),
        batch_size=mlp_config['batch_size'],
        shuffle=False
    )

    model = TripleBranchMLP(
        drug_dim=mlp_config['drug_dim'],
        gene_dim=experiment_config['n_genes']
    ).to(device)

    model, history_df, best_epoch, best_val_loss = train_torch_model_fold(
        model,
        train_loader,
        val_loader,
        y_train_np,
        mlp_config,
        device,
        writer,
        fold_idx
    )

    history_df.to_csv(
        os.path.join(output_dir, f'training_history_fold_{fold_idx:02d}.csv'),
        index=False
    )
    cv_histories.append(history_df)

    y_val, y_prob_val = predict_torch_model(model, val_loader, device)
    y_test, y_prob_test = predict_torch_model(model, test_loader, device)

    df_val_thr, best_thr_f1, best_thr_precision = find_best_thresholds(
        y_val,
        y_prob_val,
        thresholds
    )

    df_val_thr.insert(0, 'fold', fold_idx)
    df_val_thr.insert(1, 'split', 'val')
    df_val_thr.insert(2, 'threshold_type', 'grid')
    cv_val_threshold_metrics.append(df_val_thr)

    test_thr_rows = []
    for thr in thresholds:
        m_test_thr = compute_metrics_at_threshold(y_test, y_prob_test, thr)
        m_test_thr.update({
            'fold': fold_idx,
            'split': 'test',
            'threshold_type': 'grid',
        })
        test_thr_rows.append(m_test_thr)

    df_test_thr = pd.DataFrame(test_thr_rows)
    cv_test_threshold_metrics.append(df_test_thr)

    val_metrics_f1 = compute_metrics_at_threshold(y_val, y_prob_val, best_thr_f1)
    val_metrics_precision = compute_metrics_at_threshold(y_val, y_prob_val, best_thr_precision)

    test_metrics_f1 = compute_metrics_at_threshold(y_test, y_prob_test, best_thr_f1)
    test_metrics_precision = compute_metrics_at_threshold(y_test, y_prob_test, best_thr_precision)

    val_metrics_f1.update({'fold': fold_idx, 'split': 'val', 'threshold_type': 'best_f1'})
    val_metrics_precision.update({'fold': fold_idx, 'split': 'val', 'threshold_type': 'best_precision'})
    test_metrics_f1.update({'fold': fold_idx, 'split': 'test', 'threshold_type': 'best_f1'})
    test_metrics_precision.update({'fold': fold_idx, 'split': 'test', 'threshold_type': 'best_precision'})

    cv_val_results_f1.append(val_metrics_f1)
    cv_val_results_precision.append(val_metrics_precision)
    cv_test_results_f1.append(test_metrics_f1)
    cv_test_results_precision.append(test_metrics_precision)

    cv_best_thresholds.append({
        'fold': fold_idx,
        'best_threshold_f1': best_thr_f1,
        'best_threshold_precision': best_thr_precision,
        'val_f1_at_best_f1': val_metrics_f1['f1'],
        'val_precision_at_best_f1': val_metrics_f1['precision'],
        'val_recall_at_best_f1': val_metrics_f1['recall'],
        'val_precision_at_best_precision': val_metrics_precision['precision'],
        'val_recall_at_best_precision': val_metrics_precision['recall'],
        'val_f1_at_best_precision': val_metrics_precision['f1'],
        'best_epoch': best_epoch,
        'best_val_loss': best_val_loss,
    })

    log_curves_and_metrics(y_val, y_prob_val, val_metrics_f1, fold_idx, writer, tag='Val_BestF1')
    log_curves_and_metrics(y_val, y_prob_val, val_metrics_precision, fold_idx, writer, tag='Val_BestPrecision')
    log_curves_and_metrics(y_test, y_prob_test, test_metrics_f1, fold_idx, writer, tag='Test_BestF1')
    log_curves_and_metrics(y_test, y_prob_test, test_metrics_precision, fold_idx, writer, tag='Test_BestPrecision')

    predictions = df_test[
        [
            'drug_row_id',
            'drug_col_id',
            'cell_line_name',
            'study_name',
            'tissue',
            'synergy_loewe',
            'synergy_loewe_bin'
        ]
    ].copy()

    predictions['y_true'] = y_test
    predictions['y_prob'] = y_prob_test
    predictions['y_pred_best_f1'] = (y_prob_test >= best_thr_f1).astype(int)
    predictions['y_pred_best_precision'] = (y_prob_test >= best_thr_precision).astype(int)
    predictions['threshold_best_f1'] = best_thr_f1
    predictions['threshold_best_precision'] = best_thr_precision

    predictions.to_csv(
        os.path.join(output_dir, 'predictions', f'predictions_fold_{fold_idx:02d}.csv'),
        index=False
    )

    if SAVE_MODELS:
        torch.save(
            model.state_dict(),
            os.path.join(output_dir, 'models', f'triple_branch_mlp_fold_{fold_idx:02d}.pt')
        )

    n_train = len(y_train_np)
    n_val = len(y_val)
    n_test = len(y_test)

    split_info = {
        'fold': fold_idx,
        'n_train': n_train,
        'n_val': n_val,
        'n_test': n_test,
        'train_pos': int(np.sum(y_train_np)),
        'val_pos': int(np.sum(y_val)),
        'test_pos': int(np.sum(y_test)),
        'train_pos_pct': 100 * np.mean(y_train_np),
        'val_pos_pct': 100 * np.mean(y_val),
        'test_pos_pct': 100 * np.mean(y_test),
        'baseline_auprc_test': float(np.mean(y_test)),
        'n_heldout_drugs': len(test_set),
        'heldout_drugs': ';'.join(sorted(test_set)),
        'heldout_overlap_train': heldout_overlap_train,
        'best_threshold_f1': best_thr_f1,
        'best_threshold_precision': best_thr_precision,
        'best_epoch': best_epoch,
        'best_val_loss': best_val_loss,
        'n_top_genes': len(top_genes),
    }

    cv_split_info.append(split_info)

    writer.add_text('Split/Info', json.dumps(split_info, indent=2))
    writer.add_text('Genes/TopGenes_First100', ', '.join(top_genes[:100]))

    print(
        f"Fold {fold_idx} | "
        f"Train={n_train} ({100*np.mean(y_train_np):.2f}% pos), "
        f"Val={n_val} ({100*np.mean(y_val):.2f}% pos), "
        f"Test={n_test} ({100*np.mean(y_test):.2f}% pos) | "
        f"BestEpoch={best_epoch} | "
        f"ThrF1={best_thr_f1:.2f}, ThrPrec={best_thr_precision:.2f} | "
        f"AUPRC={test_metrics_f1['auprc']:.4f} "
        f"(base={test_metrics_f1['baseline_auprc']:.4f}) | "
        f"F1={test_metrics_f1['f1']:.4f} | "
        f"Prec={test_metrics_f1['precision']:.4f} | "
        f"Rec={test_metrics_f1['recall']:.4f} | "
        f"MCC={test_metrics_f1['mcc']:.4f}"
    )

    writer.close()
    torch.cuda.empty_cache()


In [ ]:
# ============================================================
# GUARDADO DE RESULTADOS
# ============================================================

df_val_f1 = pd.DataFrame(cv_val_results_f1)
df_val_precision = pd.DataFrame(cv_val_results_precision)
df_test_f1 = pd.DataFrame(cv_test_results_f1)
df_test_precision = pd.DataFrame(cv_test_results_precision)
df_split_info = pd.DataFrame(cv_split_info)
df_best_thresholds = pd.DataFrame(cv_best_thresholds)

df_val_thresholds = pd.concat(cv_val_threshold_metrics, ignore_index=True)
df_test_thresholds = pd.concat(cv_test_threshold_metrics, ignore_index=True)

df_val_f1.to_csv(os.path.join(output_dir, 'val_metrics_best_f1.csv'), index=False)
df_val_precision.to_csv(os.path.join(output_dir, 'val_metrics_best_precision.csv'), index=False)
df_test_f1.to_csv(os.path.join(output_dir, 'test_metrics_best_f1.csv'), index=False)
df_test_precision.to_csv(os.path.join(output_dir, 'test_metrics_best_precision.csv'), index=False)
df_split_info.to_csv(os.path.join(output_dir, 'split_info.csv'), index=False)
df_best_thresholds.to_csv(os.path.join(output_dir, 'best_thresholds.csv'), index=False)
df_val_thresholds.to_csv(os.path.join(output_dir, 'val_metrics_by_threshold.csv'), index=False)
df_test_thresholds.to_csv(os.path.join(output_dir, 'test_metrics_by_threshold.csv'), index=False)

if len(cv_histories) > 0:
    df_all_history = pd.concat(cv_histories, ignore_index=True)
    df_all_history.to_csv(os.path.join(output_dir, 'training_history_all_folds.csv'), index=False)


# ============================================================
# RESUMEN FINAL
# ============================================================

metric_cols = [
    'auc',
    'auprc',
    'baseline_auprc',
    'f1',
    'precision',
    'recall',
    'specificity',
    'balanced_accuracy',
    'mcc',
    'log_loss',
    'brier',
    'tp',
    'tn',
    'fp',
    'fn',
]

summary_f1 = (
    df_test_f1[metric_cols]
    .agg(['mean', 'std'])
    .T
    .reset_index()
    .rename(columns={'index': 'metric'})
)

summary_precision = (
    df_test_precision[metric_cols]
    .agg(['mean', 'std'])
    .T
    .reset_index()
    .rename(columns={'index': 'metric'})
)

summary_f1.to_csv(os.path.join(output_dir, 'summary_test_best_f1.csv'), index=False)
summary_precision.to_csv(os.path.join(output_dir, 'summary_test_best_precision.csv'), index=False)

print('\n' + '=' * 70)
print('RESULTADOS TRIPLE BRANCH MLP - LODO PARCIAL CV=10')
print('=' * 70)

print('\n[TEST - threshold elegido por mejor F1 en validacion]')
print(summary_f1)

print('\n[TEST - threshold conservador por mejor precision en validacion]')
print(summary_precision)

print('\nArchivos guardados en:', output_dir)


## MLP TRIPLE: STRATIFIED K-FOLD CV=10 (Siamese Multimodal Fusion MLP)

In [ ]:
from sklearn.model_selection import StratifiedKFold
# ============================================================
# CONFIGURACION
# ============================================================

SAVE_MODELS = False

output_dir = 'results/triple_branch_mlp_stratified_cv'
tb_dir = os.path.join(output_dir, 'tensorboard')

if os.path.exists(output_dir):
    shutil.rmtree(output_dir)

os.makedirs(output_dir, exist_ok=True)
os.makedirs(os.path.join(output_dir, 'predictions'), exist_ok=True)
os.makedirs(os.path.join(output_dir, 'figures'), exist_ok=True)
os.makedirs(tb_dir, exist_ok=True)

if SAVE_MODELS:
    os.makedirs(os.path.join(output_dir, 'models'), exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

mlp_config = {
    'lr': 1e-3,
    'batch_size': 256,
    'epochs': 80,
    'patience': 12,
    'weight_decay': 1e-4,
    'drug_dim': 1024,
    'n_genes': 500,
    'random_state': 42,
    'use_pos_weight': True,
}

experiment_config = {
    'model': 'TripleBranchMLP',
    'split_type': 'stratified_random_cv',
    'n_splits': 10,
    'n_genes': mlp_config['n_genes'],
    'random_state': mlp_config['random_state'],
    'threshold_grid_min': 0.05,
    'threshold_grid_max': 0.95,
    'threshold_grid_step': 0.01,
    'mlp_config': mlp_config,
    'save_models': SAVE_MODELS,
    'device': str(device),
    'external_lodo_splits_path': 'results/shared_splits/stratified_kfold_indices.json',
    'stratified definition': (
        'Random Stratified CV: each fold is a random sample stratified by the target variable.'
    )
}

save_experiment_config(experiment_config, output_dir)


In [ ]:
# ============================================================
# DATASET BASE
# ============================================================

base_df = X_final_df.copy()
base_df['drug_row_id'] = base_df['drug_row_id'].astype(str)
base_df['drug_col_id'] = base_df['drug_col_id'].astype(str)
base_df['synergy_loewe_bin'] = base_df['synergy_loewe_bin'].replace(-1, 0).astype(int)

thresholds = np.round(np.arange(0.05, 0.951, 0.01), 2)

shared_splits_dir = 'results/shared_splits'
os.makedirs(shared_splits_dir, exist_ok=True)

skf_splits_path = os.path.join(shared_splits_dir, 'stratified_kfold_indices.json')

if os.path.exists(skf_splits_path):
    with open(skf_splits_path, 'r') as f:
        split_indices = json.load(f)

    print('Splits StratifiedKFold cargados desde:', skf_splits_path)

else:
    skf = StratifiedKFold(
        n_splits=experiment_config['n_splits'],
        shuffle=True,
        random_state=experiment_config['random_state']
    )

    split_indices = []

    for fold_idx, (train_idx, test_idx) in enumerate(
        skf.split(base_df, base_df['synergy_loewe_bin']),
        start=1
    ):
        split_indices.append({
            'fold': fold_idx,
            'train_idx': train_idx.tolist(),
            'test_idx': test_idx.tolist()
        })

    with open(skf_splits_path, 'w') as f:
        json.dump(split_indices, f, indent=4)

    print('Splits StratifiedKFold creados y guardados en:', skf_splits_path)


In [ ]:
# ============================================================
# BUCLE CV
# ============================================================

cv_val_results_f1 = []
cv_val_results_precision = []
cv_test_results_f1 = []
cv_test_results_precision = []
cv_split_info = []
cv_val_threshold_metrics = []
cv_test_threshold_metrics = []
cv_best_thresholds = []
cv_histories = []

for split in split_indices:
    fold_idx = split['fold']
    train_idx = np.array(split['train_idx'])
    test_idx = np.array(split['test_idx'])

    writer = SummaryWriter(log_dir=os.path.join(tb_dir, f'Fold_{fold_idx}'))
    print(f'\nPROCESANDO FOLD {fold_idx}/{experiment_config["n_splits"]} - TRIPLE BRANCH MLP STRATIFIED CV')


    df_train_raw = base_df.iloc[train_idx].reset_index(drop=True)
    df_test_raw = base_df.iloc[test_idx].reset_index(drop=True)


    if len(df_train_raw) == 0 or len(df_test_raw) == 0:
        print(f'Fold {fold_idx}: sin datos suficientes, se omite.')
        writer.close()
        continue


    train_drugs = set(pd.concat([
        df_train_raw['drug_row_id'],
        df_train_raw['drug_col_id']
    ]).astype(str))

    test_drugs = set(pd.concat([
        df_test_raw['drug_row_id'],
        df_test_raw['drug_col_id']
    ]).astype(str))

    drug_overlap_train_test = len(train_drugs & test_drugs)
    drug_overlap_train_test_pct = (
        100 * drug_overlap_train_test / len(test_drugs)
        if len(test_drugs) > 0 else 0.0
    )

    df_train, df_val, df_test, top_genes = prepare_fold_data_no_leakage(
        df_train_raw,
        df_test_raw,
        data_expr,
        n_genes=experiment_config['n_genes'],
        random_state=experiment_config['random_state']
    )

    y_train_np = df_train['synergy_loewe_bin'].astype(int).to_numpy()

    if len(np.unique(y_train_np)) < 2:
        print(f'Fold {fold_idx}: train con una sola clase, se omite.')
        writer.close()
        continue

    train_loader = DataLoader(
        SynergyDataset(df_train, top_genes),
        batch_size=mlp_config['batch_size'],
        shuffle=True,

    )

    val_loader = DataLoader(
        SynergyDataset(df_val, top_genes),
        batch_size=mlp_config['batch_size'],
        shuffle=False
    )

    test_loader = DataLoader(
        SynergyDataset(df_test, top_genes),
        batch_size=mlp_config['batch_size'],
        shuffle=False
    )

    model = TripleBranchMLP(
        drug_dim=mlp_config['drug_dim'],
        gene_dim=experiment_config['n_genes']
    ).to(device)

    model, history_df, best_epoch, best_val_loss = train_torch_model_fold(
        model,
        train_loader,
        val_loader,
        y_train_np,
        mlp_config,
        device,
        writer,
        fold_idx
    )

    history_df.to_csv(
        os.path.join(output_dir, f'training_history_fold_{fold_idx:02d}.csv'),
        index=False
    )
    cv_histories.append(history_df)

    y_val, y_prob_val = predict_torch_model(model, val_loader, device)
    y_test, y_prob_test = predict_torch_model(model, test_loader, device)

    df_val_thr, best_thr_f1, best_thr_precision = find_best_thresholds(
        y_val,
        y_prob_val,
        thresholds
    )

    df_val_thr.insert(0, 'fold', fold_idx)
    df_val_thr.insert(1, 'split', 'val')
    df_val_thr.insert(2, 'threshold_type', 'grid')
    cv_val_threshold_metrics.append(df_val_thr)

    test_thr_rows = []
    for thr in thresholds:
        m_test_thr = compute_metrics_at_threshold(y_test, y_prob_test, thr)
        m_test_thr.update({
            'fold': fold_idx,
            'split': 'test',
            'threshold_type': 'grid',
        })
        test_thr_rows.append(m_test_thr)

    df_test_thr = pd.DataFrame(test_thr_rows)
    cv_test_threshold_metrics.append(df_test_thr)

    val_metrics_f1 = compute_metrics_at_threshold(y_val, y_prob_val, best_thr_f1)
    val_metrics_precision = compute_metrics_at_threshold(y_val, y_prob_val, best_thr_precision)

    test_metrics_f1 = compute_metrics_at_threshold(y_test, y_prob_test, best_thr_f1)
    test_metrics_precision = compute_metrics_at_threshold(y_test, y_prob_test, best_thr_precision)

    val_metrics_f1.update({'fold': fold_idx, 'split': 'val', 'threshold_type': 'best_f1'})
    val_metrics_precision.update({'fold': fold_idx, 'split': 'val', 'threshold_type': 'best_precision'})
    test_metrics_f1.update({'fold': fold_idx, 'split': 'test', 'threshold_type': 'best_f1'})
    test_metrics_precision.update({'fold': fold_idx, 'split': 'test', 'threshold_type': 'best_precision'})

    cv_val_results_f1.append(val_metrics_f1)
    cv_val_results_precision.append(val_metrics_precision)
    cv_test_results_f1.append(test_metrics_f1)
    cv_test_results_precision.append(test_metrics_precision)

    cv_best_thresholds.append({
        'fold': fold_idx,
        'best_threshold_f1': best_thr_f1,
        'best_threshold_precision': best_thr_precision,
        'val_f1_at_best_f1': val_metrics_f1['f1'],
        'val_precision_at_best_f1': val_metrics_f1['precision'],
        'val_recall_at_best_f1': val_metrics_f1['recall'],
        'val_precision_at_best_precision': val_metrics_precision['precision'],
        'val_recall_at_best_precision': val_metrics_precision['recall'],
        'val_f1_at_best_precision': val_metrics_precision['f1'],
        'best_epoch': best_epoch,
        'best_val_loss': best_val_loss,
    })

    log_curves_and_metrics(y_val, y_prob_val, val_metrics_f1, fold_idx, writer, tag='Val_BestF1')
    log_curves_and_metrics(y_val, y_prob_val, val_metrics_precision, fold_idx, writer, tag='Val_BestPrecision')
    log_curves_and_metrics(y_test, y_prob_test, test_metrics_f1, fold_idx, writer, tag='Test_BestF1')
    log_curves_and_metrics(y_test, y_prob_test, test_metrics_precision, fold_idx, writer, tag='Test_BestPrecision')

    predictions = df_test[
        [
            'drug_row_id',
            'drug_col_id',
            'cell_line_name',
            'study_name',
            'tissue',
            'synergy_loewe',
            'synergy_loewe_bin'
        ]
    ].copy()

    predictions['y_true'] = y_test
    predictions['y_prob'] = y_prob_test
    predictions['y_pred_best_f1'] = (y_prob_test >= best_thr_f1).astype(int)
    predictions['y_pred_best_precision'] = (y_prob_test >= best_thr_precision).astype(int)
    predictions['threshold_best_f1'] = best_thr_f1
    predictions['threshold_best_precision'] = best_thr_precision

    predictions.to_csv(
        os.path.join(output_dir, 'predictions', f'predictions_fold_{fold_idx:02d}.csv'),
        index=False
    )

    if SAVE_MODELS:
        torch.save(
            model.state_dict(),
            os.path.join(output_dir, 'models', f'triple_branch_mlp_fold_{fold_idx:02d}.pt')
        )

    n_train = len(y_train_np)
    n_val = len(y_val)
    n_test = len(y_test)

    split_info = {
        'fold': fold_idx,
        'n_train': n_train,
        'n_val': n_val,
        'n_test': n_test,
        'train_pos': int(np.sum(y_train_np)),
        'val_pos': int(np.sum(y_val)),
        'test_pos': int(np.sum(y_test)),
        'train_pos_pct': 100 * np.mean(y_train_np),
        'val_pos_pct': 100 * np.mean(y_val),
        'test_pos_pct': 100 * np.mean(y_test),
        'baseline_auprc_test': float(np.mean(y_test)),
        'n_train_drugs': len(train_drugs),
        'n_test_drugs': len(test_drugs),
        'drug_overlap_train_test': drug_overlap_train_test,
        'drug_overlap_train_test_pct': drug_overlap_train_test_pct,
        'split_indices_path': skf_splits_path,
        'best_threshold_f1': best_thr_f1,
        'best_threshold_precision': best_thr_precision,
        'best_epoch': best_epoch,
        'best_val_loss': best_val_loss,
        'n_top_genes': len(top_genes),
    }

    cv_split_info.append(split_info)

    writer.add_text('Split/Info', json.dumps(split_info, indent=2))
    writer.add_text('Genes/TopGenes_First100', ', '.join(top_genes[:100]))

    print(
        f"Fold {fold_idx} | "
        f"Train={n_train} ({100*np.mean(y_train_np):.2f}% pos), "
        f"Val={n_val} ({100*np.mean(y_val):.2f}% pos), "
        f"Test={n_test} ({100*np.mean(y_test):.2f}% pos) | "
        f"BestEpoch={best_epoch} | "
        f"ThrF1={best_thr_f1:.2f}, ThrPrec={best_thr_precision:.2f} | "
        f"AUPRC={test_metrics_f1['auprc']:.4f} "
        f"(base={test_metrics_f1['baseline_auprc']:.4f}) | "
        f"F1={test_metrics_f1['f1']:.4f} | "
        f"Prec={test_metrics_f1['precision']:.4f} | "
        f"Rec={test_metrics_f1['recall']:.4f} | "
        f"MCC={test_metrics_f1['mcc']:.4f}"
    )

    writer.close()
    torch.cuda.empty_cache()


In [ ]:
# ============================================================
# GUARDADO DE RESULTADOS
# ============================================================

df_val_f1 = pd.DataFrame(cv_val_results_f1)
df_val_precision = pd.DataFrame(cv_val_results_precision)
df_test_f1 = pd.DataFrame(cv_test_results_f1)
df_test_precision = pd.DataFrame(cv_test_results_precision)
df_split_info = pd.DataFrame(cv_split_info)
df_best_thresholds = pd.DataFrame(cv_best_thresholds)

df_val_thresholds = pd.concat(cv_val_threshold_metrics, ignore_index=True)
df_test_thresholds = pd.concat(cv_test_threshold_metrics, ignore_index=True)

df_val_f1.to_csv(os.path.join(output_dir, 'val_metrics_best_f1.csv'), index=False)
df_val_precision.to_csv(os.path.join(output_dir, 'val_metrics_best_precision.csv'), index=False)
df_test_f1.to_csv(os.path.join(output_dir, 'test_metrics_best_f1.csv'), index=False)
df_test_precision.to_csv(os.path.join(output_dir, 'test_metrics_best_precision.csv'), index=False)
df_split_info.to_csv(os.path.join(output_dir, 'split_info.csv'), index=False)
df_best_thresholds.to_csv(os.path.join(output_dir, 'best_thresholds.csv'), index=False)
df_val_thresholds.to_csv(os.path.join(output_dir, 'val_metrics_by_threshold.csv'), index=False)
df_test_thresholds.to_csv(os.path.join(output_dir, 'test_metrics_by_threshold.csv'), index=False)

if len(cv_histories) > 0:
    df_all_history = pd.concat(cv_histories, ignore_index=True)
    df_all_history.to_csv(os.path.join(output_dir, 'training_history_all_folds.csv'), index=False)


# ============================================================
# RESUMEN FINAL
# ============================================================

metric_cols = [
    'auc',
    'auprc',
    'baseline_auprc',
    'f1',
    'precision',
    'recall',
    'specificity',
    'balanced_accuracy',
    'mcc',
    'log_loss',
    'brier',
    'tp',
    'tn',
    'fp',
    'fn',
]

summary_f1 = (
    df_test_f1[metric_cols]
    .agg(['mean', 'std'])
    .T
    .reset_index()
    .rename(columns={'index': 'metric'})
)

summary_precision = (
    df_test_precision[metric_cols]
    .agg(['mean', 'std'])
    .T
    .reset_index()
    .rename(columns={'index': 'metric'})
)

summary_f1.to_csv(os.path.join(output_dir, 'summary_test_best_f1.csv'), index=False)
summary_precision.to_csv(os.path.join(output_dir, 'summary_test_best_precision.csv'), index=False)

print('\n' + '=' * 70)
print('RESULTADOS TRIPLE BRANCH MLP - STRATIFIED KFOLD CV=10')
print('=' * 70)

print('\n[TEST - threshold elegido por mejor F1 en validacion]')
print(summary_f1)

print('\n[TEST - threshold conservador por mejor precision en validacion]')
print(summary_precision)

print('\nArchivos guardados en:', output_dir)

Aun así, en GPU puede quedar alguna pequeña variabilidad dependiendo de operaciones y versiones. Para reproducibilidad estricta, CPU suele ser más determinista, aunque mucho más lento.

### MLP TRIPLE : VARIACIÓN CON INTERACCIÓN DE DROGAS (Interaction-Aware Triple-Branch MLP) Leave-Drug-Out CV=10

Se probaron interacciones explícitas mediante producto elemento a elemento y diferencia absoluta, que capturan respectivamente subestructuras compartidas y diferenciales entre fármacos. La suma no se incluyó al ser redundante con estas dos operaciones para fingerprints binarios.


In [ ]:
class SynergyInteractionDataset(Dataset):
    def __init__(self, df, gene_cols):
        d1_cols = [f'drug1_bit_{i}' for i in range(1024)]
        d2_cols = [f'drug2_bit_{i}' for i in range(1024)]

        d1 = torch.tensor(df[d1_cols].values, dtype=torch.float32)
        d2 = torch.tensor(df[d2_cols].values, dtype=torch.float32)

        self.d1 = d1
        self.d2 = d2

        d_mul = d1 * d2
        d_diff = torch.abs(d1 - d2)

        self.d_inter = torch.cat([d_mul, d_diff], dim=1)

        self.genes = torch.tensor(df[gene_cols].values, dtype=torch.float32)
        self.y = torch.tensor(
            df['synergy_loewe_bin'].values,
            dtype=torch.float32
        ).view(-1, 1)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.d1[idx], self.d2[idx], self.d_inter[idx], self.genes[idx], self.y[idx]



In [ ]:
class InteractionBranchMLP(nn.Module):
    def __init__(self, drug_dim=1024, gene_dim=500):
        super(InteractionBranchMLP, self).__init__()

        self.drug_branch = nn.Sequential(
            nn.Linear(drug_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.ReLU()
        )

        self.interaction_branch = nn.Sequential(
            nn.Linear(drug_dim * 2, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.ReLU()
        )

        self.gene_branch = nn.Sequential(
            nn.Linear(gene_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.ReLU()
        )

        self.classifier = nn.Sequential(
            nn.Linear(256 * 4, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, d1, d2, d_inter, g):
        z1 = self.drug_branch(d1)
        z2 = self.drug_branch(d2)
        zi = self.interaction_branch(d_inter)
        zg = self.gene_branch(g)

        z = torch.cat((z1, z2, zi, zg), dim=1)

        return self.classifier(z)


In [ ]:
def run_torch_epoch_interaction(model, loader, criterion, optimizer=None, device='cpu'):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss = 0.0
    y_true_all = []
    y_prob_all = []

    context = torch.enable_grad() if is_train else torch.no_grad()

    with context:
        for d1, d2, d_inter, g, y in loader:
            d1 = d1.to(device)
            d2 = d2.to(device)
            d_inter = d_inter.to(device)
            g = g.to(device)
            y = y.to(device)

            if is_train:
                optimizer.zero_grad()

            logits = model(d1, d2, d_inter, g)
            loss = criterion(logits, y)

            if is_train:
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * y.size(0)

            probs = torch.sigmoid(logits).detach().cpu().numpy().ravel()
            y_true = y.detach().cpu().numpy().ravel()

            y_prob_all.extend(probs)
            y_true_all.extend(y_true)

    avg_loss = total_loss / len(loader.dataset)

    return avg_loss, np.asarray(y_true_all), np.asarray(y_prob_all)


In [ ]:
def train_torch_interaction_model_fold(model, train_loader, val_loader, y_train, config, device, writer, fold_idx):
    n_pos = float(np.sum(y_train))
    n_neg = float(len(y_train) - n_pos)

    if config.get('use_pos_weight', True) and n_pos > 0:
        pos_weight = torch.tensor([n_neg / n_pos], dtype=torch.float32).to(device)
        criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    else:
        criterion = nn.BCEWithLogitsLoss()

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config['lr'],
        weight_decay=config['weight_decay']
    )

    best_val_loss = np.inf
    best_model_state = copy.deepcopy(model.state_dict())
    best_epoch = 0
    epochs_no_improve = 0

    history = []

    for epoch in range(1, config['epochs'] + 1):
        train_loss, y_true_train, y_prob_train = run_torch_epoch_interaction(
            model,
            train_loader,
            criterion,
            optimizer=optimizer,
            device=device
        )

        val_loss, y_true_val, y_prob_val = run_torch_epoch_interaction(
            model,
            val_loader,
            criterion,
            optimizer=None,
            device=device
        )

        val_aucpr = compute_metrics_at_threshold(
            y_true_val,
            y_prob_val,
            threshold=0.5
        )['auprc']

        writer.add_scalar('Loss/Train', train_loss, epoch)
        writer.add_scalar('Loss/Val', val_loss, epoch)
        writer.add_scalar('Metrics/Val_AUPRC_thr05', val_aucpr, epoch)

        history.append({
            'fold': fold_idx,
            'epoch': epoch,
            'train_loss': train_loss,
            'val_loss': val_loss,
            'val_auprc_thr05': val_aucpr,
        })

        print(
            f"Fold {fold_idx} | Epoch {epoch:03d}/{config['epochs']} | "
            f"TrainLoss={train_loss:.4f} | ValLoss={val_loss:.4f} | "
            f"ValAUPRC={val_aucpr:.4f}",
            end='\r'
        )

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model_state = copy.deepcopy(model.state_dict())
            best_epoch = epoch
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1

            if epochs_no_improve >= config['patience']:
                break

    print()

    model.load_state_dict(best_model_state)

    return model, pd.DataFrame(history), best_epoch, best_val_loss

In [ ]:
def predict_torch_interaction_model(model, loader, device):
    model.eval()

    y_true_all = []
    y_prob_all = []

    with torch.no_grad():
        for d1, d2, d_inter, g, y in loader:
            d1 = d1.to(device)
            d2 = d2.to(device)
            d_inter = d_inter.to(device)
            g = g.to(device)

            logits = model(d1, d2, d_inter, g)
            probs = torch.sigmoid(logits).cpu().numpy().ravel()

            y_prob_all.extend(probs)
            y_true_all.extend(y.numpy().ravel())

    return np.asarray(y_true_all), np.asarray(y_prob_all)


In [ ]:
import os
import json
import copy
import shutil
import numpy as np
import pandas as pd
import torch

from torch import nn
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
from PIL import Image

if not hasattr(Image, "Resampling"):
    class Resampling:
        LANCZOS = Image.ANTIALIAS

    Image.Resampling = Resampling


# ============================================================
# CONFIGURACION
# ============================================================

SAVE_MODELS = False

output_dir = 'results/triple_interactiondiff_branch_mlp_lodo'
tb_dir = os.path.join(output_dir, 'tensorboard')

if os.path.exists(output_dir):
    shutil.rmtree(output_dir)

os.makedirs(output_dir, exist_ok=True)
os.makedirs(os.path.join(output_dir, 'predictions'), exist_ok=True)
os.makedirs(os.path.join(output_dir, 'figures'), exist_ok=True)
os.makedirs(tb_dir, exist_ok=True)

if SAVE_MODELS:
    os.makedirs(os.path.join(output_dir, 'models'), exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

mlp_config = {
    'lr': 1e-3,
    'batch_size': 256,
    'epochs': 80,
    'patience': 12,
    'weight_decay': 1e-4,
    'drug_dim': 1024,
    'n_genes': 500,
    'random_state': 42,
    'use_pos_weight': True,
}

experiment_config = {
    'model': 'TripleInteractionDiffBranchMLP',
    'interaction_features': 'drug1*drug2 + abs(drug1-drug2)',
    'split_type': 'partial_lodo_optimized',
    'n_splits': 10,
    'n_genes': mlp_config['n_genes'],
    'random_state': mlp_config['random_state'],
    'threshold_grid_min': 0.05,
    'threshold_grid_max': 0.95,
    'threshold_grid_step': 0.01,
    'mlp_config': mlp_config,
    'save_models': SAVE_MODELS,
    'device': str(device),
    'external_lodo_splits_path': 'results/shared_splits/lodo_fold_sets.json',
    'lodo_definition': (
        'Partial Leave-Drug-Out: each test sample contains at least one held-out drug. '
        'The second drug may appear in train.'
    )
}

save_experiment_config(experiment_config, output_dir)


In [ ]:
# ============================================================
# DATASET BASE
# ============================================================

base_df = X_final_df.copy()
base_df['drug_row_id'] = base_df['drug_row_id'].astype(str)
base_df['drug_col_id'] = base_df['drug_col_id'].astype(str)
base_df['synergy_loewe_bin'] = base_df['synergy_loewe_bin'].replace(-1, 0).astype(int)

thresholds = np.round(np.arange(0.05, 0.951, 0.01), 2)


# ============================================================
# CARGAR SPLITS LODO COMPARTIDOS
# ============================================================

shared_splits_path = experiment_config['external_lodo_splits_path']

with open(shared_splits_path, 'r') as f:
    fold_sets_dict = json.load(f)

fold_sets = [
    set(fold_sets_dict[str(i + 1)])
    for i in range(len(fold_sets_dict))
]

assert len(fold_sets) == experiment_config['n_splits']

partition_rows = []

for fold_idx, fold_set in enumerate(fold_sets, start=1):
    mask_test = (
        base_df['drug_row_id'].isin(fold_set)
        | base_df['drug_col_id'].isin(fold_set)
    )

    fold_df = base_df[mask_test]

    n_examples = len(fold_df)
    pos_rate = fold_df['synergy_loewe_bin'].mean() if n_examples > 0 else np.nan

    partition_rows.append({
        'fold': fold_idx,
        'n_examples_induced_test': n_examples,
        'positive_pct_induced_test': pos_rate * 100,
        'baseline_auprc_induced_test': pos_rate,
        'n_heldout_drugs': len(fold_set)
    })

fold_partition_summary = pd.DataFrame(partition_rows)
fold_partition_summary.to_csv(
    os.path.join(output_dir, 'lodo_partition_summary.csv'),
    index=False
)

print('\nSplits LODO compartidos cargados desde:', shared_splits_path)
print(fold_partition_summary)


# ============================================================
# SOLAPAMIENTO TEST
# ============================================================

test_overlap_info, overlap_df, overlap_counts = summarize_test_overlap(
    base_df,
    fold_sets,
    output_path=output_dir
)

print('\nSolapamiento entre conjuntos test:')
print(test_overlap_info)
print('\nDistribucion de apariciones por muestra en test:')
print(overlap_counts)


In [ ]:
# ============================================================
# BUCLE CV
# ============================================================

cv_val_results_f1 = []
cv_val_results_precision = []
cv_test_results_f1 = []
cv_test_results_precision = []
cv_split_info = []
cv_val_threshold_metrics = []
cv_test_threshold_metrics = []
cv_best_thresholds = []
cv_histories = []

for fold_idx, test_set in enumerate(fold_sets, start=1):
    writer = SummaryWriter(log_dir=os.path.join(tb_dir, f'Fold_{fold_idx}'))

    print(f'\nPROCESANDO FOLD {fold_idx}/{experiment_config["n_splits"]} - TRIPLE BRANCH MLP LODO')

    mask_test = (
        base_df['drug_row_id'].isin(test_set)
        | base_df['drug_col_id'].isin(test_set)
    )

    df_test_raw = base_df[mask_test].reset_index(drop=True)
    df_train_raw = base_df[~mask_test].reset_index(drop=True)

    if len(df_train_raw) == 0 or len(df_test_raw) == 0:
        print(f'Fold {fold_idx}: sin datos suficientes, se omite.')
        writer.close()
        continue

    assert (
        df_test_raw['drug_row_id'].isin(test_set)
        | df_test_raw['drug_col_id'].isin(test_set)
    ).all(), f'Fold {fold_idx}: hay muestras test sin droga held-out.'

    train_drugs = set(pd.concat([
        df_train_raw['drug_row_id'],
        df_train_raw['drug_col_id']
    ]).astype(str))

    heldout_overlap_train = len(train_drugs & test_set)
    assert heldout_overlap_train == 0, f'Fold {fold_idx}: drogas held-out aparecen en train.'

    df_train, df_val, df_test, top_genes = prepare_fold_data_no_leakage(
        df_train_raw,
        df_test_raw,
        data_expr,
        n_genes=experiment_config['n_genes'],
        random_state=experiment_config['random_state']
    )

    y_train_np = df_train['synergy_loewe_bin'].astype(int).to_numpy()

    if len(np.unique(y_train_np)) < 2:
        print(f'Fold {fold_idx}: train con una sola clase, se omite.')
        writer.close()
        continue

    train_loader = DataLoader(
        SynergyInteractionDataset(df_train, top_genes),
        batch_size=mlp_config['batch_size'],
        shuffle=True,

    )

    val_loader = DataLoader(
        SynergyInteractionDataset(df_val, top_genes),
        batch_size=mlp_config['batch_size'],
        shuffle=False
    )

    test_loader = DataLoader(
        SynergyInteractionDataset(df_test, top_genes),
        batch_size=mlp_config['batch_size'],
        shuffle=False
    )

    model = InteractionBranchMLP(
    drug_dim=mlp_config['drug_dim'],
    gene_dim=experiment_config['n_genes']
    ).to(device)

    model, history_df, best_epoch, best_val_loss = train_torch_interaction_model_fold(
        model,
        train_loader,
        val_loader,
        y_train_np,
        mlp_config,
        device,
        writer,
        fold_idx
    )

    history_df.to_csv(
        os.path.join(output_dir, f'training_history_fold_{fold_idx:02d}.csv'),
        index=False
    )
    cv_histories.append(history_df)

    y_val, y_prob_val = predict_torch_interaction_model(model, val_loader, device)
    y_test, y_prob_test = predict_torch_interaction_model(model, test_loader, device)

    df_val_thr, best_thr_f1, best_thr_precision = find_best_thresholds(
        y_val,
        y_prob_val,
        thresholds
    )

    df_val_thr.insert(0, 'fold', fold_idx)
    df_val_thr.insert(1, 'split', 'val')
    df_val_thr.insert(2, 'threshold_type', 'grid')
    cv_val_threshold_metrics.append(df_val_thr)

    test_thr_rows = []
    for thr in thresholds:
        m_test_thr = compute_metrics_at_threshold(y_test, y_prob_test, thr)
        m_test_thr.update({
            'fold': fold_idx,
            'split': 'test',
            'threshold_type': 'grid',
        })
        test_thr_rows.append(m_test_thr)

    df_test_thr = pd.DataFrame(test_thr_rows)
    cv_test_threshold_metrics.append(df_test_thr)

    val_metrics_f1 = compute_metrics_at_threshold(y_val, y_prob_val, best_thr_f1)
    val_metrics_precision = compute_metrics_at_threshold(y_val, y_prob_val, best_thr_precision)

    test_metrics_f1 = compute_metrics_at_threshold(y_test, y_prob_test, best_thr_f1)
    test_metrics_precision = compute_metrics_at_threshold(y_test, y_prob_test, best_thr_precision)

    val_metrics_f1.update({'fold': fold_idx, 'split': 'val', 'threshold_type': 'best_f1'})
    val_metrics_precision.update({'fold': fold_idx, 'split': 'val', 'threshold_type': 'best_precision'})
    test_metrics_f1.update({'fold': fold_idx, 'split': 'test', 'threshold_type': 'best_f1'})
    test_metrics_precision.update({'fold': fold_idx, 'split': 'test', 'threshold_type': 'best_precision'})

    cv_val_results_f1.append(val_metrics_f1)
    cv_val_results_precision.append(val_metrics_precision)
    cv_test_results_f1.append(test_metrics_f1)
    cv_test_results_precision.append(test_metrics_precision)

    cv_best_thresholds.append({
        'fold': fold_idx,
        'best_threshold_f1': best_thr_f1,
        'best_threshold_precision': best_thr_precision,
        'val_f1_at_best_f1': val_metrics_f1['f1'],
        'val_precision_at_best_f1': val_metrics_f1['precision'],
        'val_recall_at_best_f1': val_metrics_f1['recall'],
        'val_precision_at_best_precision': val_metrics_precision['precision'],
        'val_recall_at_best_precision': val_metrics_precision['recall'],
        'val_f1_at_best_precision': val_metrics_precision['f1'],
        'best_epoch': best_epoch,
        'best_val_loss': best_val_loss,
    })

    log_curves_and_metrics(y_val, y_prob_val, val_metrics_f1, fold_idx, writer, tag='Val_BestF1')
    log_curves_and_metrics(y_val, y_prob_val, val_metrics_precision, fold_idx, writer, tag='Val_BestPrecision')
    log_curves_and_metrics(y_test, y_prob_test, test_metrics_f1, fold_idx, writer, tag='Test_BestF1')
    log_curves_and_metrics(y_test, y_prob_test, test_metrics_precision, fold_idx, writer, tag='Test_BestPrecision')

    predictions = df_test[
        [
            'drug_row_id',
            'drug_col_id',
            'cell_line_name',
            'study_name',
            'tissue',
            'synergy_loewe',
            'synergy_loewe_bin'
        ]
    ].copy()

    predictions['y_true'] = y_test
    predictions['y_prob'] = y_prob_test
    predictions['y_pred_best_f1'] = (y_prob_test >= best_thr_f1).astype(int)
    predictions['y_pred_best_precision'] = (y_prob_test >= best_thr_precision).astype(int)
    predictions['threshold_best_f1'] = best_thr_f1
    predictions['threshold_best_precision'] = best_thr_precision

    predictions.to_csv(
        os.path.join(output_dir, 'predictions', f'predictions_fold_{fold_idx:02d}.csv'),
        index=False
    )

    if SAVE_MODELS:
        torch.save(
            model.state_dict(),
            os.path.join(output_dir, 'models', f'triple_branch_mlp_fold_{fold_idx:02d}.pt')
        )

    n_train = len(y_train_np)
    n_val = len(y_val)
    n_test = len(y_test)

    split_info = {
        'fold': fold_idx,
        'n_train': n_train,
        'n_val': n_val,
        'n_test': n_test,
        'train_pos': int(np.sum(y_train_np)),
        'val_pos': int(np.sum(y_val)),
        'test_pos': int(np.sum(y_test)),
        'train_pos_pct': 100 * np.mean(y_train_np),
        'val_pos_pct': 100 * np.mean(y_val),
        'test_pos_pct': 100 * np.mean(y_test),
        'baseline_auprc_test': float(np.mean(y_test)),
        'n_heldout_drugs': len(test_set),
        'heldout_drugs': ';'.join(sorted(test_set)),
        'heldout_overlap_train': heldout_overlap_train,
        'best_threshold_f1': best_thr_f1,
        'best_threshold_precision': best_thr_precision,
        'best_epoch': best_epoch,
        'best_val_loss': best_val_loss,
        'n_top_genes': len(top_genes),
    }

    cv_split_info.append(split_info)

    writer.add_text('Split/Info', json.dumps(split_info, indent=2))
    writer.add_text('Genes/TopGenes_First100', ', '.join(top_genes[:100]))

    print(
        f"Fold {fold_idx} | "
        f"Train={n_train} ({100*np.mean(y_train_np):.2f}% pos), "
        f"Val={n_val} ({100*np.mean(y_val):.2f}% pos), "
        f"Test={n_test} ({100*np.mean(y_test):.2f}% pos) | "
        f"BestEpoch={best_epoch} | "
        f"ThrF1={best_thr_f1:.2f}, ThrPrec={best_thr_precision:.2f} | "
        f"AUPRC={test_metrics_f1['auprc']:.4f} "
        f"(base={test_metrics_f1['baseline_auprc']:.4f}) | "
        f"F1={test_metrics_f1['f1']:.4f} | "
        f"Prec={test_metrics_f1['precision']:.4f} | "
        f"Rec={test_metrics_f1['recall']:.4f} | "
        f"MCC={test_metrics_f1['mcc']:.4f}"
    )

    writer.close()
    torch.cuda.empty_cache()


In [ ]:
# ============================================================
# GUARDADO DE RESULTADOS
# ============================================================

df_val_f1 = pd.DataFrame(cv_val_results_f1)
df_val_precision = pd.DataFrame(cv_val_results_precision)
df_test_f1 = pd.DataFrame(cv_test_results_f1)
df_test_precision = pd.DataFrame(cv_test_results_precision)
df_split_info = pd.DataFrame(cv_split_info)
df_best_thresholds = pd.DataFrame(cv_best_thresholds)

df_val_thresholds = pd.concat(cv_val_threshold_metrics, ignore_index=True)
df_test_thresholds = pd.concat(cv_test_threshold_metrics, ignore_index=True)

df_val_f1.to_csv(os.path.join(output_dir, 'val_metrics_best_f1.csv'), index=False)
df_val_precision.to_csv(os.path.join(output_dir, 'val_metrics_best_precision.csv'), index=False)
df_test_f1.to_csv(os.path.join(output_dir, 'test_metrics_best_f1.csv'), index=False)
df_test_precision.to_csv(os.path.join(output_dir, 'test_metrics_best_precision.csv'), index=False)
df_split_info.to_csv(os.path.join(output_dir, 'split_info.csv'), index=False)
df_best_thresholds.to_csv(os.path.join(output_dir, 'best_thresholds.csv'), index=False)
df_val_thresholds.to_csv(os.path.join(output_dir, 'val_metrics_by_threshold.csv'), index=False)
df_test_thresholds.to_csv(os.path.join(output_dir, 'test_metrics_by_threshold.csv'), index=False)

if len(cv_histories) > 0:
    df_all_history = pd.concat(cv_histories, ignore_index=True)
    df_all_history.to_csv(os.path.join(output_dir, 'training_history_all_folds.csv'), index=False)


# ============================================================
# RESUMEN FINAL
# ============================================================

metric_cols = [
    'auc',
    'auprc',
    'baseline_auprc',
    'f1',
    'precision',
    'recall',
    'specificity',
    'balanced_accuracy',
    'mcc',
    'log_loss',
    'brier',
    'tp',
    'tn',
    'fp',
    'fn',
]

summary_f1 = (
    df_test_f1[metric_cols]
    .agg(['mean', 'std'])
    .T
    .reset_index()
    .rename(columns={'index': 'metric'})
)

summary_precision = (
    df_test_precision[metric_cols]
    .agg(['mean', 'std'])
    .T
    .reset_index()
    .rename(columns={'index': 'metric'})
)

summary_f1.to_csv(os.path.join(output_dir, 'summary_test_best_f1.csv'), index=False)
summary_precision.to_csv(os.path.join(output_dir, 'summary_test_best_precision.csv'), index=False)

print('\n' + '=' * 70)
print('RESULTADOS TRIPLE BRANCH MLP - LODO PARCIAL CV=10')
print('=' * 70)

print('\n[TEST - threshold elegido por mejor F1 en validacion]')
print(summary_f1)

print('\n[TEST - threshold conservador por mejor precision en validacion]')
print(summary_precision)

print('\nArchivos guardados en:', output_dir)


## MLP TRIPLE : VARIACIÓN CON INTERACCIÓN DE DROGAS (Interaction-Aware Triple-Branch MLP) Stratified K-Fold CV=10



In [ ]:
import os
import json
import copy
import shutil
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import StratifiedKFold

from torch import nn
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
from PIL import Image

if not hasattr(Image, "Resampling"):
    class Resampling:
        LANCZOS = Image.ANTIALIAS

    Image.Resampling = Resampling


# ============================================================
# CONFIGURACION
# ============================================================

SAVE_MODELS = False

output_dir = 'results/triple_interactiondiff_branch_mlp_stratified_cv'
tb_dir = os.path.join(output_dir, 'tensorboard')

if os.path.exists(output_dir):
    shutil.rmtree(output_dir)

os.makedirs(output_dir, exist_ok=True)
os.makedirs(os.path.join(output_dir, 'predictions'), exist_ok=True)
os.makedirs(os.path.join(output_dir, 'figures'), exist_ok=True)
os.makedirs(tb_dir, exist_ok=True)

if SAVE_MODELS:
    os.makedirs(os.path.join(output_dir, 'models'), exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

mlp_config = {
    'lr': 1e-3,
    'batch_size': 256,
    'epochs': 80,
    'patience': 12,
    'weight_decay': 1e-4,
    'drug_dim': 1024,
    'n_genes': 500,
    'random_state': 42,
    'use_pos_weight': True,
}

experiment_config = {
    'model': 'TripleInteractionDiffBranchMLP',
    'interaction_features': 'drug1*drug2 + abs(drug1-drug2)',
    'split_type': 'stratified_kfold',
    'n_splits': 10,
    'n_genes': mlp_config['n_genes'],
    'random_state': mlp_config['random_state'],
    'threshold_grid_min': 0.05,
    'threshold_grid_max': 0.95,
    'threshold_grid_step': 0.01,
    'mlp_config': mlp_config,
    'save_models': SAVE_MODELS,
    'device': str(device),
    'external_lodo_splits_path': 'results/shared_splits/stratified_kfold_indices.json',
    'stratified_definition': (
        'Random Stratified CV: each fold is a random sample stratified by the target variable.'
    )
}

save_experiment_config(experiment_config, output_dir)


In [ ]:
# ============================================================
# DATASET BASE
# ============================================================

base_df = X_final_df.copy()
base_df['drug_row_id'] = base_df['drug_row_id'].astype(str)
base_df['drug_col_id'] = base_df['drug_col_id'].astype(str)
base_df['synergy_loewe_bin'] = base_df['synergy_loewe_bin'].replace(-1, 0).astype(int)

thresholds = np.round(np.arange(0.05, 0.951, 0.01), 2)

shared_splits_dir = 'results/shared_splits'
os.makedirs(shared_splits_dir, exist_ok=True)
skf_splits_path = os.path.join(shared_splits_dir, 'stratified_kfold_indices.json')

if os.path.exists(skf_splits_path):
    with open(skf_splits_path, 'r') as f:
        split_indices = json.load(f)

    print('Splits StratifiedKFold cargados desde:', skf_splits_path)

else:
    skf = StratifiedKFold(
        n_splits=experiment_config['n_splits'],
        shuffle=True,
        random_state=experiment_config['random_state']
    )

    split_indices = []

    for fold_idx, (train_idx, test_idx) in enumerate(
        skf.split(base_df, base_df['synergy_loewe_bin']),
        start=1
    ):
        split_indices.append({
            'fold': fold_idx,
            'train_idx': train_idx.tolist(),
            'test_idx': test_idx.tolist()
        })

    with open(skf_splits_path, 'w') as f:
        json.dump(split_indices, f, indent=4)

    print('Splits StratifiedKFold creados y guardados en:', skf_splits_path)

In [ ]:
# ============================================================
# BUCLE CV
# ============================================================

cv_val_results_f1 = []
cv_val_results_precision = []
cv_test_results_f1 = []
cv_test_results_precision = []
cv_split_info = []
cv_val_threshold_metrics = []
cv_test_threshold_metrics = []
cv_best_thresholds = []
cv_histories = []



for split in split_indices:
    fold_idx = split['fold']
    train_idx = np.array(split['train_idx'])
    test_idx = np.array(split['test_idx'])

    writer = SummaryWriter(log_dir=os.path.join(tb_dir, f'Fold_{fold_idx}'))
    print(f'\nPROCESANDO FOLD {fold_idx}/{experiment_config["n_splits"]} - TRIPLE BRANCH MLP STRATIFIED CV')


    df_train_raw = base_df.iloc[train_idx].reset_index(drop=True)
    df_test_raw = base_df.iloc[test_idx].reset_index(drop=True)


    if len(df_train_raw) == 0 or len(df_test_raw) == 0:
        print(f'Fold {fold_idx}: sin datos suficientes, se omite.')
        writer.close()
        continue

    df_train, df_val, df_test, top_genes = prepare_fold_data_no_leakage(
        df_train_raw,
        df_test_raw,
        data_expr,
        n_genes=experiment_config['n_genes'],
        random_state=experiment_config['random_state']
    )

    y_train_np = df_train['synergy_loewe_bin'].astype(int).to_numpy()

    if len(np.unique(y_train_np)) < 2:
        print(f'Fold {fold_idx}: train con una sola clase, se omite.')
        writer.close()
        continue

    train_loader = DataLoader(
        SynergyInteractionDataset(df_train, top_genes),
        batch_size=mlp_config['batch_size'],
        shuffle=True,

    )

    val_loader = DataLoader(
        SynergyInteractionDataset(df_val, top_genes),
        batch_size=mlp_config['batch_size'],
        shuffle=False
    )

    test_loader = DataLoader(
        SynergyInteractionDataset(df_test, top_genes),
        batch_size=mlp_config['batch_size'],
        shuffle=False
    )

    model = InteractionBranchMLP(
    drug_dim=mlp_config['drug_dim'],
    gene_dim=experiment_config['n_genes']
    ).to(device)

    model, history_df, best_epoch, best_val_loss = train_torch_interaction_model_fold(
        model,
        train_loader,
        val_loader,
        y_train_np,
        mlp_config,
        device,
        writer,
        fold_idx
    )

    history_df.to_csv(
        os.path.join(output_dir, f'training_history_fold_{fold_idx:02d}.csv'),
        index=False
    )
    cv_histories.append(history_df)

    y_val, y_prob_val = predict_torch_interaction_model(model, val_loader, device)
    y_test, y_prob_test = predict_torch_interaction_model(model, test_loader, device)

    df_val_thr, best_thr_f1, best_thr_precision = find_best_thresholds(
        y_val,
        y_prob_val,
        thresholds
    )

    df_val_thr.insert(0, 'fold', fold_idx)
    df_val_thr.insert(1, 'split', 'val')
    df_val_thr.insert(2, 'threshold_type', 'grid')
    cv_val_threshold_metrics.append(df_val_thr)

    test_thr_rows = []
    for thr in thresholds:
        m_test_thr = compute_metrics_at_threshold(y_test, y_prob_test, thr)
        m_test_thr.update({
            'fold': fold_idx,
            'split': 'test',
            'threshold_type': 'grid',
        })
        test_thr_rows.append(m_test_thr)

    df_test_thr = pd.DataFrame(test_thr_rows)
    cv_test_threshold_metrics.append(df_test_thr)

    val_metrics_f1 = compute_metrics_at_threshold(y_val, y_prob_val, best_thr_f1)
    val_metrics_precision = compute_metrics_at_threshold(y_val, y_prob_val, best_thr_precision)

    test_metrics_f1 = compute_metrics_at_threshold(y_test, y_prob_test, best_thr_f1)
    test_metrics_precision = compute_metrics_at_threshold(y_test, y_prob_test, best_thr_precision)

    val_metrics_f1.update({'fold': fold_idx, 'split': 'val', 'threshold_type': 'best_f1'})
    val_metrics_precision.update({'fold': fold_idx, 'split': 'val', 'threshold_type': 'best_precision'})
    test_metrics_f1.update({'fold': fold_idx, 'split': 'test', 'threshold_type': 'best_f1'})
    test_metrics_precision.update({'fold': fold_idx, 'split': 'test', 'threshold_type': 'best_precision'})

    cv_val_results_f1.append(val_metrics_f1)
    cv_val_results_precision.append(val_metrics_precision)
    cv_test_results_f1.append(test_metrics_f1)
    cv_test_results_precision.append(test_metrics_precision)

    cv_best_thresholds.append({
        'fold': fold_idx,
        'best_threshold_f1': best_thr_f1,
        'best_threshold_precision': best_thr_precision,
        'val_f1_at_best_f1': val_metrics_f1['f1'],
        'val_precision_at_best_f1': val_metrics_f1['precision'],
        'val_recall_at_best_f1': val_metrics_f1['recall'],
        'val_precision_at_best_precision': val_metrics_precision['precision'],
        'val_recall_at_best_precision': val_metrics_precision['recall'],
        'val_f1_at_best_precision': val_metrics_precision['f1'],
        'best_epoch': best_epoch,
        'best_val_loss': best_val_loss,
    })

    log_curves_and_metrics(y_val, y_prob_val, val_metrics_f1, fold_idx, writer, tag='Val_BestF1')
    log_curves_and_metrics(y_val, y_prob_val, val_metrics_precision, fold_idx, writer, tag='Val_BestPrecision')
    log_curves_and_metrics(y_test, y_prob_test, test_metrics_f1, fold_idx, writer, tag='Test_BestF1')
    log_curves_and_metrics(y_test, y_prob_test, test_metrics_precision, fold_idx, writer, tag='Test_BestPrecision')

    predictions = df_test[
        [
            'drug_row_id',
            'drug_col_id',
            'cell_line_name',
            'study_name',
            'tissue',
            'synergy_loewe',
            'synergy_loewe_bin'
        ]
    ].copy()

    predictions['y_true'] = y_test
    predictions['y_prob'] = y_prob_test
    predictions['y_pred_best_f1'] = (y_prob_test >= best_thr_f1).astype(int)
    predictions['y_pred_best_precision'] = (y_prob_test >= best_thr_precision).astype(int)
    predictions['threshold_best_f1'] = best_thr_f1
    predictions['threshold_best_precision'] = best_thr_precision

    predictions.to_csv(
        os.path.join(output_dir, 'predictions', f'predictions_fold_{fold_idx:02d}.csv'),
        index=False
    )

    if SAVE_MODELS:
        torch.save(
            model.state_dict(),
            os.path.join(output_dir, 'models', f'triple_branch_mlp_fold_{fold_idx:02d}.pt')
        )

    n_train = len(y_train_np)
    n_val = len(y_val)
    n_test = len(y_test)


    train_drugs = set(pd.concat([
        df_train_raw['drug_row_id'],
        df_train_raw['drug_col_id']
    ]).astype(str))

    test_drugs = set(pd.concat([
        df_test_raw['drug_row_id'],
        df_test_raw['drug_col_id']
    ]).astype(str))

    drug_overlap_train_test = len(train_drugs & test_drugs)
    drug_overlap_train_test_pct = (
        100 * drug_overlap_train_test / len(test_drugs)
        if len(test_drugs) > 0 else 0.0
    )

    split_info = {
        'fold': fold_idx,
        'n_train': n_train,
        'n_val': n_val,
        'n_test': n_test,
        'train_pos': int(np.sum(y_train_np)),
        'val_pos': int(np.sum(y_val)),
        'test_pos': int(np.sum(y_test)),
        'train_pos_pct': 100 * np.mean(y_train_np),
        'val_pos_pct': 100 * np.mean(y_val),
        'test_pos_pct': 100 * np.mean(y_test),
        'baseline_auprc_test': float(np.mean(y_test)),
        'n_train_drugs': len(train_drugs),
        'n_test_drugs': len(test_drugs),
        'drug_overlap_train_test': drug_overlap_train_test,
        'drug_overlap_train_test_pct': drug_overlap_train_test_pct,
        'split_indices_path': skf_splits_path,
        'best_threshold_f1': best_thr_f1,
        'best_threshold_precision': best_thr_precision,
        'best_epoch': best_epoch,
        'best_val_loss': best_val_loss,
        'n_top_genes': len(top_genes),
    }

    cv_split_info.append(split_info)

    writer.add_text('Split/Info', json.dumps(split_info, indent=2))
    writer.add_text('Genes/TopGenes_First100', ', '.join(top_genes[:100]))

    print(
        f"Fold {fold_idx} | "
        f"Train={n_train} ({100*np.mean(y_train_np):.2f}% pos), "
        f"Val={n_val} ({100*np.mean(y_val):.2f}% pos), "
        f"Test={n_test} ({100*np.mean(y_test):.2f}% pos) | "
        f"BestEpoch={best_epoch} | "
        f"ThrF1={best_thr_f1:.2f}, ThrPrec={best_thr_precision:.2f} | "
        f"AUPRC={test_metrics_f1['auprc']:.4f} "
        f"(base={test_metrics_f1['baseline_auprc']:.4f}) | "
        f"F1={test_metrics_f1['f1']:.4f} | "
        f"Prec={test_metrics_f1['precision']:.4f} | "
        f"Rec={test_metrics_f1['recall']:.4f} | "
        f"MCC={test_metrics_f1['mcc']:.4f}"
    )

    writer.close()
    torch.cuda.empty_cache()


In [ ]:
# ============================================================
# GUARDADO DE RESULTADOS
# ============================================================

df_val_f1 = pd.DataFrame(cv_val_results_f1)
df_val_precision = pd.DataFrame(cv_val_results_precision)
df_test_f1 = pd.DataFrame(cv_test_results_f1)
df_test_precision = pd.DataFrame(cv_test_results_precision)
df_split_info = pd.DataFrame(cv_split_info)
df_best_thresholds = pd.DataFrame(cv_best_thresholds)

df_val_thresholds = pd.concat(cv_val_threshold_metrics, ignore_index=True)
df_test_thresholds = pd.concat(cv_test_threshold_metrics, ignore_index=True)

df_val_f1.to_csv(os.path.join(output_dir, 'val_metrics_best_f1.csv'), index=False)
df_val_precision.to_csv(os.path.join(output_dir, 'val_metrics_best_precision.csv'), index=False)
df_test_f1.to_csv(os.path.join(output_dir, 'test_metrics_best_f1.csv'), index=False)
df_test_precision.to_csv(os.path.join(output_dir, 'test_metrics_best_precision.csv'), index=False)
df_split_info.to_csv(os.path.join(output_dir, 'split_info.csv'), index=False)
df_best_thresholds.to_csv(os.path.join(output_dir, 'best_thresholds.csv'), index=False)
df_val_thresholds.to_csv(os.path.join(output_dir, 'val_metrics_by_threshold.csv'), index=False)
df_test_thresholds.to_csv(os.path.join(output_dir, 'test_metrics_by_threshold.csv'), index=False)

if len(cv_histories) > 0:
    df_all_history = pd.concat(cv_histories, ignore_index=True)
    df_all_history.to_csv(os.path.join(output_dir, 'training_history_all_folds.csv'), index=False)


# ============================================================
# RESUMEN FINAL
# ============================================================

metric_cols = [
    'auc',
    'auprc',
    'baseline_auprc',
    'f1',
    'precision',
    'recall',
    'specificity',
    'balanced_accuracy',
    'mcc',
    'log_loss',
    'brier',
    'tp',
    'tn',
    'fp',
    'fn',
]

summary_f1 = (
    df_test_f1[metric_cols]
    .agg(['mean', 'std'])
    .T
    .reset_index()
    .rename(columns={'index': 'metric'})
)

summary_precision = (
    df_test_precision[metric_cols]
    .agg(['mean', 'std'])
    .T
    .reset_index()
    .rename(columns={'index': 'metric'})
)

summary_f1.to_csv(os.path.join(output_dir, 'summary_test_best_f1.csv'), index=False)
summary_precision.to_csv(os.path.join(output_dir, 'summary_test_best_precision.csv'), index=False)

print('\n' + '=' * 70)
print('RESULTADOS TRIPLE INTERACTION-DIFF BRANCH  - STRATIFIED KFOLD CV=10')
print('=' * 70)

print('\n[TEST - threshold elegido por mejor F1 en validacion]')
print(summary_f1)

print('\n[TEST - threshold conservador por mejor precision en validacion]')
print(summary_precision)

print('\nArchivos guardados en:', output_dir)


La incorporación explícita de interacciones simples entre fingerprints, mediante producto elemento a elemento y diferencia absoluta, no mejoró el rendimiento respecto a la arquitectura con ramas independientes.


Puede deberse a varias razones:

- d1 * d2 y abs(d1 - d2) son interacciones muy simples sobre fingerprints binarios.
- La sinergia puede depender más de mecanismos biológicos, targets, vías celulares o contexto de expresión que de similitud/diferencia estructural directa.
- La rama adicional aumenta parámetros y puede dificultar la generalización.
- El modelo base ya aprende interacciones no lineales al concatenar z1, z2 y zg en el clasificador final.
- El escenario LDO es más exigente y puede penalizar arquitecturas más complejas.

## MLP : Leave-Drug-Out CV=10

In [ ]:
class TabularSynergyDataset(Dataset):
    def __init__(self, df, gene_cols):
        d1_cols = [f'drug1_bit_{i}' for i in range(1024)]
        d2_cols = [f'drug2_bit_{i}' for i in range(1024)]

        feature_cols = d1_cols + d2_cols + gene_cols

        self.x = torch.tensor(df[feature_cols].values, dtype=torch.float32)
        self.y = torch.tensor(
            df['synergy_loewe_bin'].values,
            dtype=torch.float32
        ).view(-1, 1)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]


import torch
import torch.nn as nn
import torch.nn.functional as F

class DrugSynergyMLP(nn.Module):
    def __init__(self, n_genes=500, input_dim=None):
        super(DrugSynergyMLP, self).__init__()

        # Compatibilidad: si se pasa input_dim, tiene prioridad.
        if input_dim is None:
            input_dim = n_genes

        # Capa 1: Entrada -> 512
        self.fc1 = nn.Linear(input_dim, 512)
        self.bn1 = nn.BatchNorm1d(512)
        self.dropout1 = nn.Dropout(0.4)

        # Capa 2: 512 -> 256
        self.fc2 = nn.Linear(512, 256)
        self.bn2 = nn.BatchNorm1d(256)
        self.dropout2 = nn.Dropout(0.4)

        # Capa 3: 256 -> 128
        self.fc3 = nn.Linear(256, 128)
        self.bn3 = nn.BatchNorm1d(128)

        # Capa de salida (Clasificación binaria: Sinergia vs Antagonismo)
        # Usamos 1 neurona con Sigmoid al final (o BCEWithLogitsLoss)
        self.output = nn.Linear(128, 1)

    def forward(self, x):
        # Bloque 1
        x = F.relu(self.bn1(self.fc1(x)))
        x = self.dropout1(x)

        # Bloque 2
        x = F.relu(self.bn2(self.fc2(x)))
        x = self.dropout2(x)

        # Bloque 3
        x = F.relu(self.bn3(self.fc3(x)))

        # Salida
        return self.output(x)

In [ ]:
def run_torch_epoch_tabular(model, loader, criterion, optimizer=None, device='cpu'):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss = 0.0
    y_true_all = []
    y_prob_all = []

    context = torch.enable_grad() if is_train else torch.no_grad()

    with context:
        for x, y in loader:
            x = x.to(device)
            y = y.to(device)

            if is_train:
                optimizer.zero_grad()

            logits = model(x)
            loss = criterion(logits, y)

            if is_train:
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * y.size(0)

            probs = torch.sigmoid(logits).detach().cpu().numpy().ravel()
            y_true = y.detach().cpu().numpy().ravel()

            y_prob_all.extend(probs)
            y_true_all.extend(y_true)

    avg_loss = total_loss / len(loader.dataset)

    return avg_loss, np.asarray(y_true_all), np.asarray(y_prob_all)


In [ ]:
def train_torch_tabular_model_fold(model, train_loader, val_loader, y_train, config, device, writer, fold_idx):
    n_pos = float(np.sum(y_train))
    n_neg = float(len(y_train) - n_pos)

    if config.get('use_pos_weight', True) and n_pos > 0:
        pos_weight = torch.tensor([n_neg / n_pos], dtype=torch.float32).to(device)
        criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    else:
        criterion = nn.BCEWithLogitsLoss()

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config['lr'],
        weight_decay=config['weight_decay']
    )

    best_val_loss = np.inf
    best_model_state = copy.deepcopy(model.state_dict())
    best_epoch = 0
    epochs_no_improve = 0

    history = []

    for epoch in range(1, config['epochs'] + 1):
        train_loss, y_true_train, y_prob_train = run_torch_epoch_tabular(
            model,
            train_loader,
            criterion,
            optimizer=optimizer,
            device=device
        )

        val_loss, y_true_val, y_prob_val = run_torch_epoch_tabular(
            model,
            val_loader,
            criterion,
            optimizer=None,
            device=device
        )

        val_auprc = compute_metrics_at_threshold(
            y_true_val,
            y_prob_val,
            threshold=0.5
        )['auprc']

        writer.add_scalar('Loss/Train', train_loss, epoch)
        writer.add_scalar('Loss/Val', val_loss, epoch)
        writer.add_scalar('Metrics/Val_AUPRC_thr05', val_auprc, epoch)

        history.append({
            'fold': fold_idx,
            'epoch': epoch,
            'train_loss': train_loss,
            'val_loss': val_loss,
            'val_auprc_thr05': val_auprc,
        })

        print(
            f"Fold {fold_idx} | Epoch {epoch:03d}/{config['epochs']} | "
            f"TrainLoss={train_loss:.4f} | ValLoss={val_loss:.4f} | "
            f"ValAUPRC={val_auprc:.4f}",
            end='\r'
        )

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model_state = copy.deepcopy(model.state_dict())
            best_epoch = epoch
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1

            if epochs_no_improve >= config['patience']:
                break

    print()

    model.load_state_dict(best_model_state)

    return model, pd.DataFrame(history), best_epoch, best_val_loss


In [ ]:
def predict_torch_tabular_model(model, loader, device):
    model.eval()

    y_true_all = []
    y_prob_all = []

    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)

            logits = model(x)
            probs = torch.sigmoid(logits).cpu().numpy().ravel()

            y_prob_all.extend(probs)
            y_true_all.extend(y.numpy().ravel())

    return np.asarray(y_true_all), np.asarray(y_prob_all)


In [ ]:
import os
import json
import copy
import shutil
import numpy as np
import pandas as pd
import torch

from torch import nn
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
from PIL import Image

if not hasattr(Image, "Resampling"):
    class Resampling:
        LANCZOS = Image.ANTIALIAS

    Image.Resampling = Resampling


# ============================================================
# CONFIGURACION
# ============================================================

SAVE_MODELS = False

output_dir = 'results/basic_mlp_lodo'
tb_dir = os.path.join(output_dir, 'tensorboard')

if os.path.exists(output_dir):
    shutil.rmtree(output_dir)

os.makedirs(output_dir, exist_ok=True)
os.makedirs(os.path.join(output_dir, 'predictions'), exist_ok=True)
os.makedirs(os.path.join(output_dir, 'figures'), exist_ok=True)
os.makedirs(tb_dir, exist_ok=True)

if SAVE_MODELS:
    os.makedirs(os.path.join(output_dir, 'models'), exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

mlp_config = {
    'lr': 1e-3,
    'batch_size': 256,
    'epochs': 80,
    'patience': 12,
    'weight_decay': 1e-4,
    'drug_dim': 1024,
    'n_genes': 500,
    'random_state': 42,
    'use_pos_weight': True,
}

experiment_config = {
    'model': 'EarlyFusionMLP',
    'split_type': 'partial_lodo_optimized',
    'n_splits': 10,
    'n_genes': mlp_config['n_genes'],
    'random_state': mlp_config['random_state'],
    'threshold_grid_min': 0.05,
    'threshold_grid_max': 0.95,
    'threshold_grid_step': 0.01,
    'mlp_config': mlp_config,
    'save_models': SAVE_MODELS,
    'device': str(device),
    'external_lodo_splits_path': 'results/shared_splits/lodo_fold_sets.json',
    'lodo_definition': (
        'Partial Leave-Drug-Out: each test sample contains at least one held-out drug. '
        'The second drug may appear in train.'
    ),
    'input_features': 'drug1_fp + drug2_fp + selected_gene_expression'
}

save_experiment_config(experiment_config, output_dir)


In [ ]:
# ============================================================
# DATASET BASE
# ============================================================

base_df = X_final_df.copy()
base_df['drug_row_id'] = base_df['drug_row_id'].astype(str)
base_df['drug_col_id'] = base_df['drug_col_id'].astype(str)
base_df['synergy_loewe_bin'] = base_df['synergy_loewe_bin'].replace(-1, 0).astype(int)

thresholds = np.round(np.arange(0.05, 0.951, 0.01), 2)


# ============================================================
# CARGAR SPLITS LODO COMPARTIDOS
# ============================================================

shared_splits_path = experiment_config['external_lodo_splits_path']

with open(shared_splits_path, 'r') as f:
    fold_sets_dict = json.load(f)

fold_sets = [
    set(fold_sets_dict[str(i + 1)])
    for i in range(len(fold_sets_dict))
]

assert len(fold_sets) == experiment_config['n_splits']

partition_rows = []

for fold_idx, fold_set in enumerate(fold_sets, start=1):
    mask_test = (
        base_df['drug_row_id'].isin(fold_set)
        | base_df['drug_col_id'].isin(fold_set)
    )

    fold_df = base_df[mask_test]

    n_examples = len(fold_df)
    pos_rate = fold_df['synergy_loewe_bin'].mean() if n_examples > 0 else np.nan

    partition_rows.append({
        'fold': fold_idx,
        'n_examples_induced_test': n_examples,
        'positive_pct_induced_test': pos_rate * 100,
        'baseline_auprc_induced_test': pos_rate,
        'n_heldout_drugs': len(fold_set)
    })

fold_partition_summary = pd.DataFrame(partition_rows)
fold_partition_summary.to_csv(
    os.path.join(output_dir, 'lodo_partition_summary.csv'),
    index=False
)

print('\nSplits LODO compartidos cargados desde:', shared_splits_path)
print(fold_partition_summary)


# ============================================================
# SOLAPAMIENTO TEST
# ============================================================

test_overlap_info, overlap_df, overlap_counts = summarize_test_overlap(
    base_df,
    fold_sets,
    output_path=output_dir
)

print('\nSolapamiento entre conjuntos test:')
print(test_overlap_info)
print('\nDistribucion de apariciones por muestra en test:')
print(overlap_counts)


In [ ]:
# ============================================================
# BUCLE CV
# ============================================================

cv_val_results_f1 = []
cv_val_results_precision = []
cv_test_results_f1 = []
cv_test_results_precision = []
cv_split_info = []
cv_val_threshold_metrics = []
cv_test_threshold_metrics = []
cv_best_thresholds = []
cv_histories = []


for fold_idx, test_set in enumerate(fold_sets, start=1):
    writer = SummaryWriter(log_dir=os.path.join(tb_dir, f'Fold_{fold_idx}'))

    print(f'\nPROCESANDO FOLD {fold_idx}/{experiment_config["n_splits"]} - BASIC MLP LDO CV')

    mask_test = (
        base_df['drug_row_id'].isin(test_set)
        | base_df['drug_col_id'].isin(test_set)
    )

    df_test_raw = base_df[mask_test].reset_index(drop=True)
    df_train_raw = base_df[~mask_test].reset_index(drop=True)

    if len(df_train_raw) == 0 or len(df_test_raw) == 0:
        print(f'Fold {fold_idx}: sin datos suficientes, se omite.')
        writer.close()
        continue

    assert (
        df_test_raw['drug_row_id'].isin(test_set)
        | df_test_raw['drug_col_id'].isin(test_set)
    ).all(), f'Fold {fold_idx}: hay muestras test sin droga held-out.'

    train_drugs = set(pd.concat([
        df_train_raw['drug_row_id'],
        df_train_raw['drug_col_id']
    ]).astype(str))

    heldout_overlap_train = len(train_drugs & test_set)
    assert heldout_overlap_train == 0, f'Fold {fold_idx}: drogas held-out aparecen en train.'

    df_train, df_val, df_test, top_genes = prepare_fold_data_no_leakage(
        df_train_raw,
        df_test_raw,
        data_expr,
        n_genes=experiment_config['n_genes'],
        random_state=experiment_config['random_state']
    )

    y_train_np = df_train['synergy_loewe_bin'].astype(int).to_numpy()

    if len(np.unique(y_train_np)) < 2:
        print(f'Fold {fold_idx}: train con una sola clase, se omite.')
        writer.close()
        continue

    train_loader = DataLoader(
        TabularSynergyDataset(df_train, top_genes),
        batch_size=mlp_config['batch_size'],
        shuffle=True,

    )

    val_loader = DataLoader(
        TabularSynergyDataset(df_val, top_genes),
        batch_size=mlp_config['batch_size'],
        shuffle=False
    )

    test_loader = DataLoader(
        TabularSynergyDataset(df_test, top_genes),
        batch_size=mlp_config['batch_size'],
        shuffle=False
    )

    input_dim = 1024 + 1024 + experiment_config['n_genes']

    model = DrugSynergyMLP(
        input_dim=input_dim
    ).to(device)

    model, history_df, best_epoch, best_val_loss = train_torch_tabular_model_fold(
        model,
        train_loader,
        val_loader,
        y_train_np,
        mlp_config,
        device,
        writer,
        fold_idx
    )

    history_df.to_csv(
        os.path.join(output_dir, f'training_history_fold_{fold_idx:02d}.csv'),
        index=False
    )
    cv_histories.append(history_df)

    y_val, y_prob_val = predict_torch_tabular_model(model, val_loader, device)
    y_test, y_prob_test = predict_torch_tabular_model(model, test_loader, device)

    df_val_thr, best_thr_f1, best_thr_precision = find_best_thresholds(
        y_val,
        y_prob_val,
        thresholds
    )

    df_val_thr.insert(0, 'fold', fold_idx)
    df_val_thr.insert(1, 'split', 'val')
    df_val_thr.insert(2, 'threshold_type', 'grid')
    cv_val_threshold_metrics.append(df_val_thr)

    test_thr_rows = []
    for thr in thresholds:
        m_test_thr = compute_metrics_at_threshold(y_test, y_prob_test, thr)
        m_test_thr.update({
            'fold': fold_idx,
            'split': 'test',
            'threshold_type': 'grid',
        })
        test_thr_rows.append(m_test_thr)

    df_test_thr = pd.DataFrame(test_thr_rows)
    cv_test_threshold_metrics.append(df_test_thr)

    val_metrics_f1 = compute_metrics_at_threshold(y_val, y_prob_val, best_thr_f1)
    val_metrics_precision = compute_metrics_at_threshold(y_val, y_prob_val, best_thr_precision)

    test_metrics_f1 = compute_metrics_at_threshold(y_test, y_prob_test, best_thr_f1)
    test_metrics_precision = compute_metrics_at_threshold(y_test, y_prob_test, best_thr_precision)

    val_metrics_f1.update({'fold': fold_idx, 'split': 'val', 'threshold_type': 'best_f1'})
    val_metrics_precision.update({'fold': fold_idx, 'split': 'val', 'threshold_type': 'best_precision'})
    test_metrics_f1.update({'fold': fold_idx, 'split': 'test', 'threshold_type': 'best_f1'})
    test_metrics_precision.update({'fold': fold_idx, 'split': 'test', 'threshold_type': 'best_precision'})

    cv_val_results_f1.append(val_metrics_f1)
    cv_val_results_precision.append(val_metrics_precision)
    cv_test_results_f1.append(test_metrics_f1)
    cv_test_results_precision.append(test_metrics_precision)

    cv_best_thresholds.append({
        'fold': fold_idx,
        'best_threshold_f1': best_thr_f1,
        'best_threshold_precision': best_thr_precision,
        'val_f1_at_best_f1': val_metrics_f1['f1'],
        'val_precision_at_best_f1': val_metrics_f1['precision'],
        'val_recall_at_best_f1': val_metrics_f1['recall'],
        'val_precision_at_best_precision': val_metrics_precision['precision'],
        'val_recall_at_best_precision': val_metrics_precision['recall'],
        'val_f1_at_best_precision': val_metrics_precision['f1'],
        'best_epoch': best_epoch,
        'best_val_loss': best_val_loss,
    })

    log_curves_and_metrics(y_val, y_prob_val, val_metrics_f1, fold_idx, writer, tag='Val_BestF1')
    log_curves_and_metrics(y_val, y_prob_val, val_metrics_precision, fold_idx, writer, tag='Val_BestPrecision')
    log_curves_and_metrics(y_test, y_prob_test, test_metrics_f1, fold_idx, writer, tag='Test_BestF1')
    log_curves_and_metrics(y_test, y_prob_test, test_metrics_precision, fold_idx, writer, tag='Test_BestPrecision')

    predictions = df_test[
        [
            'drug_row_id',
            'drug_col_id',
            'cell_line_name',
            'study_name',
            'tissue',
            'synergy_loewe',
            'synergy_loewe_bin'
        ]
    ].copy()

    predictions['y_true'] = y_test
    predictions['y_prob'] = y_prob_test
    predictions['y_pred_best_f1'] = (y_prob_test >= best_thr_f1).astype(int)
    predictions['y_pred_best_precision'] = (y_prob_test >= best_thr_precision).astype(int)
    predictions['threshold_best_f1'] = best_thr_f1
    predictions['threshold_best_precision'] = best_thr_precision

    predictions.to_csv(
        os.path.join(output_dir, 'predictions', f'predictions_fold_{fold_idx:02d}.csv'),
        index=False
    )

    if SAVE_MODELS:
        torch.save(
            model.state_dict(),
            os.path.join(output_dir, 'models', f'triple_branch_mlp_fold_{fold_idx:02d}.pt')
        )

    n_train = len(y_train_np)
    n_val = len(y_val)
    n_test = len(y_test)

    split_info = {
        'fold': fold_idx,
        'n_train': n_train,
        'n_val': n_val,
        'n_test': n_test,
        'train_pos': int(np.sum(y_train_np)),
        'val_pos': int(np.sum(y_val)),
        'test_pos': int(np.sum(y_test)),
        'train_pos_pct': 100 * np.mean(y_train_np),
        'val_pos_pct': 100 * np.mean(y_val),
        'test_pos_pct': 100 * np.mean(y_test),
        'baseline_auprc_test': float(np.mean(y_test)),
        'n_heldout_drugs': len(test_set),
        'heldout_drugs': ';'.join(sorted(test_set)),
        'heldout_overlap_train': heldout_overlap_train,
        'best_threshold_f1': best_thr_f1,
        'best_threshold_precision': best_thr_precision,
        'best_epoch': best_epoch,
        'best_val_loss': best_val_loss,
        'n_top_genes': len(top_genes),
    }

    cv_split_info.append(split_info)

    writer.add_text('Split/Info', json.dumps(split_info, indent=2))
    writer.add_text('Genes/TopGenes_First100', ', '.join(top_genes[:100]))

    print(
        f"Fold {fold_idx} | "
        f"Train={n_train} ({100*np.mean(y_train_np):.2f}% pos), "
        f"Val={n_val} ({100*np.mean(y_val):.2f}% pos), "
        f"Test={n_test} ({100*np.mean(y_test):.2f}% pos) | "
        f"BestEpoch={best_epoch} | "
        f"ThrF1={best_thr_f1:.2f}, ThrPrec={best_thr_precision:.2f} | "
        f"AUPRC={test_metrics_f1['auprc']:.4f} "
        f"(base={test_metrics_f1['baseline_auprc']:.4f}) | "
        f"F1={test_metrics_f1['f1']:.4f} | "
        f"Prec={test_metrics_f1['precision']:.4f} | "
        f"Rec={test_metrics_f1['recall']:.4f} | "
        f"MCC={test_metrics_f1['mcc']:.4f}"
    )

    writer.close()
    torch.cuda.empty_cache()


In [ ]:
# ============================================================
# GUARDADO DE RESULTADOS
# ============================================================

df_val_f1 = pd.DataFrame(cv_val_results_f1)
df_val_precision = pd.DataFrame(cv_val_results_precision)
df_test_f1 = pd.DataFrame(cv_test_results_f1)
df_test_precision = pd.DataFrame(cv_test_results_precision)
df_split_info = pd.DataFrame(cv_split_info)
df_best_thresholds = pd.DataFrame(cv_best_thresholds)

df_val_thresholds = pd.concat(cv_val_threshold_metrics, ignore_index=True)
df_test_thresholds = pd.concat(cv_test_threshold_metrics, ignore_index=True)

df_val_f1.to_csv(os.path.join(output_dir, 'val_metrics_best_f1.csv'), index=False)
df_val_precision.to_csv(os.path.join(output_dir, 'val_metrics_best_precision.csv'), index=False)
df_test_f1.to_csv(os.path.join(output_dir, 'test_metrics_best_f1.csv'), index=False)
df_test_precision.to_csv(os.path.join(output_dir, 'test_metrics_best_precision.csv'), index=False)
df_split_info.to_csv(os.path.join(output_dir, 'split_info.csv'), index=False)
df_best_thresholds.to_csv(os.path.join(output_dir, 'best_thresholds.csv'), index=False)
df_val_thresholds.to_csv(os.path.join(output_dir, 'val_metrics_by_threshold.csv'), index=False)
df_test_thresholds.to_csv(os.path.join(output_dir, 'test_metrics_by_threshold.csv'), index=False)

if len(cv_histories) > 0:
    df_all_history = pd.concat(cv_histories, ignore_index=True)
    df_all_history.to_csv(os.path.join(output_dir, 'training_history_all_folds.csv'), index=False)


# ============================================================
# RESUMEN FINAL
# ============================================================

metric_cols = [
    'auc',
    'auprc',
    'baseline_auprc',
    'f1',
    'precision',
    'recall',
    'specificity',
    'balanced_accuracy',
    'mcc',
    'log_loss',
    'brier',
    'tp',
    'tn',
    'fp',
    'fn',
]

summary_f1 = (
    df_test_f1[metric_cols]
    .agg(['mean', 'std'])
    .T
    .reset_index()
    .rename(columns={'index': 'metric'})
)

summary_precision = (
    df_test_precision[metric_cols]
    .agg(['mean', 'std'])
    .T
    .reset_index()
    .rename(columns={'index': 'metric'})
)

summary_f1.to_csv(os.path.join(output_dir, 'summary_test_best_f1.csv'), index=False)
summary_precision.to_csv(os.path.join(output_dir, 'summary_test_best_precision.csv'), index=False)

print('\n' + '=' * 70)
print('RESULTADOS MLP TABULAR - LODO PARCIAL CV=10')
print('=' * 70)

print('\n[TEST - threshold elegido por mejor F1 en validacion]')
print(summary_f1)

print('\n[TEST - threshold conservador por mejor precision en validacion]')
print(summary_precision)

print('\nArchivos guardados en:', output_dir)


## MLP : Stratified K-Fold CV=10

In [ ]:
import os
import json
import copy
import shutil
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import StratifiedKFold
from torch import nn
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
from PIL import Image

if not hasattr(Image, "Resampling"):
    class Resampling:
        LANCZOS = Image.ANTIALIAS

    Image.Resampling = Resampling


# ============================================================
# CONFIGURACION
# ============================================================

SAVE_MODELS = False

output_dir = 'results/basic_mlp_stratified_cv'
tb_dir = os.path.join(output_dir, 'tensorboard')

if os.path.exists(output_dir):
    shutil.rmtree(output_dir)

os.makedirs(output_dir, exist_ok=True)
os.makedirs(os.path.join(output_dir, 'predictions'), exist_ok=True)
os.makedirs(os.path.join(output_dir, 'figures'), exist_ok=True)
os.makedirs(tb_dir, exist_ok=True)

if SAVE_MODELS:
    os.makedirs(os.path.join(output_dir, 'models'), exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

mlp_config = {
    'lr': 1e-3,
    'batch_size': 256,
    'epochs': 80,
    'patience': 12,
    'weight_decay': 1e-4,
    'drug_dim': 1024,
    'n_genes': 500,
    'random_state': 42,
    'use_pos_weight': True,
}

experiment_config = {
    'model': 'EarlyFusionMLP',
    'split_type': 'stratified_kfold',
    'n_splits': 10,
    'n_genes': mlp_config['n_genes'],
    'random_state': mlp_config['random_state'],
    'threshold_grid_min': 0.05,
    'threshold_grid_max': 0.95,
    'threshold_grid_step': 0.01,
    'mlp_config': mlp_config,
    'save_models': SAVE_MODELS,
    'device': str(device),
    'external_split_indices_path': 'results/shared_splits/stratified_kfold_indices.json',
    'stratified_definition': (
        'Random Stratified CV: each fold is a random sample stratified by the target variable.'
    ),
    'input_features': 'drug1_fp + drug2_fp + selected_gene_expression'
}

save_experiment_config(experiment_config, output_dir)


In [ ]:
# ============================================================
# DATASET BASE
# ============================================================

base_df = X_final_df.copy()
base_df['drug_row_id'] = base_df['drug_row_id'].astype(str)
base_df['drug_col_id'] = base_df['drug_col_id'].astype(str)
base_df['synergy_loewe_bin'] = base_df['synergy_loewe_bin'].replace(-1, 0).astype(int)

thresholds = np.round(np.arange(0.05, 0.951, 0.01), 2)

shared_splits_dir = 'results/shared_splits'
os.makedirs(shared_splits_dir, exist_ok=True)
skf_splits_path = os.path.join(shared_splits_dir, 'stratified_kfold_indices.json')

if os.path.exists(skf_splits_path):
    with open(skf_splits_path, 'r') as f:
        split_indices = json.load(f)

    print('Splits StratifiedKFold cargados desde:', skf_splits_path)

else:
    skf = StratifiedKFold(
        n_splits=experiment_config['n_splits'],
        shuffle=True,
        random_state=experiment_config['random_state']
    )

    split_indices = []

    for fold_idx, (train_idx, test_idx) in enumerate(
        skf.split(base_df, base_df['synergy_loewe_bin']),
        start=1
    ):
        split_indices.append({
            'fold': fold_idx,
            'train_idx': train_idx.tolist(),
            'test_idx': test_idx.tolist()
        })

    with open(skf_splits_path, 'w') as f:
        json.dump(split_indices, f, indent=4)

    print('Splits StratifiedKFold creados y guardados en:', skf_splits_path)

In [ ]:
# ============================================================
# BUCLE CV
# ============================================================

cv_val_results_f1 = []
cv_val_results_precision = []
cv_test_results_f1 = []
cv_test_results_precision = []
cv_split_info = []
cv_val_threshold_metrics = []
cv_test_threshold_metrics = []
cv_best_thresholds = []
cv_histories = []



for split in split_indices:
    fold_idx = split['fold']
    train_idx = np.array(split['train_idx'])
    test_idx = np.array(split['test_idx'])

    writer = SummaryWriter(log_dir=os.path.join(tb_dir, f'Fold_{fold_idx}'))
    print(f'\nPROCESANDO FOLD {fold_idx}/{experiment_config["n_splits"]} - BASIC MLP STRATIFIED CV')


    df_train_raw = base_df.iloc[train_idx].reset_index(drop=True)
    df_test_raw = base_df.iloc[test_idx].reset_index(drop=True)


    if len(df_train_raw) == 0 or len(df_test_raw) == 0:
        print(f'Fold {fold_idx}: sin datos suficientes, se omite.')
        writer.close()
        continue

    df_train, df_val, df_test, top_genes = prepare_fold_data_no_leakage(
        df_train_raw,
        df_test_raw,
        data_expr,
        n_genes=experiment_config['n_genes'],
        random_state=experiment_config['random_state']
    )

    y_train_np = df_train['synergy_loewe_bin'].astype(int).to_numpy()

    if len(np.unique(y_train_np)) < 2:
        print(f'Fold {fold_idx}: train con una sola clase, se omite.')
        writer.close()
        continue

    train_loader = DataLoader(
        TabularSynergyDataset(df_train, top_genes),
        batch_size=mlp_config['batch_size'],
        shuffle=True,

    )

    val_loader = DataLoader(
        TabularSynergyDataset(df_val, top_genes),
        batch_size=mlp_config['batch_size'],
        shuffle=False
    )

    test_loader = DataLoader(
        TabularSynergyDataset(df_test, top_genes),
        batch_size=mlp_config['batch_size'],
        shuffle=False
    )

    input_dim = 1024 + 1024 + experiment_config['n_genes']

    model = DrugSynergyMLP(
        input_dim=input_dim
    ).to(device)

    model, history_df, best_epoch, best_val_loss = train_torch_tabular_model_fold(
        model,
        train_loader,
        val_loader,
        y_train_np,
        mlp_config,
        device,
        writer,
        fold_idx
    )

    history_df.to_csv(
        os.path.join(output_dir, f'training_history_fold_{fold_idx:02d}.csv'),
        index=False
    )
    cv_histories.append(history_df)

    y_val, y_prob_val = predict_torch_tabular_model(model, val_loader, device)
    y_test, y_prob_test = predict_torch_tabular_model(model, test_loader, device)

    df_val_thr, best_thr_f1, best_thr_precision = find_best_thresholds(
        y_val,
        y_prob_val,
        thresholds
    )

    df_val_thr.insert(0, 'fold', fold_idx)
    df_val_thr.insert(1, 'split', 'val')
    df_val_thr.insert(2, 'threshold_type', 'grid')
    cv_val_threshold_metrics.append(df_val_thr)

    test_thr_rows = []
    for thr in thresholds:
        m_test_thr = compute_metrics_at_threshold(y_test, y_prob_test, thr)
        m_test_thr.update({
            'fold': fold_idx,
            'split': 'test',
            'threshold_type': 'grid',
        })
        test_thr_rows.append(m_test_thr)

    df_test_thr = pd.DataFrame(test_thr_rows)
    cv_test_threshold_metrics.append(df_test_thr)

    val_metrics_f1 = compute_metrics_at_threshold(y_val, y_prob_val, best_thr_f1)
    val_metrics_precision = compute_metrics_at_threshold(y_val, y_prob_val, best_thr_precision)

    test_metrics_f1 = compute_metrics_at_threshold(y_test, y_prob_test, best_thr_f1)
    test_metrics_precision = compute_metrics_at_threshold(y_test, y_prob_test, best_thr_precision)

    val_metrics_f1.update({'fold': fold_idx, 'split': 'val', 'threshold_type': 'best_f1'})
    val_metrics_precision.update({'fold': fold_idx, 'split': 'val', 'threshold_type': 'best_precision'})
    test_metrics_f1.update({'fold': fold_idx, 'split': 'test', 'threshold_type': 'best_f1'})
    test_metrics_precision.update({'fold': fold_idx, 'split': 'test', 'threshold_type': 'best_precision'})

    cv_val_results_f1.append(val_metrics_f1)
    cv_val_results_precision.append(val_metrics_precision)
    cv_test_results_f1.append(test_metrics_f1)
    cv_test_results_precision.append(test_metrics_precision)

    cv_best_thresholds.append({
        'fold': fold_idx,
        'best_threshold_f1': best_thr_f1,
        'best_threshold_precision': best_thr_precision,
        'val_f1_at_best_f1': val_metrics_f1['f1'],
        'val_precision_at_best_f1': val_metrics_f1['precision'],
        'val_recall_at_best_f1': val_metrics_f1['recall'],
        'val_precision_at_best_precision': val_metrics_precision['precision'],
        'val_recall_at_best_precision': val_metrics_precision['recall'],
        'val_f1_at_best_precision': val_metrics_precision['f1'],
        'best_epoch': best_epoch,
        'best_val_loss': best_val_loss,
    })

    log_curves_and_metrics(y_val, y_prob_val, val_metrics_f1, fold_idx, writer, tag='Val_BestF1')
    log_curves_and_metrics(y_val, y_prob_val, val_metrics_precision, fold_idx, writer, tag='Val_BestPrecision')
    log_curves_and_metrics(y_test, y_prob_test, test_metrics_f1, fold_idx, writer, tag='Test_BestF1')
    log_curves_and_metrics(y_test, y_prob_test, test_metrics_precision, fold_idx, writer, tag='Test_BestPrecision')

    predictions = df_test[
        [
            'drug_row_id',
            'drug_col_id',
            'cell_line_name',
            'study_name',
            'tissue',
            'synergy_loewe',
            'synergy_loewe_bin'
        ]
    ].copy()

    predictions['y_true'] = y_test
    predictions['y_prob'] = y_prob_test
    predictions['y_pred_best_f1'] = (y_prob_test >= best_thr_f1).astype(int)
    predictions['y_pred_best_precision'] = (y_prob_test >= best_thr_precision).astype(int)
    predictions['threshold_best_f1'] = best_thr_f1
    predictions['threshold_best_precision'] = best_thr_precision

    predictions.to_csv(
        os.path.join(output_dir, 'predictions', f'predictions_fold_{fold_idx:02d}.csv'),
        index=False
    )

    if SAVE_MODELS:
        torch.save(
            model.state_dict(),
            os.path.join(output_dir, 'models', f'triple_branch_mlp_fold_{fold_idx:02d}.pt')
        )

    n_train = len(y_train_np)
    n_val = len(y_val)
    n_test = len(y_test)


    train_drugs = set(pd.concat([
        df_train_raw['drug_row_id'],
        df_train_raw['drug_col_id']
    ]).astype(str))

    test_drugs = set(pd.concat([
        df_test_raw['drug_row_id'],
        df_test_raw['drug_col_id']
    ]).astype(str))

    drug_overlap_train_test = len(train_drugs & test_drugs)
    drug_overlap_train_test_pct = (
        100 * drug_overlap_train_test / len(test_drugs)
        if len(test_drugs) > 0 else 0.0
    )

    split_info = {
        'fold': fold_idx,
        'n_train': n_train,
        'n_val': n_val,
        'n_test': n_test,
        'train_pos': int(np.sum(y_train_np)),
        'val_pos': int(np.sum(y_val)),
        'test_pos': int(np.sum(y_test)),
        'train_pos_pct': 100 * np.mean(y_train_np),
        'val_pos_pct': 100 * np.mean(y_val),
        'test_pos_pct': 100 * np.mean(y_test),
        'baseline_auprc_test': float(np.mean(y_test)),
        'n_train_drugs': len(train_drugs),
        'n_test_drugs': len(test_drugs),
        'drug_overlap_train_test': drug_overlap_train_test,
        'drug_overlap_train_test_pct': drug_overlap_train_test_pct,
        'split_indices_path': skf_splits_path,
        'best_threshold_f1': best_thr_f1,
        'best_threshold_precision': best_thr_precision,
        'best_epoch': best_epoch,
        'best_val_loss': best_val_loss,
        'n_top_genes': len(top_genes),
    }

    cv_split_info.append(split_info)

    writer.add_text('Split/Info', json.dumps(split_info, indent=2))
    writer.add_text('Genes/TopGenes_First100', ', '.join(top_genes[:100]))

    print(
        f"Fold {fold_idx} | "
        f"Train={n_train} ({100*np.mean(y_train_np):.2f}% pos), "
        f"Val={n_val} ({100*np.mean(y_val):.2f}% pos), "
        f"Test={n_test} ({100*np.mean(y_test):.2f}% pos) | "
        f"BestEpoch={best_epoch} | "
        f"ThrF1={best_thr_f1:.2f}, ThrPrec={best_thr_precision:.2f} | "
        f"AUPRC={test_metrics_f1['auprc']:.4f} "
        f"(base={test_metrics_f1['baseline_auprc']:.4f}) | "
        f"F1={test_metrics_f1['f1']:.4f} | "
        f"Prec={test_metrics_f1['precision']:.4f} | "
        f"Rec={test_metrics_f1['recall']:.4f} | "
        f"MCC={test_metrics_f1['mcc']:.4f}"
    )

    writer.close()
    torch.cuda.empty_cache()


In [ ]:
# ============================================================
# GUARDADO DE RESULTADOS
# ============================================================

df_val_f1 = pd.DataFrame(cv_val_results_f1)
df_val_precision = pd.DataFrame(cv_val_results_precision)
df_test_f1 = pd.DataFrame(cv_test_results_f1)
df_test_precision = pd.DataFrame(cv_test_results_precision)
df_split_info = pd.DataFrame(cv_split_info)
df_best_thresholds = pd.DataFrame(cv_best_thresholds)

df_val_thresholds = pd.concat(cv_val_threshold_metrics, ignore_index=True)
df_test_thresholds = pd.concat(cv_test_threshold_metrics, ignore_index=True)

df_val_f1.to_csv(os.path.join(output_dir, 'val_metrics_best_f1.csv'), index=False)
df_val_precision.to_csv(os.path.join(output_dir, 'val_metrics_best_precision.csv'), index=False)
df_test_f1.to_csv(os.path.join(output_dir, 'test_metrics_best_f1.csv'), index=False)
df_test_precision.to_csv(os.path.join(output_dir, 'test_metrics_best_precision.csv'), index=False)
df_split_info.to_csv(os.path.join(output_dir, 'split_info.csv'), index=False)
df_best_thresholds.to_csv(os.path.join(output_dir, 'best_thresholds.csv'), index=False)
df_val_thresholds.to_csv(os.path.join(output_dir, 'val_metrics_by_threshold.csv'), index=False)
df_test_thresholds.to_csv(os.path.join(output_dir, 'test_metrics_by_threshold.csv'), index=False)

if len(cv_histories) > 0:
    df_all_history = pd.concat(cv_histories, ignore_index=True)
    df_all_history.to_csv(os.path.join(output_dir, 'training_history_all_folds.csv'), index=False)


# ============================================================
# RESUMEN FINAL
# ============================================================

metric_cols = [
    'auc',
    'auprc',
    'baseline_auprc',
    'f1',
    'precision',
    'recall',
    'specificity',
    'balanced_accuracy',
    'mcc',
    'log_loss',
    'brier',
    'tp',
    'tn',
    'fp',
    'fn',
]

summary_f1 = (
    df_test_f1[metric_cols]
    .agg(['mean', 'std'])
    .T
    .reset_index()
    .rename(columns={'index': 'metric'})
)

summary_precision = (
    df_test_precision[metric_cols]
    .agg(['mean', 'std'])
    .T
    .reset_index()
    .rename(columns={'index': 'metric'})
)

summary_f1.to_csv(os.path.join(output_dir, 'summary_test_best_f1.csv'), index=False)
summary_precision.to_csv(os.path.join(output_dir, 'summary_test_best_precision.csv'), index=False)

print('\n' + '=' * 70)
print('RESULTADOS MLP TABULAR - STRATIFIED K-FOLD CV=10')
print('=' * 70)

print('\n[TEST - threshold elegido por mejor F1 en validacion]')
print(summary_f1)

print('\n[TEST - threshold conservador por mejor precision en validacion]')
print(summary_precision)

print('\nArchivos guardados en:', output_dir)



## TRANSFORMER + MLP Leave-Drug-Out CV=10


In [ ]:
import torch
import torch.nn as nn


class DrugPairEncoder(nn.Module):
    def __init__(self, drug_dim=1024, embed_dim=128, n_heads=4, n_layers=2, ff_dim=256, dropout=0.2):
        super().__init__()
        self.drug_proj = nn.Sequential(
            nn.Linear(drug_dim, embed_dim),
            nn.LayerNorm(embed_dim),
            nn.GELU(),
        )
        self.cls_token = nn.Parameter(torch.randn(1, 1, embed_dim) * 0.02)
        self.pos_embed = nn.Parameter(torch.randn(1, 3, embed_dim) * 0.02)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=n_heads,
            dim_feedforward=ff_dim,
            dropout=dropout,
            batch_first=True,
            activation="gelu",
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)

    def forward(self, d1, d2):
        batch_size = d1.size(0)
        d1_tok = self.drug_proj(d1).unsqueeze(1)
        d2_tok = self.drug_proj(d2).unsqueeze(1)
        cls_tok = self.cls_token.expand(batch_size, -1, -1)

        tokens = torch.cat((cls_tok, d1_tok, d2_tok), dim=1)
        tokens = tokens + self.pos_embed[:, :tokens.size(1), :]
        encoded = self.encoder(tokens)

        cls_repr = encoded[:, 0, :]
        pair_repr = encoded[:, 1:, :].reshape(batch_size, -1)
        return cls_repr, pair_repr


class GeneEncoder(nn.Module):
    def __init__(self, gene_dim, embed_dim=128, dropout=0.2): 
        super().__init__()
        self.block = nn.Sequential(
            nn.Linear(gene_dim, embed_dim),
            nn.LayerNorm(embed_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(embed_dim, embed_dim),
            nn.LayerNorm(embed_dim),
            nn.GELU(),
        )

    def forward(self, g):
        return self.block(g)


class TransformerFusionPredictor(nn.Module):

    def __init__(self, input_size, hidden_size):
        super().__init__()
        mid_size = max(hidden_size // 2, 32)
        self.network = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.BatchNorm1d(hidden_size),
            nn.GELU(),
            nn.Dropout(0.3), # 0.3
            nn.Linear(hidden_size, mid_size),
            nn.BatchNorm1d(mid_size),
            nn.GELU(),
            nn.Dropout(0.2), # 0.2
            nn.Linear(mid_size, 1),
        )

    def forward(self, feat):
        return self.network(feat)


class DrugPairTransformerMLP(nn.Module):
    def __init__(self, hiddim, n_genes=500, drug_dim=1024, embed_dim=128, n_heads=4, n_layers=2):
        super().__init__()
        self.drug_encoder = DrugPairEncoder(
            drug_dim=drug_dim,
            embed_dim=embed_dim,
            n_heads=n_heads,
            n_layers=n_layers,
            ff_dim=embed_dim * 2,
            dropout=0.2,
        )
        self.gene_encoder = GeneEncoder(n_genes, embed_dim=embed_dim, dropout=0.2)
        self.predictor = TransformerFusionPredictor(input_size=(embed_dim * 4), hidden_size=hiddim)

    def forward(self, d1, d2, g):
        cls_repr, pair_repr = self.drug_encoder(d1, d2)
        gene_repr = self.gene_encoder(g)
        fused = torch.cat((cls_repr, pair_repr, gene_repr), dim=1)
        return self.predictor(fused)

Se implementó una arquitectura inspirada en MADSP, adaptada a las variables disponibles en este trabajo. En lugar de incorporar información de targets, pathways o matrices gen-gen, el modelo representa los dos fármacos como tokens derivados de sus Morgan fingerprints y aplica un encoder Transformer para capturar interacciones entre ellos. La representación resultante se fusiona con una codificación de la expresión génica celular mediante una MLP final.


Transformer-based drug-pair encoder + gene fusion MLP

Utilizamos el mismo SynergyDataset y las mismas funciones de entrenamiento/predicción que usabas para la TripleBranchMLP basica, porque el forward recibe model(d1,d2,g) igual que la TripleBranchMLP


In [ ]:
import os
import json
import copy
import shutil
import numpy as np
import pandas as pd
import torch

from torch import nn
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
from PIL import Image

if not hasattr(Image, "Resampling"):
    class Resampling:
        LANCZOS = Image.ANTIALIAS

    Image.Resampling = Resampling


# ============================================================
# CONFIGURACION
# ============================================================

SAVE_MODELS = False

output_dir = 'results/drug_pair_transformer_mlp_lodo'
tb_dir = os.path.join(output_dir, 'tensorboard')

if os.path.exists(output_dir):
    shutil.rmtree(output_dir)

os.makedirs(output_dir, exist_ok=True)
os.makedirs(os.path.join(output_dir, 'predictions'), exist_ok=True)
os.makedirs(os.path.join(output_dir, 'figures'), exist_ok=True)
os.makedirs(tb_dir, exist_ok=True)

if SAVE_MODELS:
    os.makedirs(os.path.join(output_dir, 'models'), exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

mlp_config = {
    'lr': 1e-3,
    'batch_size': 256,
    'epochs': 80,
    'patience': 12,
    'weight_decay': 1e-4,
    'drug_dim': 1024,
    'n_genes': 500,
    'random_state': 42,
    'use_pos_weight': True,
    'hidden_dim': 256,
    'embed_dim': 128,
    'n_heads': 4,
    'n_layers': 2,
}


experiment_config = {
    'model': 'DrugPairTransformerMLP',
    'architecture': (
        'Transformer encoder over [CLS, drug1, drug2] tokens fused with gene expression encoder.'
    ),
    'split_type': 'partial_lodo_optimized',
    'n_splits': 10,
    'n_genes': mlp_config['n_genes'],
    'random_state': mlp_config['random_state'],
    'threshold_grid_min': 0.05,
    'threshold_grid_max': 0.95,
    'threshold_grid_step': 0.01,
    'mlp_config': mlp_config,
    'save_models': SAVE_MODELS,
    'device': str(device),
    'external_lodo_splits_path': 'results/shared_splits/lodo_fold_sets.json',
    'lodo_definition': (
        'Partial Leave-Drug-Out: each test sample contains at least one held-out drug. '
        'The second drug may appear in train.'
    )
}


save_experiment_config(experiment_config, output_dir)




# ============================================================
# DATASET BASE
# ============================================================

base_df = X_final_df.copy()
base_df['drug_row_id'] = base_df['drug_row_id'].astype(str)
base_df['drug_col_id'] = base_df['drug_col_id'].astype(str)
base_df['synergy_loewe_bin'] = base_df['synergy_loewe_bin'].replace(-1, 0).astype(int)

thresholds = np.round(np.arange(0.05, 0.951, 0.01), 2)


# ============================================================
# CARGAR SPLITS LODO COMPARTIDOS
# ============================================================

shared_splits_path = experiment_config['external_lodo_splits_path']

with open(shared_splits_path, 'r') as f:
    fold_sets_dict = json.load(f)

fold_sets = [
    set(fold_sets_dict[str(i + 1)])
    for i in range(len(fold_sets_dict))
]

assert len(fold_sets) == experiment_config['n_splits']

partition_rows = []

for fold_idx, fold_set in enumerate(fold_sets, start=1):
    mask_test = (
        base_df['drug_row_id'].isin(fold_set)
        | base_df['drug_col_id'].isin(fold_set)
    )

    fold_df = base_df[mask_test]

    n_examples = len(fold_df)
    pos_rate = fold_df['synergy_loewe_bin'].mean() if n_examples > 0 else np.nan

    partition_rows.append({
        'fold': fold_idx,
        'n_examples_induced_test': n_examples,
        'positive_pct_induced_test': pos_rate * 100,
        'baseline_auprc_induced_test': pos_rate,
        'n_heldout_drugs': len(fold_set)
    })

fold_partition_summary = pd.DataFrame(partition_rows)
fold_partition_summary.to_csv(
    os.path.join(output_dir, 'lodo_partition_summary.csv'),
    index=False
)

print('\nSplits LODO compartidos cargados desde:', shared_splits_path)
print(fold_partition_summary)


# ============================================================
# SOLAPAMIENTO TEST
# ============================================================

test_overlap_info, overlap_df, overlap_counts = summarize_test_overlap(
    base_df,
    fold_sets,
    output_path=output_dir
)

print('\nSolapamiento entre conjuntos test:')
print(test_overlap_info)
print('\nDistribucion de apariciones por muestra en test:')
print(overlap_counts)


In [ ]:
# ============================================================
# BUCLE CV
# ============================================================

cv_val_results_f1 = []
cv_val_results_precision = []
cv_test_results_f1 = []
cv_test_results_precision = []
cv_split_info = []
cv_val_threshold_metrics = []
cv_test_threshold_metrics = []
cv_best_thresholds = []
cv_histories = []



for fold_idx, test_set in enumerate(fold_sets, start=1):
    writer = SummaryWriter(log_dir=os.path.join(tb_dir, f'Fold_{fold_idx}'))

    print(f'\nPROCESANDO FOLD {fold_idx}/{experiment_config["n_splits"]} - TRANSFORMER + MLP LDO CV')

    mask_test = (
        base_df['drug_row_id'].isin(test_set)
        | base_df['drug_col_id'].isin(test_set)
    )

    df_test_raw = base_df[mask_test].reset_index(drop=True)
    df_train_raw = base_df[~mask_test].reset_index(drop=True)

    if len(df_train_raw) == 0 or len(df_test_raw) == 0:
        print(f'Fold {fold_idx}: sin datos suficientes, se omite.')
        writer.close()
        continue

    assert (
        df_test_raw['drug_row_id'].isin(test_set)
        | df_test_raw['drug_col_id'].isin(test_set)
    ).all(), f'Fold {fold_idx}: hay muestras test sin droga held-out.'

    train_drugs = set(pd.concat([
        df_train_raw['drug_row_id'],
        df_train_raw['drug_col_id']
    ]).astype(str))

    heldout_overlap_train = len(train_drugs & test_set)
    assert heldout_overlap_train == 0, f'Fold {fold_idx}: drogas held-out aparecen en train.'

    df_train, df_val, df_test, top_genes = prepare_fold_data_no_leakage(
        df_train_raw,
        df_test_raw,
        data_expr,
        n_genes=experiment_config['n_genes'],
        random_state=experiment_config['random_state']
    )

    y_train_np = df_train['synergy_loewe_bin'].astype(int).to_numpy()

    if len(np.unique(y_train_np)) < 2:
        print(f'Fold {fold_idx}: train con una sola clase, se omite.')
        writer.close()
        continue

    train_loader = DataLoader(
        SynergyDataset(df_train, top_genes),
        batch_size=mlp_config['batch_size'],
        shuffle=True,

    )

    val_loader = DataLoader(
        SynergyDataset(df_val, top_genes),
        batch_size=mlp_config['batch_size'],
        shuffle=False
    )

    test_loader = DataLoader(
        SynergyDataset(df_test, top_genes),
        batch_size=mlp_config['batch_size'],
        shuffle=False
    )

    model = DrugPairTransformerMLP(
        hiddim=mlp_config['hidden_dim'],
        n_genes=experiment_config['n_genes'],
        drug_dim=mlp_config['drug_dim'],
        embed_dim=mlp_config['embed_dim'],
        n_heads=mlp_config['n_heads'],
        n_layers=mlp_config['n_layers']
    ).to(device)


    model, history_df, best_epoch, best_val_loss = train_torch_model_fold(
        model,
        train_loader,
        val_loader,
        y_train_np,
        mlp_config,
        device,
        writer,
        fold_idx
    )

    history_df.to_csv(
        os.path.join(output_dir, f'training_history_fold_{fold_idx:02d}.csv'),
        index=False
    )
    cv_histories.append(history_df)

    y_val, y_prob_val = predict_torch_model(model, val_loader, device)
    y_test, y_prob_test = predict_torch_model(model, test_loader, device)

    df_val_thr, best_thr_f1, best_thr_precision = find_best_thresholds(
        y_val,
        y_prob_val,
        thresholds
    )

    df_val_thr.insert(0, 'fold', fold_idx)
    df_val_thr.insert(1, 'split', 'val')
    df_val_thr.insert(2, 'threshold_type', 'grid')
    cv_val_threshold_metrics.append(df_val_thr)

    test_thr_rows = []
    for thr in thresholds:
        m_test_thr = compute_metrics_at_threshold(y_test, y_prob_test, thr)
        m_test_thr.update({
            'fold': fold_idx,
            'split': 'test',
            'threshold_type': 'grid',
        })
        test_thr_rows.append(m_test_thr)

    df_test_thr = pd.DataFrame(test_thr_rows)
    cv_test_threshold_metrics.append(df_test_thr)

    val_metrics_f1 = compute_metrics_at_threshold(y_val, y_prob_val, best_thr_f1)
    val_metrics_precision = compute_metrics_at_threshold(y_val, y_prob_val, best_thr_precision)

    test_metrics_f1 = compute_metrics_at_threshold(y_test, y_prob_test, best_thr_f1)
    test_metrics_precision = compute_metrics_at_threshold(y_test, y_prob_test, best_thr_precision)

    val_metrics_f1.update({'fold': fold_idx, 'split': 'val', 'threshold_type': 'best_f1'})
    val_metrics_precision.update({'fold': fold_idx, 'split': 'val', 'threshold_type': 'best_precision'})
    test_metrics_f1.update({'fold': fold_idx, 'split': 'test', 'threshold_type': 'best_f1'})
    test_metrics_precision.update({'fold': fold_idx, 'split': 'test', 'threshold_type': 'best_precision'})

    cv_val_results_f1.append(val_metrics_f1)
    cv_val_results_precision.append(val_metrics_precision)
    cv_test_results_f1.append(test_metrics_f1)
    cv_test_results_precision.append(test_metrics_precision)

    cv_best_thresholds.append({
        'fold': fold_idx,
        'best_threshold_f1': best_thr_f1,
        'best_threshold_precision': best_thr_precision,
        'val_f1_at_best_f1': val_metrics_f1['f1'],
        'val_precision_at_best_f1': val_metrics_f1['precision'],
        'val_recall_at_best_f1': val_metrics_f1['recall'],
        'val_precision_at_best_precision': val_metrics_precision['precision'],
        'val_recall_at_best_precision': val_metrics_precision['recall'],
        'val_f1_at_best_precision': val_metrics_precision['f1'],
        'best_epoch': best_epoch,
        'best_val_loss': best_val_loss,
    })

    log_curves_and_metrics(y_val, y_prob_val, val_metrics_f1, fold_idx, writer, tag='Val_BestF1')
    log_curves_and_metrics(y_val, y_prob_val, val_metrics_precision, fold_idx, writer, tag='Val_BestPrecision')
    log_curves_and_metrics(y_test, y_prob_test, test_metrics_f1, fold_idx, writer, tag='Test_BestF1')
    log_curves_and_metrics(y_test, y_prob_test, test_metrics_precision, fold_idx, writer, tag='Test_BestPrecision')

    predictions = df_test[
        [
            'drug_row_id',
            'drug_col_id',
            'cell_line_name',
            'study_name',
            'tissue',
            'synergy_loewe',
            'synergy_loewe_bin'
        ]
    ].copy()

    predictions['y_true'] = y_test
    predictions['y_prob'] = y_prob_test
    predictions['y_pred_best_f1'] = (y_prob_test >= best_thr_f1).astype(int)
    predictions['y_pred_best_precision'] = (y_prob_test >= best_thr_precision).astype(int)
    predictions['threshold_best_f1'] = best_thr_f1
    predictions['threshold_best_precision'] = best_thr_precision

    predictions.to_csv(
        os.path.join(output_dir, 'predictions', f'predictions_fold_{fold_idx:02d}.csv'),
        index=False
    )

    if SAVE_MODELS:
        torch.save(
            model.state_dict(),
            os.path.join(output_dir, 'models', f'triple_branch_mlp_fold_{fold_idx:02d}.pt')
        )

    n_train = len(y_train_np)
    n_val = len(y_val)
    n_test = len(y_test)

    split_info = {
        'fold': fold_idx,
        'n_train': n_train,
        'n_val': n_val,
        'n_test': n_test,
        'train_pos': int(np.sum(y_train_np)),
        'val_pos': int(np.sum(y_val)),
        'test_pos': int(np.sum(y_test)),
        'train_pos_pct': 100 * np.mean(y_train_np),
        'val_pos_pct': 100 * np.mean(y_val),
        'test_pos_pct': 100 * np.mean(y_test),
        'baseline_auprc_test': float(np.mean(y_test)),
        'n_heldout_drugs': len(test_set),
        'heldout_drugs': ';'.join(sorted(test_set)),
        'heldout_overlap_train': heldout_overlap_train,
        'best_threshold_f1': best_thr_f1,
        'best_threshold_precision': best_thr_precision,
        'best_epoch': best_epoch,
        'best_val_loss': best_val_loss,
        'n_top_genes': len(top_genes),
    }

    cv_split_info.append(split_info)

    writer.add_text('Split/Info', json.dumps(split_info, indent=2))
    writer.add_text('Genes/TopGenes_First100', ', '.join(top_genes[:100]))

    print(
        f"Fold {fold_idx} | "
        f"Train={n_train} ({100*np.mean(y_train_np):.2f}% pos), "
        f"Val={n_val} ({100*np.mean(y_val):.2f}% pos), "
        f"Test={n_test} ({100*np.mean(y_test):.2f}% pos) | "
        f"BestEpoch={best_epoch} | "
        f"ThrF1={best_thr_f1:.2f}, ThrPrec={best_thr_precision:.2f} | "
        f"AUPRC={test_metrics_f1['auprc']:.4f} "
        f"(base={test_metrics_f1['baseline_auprc']:.4f}) | "
        f"F1={test_metrics_f1['f1']:.4f} | "
        f"Prec={test_metrics_f1['precision']:.4f} | "
        f"Rec={test_metrics_f1['recall']:.4f} | "
        f"MCC={test_metrics_f1['mcc']:.4f}"
    )

    writer.close()
    torch.cuda.empty_cache()

In [ ]:
# ============================================================
# GUARDADO DE RESULTADOS
# ============================================================

df_val_f1 = pd.DataFrame(cv_val_results_f1)
df_val_precision = pd.DataFrame(cv_val_results_precision)
df_test_f1 = pd.DataFrame(cv_test_results_f1)
df_test_precision = pd.DataFrame(cv_test_results_precision)
df_split_info = pd.DataFrame(cv_split_info)
df_best_thresholds = pd.DataFrame(cv_best_thresholds)

df_val_thresholds = pd.concat(cv_val_threshold_metrics, ignore_index=True)
df_test_thresholds = pd.concat(cv_test_threshold_metrics, ignore_index=True)

df_val_f1.to_csv(os.path.join(output_dir, 'val_metrics_best_f1.csv'), index=False)
df_val_precision.to_csv(os.path.join(output_dir, 'val_metrics_best_precision.csv'), index=False)
df_test_f1.to_csv(os.path.join(output_dir, 'test_metrics_best_f1.csv'), index=False)
df_test_precision.to_csv(os.path.join(output_dir, 'test_metrics_best_precision.csv'), index=False)
df_split_info.to_csv(os.path.join(output_dir, 'split_info.csv'), index=False)
df_best_thresholds.to_csv(os.path.join(output_dir, 'best_thresholds.csv'), index=False)
df_val_thresholds.to_csv(os.path.join(output_dir, 'val_metrics_by_threshold.csv'), index=False)
df_test_thresholds.to_csv(os.path.join(output_dir, 'test_metrics_by_threshold.csv'), index=False)

if len(cv_histories) > 0:
    df_all_history = pd.concat(cv_histories, ignore_index=True)
    df_all_history.to_csv(os.path.join(output_dir, 'training_history_all_folds.csv'), index=False)




# ============================================================
# RESUMEN FINAL
# ============================================================

metric_cols = [
    'auc',
    'auprc',
    'baseline_auprc',
    'f1',
    'precision',
    'recall',
    'specificity',
    'balanced_accuracy',
    'mcc',
    'log_loss',
    'brier',
    'tp',
    'tn',
    'fp',
    'fn',
]

summary_f1 = (
    df_test_f1[metric_cols]
    .agg(['mean', 'std'])
    .T
    .reset_index()
    .rename(columns={'index': 'metric'})
)

summary_precision = (
    df_test_precision[metric_cols]
    .agg(['mean', 'std'])
    .T
    .reset_index()
    .rename(columns={'index': 'metric'})
)

summary_f1.to_csv(os.path.join(output_dir, 'summary_test_best_f1.csv'), index=False)
summary_precision.to_csv(os.path.join(output_dir, 'summary_test_best_precision.csv'), index=False)

print('\n' + '=' * 70)
print('RESULTADOS DRUG PAIR TRANSFORMER MLP - LODO PARCIAL CV=10')
print('=' * 70)

print('\n[TEST - threshold elegido por mejor F1 en validacion]')
print(summary_f1)

print('\n[TEST - threshold conservador por mejor precision en validacion]')
print(summary_precision)

print('\nArchivos guardados en:', output_dir)


## TRANSFORMER + MLP Stratified K-Fold CV=10

In [ ]:
import os
import json
import copy
import shutil
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import StratifiedKFold
from torch import nn
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
from PIL import Image

if not hasattr(Image, "Resampling"):
    class Resampling:
        LANCZOS = Image.ANTIALIAS

    Image.Resampling = Resampling


# ============================================================
# CONFIGURACION
# ============================================================

SAVE_MODELS = False

output_dir = 'results/drug_pair_transformer_mlp_stratified_cv'
tb_dir = os.path.join(output_dir, 'tensorboard')

if os.path.exists(output_dir):
    shutil.rmtree(output_dir)

os.makedirs(output_dir, exist_ok=True)
os.makedirs(os.path.join(output_dir, 'predictions'), exist_ok=True)
os.makedirs(os.path.join(output_dir, 'figures'), exist_ok=True)
os.makedirs(tb_dir, exist_ok=True)

if SAVE_MODELS:
    os.makedirs(os.path.join(output_dir, 'models'), exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

mlp_config = {
    'lr': 1e-3,
    'batch_size': 256,
    'epochs': 80,
    'patience': 12,
    'weight_decay': 1e-4,
    'drug_dim': 1024,
    'n_genes': 500,
    'random_state': 42,
    'use_pos_weight': True,
    'hidden_dim': 256,
    'embed_dim': 128,
    'n_heads': 4,
    'n_layers': 2,
}


experiment_config = {
    'model': 'DrugPairTransformerMLP',
    'architecture': (
        'Transformer encoder over [CLS, drug1, drug2] tokens fused with gene expression encoder.'
    ),
    'split_type': 'stratified_kfold',
    'n_splits': 10,
    'n_genes': mlp_config['n_genes'],
    'random_state': mlp_config['random_state'],
    'threshold_grid_min': 0.05,
    'threshold_grid_max': 0.95,
    'threshold_grid_step': 0.01,
    'mlp_config': mlp_config,
    'save_models': SAVE_MODELS,
    'device': str(device),
    'external_split_indices_path': 'results/shared_splits/stratified_kfold_indices.json',
    'stratified_definition': (
        'Random Stratified CV: each fold is a random sample stratified by the target variable.'
    )
}


save_experiment_config(experiment_config, output_dir)




# ============================================================
# DATASET BASE
# ============================================================

base_df = X_final_df.copy()
base_df['drug_row_id'] = base_df['drug_row_id'].astype(str)
base_df['drug_col_id'] = base_df['drug_col_id'].astype(str)
base_df['synergy_loewe_bin'] = base_df['synergy_loewe_bin'].replace(-1, 0).astype(int)

thresholds = np.round(np.arange(0.05, 0.951, 0.01), 2)


shared_splits_dir = 'results/shared_splits'
os.makedirs(shared_splits_dir, exist_ok=True)

skf_splits_path = os.path.join(shared_splits_dir, 'stratified_kfold_indices.json')

if os.path.exists(skf_splits_path):
    with open(skf_splits_path, 'r') as f:
        split_indices = json.load(f)

    print('Splits StratifiedKFold cargados desde:', skf_splits_path)

else:
    skf = StratifiedKFold(
        n_splits=experiment_config['n_splits'],
        shuffle=True,
        random_state=experiment_config['random_state']
    )

    split_indices = []

    for fold_idx, (train_idx, test_idx) in enumerate(
        skf.split(base_df, base_df['synergy_loewe_bin']),
        start=1
    ):
        split_indices.append({
            'fold': fold_idx,
            'train_idx': train_idx.tolist(),
            'test_idx': test_idx.tolist()
        })

    with open(skf_splits_path, 'w') as f:
        json.dump(split_indices, f, indent=4)

    print('Splits StratifiedKFold creados y guardados en:', skf_splits_path)


In [ ]:
# ============================================================
# BUCLE CV
# ============================================================

cv_val_results_f1 = []
cv_val_results_precision = []
cv_test_results_f1 = []
cv_test_results_precision = []
cv_split_info = []
cv_val_threshold_metrics = []
cv_test_threshold_metrics = []
cv_best_thresholds = []
cv_histories = []


for split in split_indices:

    fold_idx = split['fold']
    train_idx = np.array(split['train_idx'])
    test_idx = np.array(split['test_idx'])
    writer = SummaryWriter(log_dir=os.path.join(tb_dir, f'Fold_{fold_idx}'))

    print(f'\nPROCESANDO FOLD {fold_idx}/{experiment_config["n_splits"]} - TTRANSFORMER + MLP STRATIFIED CV')

    df_train_raw = base_df.iloc[train_idx].reset_index(drop=True)
    df_test_raw = base_df.iloc[test_idx].reset_index(drop=True)

    if len(df_train_raw) == 0 or len(df_test_raw) == 0:
        print(f'Fold {fold_idx}: sin datos suficientes, se omite.')
        writer.close()
        continue

    train_drugs = set(pd.concat([
        df_train_raw['drug_row_id'],
        df_train_raw['drug_col_id']
    ]).astype(str))

    test_drugs = set(pd.concat([
        df_test_raw['drug_row_id'],
        df_test_raw['drug_col_id']
    ]).astype(str))

    drug_overlap_train_test = len(train_drugs & test_drugs)
    drug_overlap_train_test_pct = (
        100 * drug_overlap_train_test / len(test_drugs)
        if len(test_drugs) > 0 else 0.0
    )

    df_train, df_val, df_test, top_genes = prepare_fold_data_no_leakage(
        df_train_raw,
        df_test_raw,
        data_expr,
        n_genes=experiment_config['n_genes'],
        random_state=experiment_config['random_state']
    )

    y_train_np = df_train['synergy_loewe_bin'].astype(int).to_numpy()

    if len(np.unique(y_train_np)) < 2:
        print(f'Fold {fold_idx}: train con una sola clase, se omite.')
        writer.close()
        continue

    train_loader = DataLoader(
        SynergyDataset(df_train, top_genes),
        batch_size=mlp_config['batch_size'],
        shuffle=True,

    )

    val_loader = DataLoader(
        SynergyDataset(df_val, top_genes),
        batch_size=mlp_config['batch_size'],
        shuffle=False
    )

    test_loader = DataLoader(
        SynergyDataset(df_test, top_genes),
        batch_size=mlp_config['batch_size'],
        shuffle=False
    )

    model = DrugPairTransformerMLP(
        hiddim=mlp_config['hidden_dim'],
        n_genes=experiment_config['n_genes'],
        drug_dim=mlp_config['drug_dim'],
        embed_dim=mlp_config['embed_dim'],
        n_heads=mlp_config['n_heads'],
        n_layers=mlp_config['n_layers']
    ).to(device)


    model, history_df, best_epoch, best_val_loss = train_torch_model_fold(
        model,
        train_loader,
        val_loader,
        y_train_np,
        mlp_config,
        device,
        writer,
        fold_idx
    )

    history_df.to_csv(
        os.path.join(output_dir, f'training_history_fold_{fold_idx:02d}.csv'),
        index=False
    )
    cv_histories.append(history_df)

    y_val, y_prob_val = predict_torch_model(model, val_loader, device)
    y_test, y_prob_test = predict_torch_model(model, test_loader, device)

    df_val_thr, best_thr_f1, best_thr_precision = find_best_thresholds(
        y_val,
        y_prob_val,
        thresholds
    )

    df_val_thr.insert(0, 'fold', fold_idx)
    df_val_thr.insert(1, 'split', 'val')
    df_val_thr.insert(2, 'threshold_type', 'grid')
    cv_val_threshold_metrics.append(df_val_thr)

    test_thr_rows = []
    for thr in thresholds:
        m_test_thr = compute_metrics_at_threshold(y_test, y_prob_test, thr)
        m_test_thr.update({
            'fold': fold_idx,
            'split': 'test',
            'threshold_type': 'grid',
        })
        test_thr_rows.append(m_test_thr)

    df_test_thr = pd.DataFrame(test_thr_rows)
    cv_test_threshold_metrics.append(df_test_thr)

    val_metrics_f1 = compute_metrics_at_threshold(y_val, y_prob_val, best_thr_f1)
    val_metrics_precision = compute_metrics_at_threshold(y_val, y_prob_val, best_thr_precision)

    test_metrics_f1 = compute_metrics_at_threshold(y_test, y_prob_test, best_thr_f1)
    test_metrics_precision = compute_metrics_at_threshold(y_test, y_prob_test, best_thr_precision)

    val_metrics_f1.update({'fold': fold_idx, 'split': 'val', 'threshold_type': 'best_f1'})
    val_metrics_precision.update({'fold': fold_idx, 'split': 'val', 'threshold_type': 'best_precision'})
    test_metrics_f1.update({'fold': fold_idx, 'split': 'test', 'threshold_type': 'best_f1'})
    test_metrics_precision.update({'fold': fold_idx, 'split': 'test', 'threshold_type': 'best_precision'})

    cv_val_results_f1.append(val_metrics_f1)
    cv_val_results_precision.append(val_metrics_precision)
    cv_test_results_f1.append(test_metrics_f1)
    cv_test_results_precision.append(test_metrics_precision)

    cv_best_thresholds.append({
        'fold': fold_idx,
        'best_threshold_f1': best_thr_f1,
        'best_threshold_precision': best_thr_precision,
        'val_f1_at_best_f1': val_metrics_f1['f1'],
        'val_precision_at_best_f1': val_metrics_f1['precision'],
        'val_recall_at_best_f1': val_metrics_f1['recall'],
        'val_precision_at_best_precision': val_metrics_precision['precision'],
        'val_recall_at_best_precision': val_metrics_precision['recall'],
        'val_f1_at_best_precision': val_metrics_precision['f1'],
        'best_epoch': best_epoch,
        'best_val_loss': best_val_loss,
    })

    log_curves_and_metrics(y_val, y_prob_val, val_metrics_f1, fold_idx, writer, tag='Val_BestF1')
    log_curves_and_metrics(y_val, y_prob_val, val_metrics_precision, fold_idx, writer, tag='Val_BestPrecision')
    log_curves_and_metrics(y_test, y_prob_test, test_metrics_f1, fold_idx, writer, tag='Test_BestF1')
    log_curves_and_metrics(y_test, y_prob_test, test_metrics_precision, fold_idx, writer, tag='Test_BestPrecision')

    predictions = df_test[
        [
            'drug_row_id',
            'drug_col_id',
            'cell_line_name',
            'study_name',
            'tissue',
            'synergy_loewe',
            'synergy_loewe_bin'
        ]
    ].copy()

    predictions['y_true'] = y_test
    predictions['y_prob'] = y_prob_test
    predictions['y_pred_best_f1'] = (y_prob_test >= best_thr_f1).astype(int)
    predictions['y_pred_best_precision'] = (y_prob_test >= best_thr_precision).astype(int)
    predictions['threshold_best_f1'] = best_thr_f1
    predictions['threshold_best_precision'] = best_thr_precision

    predictions.to_csv(
        os.path.join(output_dir, 'predictions', f'predictions_fold_{fold_idx:02d}.csv'),
        index=False
    )

    if SAVE_MODELS:
        torch.save(
            model.state_dict(),
            os.path.join(output_dir, 'models', f'triple_branch_mlp_fold_{fold_idx:02d}.pt')
        )

    n_train = len(y_train_np)
    n_val = len(y_val)
    n_test = len(y_test)

    split_info = {
        'fold': fold_idx,
        'n_train': n_train,
        'n_val': n_val,
        'n_test': n_test,
        'train_pos': int(np.sum(y_train_np)),
        'val_pos': int(np.sum(y_val)),
        'test_pos': int(np.sum(y_test)),
        'train_pos_pct': 100 * np.mean(y_train_np),
        'val_pos_pct': 100 * np.mean(y_val),
        'test_pos_pct': 100 * np.mean(y_test),
        'baseline_auprc_test': float(np.mean(y_test)),
        'n_train_drugs': len(train_drugs),
        'n_test_drugs': len(test_drugs),
        'drug_overlap_train_test': drug_overlap_train_test,
        'drug_overlap_train_test_pct': drug_overlap_train_test_pct,
        'split_indices_path': skf_splits_path,
        'best_threshold_f1': best_thr_f1,
        'best_threshold_precision': best_thr_precision,
        'best_epoch': best_epoch,
        'best_val_loss': best_val_loss,
        'n_top_genes': len(top_genes),
    }

    cv_split_info.append(split_info)

    writer.add_text('Split/Info', json.dumps(split_info, indent=2))
    writer.add_text('Genes/TopGenes_First100', ', '.join(top_genes[:100]))

    print(
        f"Fold {fold_idx} | "
        f"Train={n_train} ({100*np.mean(y_train_np):.2f}% pos), "
        f"Val={n_val} ({100*np.mean(y_val):.2f}% pos), "
        f"Test={n_test} ({100*np.mean(y_test):.2f}% pos) | "
        f"BestEpoch={best_epoch} | "
        f"ThrF1={best_thr_f1:.2f}, ThrPrec={best_thr_precision:.2f} | "
        f"AUPRC={test_metrics_f1['auprc']:.4f} "
        f"(base={test_metrics_f1['baseline_auprc']:.4f}) | "
        f"F1={test_metrics_f1['f1']:.4f} | "
        f"Prec={test_metrics_f1['precision']:.4f} | "
        f"Rec={test_metrics_f1['recall']:.4f} | "
        f"MCC={test_metrics_f1['mcc']:.4f}"
    )

    writer.close()
    torch.cuda.empty_cache()

In [ ]:
# ============================================================
# GUARDADO DE RESULTADOS
# ============================================================

df_val_f1 = pd.DataFrame(cv_val_results_f1)
df_val_precision = pd.DataFrame(cv_val_results_precision)
df_test_f1 = pd.DataFrame(cv_test_results_f1)
df_test_precision = pd.DataFrame(cv_test_results_precision)
df_split_info = pd.DataFrame(cv_split_info)
df_best_thresholds = pd.DataFrame(cv_best_thresholds)

df_val_thresholds = pd.concat(cv_val_threshold_metrics, ignore_index=True)
df_test_thresholds = pd.concat(cv_test_threshold_metrics, ignore_index=True)

df_val_f1.to_csv(os.path.join(output_dir, 'val_metrics_best_f1.csv'), index=False)
df_val_precision.to_csv(os.path.join(output_dir, 'val_metrics_best_precision.csv'), index=False)
df_test_f1.to_csv(os.path.join(output_dir, 'test_metrics_best_f1.csv'), index=False)
df_test_precision.to_csv(os.path.join(output_dir, 'test_metrics_best_precision.csv'), index=False)
df_split_info.to_csv(os.path.join(output_dir, 'split_info.csv'), index=False)
df_best_thresholds.to_csv(os.path.join(output_dir, 'best_thresholds.csv'), index=False)
df_val_thresholds.to_csv(os.path.join(output_dir, 'val_metrics_by_threshold.csv'), index=False)
df_test_thresholds.to_csv(os.path.join(output_dir, 'test_metrics_by_threshold.csv'), index=False)

if len(cv_histories) > 0:
    df_all_history = pd.concat(cv_histories, ignore_index=True)
    df_all_history.to_csv(os.path.join(output_dir, 'training_history_all_folds.csv'), index=False)


# ============================================================
# RESUMEN FINAL
# ============================================================

metric_cols = [
    'auc',
    'auprc',
    'baseline_auprc',
    'f1',
    'precision',
    'recall',
    'specificity',
    'balanced_accuracy',
    'mcc',
    'log_loss',
    'brier',
    'tp',
    'tn',
    'fp',
    'fn',
]

summary_f1 = (
    df_test_f1[metric_cols]
    .agg(['mean', 'std'])
    .T
    .reset_index()
    .rename(columns={'index': 'metric'})
)

summary_precision = (
    df_test_precision[metric_cols]
    .agg(['mean', 'std'])
    .T
    .reset_index()
    .rename(columns={'index': 'metric'})
)

summary_f1.to_csv(os.path.join(output_dir, 'summary_test_best_f1.csv'), index=False)
summary_precision.to_csv(os.path.join(output_dir, 'summary_test_best_precision.csv'), index=False)

print('\n' + '=' * 70)
print('RESULTADOS DRUG PAIR TRANSFORMER MLP - LODO PARCIAL CV=10')
print('=' * 70)

print('\n[TEST - threshold elegido por mejor F1 en validacion]')
print(summary_f1)

print('\n[TEST - threshold conservador por mejor precision en validacion]')
print(summary_precision)

print('\nArchivos guardados en:', output_dir)


## GNNs : Leave-Drug-Out CV=10



In [ ]:
from rdkit import Chem
import torch.nn.functional as F
from torch_geometric.data import Data, Batch
from torch_geometric.nn import GINConv, global_mean_pool, global_max_pool


In [ ]:
def build_id_to_smiles(data_smiles_reduced):
    lookup = data_smiles_reduced[['drugbank_id', 'cid', 'isomeric_smiles']].copy()

    id_to_smiles = {}

    for _, row in lookup.iterrows():
        smiles = row['isomeric_smiles']

        if pd.isna(smiles):
            continue

        if pd.notna(row['drugbank_id']):
            id_to_smiles[str(row['drugbank_id']).strip()] = smiles

        if pd.notna(row['cid']):
            id_to_smiles[str(row['cid']).strip()] = smiles

    return id_to_smiles


id_to_smiles = build_id_to_smiles(data_smiles_reduced)

X_graph_df = X_final_df.copy()
X_graph_df['smiles_row'] = X_graph_df['drug_row_id'].astype(str).map(id_to_smiles)
X_graph_df['smiles_col'] = X_graph_df['drug_col_id'].astype(str).map(id_to_smiles)

print(X_graph_df[['smiles_row', 'smiles_col']].isna().sum())

X_graph_df = X_graph_df.dropna(subset=['smiles_row', 'smiles_col']).reset_index(drop=True)


Pasamos los Smiles a grafo:

In [ ]:
ATOM_LIST = [
    'C', 'N', 'O', 'S', 'F', 'P', 'Cl', 'Br', 'I',
    'B', 'Si', 'Se', 'other'
]

def atom_features(atom):
    symbol = atom.GetSymbol()
    symbol_idx = ATOM_LIST.index(symbol) if symbol in ATOM_LIST else ATOM_LIST.index('other')

    features = []

    features += [1 if i == symbol_idx else 0 for i in range(len(ATOM_LIST))]
    features += [
        atom.GetDegree(),
        atom.GetFormalCharge(),
        int(atom.GetIsAromatic()),
        int(atom.GetHybridization()),
        atom.GetTotalNumHs()
    ]

    return features


def smiles_to_graph(smiles):
    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        return None

    x = torch.tensor(
        [atom_features(atom) for atom in mol.GetAtoms()],
        dtype=torch.float32
    )

    edge_index = []

    for bond in mol.GetBonds():
        i = bond.GetBeginAtomIdx()
        j = bond.GetEndAtomIdx()

        edge_index.append([i, j])
        edge_index.append([j, i])

    if len(edge_index) == 0:
        edge_index = torch.empty((2, 0), dtype=torch.long)
    else:
        edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()

    return Data(x=x, edge_index=edge_index)


Dataset GNN:

In [ ]:
class GraphSynergyDataset(Dataset):
    def __init__(self, df, gene_cols, graph_cache):
        self.df = df.reset_index(drop=True)
        self.gene_cols = gene_cols
        self.graph_cache = graph_cache

        self.genes = torch.tensor(
            self.df[gene_cols].values,
            dtype=torch.float32
        )

        self.y = torch.tensor(
            self.df['synergy_loewe_bin'].values,
            dtype=torch.float32
        ).view(-1, 1)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        smiles1 = self.df.loc[idx, 'smiles_row']
        smiles2 = self.df.loc[idx, 'smiles_col']

        g1 = self.graph_cache[smiles1]
        g2 = self.graph_cache[smiles2]

        genes = self.genes[idx]
        y = self.y[idx]

        return g1, g2, genes, y


In [ ]:
def graph_collate_fn(batch):
    g1_list, g2_list, genes, y = zip(*batch)

    batch_g1 = Batch.from_data_list(g1_list)
    batch_g2 = Batch.from_data_list(g2_list)

    genes = torch.stack(genes)
    y = torch.stack(y)

    return batch_g1, batch_g2, genes, y


Modelo GNN

In [ ]:
class GINDrugEncoder(nn.Module):
    def __init__(self, node_dim, hidden_dim=128, out_dim=128, dropout=0.2):
        super().__init__()

        self.conv1 = GINConv(
            nn.Sequential(
                nn.Linear(node_dim, hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, hidden_dim)
            )
        )

        self.conv2 = GINConv(
            nn.Sequential(
                nn.Linear(hidden_dim, hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, out_dim)
            )
        )

        self.bn1 = nn.BatchNorm1d(hidden_dim)
        self.bn2 = nn.BatchNorm1d(out_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch

        x = self.conv1(x, edge_index)
        x = self.bn1(x)
        x = F.relu(x)
        x = self.dropout(x)

        x = self.conv2(x, edge_index)
        x = self.bn2(x)
        x = F.relu(x)

        mean_pool = global_mean_pool(x, batch)
        max_pool = global_max_pool(x, batch)

        return torch.cat([mean_pool, max_pool], dim=1)


In [ ]:
class GraphDrugSynergyModel(nn.Module):
    def __init__(self, node_dim, gene_dim=500, drug_emb_dim=256, gene_emb_dim=128, hidden_dim=256):
        super().__init__()

        self.drug_encoder = GINDrugEncoder(
            node_dim=node_dim,
            hidden_dim=128,
            out_dim=128,
            dropout=0.2
        )

        self.gene_encoder = nn.Sequential(
            nn.Linear(gene_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, gene_emb_dim),
            nn.ReLU()
        )

        fusion_dim = drug_emb_dim * 4 + gene_emb_dim
        # d1, d2, abs(d1-d2), d1*d2, gene

        self.classifier = nn.Sequential(
            nn.Linear(fusion_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, g1, g2, genes):
        z1 = self.drug_encoder(g1)
        z2 = self.drug_encoder(g2)
        zg = self.gene_encoder(genes)

        z = torch.cat([
            z1,
            z2,
            torch.abs(z1 - z2),
            z1 * z2,
            zg
        ], dim=1)

        return self.classifier(z)


In [ ]:
def run_torch_epoch_graph(model, loader, criterion, optimizer=None, device='cpu'):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss = 0.0
    y_true_all = []
    y_prob_all = []

    context = torch.enable_grad() if is_train else torch.no_grad()

    with context:
        for g1, g2, genes, y in loader:
            g1 = g1.to(device)
            g2 = g2.to(device)
            genes = genes.to(device)
            y = y.to(device)

            if is_train:
                optimizer.zero_grad()

            logits = model(g1, g2, genes)
            loss = criterion(logits, y)

            if is_train:
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * y.size(0)

            y_true_all.extend(y.detach().cpu().numpy().ravel())
            y_prob_all.extend(torch.sigmoid(logits).detach().cpu().numpy().ravel())

    return total_loss / len(loader.dataset), np.asarray(y_true_all), np.asarray(y_prob_all)


In [ ]:
def train_torch_graph_model_fold(model, train_loader, val_loader, y_train, config, device, writer, fold_idx):
    n_pos = float(np.sum(y_train))
    n_neg = float(len(y_train) - n_pos)

    if config.get('use_pos_weight', True) and n_pos > 0:
        pos_weight = torch.tensor([n_neg / n_pos], dtype=torch.float32).to(device)
        criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    else:
        criterion = nn.BCEWithLogitsLoss()

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config['lr'],
        weight_decay=config['weight_decay']
    )

    best_val_loss = np.inf
    best_model_state = copy.deepcopy(model.state_dict())
    best_epoch = 0
    epochs_no_improve = 0
    history = []

    for epoch in range(1, config['epochs'] + 1):
        train_loss, _, _ = run_torch_epoch_graph(
            model, train_loader, criterion, optimizer=optimizer, device=device
        )

        val_loss, y_true_val, y_prob_val = run_torch_epoch_graph(
            model, val_loader, criterion, optimizer=None, device=device
        )

        val_auprc = compute_metrics_at_threshold(y_true_val, y_prob_val, threshold=0.5)['auprc']

        writer.add_scalar('Loss/Train', train_loss, epoch)
        writer.add_scalar('Loss/Val', val_loss, epoch)
        writer.add_scalar('Metrics/Val_AUPRC_thr05', val_auprc, epoch)

        history.append({
            'fold': fold_idx,
            'epoch': epoch,
            'train_loss': train_loss,
            'val_loss': val_loss,
            'val_auprc_thr05': val_auprc,
        })

        print(
            f"Fold {fold_idx} | Epoch {epoch:03d}/{config['epochs']} | "
            f"TrainLoss={train_loss:.4f} | ValLoss={val_loss:.4f} | "
            f"ValAUPRC={val_auprc:.4f}",
            end='\r'
        )

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model_state = copy.deepcopy(model.state_dict())
            best_epoch = epoch
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= config['patience']:
                break

    print()
    model.load_state_dict(best_model_state)

    return model, pd.DataFrame(history), best_epoch, best_val_loss


In [ ]:
def predict_torch_graph_model(model, loader, device):
    model.eval()

    y_true_all = []
    y_prob_all = []

    with torch.no_grad():
        for g1, g2, genes, y in loader:
            g1 = g1.to(device)
            g2 = g2.to(device)
            genes = genes.to(device)

            logits = model(g1, g2, genes)
            probs = torch.sigmoid(logits).cpu().numpy().ravel()

            y_prob_all.extend(probs)
            y_true_all.extend(y.numpy().ravel())

    return np.asarray(y_true_all), np.asarray(y_prob_all)


In [ ]:
import os
import json
import copy
import shutil
import numpy as np
import pandas as pd
import torch

from torch import nn
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
from PIL import Image

if not hasattr(Image, "Resampling"):
    class Resampling:
        LANCZOS = Image.ANTIALIAS

    Image.Resampling = Resampling


# ============================================================
# CONFIGURACION
# ============================================================

SAVE_MODELS = False

output_dir = 'results/gnn_drug_synergy_lodo'
tb_dir = os.path.join(output_dir, 'tensorboard')

if os.path.exists(output_dir):
    shutil.rmtree(output_dir)

os.makedirs(output_dir, exist_ok=True)
os.makedirs(os.path.join(output_dir, 'predictions'), exist_ok=True)
os.makedirs(os.path.join(output_dir, 'figures'), exist_ok=True)
os.makedirs(tb_dir, exist_ok=True)

if SAVE_MODELS:
    os.makedirs(os.path.join(output_dir, 'models'), exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

mlp_config = {
    'lr': 1e-3,
    'batch_size': 256,
    'epochs': 80,
    'patience': 12,
    'weight_decay': 1e-4,
    'n_genes': 500,
    'random_state': 42,
    'use_pos_weight': True,
    'hidden_dim': 256
}


experiment_config = {
    'model': 'GraphDrugSynergyModel',
    'architecture': (
        'GIN-based molecular graph encoder for each drug, fused with selected gene expression.'
    ),
    'split_type': 'partial_lodo_optimized',
    'n_splits': 10,
    'n_genes': mlp_config['n_genes'],
    'random_state': mlp_config['random_state'],
    'threshold_grid_min': 0.05,
    'threshold_grid_max': 0.95,
    'threshold_grid_step': 0.01,
    'mlp_config': mlp_config,
    'save_models': SAVE_MODELS,
    'device': str(device),
    'external_lodo_splits_path': 'results/shared_splits/lodo_fold_sets.json',
}



save_experiment_config(experiment_config, output_dir)


In [ ]:


# ============================================================
# DATASET BASE
# ============================================================

base_df = X_graph_df.copy()
base_df['drug_row_id'] = base_df['drug_row_id'].astype(str)
base_df['drug_col_id'] = base_df['drug_col_id'].astype(str)
base_df['synergy_loewe_bin'] = base_df['synergy_loewe_bin'].replace(-1, 0).astype(int)

print(base_df[['smiles_row', 'smiles_col']].isna().sum())
unique_smiles = pd.concat([
    base_df['smiles_row'],
    base_df['smiles_col']
]).dropna().unique()

graph_cache = {
    smiles: smiles_to_graph(smiles)
    for smiles in unique_smiles
}

graph_cache = {
    smiles: graph
    for smiles, graph in graph_cache.items()
    if graph is not None
}

print("Grafos validos:", len(graph_cache), "de", len(unique_smiles))

valid_smiles = set(graph_cache.keys())

before = len(base_df)

base_df = base_df[
    base_df['smiles_row'].isin(valid_smiles)
    & base_df['smiles_col'].isin(valid_smiles)
].reset_index(drop=True)

print("Filas eliminadas por grafos invalidos:", before - len(base_df))

thresholds = np.round(np.arange(0.05, 0.951, 0.01), 2)

shared_splits_path = experiment_config['external_lodo_splits_path']

with open(shared_splits_path, 'r') as f:
    fold_sets_dict = json.load(f)

fold_sets = [
    set(fold_sets_dict[str(i + 1)])
    for i in range(len(fold_sets_dict))
]

assert len(fold_sets) == experiment_config['n_splits']

partition_rows = []


for fold_idx, fold_set in enumerate(fold_sets, start=1):
    mask_test = (
        base_df['drug_row_id'].isin(fold_set)
        | base_df['drug_col_id'].isin(fold_set)
    )

    fold_df = base_df[mask_test]

    n_examples = len(fold_df)
    pos_rate = fold_df['synergy_loewe_bin'].mean() if n_examples > 0 else np.nan

    partition_rows.append({
        'fold': fold_idx,
        'n_examples_induced_test': n_examples,
        'positive_pct_induced_test': pos_rate * 100,
        'baseline_auprc_induced_test': pos_rate,
        'n_heldout_drugs': len(fold_set)
    })

fold_partition_summary = pd.DataFrame(partition_rows)
fold_partition_summary.to_csv(
    os.path.join(output_dir, 'lodo_partition_summary.csv'),
    index=False
)

print('\nSplits LODO compartidos cargados desde:', shared_splits_path)
print(fold_partition_summary)


# ============================================================
# SOLAPAMIENTO TEST
# ============================================================

test_overlap_info, overlap_df, overlap_counts = summarize_test_overlap(
    base_df,
    fold_sets,
    output_path=output_dir
)

print('\nSolapamiento entre conjuntos test:')
print(test_overlap_info)
print('\nDistribucion de apariciones por muestra en test:')
print(overlap_counts)

In [ ]:
import warnings

warnings.filterwarnings(
    "ignore",
    message=".*torch-scatter.*",
    category=UserWarning
)
# ============================================================
# BUCLE CV
# ============================================================

cv_val_results_f1 = []
cv_val_results_precision = []
cv_test_results_f1 = []
cv_test_results_precision = []
cv_split_info = []
cv_val_threshold_metrics = []
cv_test_threshold_metrics = []
cv_best_thresholds = []
cv_histories = []


unique_smiles = pd.concat([
    base_df['smiles_row'],
    base_df['smiles_col']
]).dropna().unique()


print(base_df[['smiles_row', 'smiles_col']].isna().sum())

graph_cache = {
    smiles: smiles_to_graph(smiles)
    for smiles in unique_smiles
}

graph_cache = {
    smiles: graph
    for smiles, graph in graph_cache.items()
    if graph is not None
}
print("Grafos validos:", len(graph_cache), "de", len(unique_smiles))

valid_smiles = set(graph_cache.keys())

before = len(base_df)

base_df = base_df[
    base_df['smiles_row'].isin(valid_smiles)
    & base_df['smiles_col'].isin(valid_smiles)
].reset_index(drop=True)

print("Filas eliminadas por grafos invalidos:", before - len(base_df))


for fold_idx, test_set in enumerate(fold_sets, start=1):
    writer = SummaryWriter(log_dir=os.path.join(tb_dir, f'Fold_{fold_idx}'))

    print(f'\nPROCESANDO FOLD {fold_idx}/{experiment_config["n_splits"]} - GNN DRUG SYNERGY LODO CV')


    mask_test = (
        base_df['drug_row_id'].isin(test_set)
        | base_df['drug_col_id'].isin(test_set)
    )

    df_test_raw = base_df[mask_test].reset_index(drop=True)
    df_train_raw = base_df[~mask_test].reset_index(drop=True)

    if len(df_train_raw) == 0 or len(df_test_raw) == 0:
        print(f'Fold {fold_idx}: sin datos suficientes, se omite.')
        writer.close()
        continue

    assert (
        df_test_raw['drug_row_id'].isin(test_set)
        | df_test_raw['drug_col_id'].isin(test_set)
    ).all(), f'Fold {fold_idx}: hay muestras test sin droga held-out.'

    train_drugs = set(pd.concat([
        df_train_raw['drug_row_id'],
        df_train_raw['drug_col_id']
    ]).astype(str))

    heldout_overlap_train = len(train_drugs & test_set)
    assert heldout_overlap_train == 0, f'Fold {fold_idx}: drogas held-out aparecen en train.'

    df_train, df_val, df_test, top_genes = prepare_fold_data_no_leakage(
        df_train_raw,
        df_test_raw,
        data_expr,
        n_genes=experiment_config['n_genes'],
        random_state=experiment_config['random_state']
    )

    y_train_np = df_train['synergy_loewe_bin'].astype(int).to_numpy()

    if len(np.unique(y_train_np)) < 2:
        print(f'Fold {fold_idx}: train con una sola clase, se omite.')
        writer.close()
        continue

    train_loader = DataLoader(
        GraphSynergyDataset(df_train, top_genes, graph_cache),
        batch_size=mlp_config['batch_size'],
        shuffle=True,
        collate_fn=graph_collate_fn
    )

    val_loader = DataLoader(
        GraphSynergyDataset(df_val, top_genes, graph_cache),
        batch_size=mlp_config['batch_size'],
        shuffle=False,
        collate_fn=graph_collate_fn
    )

    test_loader = DataLoader(
        GraphSynergyDataset(df_test, top_genes, graph_cache),
        batch_size=mlp_config['batch_size'],
        shuffle=False,
        collate_fn=graph_collate_fn
    )

    node_dim = next(iter(graph_cache.values())).x.shape[1]

    model = GraphDrugSynergyModel(
        node_dim=node_dim,
        gene_dim=experiment_config['n_genes'],
        hidden_dim=mlp_config['hidden_dim']
    ).to(device)
    
    model, history_df, best_epoch, best_val_loss = train_torch_graph_model_fold(
        model,
        train_loader,
        val_loader,
        y_train_np,
        mlp_config,
        device,
        writer,
        fold_idx
    )


    history_df.to_csv(
        os.path.join(output_dir, f'training_history_fold_{fold_idx:02d}.csv'),
        index=False
    )
    cv_histories.append(history_df)

    y_val, y_prob_val = predict_torch_graph_model(model, val_loader, device)
    y_test, y_prob_test = predict_torch_graph_model(model, test_loader, device)

    df_val_thr, best_thr_f1, best_thr_precision = find_best_thresholds(
        y_val,
        y_prob_val,
        thresholds
    )

    df_val_thr.insert(0, 'fold', fold_idx)
    df_val_thr.insert(1, 'split', 'val')
    df_val_thr.insert(2, 'threshold_type', 'grid')
    cv_val_threshold_metrics.append(df_val_thr)

    test_thr_rows = []
    for thr in thresholds:
        m_test_thr = compute_metrics_at_threshold(y_test, y_prob_test, thr)
        m_test_thr.update({
            'fold': fold_idx,
            'split': 'test',
            'threshold_type': 'grid',
        })
        test_thr_rows.append(m_test_thr)

    df_test_thr = pd.DataFrame(test_thr_rows)
    cv_test_threshold_metrics.append(df_test_thr)

    val_metrics_f1 = compute_metrics_at_threshold(y_val, y_prob_val, best_thr_f1)
    val_metrics_precision = compute_metrics_at_threshold(y_val, y_prob_val, best_thr_precision)

    test_metrics_f1 = compute_metrics_at_threshold(y_test, y_prob_test, best_thr_f1)
    test_metrics_precision = compute_metrics_at_threshold(y_test, y_prob_test, best_thr_precision)

    val_metrics_f1.update({'fold': fold_idx, 'split': 'val', 'threshold_type': 'best_f1'})
    val_metrics_precision.update({'fold': fold_idx, 'split': 'val', 'threshold_type': 'best_precision'})
    test_metrics_f1.update({'fold': fold_idx, 'split': 'test', 'threshold_type': 'best_f1'})
    test_metrics_precision.update({'fold': fold_idx, 'split': 'test', 'threshold_type': 'best_precision'})

    cv_val_results_f1.append(val_metrics_f1)
    cv_val_results_precision.append(val_metrics_precision)
    cv_test_results_f1.append(test_metrics_f1)
    cv_test_results_precision.append(test_metrics_precision)

    cv_best_thresholds.append({
        'fold': fold_idx,
        'best_threshold_f1': best_thr_f1,
        'best_threshold_precision': best_thr_precision,
        'val_f1_at_best_f1': val_metrics_f1['f1'],
        'val_precision_at_best_f1': val_metrics_f1['precision'],
        'val_recall_at_best_f1': val_metrics_f1['recall'],
        'val_precision_at_best_precision': val_metrics_precision['precision'],
        'val_recall_at_best_precision': val_metrics_precision['recall'],
        'val_f1_at_best_precision': val_metrics_precision['f1'],
        'best_epoch': best_epoch,
        'best_val_loss': best_val_loss,
    })

    log_curves_and_metrics(y_val, y_prob_val, val_metrics_f1, fold_idx, writer, tag='Val_BestF1')
    log_curves_and_metrics(y_val, y_prob_val, val_metrics_precision, fold_idx, writer, tag='Val_BestPrecision')
    log_curves_and_metrics(y_test, y_prob_test, test_metrics_f1, fold_idx, writer, tag='Test_BestF1')
    log_curves_and_metrics(y_test, y_prob_test, test_metrics_precision, fold_idx, writer, tag='Test_BestPrecision')

    predictions = df_test[
        [
            'drug_row_id',
            'drug_col_id',
            'cell_line_name',
            'study_name',
            'tissue',
            'synergy_loewe',
            'synergy_loewe_bin'
        ]
    ].copy()

    predictions['y_true'] = y_test
    predictions['y_prob'] = y_prob_test
    predictions['y_pred_best_f1'] = (y_prob_test >= best_thr_f1).astype(int)
    predictions['y_pred_best_precision'] = (y_prob_test >= best_thr_precision).astype(int)
    predictions['threshold_best_f1'] = best_thr_f1
    predictions['threshold_best_precision'] = best_thr_precision

    predictions.to_csv(
        os.path.join(output_dir, 'predictions', f'predictions_fold_{fold_idx:02d}.csv'),
        index=False
    )

    if SAVE_MODELS:
        torch.save(
            model.state_dict(),
            os.path.join(output_dir, 'models', f'triple_branch_mlp_fold_{fold_idx:02d}.pt')
        )

    n_train = len(y_train_np)
    n_val = len(y_val)
    n_test = len(y_test)

    split_info = {
        'fold': fold_idx,
        'n_train': n_train,
        'n_val': n_val,
        'n_test': n_test,
        'train_pos': int(np.sum(y_train_np)),
        'val_pos': int(np.sum(y_val)),
        'test_pos': int(np.sum(y_test)),
        'train_pos_pct': 100 * np.mean(y_train_np),
        'val_pos_pct': 100 * np.mean(y_val),
        'test_pos_pct': 100 * np.mean(y_test),
        'baseline_auprc_test': float(np.mean(y_test)),
        'n_heldout_drugs': len(test_set),
        'heldout_drugs': ';'.join(sorted(test_set)),
        'heldout_overlap_train': heldout_overlap_train,
        'best_threshold_f1': best_thr_f1,
        'best_threshold_precision': best_thr_precision,
        'best_epoch': best_epoch,
        'best_val_loss': best_val_loss,
        'n_top_genes': len(top_genes),
    }

    cv_split_info.append(split_info)

    writer.add_text('Split/Info', json.dumps(split_info, indent=2))
    writer.add_text('Genes/TopGenes_First100', ', '.join(top_genes[:100]))

    print(
        f"Fold {fold_idx} | "
        f"Train={n_train} ({100*np.mean(y_train_np):.2f}% pos), "
        f"Val={n_val} ({100*np.mean(y_val):.2f}% pos), "
        f"Test={n_test} ({100*np.mean(y_test):.2f}% pos) | "
        f"BestEpoch={best_epoch} | "
        f"ThrF1={best_thr_f1:.2f}, ThrPrec={best_thr_precision:.2f} | "
        f"AUPRC={test_metrics_f1['auprc']:.4f} "
        f"(base={test_metrics_f1['baseline_auprc']:.4f}) | "
        f"F1={test_metrics_f1['f1']:.4f} | "
        f"Prec={test_metrics_f1['precision']:.4f} | "
        f"Rec={test_metrics_f1['recall']:.4f} | "
        f"MCC={test_metrics_f1['mcc']:.4f}"
    )

    writer.close()
    torch.cuda.empty_cache()

In [ ]:

    
# ============================================================
# GUARDADO DE RESULTADOS
# ============================================================

df_val_f1 = pd.DataFrame(cv_val_results_f1)
df_val_precision = pd.DataFrame(cv_val_results_precision)
df_test_f1 = pd.DataFrame(cv_test_results_f1)
df_test_precision = pd.DataFrame(cv_test_results_precision)
df_split_info = pd.DataFrame(cv_split_info)
df_best_thresholds = pd.DataFrame(cv_best_thresholds)

df_val_thresholds = pd.concat(cv_val_threshold_metrics, ignore_index=True)
df_test_thresholds = pd.concat(cv_test_threshold_metrics, ignore_index=True)

df_val_f1.to_csv(os.path.join(output_dir, 'val_metrics_best_f1.csv'), index=False)
df_val_precision.to_csv(os.path.join(output_dir, 'val_metrics_best_precision.csv'), index=False)
df_test_f1.to_csv(os.path.join(output_dir, 'test_metrics_best_f1.csv'), index=False)
df_test_precision.to_csv(os.path.join(output_dir, 'test_metrics_best_precision.csv'), index=False)
df_split_info.to_csv(os.path.join(output_dir, 'split_info.csv'), index=False)
df_best_thresholds.to_csv(os.path.join(output_dir, 'best_thresholds.csv'), index=False)
df_val_thresholds.to_csv(os.path.join(output_dir, 'val_metrics_by_threshold.csv'), index=False)
df_test_thresholds.to_csv(os.path.join(output_dir, 'test_metrics_by_threshold.csv'), index=False)

if len(cv_histories) > 0:
    df_all_history = pd.concat(cv_histories, ignore_index=True)
    df_all_history.to_csv(os.path.join(output_dir, 'training_history_all_folds.csv'), index=False)




# ============================================================
# RESUMEN FINAL
# ============================================================

metric_cols = [
    'auc',
    'auprc',
    'baseline_auprc',
    'f1',
    'precision',
    'recall',
    'specificity',
    'balanced_accuracy',
    'mcc',
    'log_loss',
    'brier',
    'tp',
    'tn',
    'fp',
    'fn',
]

summary_f1 = (
    df_test_f1[metric_cols]
    .agg(['mean', 'std'])
    .T
    .reset_index()
    .rename(columns={'index': 'metric'})
)

summary_precision = (
    df_test_precision[metric_cols]
    .agg(['mean', 'std'])
    .T
    .reset_index()
    .rename(columns={'index': 'metric'})
)

summary_f1.to_csv(os.path.join(output_dir, 'summary_test_best_f1.csv'), index=False)
summary_precision.to_csv(os.path.join(output_dir, 'summary_test_best_precision.csv'), index=False)

print('\n' + '=' * 70)
print('RESULTADOS GNN DRUG SYNERGY MODEL - LODO PARCIAL CV=10')
print('=' * 70)

print('\n[TEST - threshold elegido por mejor F1 en validacion]')
print(summary_f1)

print('\n[TEST - threshold conservador por mejor precision en validacion]')
print(summary_precision)

print('\nArchivos guardados en:', output_dir)


## GNN: Stratified K-Fold CV=10


In [ ]:
import os
import json
import copy
import shutil
import numpy as np
import pandas as pd
import torch

from torch import nn
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
from PIL import Image

if not hasattr(Image, "Resampling"):
    class Resampling:
        LANCZOS = Image.ANTIALIAS

    Image.Resampling = Resampling


# ============================================================
# CONFIGURACION
# ============================================================

SAVE_MODELS = False

output_dir = 'results/gnn_drug_synergy_stratified_cv'
tb_dir = os.path.join(output_dir, 'tensorboard')

if os.path.exists(output_dir):
    shutil.rmtree(output_dir)

os.makedirs(output_dir, exist_ok=True)
os.makedirs(os.path.join(output_dir, 'predictions'), exist_ok=True)
os.makedirs(os.path.join(output_dir, 'figures'), exist_ok=True)
os.makedirs(tb_dir, exist_ok=True)

if SAVE_MODELS:
    os.makedirs(os.path.join(output_dir, 'models'), exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

mlp_config = {
    'lr': 1e-3,
    'batch_size': 256,
    'epochs': 80,
    'patience': 12,
    'weight_decay': 1e-4,
    'n_genes': 500,
    'random_state': 42,
    'use_pos_weight': True,
    'hidden_dim': 256
}


experiment_config = {
    'model': 'GraphDrugSynergyModel',
    'architecture': (
        'GIN-based molecular graph encoder for each drug, fused with selected gene expression.'
    ),
    'split_type': 'stratified_kfold',
    'n_splits': 10,
    'n_genes': mlp_config['n_genes'],
    'random_state': mlp_config['random_state'],
    'threshold_grid_min': 0.05,
    'threshold_grid_max': 0.95,
    'threshold_grid_step': 0.01,
    'mlp_config': mlp_config,
    'save_models': SAVE_MODELS,
    'device': str(device),
    'external_split_indices_path': 'results/shared_splits/stratified_kfold_indices.json',
}



save_experiment_config(experiment_config, output_dir)


In [ ]:


# ============================================================
# DATASET BASE
# ============================================================

base_df = X_graph_df.copy()
base_df['drug_row_id'] = base_df['drug_row_id'].astype(str)
base_df['drug_col_id'] = base_df['drug_col_id'].astype(str)
base_df['synergy_loewe_bin'] = base_df['synergy_loewe_bin'].replace(-1, 0).astype(int)

print(base_df[['smiles_row', 'smiles_col']].isna().sum())
unique_smiles = pd.concat([
    base_df['smiles_row'],
    base_df['smiles_col']
]).dropna().unique()

graph_cache = {
    smiles: smiles_to_graph(smiles)
    for smiles in unique_smiles
}

graph_cache = {
    smiles: graph
    for smiles, graph in graph_cache.items()
    if graph is not None
}

print("Grafos validos:", len(graph_cache), "de", len(unique_smiles))

valid_smiles = set(graph_cache.keys())

before = len(base_df)

base_df = base_df[
    base_df['smiles_row'].isin(valid_smiles)
    & base_df['smiles_col'].isin(valid_smiles)
].reset_index(drop=True)

print("Filas eliminadas por grafos invalidos:", before - len(base_df))

thresholds = np.round(np.arange(0.05, 0.951, 0.01), 2)

shared_splits_path = experiment_config['external_split_indices_path']

with open(shared_splits_path, 'r') as f:
    split_indices = json.load(f)

assert len(split_indices) == experiment_config['n_splits'], (
    f"Se esperaban {experiment_config['n_splits']} folds, "
    f"pero se cargaron {len(split_indices)}."
)

partition_rows = []

for split in split_indices:
    fold_idx = split['fold']
    test_idx = np.array(split['test_idx'])

    fold_df = base_df.iloc[test_idx]

    n_examples = len(fold_df)
    pos_rate = fold_df['synergy_loewe_bin'].mean() if n_examples > 0 else np.nan

    partition_rows.append({
        'fold': fold_idx,
        'n_examples_test': n_examples,
        'positive_pct_test': pos_rate * 100,
        'baseline_auprc_test': pos_rate
    })

fold_partition_summary = pd.DataFrame(partition_rows)
fold_partition_summary.to_csv(
    os.path.join(output_dir, 'stratified_kfold_partition_summary.csv'),
    index=False
)

print('\nSplits StratifiedKFold cargados desde:', shared_splits_path)
print(fold_partition_summary)


In [ ]:
import warnings

warnings.filterwarnings(
    "ignore",
    message=".*torch-scatter.*",
    category=UserWarning
)
# ============================================================
# BUCLE CV
# ============================================================

cv_val_results_f1 = []
cv_val_results_precision = []
cv_test_results_f1 = []
cv_test_results_precision = []
cv_split_info = []
cv_val_threshold_metrics = []
cv_test_threshold_metrics = []
cv_best_thresholds = []
cv_histories = []


unique_smiles = pd.concat([
    base_df['smiles_row'],
    base_df['smiles_col']
]).dropna().unique()


print(base_df[['smiles_row', 'smiles_col']].isna().sum())

graph_cache = {
    smiles: smiles_to_graph(smiles)
    for smiles in unique_smiles
}

graph_cache = {
    smiles: graph
    for smiles, graph in graph_cache.items()
    if graph is not None
}
print("Grafos validos:", len(graph_cache), "de", len(unique_smiles))

valid_smiles = set(graph_cache.keys())

before = len(base_df)

base_df = base_df[
    base_df['smiles_row'].isin(valid_smiles)
    & base_df['smiles_col'].isin(valid_smiles)
].reset_index(drop=True)

print("Filas eliminadas por grafos invalidos:", before - len(base_df))


for split in split_indices:
    fold_idx = split['fold']
    train_idx = np.array(split['train_idx'])
    test_idx = np.array(split['test_idx'])

    writer = SummaryWriter(log_dir=os.path.join(tb_dir, f'Fold_{fold_idx}'))

    print(f'\nPROCESANDO FOLD {fold_idx}/{experiment_config["n_splits"]} - GNN DRUG SYNERGY STRATIFIED CV')



    df_train_raw = base_df.iloc[train_idx].reset_index(drop=True)
    df_test_raw = base_df.iloc[test_idx].reset_index(drop=True)

    train_drugs = set(pd.concat([
        df_train_raw['drug_row_id'],
        df_train_raw['drug_col_id']
    ]).astype(str))

    test_drugs = set(pd.concat([
        df_test_raw['drug_row_id'],
        df_test_raw['drug_col_id']
    ]).astype(str))

    drug_overlap_train_test = len(train_drugs & test_drugs)
    drug_overlap_train_test_pct = (
        100 * drug_overlap_train_test / len(test_drugs)
        if len(test_drugs) > 0 else 0.0
    )

    if len(df_train_raw) == 0 or len(df_test_raw) == 0:
        print(f'Fold {fold_idx}: sin datos suficientes, se omite.')
        writer.close()
        continue


    df_train, df_val, df_test, top_genes = prepare_fold_data_no_leakage(
        df_train_raw,
        df_test_raw,
        data_expr,
        n_genes=experiment_config['n_genes'],
        random_state=experiment_config['random_state']
    )

    y_train_np = df_train['synergy_loewe_bin'].astype(int).to_numpy()

    if len(np.unique(y_train_np)) < 2:
        print(f'Fold {fold_idx}: train con una sola clase, se omite.')
        writer.close()
        continue

    train_loader = DataLoader(
        GraphSynergyDataset(df_train, top_genes, graph_cache),
        batch_size=mlp_config['batch_size'],
        shuffle=True,
        collate_fn=graph_collate_fn
    )

    val_loader = DataLoader(
        GraphSynergyDataset(df_val, top_genes, graph_cache),
        batch_size=mlp_config['batch_size'],
        shuffle=False,
        collate_fn=graph_collate_fn
    )

    test_loader = DataLoader(
        GraphSynergyDataset(df_test, top_genes, graph_cache),
        batch_size=mlp_config['batch_size'],
        shuffle=False,
        collate_fn=graph_collate_fn
    )

    node_dim = next(iter(graph_cache.values())).x.shape[1]

    model = GraphDrugSynergyModel(
        node_dim=node_dim,
        gene_dim=experiment_config['n_genes'],
        hidden_dim=mlp_config['hidden_dim']
    ).to(device)
    
    model, history_df, best_epoch, best_val_loss = train_torch_graph_model_fold(
        model,
        train_loader,
        val_loader,
        y_train_np,
        mlp_config,
        device,
        writer,
        fold_idx
    )


    history_df.to_csv(
        os.path.join(output_dir, f'training_history_fold_{fold_idx:02d}.csv'),
        index=False
    )
    cv_histories.append(history_df)

    y_val, y_prob_val = predict_torch_graph_model(model, val_loader, device)
    y_test, y_prob_test = predict_torch_graph_model(model, test_loader, device)

    df_val_thr, best_thr_f1, best_thr_precision = find_best_thresholds(
        y_val,
        y_prob_val,
        thresholds
    )

    df_val_thr.insert(0, 'fold', fold_idx)
    df_val_thr.insert(1, 'split', 'val')
    df_val_thr.insert(2, 'threshold_type', 'grid')
    cv_val_threshold_metrics.append(df_val_thr)

    test_thr_rows = []
    for thr in thresholds:
        m_test_thr = compute_metrics_at_threshold(y_test, y_prob_test, thr)
        m_test_thr.update({
            'fold': fold_idx,
            'split': 'test',
            'threshold_type': 'grid',
        })
        test_thr_rows.append(m_test_thr)

    df_test_thr = pd.DataFrame(test_thr_rows)
    cv_test_threshold_metrics.append(df_test_thr)

    val_metrics_f1 = compute_metrics_at_threshold(y_val, y_prob_val, best_thr_f1)
    val_metrics_precision = compute_metrics_at_threshold(y_val, y_prob_val, best_thr_precision)

    test_metrics_f1 = compute_metrics_at_threshold(y_test, y_prob_test, best_thr_f1)
    test_metrics_precision = compute_metrics_at_threshold(y_test, y_prob_test, best_thr_precision)

    val_metrics_f1.update({'fold': fold_idx, 'split': 'val', 'threshold_type': 'best_f1'})
    val_metrics_precision.update({'fold': fold_idx, 'split': 'val', 'threshold_type': 'best_precision'})
    test_metrics_f1.update({'fold': fold_idx, 'split': 'test', 'threshold_type': 'best_f1'})
    test_metrics_precision.update({'fold': fold_idx, 'split': 'test', 'threshold_type': 'best_precision'})

    cv_val_results_f1.append(val_metrics_f1)
    cv_val_results_precision.append(val_metrics_precision)
    cv_test_results_f1.append(test_metrics_f1)
    cv_test_results_precision.append(test_metrics_precision)

    cv_best_thresholds.append({
        'fold': fold_idx,
        'best_threshold_f1': best_thr_f1,
        'best_threshold_precision': best_thr_precision,
        'val_f1_at_best_f1': val_metrics_f1['f1'],
        'val_precision_at_best_f1': val_metrics_f1['precision'],
        'val_recall_at_best_f1': val_metrics_f1['recall'],
        'val_precision_at_best_precision': val_metrics_precision['precision'],
        'val_recall_at_best_precision': val_metrics_precision['recall'],
        'val_f1_at_best_precision': val_metrics_precision['f1'],
        'best_epoch': best_epoch,
        'best_val_loss': best_val_loss,
    })

    log_curves_and_metrics(y_val, y_prob_val, val_metrics_f1, fold_idx, writer, tag='Val_BestF1')
    log_curves_and_metrics(y_val, y_prob_val, val_metrics_precision, fold_idx, writer, tag='Val_BestPrecision')
    log_curves_and_metrics(y_test, y_prob_test, test_metrics_f1, fold_idx, writer, tag='Test_BestF1')
    log_curves_and_metrics(y_test, y_prob_test, test_metrics_precision, fold_idx, writer, tag='Test_BestPrecision')

    predictions = df_test[
        [
            'drug_row_id',
            'drug_col_id',
            'cell_line_name',
            'study_name',
            'tissue',
            'synergy_loewe',
            'synergy_loewe_bin'
        ]
    ].copy()

    predictions['y_true'] = y_test
    predictions['y_prob'] = y_prob_test
    predictions['y_pred_best_f1'] = (y_prob_test >= best_thr_f1).astype(int)
    predictions['y_pred_best_precision'] = (y_prob_test >= best_thr_precision).astype(int)
    predictions['threshold_best_f1'] = best_thr_f1
    predictions['threshold_best_precision'] = best_thr_precision

    predictions.to_csv(
        os.path.join(output_dir, 'predictions', f'predictions_fold_{fold_idx:02d}.csv'),
        index=False
    )

    if SAVE_MODELS:
        torch.save(
            model.state_dict(),
            os.path.join(output_dir, 'models', f'triple_branch_mlp_fold_{fold_idx:02d}.pt')
        )

    n_train = len(y_train_np)
    n_val = len(y_val)
    n_test = len(y_test)

    split_info = {
        'fold': fold_idx,
        'n_train': n_train,
        'n_val': n_val,
        'n_test': n_test,
        'train_pos': int(np.sum(y_train_np)),
        'val_pos': int(np.sum(y_val)),
        'test_pos': int(np.sum(y_test)),
        'train_pos_pct': 100 * np.mean(y_train_np),
        'val_pos_pct': 100 * np.mean(y_val),
        'test_pos_pct': 100 * np.mean(y_test),
        'baseline_auprc_test': float(np.mean(y_test)),
        'n_train_drugs': len(train_drugs),
        'n_test_drugs': len(test_drugs),
        'drug_overlap_train_test': drug_overlap_train_test,
        'drug_overlap_train_test_pct': drug_overlap_train_test_pct,
        'split_indices_path': shared_splits_path,
        'best_threshold_f1': best_thr_f1,
        'best_threshold_precision': best_thr_precision,
        'best_epoch': best_epoch,
        'best_val_loss': best_val_loss,
        'n_top_genes': len(top_genes),
    }

    cv_split_info.append(split_info)

    writer.add_text('Split/Info', json.dumps(split_info, indent=2))
    writer.add_text('Genes/TopGenes_First100', ', '.join(top_genes[:100]))

    print(
        f"Fold {fold_idx} | "
        f"Train={n_train} ({100*np.mean(y_train_np):.2f}% pos), "
        f"Val={n_val} ({100*np.mean(y_val):.2f}% pos), "
        f"Test={n_test} ({100*np.mean(y_test):.2f}% pos) | "
        f"BestEpoch={best_epoch} | "
        f"ThrF1={best_thr_f1:.2f}, ThrPrec={best_thr_precision:.2f} | "
        f"AUPRC={test_metrics_f1['auprc']:.4f} "
        f"(base={test_metrics_f1['baseline_auprc']:.4f}) | "
        f"F1={test_metrics_f1['f1']:.4f} | "
        f"Prec={test_metrics_f1['precision']:.4f} | "
        f"Rec={test_metrics_f1['recall']:.4f} | "
        f"MCC={test_metrics_f1['mcc']:.4f}"
    )

    writer.close()
    torch.cuda.empty_cache()

In [ ]:

    
# ============================================================
# GUARDADO DE RESULTADOS
# ============================================================

df_val_f1 = pd.DataFrame(cv_val_results_f1)
df_val_precision = pd.DataFrame(cv_val_results_precision)
df_test_f1 = pd.DataFrame(cv_test_results_f1)
df_test_precision = pd.DataFrame(cv_test_results_precision)
df_split_info = pd.DataFrame(cv_split_info)
df_best_thresholds = pd.DataFrame(cv_best_thresholds)

df_val_thresholds = pd.concat(cv_val_threshold_metrics, ignore_index=True)
df_test_thresholds = pd.concat(cv_test_threshold_metrics, ignore_index=True)

df_val_f1.to_csv(os.path.join(output_dir, 'val_metrics_best_f1.csv'), index=False)
df_val_precision.to_csv(os.path.join(output_dir, 'val_metrics_best_precision.csv'), index=False)
df_test_f1.to_csv(os.path.join(output_dir, 'test_metrics_best_f1.csv'), index=False)
df_test_precision.to_csv(os.path.join(output_dir, 'test_metrics_best_precision.csv'), index=False)
df_split_info.to_csv(os.path.join(output_dir, 'split_info.csv'), index=False)
df_best_thresholds.to_csv(os.path.join(output_dir, 'best_thresholds.csv'), index=False)
df_val_thresholds.to_csv(os.path.join(output_dir, 'val_metrics_by_threshold.csv'), index=False)
df_test_thresholds.to_csv(os.path.join(output_dir, 'test_metrics_by_threshold.csv'), index=False)

if len(cv_histories) > 0:
    df_all_history = pd.concat(cv_histories, ignore_index=True)
    df_all_history.to_csv(os.path.join(output_dir, 'training_history_all_folds.csv'), index=False)




# ============================================================
# RESUMEN FINAL
# ============================================================

metric_cols = [
    'auc',
    'auprc',
    'baseline_auprc',
    'f1',
    'precision',
    'recall',
    'specificity',
    'balanced_accuracy',
    'mcc',
    'log_loss',
    'brier',
    'tp',
    'tn',
    'fp',
    'fn',
]

summary_f1 = (
    df_test_f1[metric_cols]
    .agg(['mean', 'std'])
    .T
    .reset_index()
    .rename(columns={'index': 'metric'})
)

summary_precision = (
    df_test_precision[metric_cols]
    .agg(['mean', 'std'])
    .T
    .reset_index()
    .rename(columns={'index': 'metric'})
)

summary_f1.to_csv(os.path.join(output_dir, 'summary_test_best_f1.csv'), index=False)
summary_precision.to_csv(os.path.join(output_dir, 'summary_test_best_precision.csv'), index=False)

print('\n' + '=' * 70)
print('RESULTADOS GNN DRUG SYNERGY MODEL - STRATIFIED K-FOLD CV=10')
print('=' * 70)

print('\n[TEST - threshold elegido por mejor F1 en validacion]')
print(summary_f1)

print('\n[TEST - threshold conservador por mejor precision en validacion]')
print(summary_precision)

print('\nArchivos guardados en:', output_dir)


## LDO sin solapamiento. 1 FOLD

Comparación pequeña adicional:

LDO parcial con solapamiento como evaluación principal.
Un holdout LDO único sin solapamiento, aunque sea solo train/test, no 10 folds.

Seleccionar 15-20% de fármacos held-out.
Test = muestras con al menos uno de esos fármacos.
Train = muestras sin esos fármacos.

1. Seleccionas un porcentaje de fármacos held-out, por ejemplo 15% o 20%.
2. Todas las muestras que contienen alguno de esos fármacos van a test.
3. Todas las muestras que no contienen esos fármacos van a train.
4. Los fármacos held-out no aparecen en train.

In [ ]:
def evaluate_lodo_holdout(df, heldout_drugs, target_pos_rate=None, test_size_target=0.2):
    if target_pos_rate is None:
        target_pos_rate = df['synergy_loewe_bin'].mean()

    mask_test = (
        df['drug_row_id'].astype(str).isin(heldout_drugs)
        | df['drug_col_id'].astype(str).isin(heldout_drugs)
    )

    df_test = df[mask_test]
    df_train = df[~mask_test]

    n_total = len(df)
    n_test = len(df_test)
    test_frac = n_test / n_total if n_total > 0 else 0.0
    pos_rate = df_test['synergy_loewe_bin'].mean() if n_test > 0 else 0.0

    size_error = abs(test_frac - test_size_target)
    pos_error = abs(pos_rate - target_pos_rate)

    score = size_error + 2 * pos_error

    return score, n_test, test_frac, pos_rate


def build_lodo_holdout_random_search(
    df,
    drug_test_frac=0.15,
    target_test_frac=0.2,
    n_iter=5000,
    random_state=42
):
    rng = np.random.default_rng(random_state)

    drugs = pd.Index(
        pd.concat([df['drug_row_id'], df['drug_col_id']], ignore_index=True)
        .astype(str)
        .unique()
    ).to_numpy()

    n_test_drugs = max(1, int(round(len(drugs) * drug_test_frac)))

    best_score = np.inf
    best_heldout = None
    best_info = None

    for _ in range(n_iter):
        heldout = set(rng.choice(drugs, size=n_test_drugs, replace=False).tolist())

        score, n_test, test_frac, pos_rate = evaluate_lodo_holdout(
            df,
            heldout,
            target_pos_rate=df['synergy_loewe_bin'].mean(),
            test_size_target=target_test_frac
        )

        if score < best_score:
            best_score = score
            best_heldout = heldout
            best_info = {
                'score': score,
                'n_test': n_test,
                'test_frac': test_frac,
                'positive_pct': 100 * pos_rate,
                'n_heldout_drugs': len(heldout)
            }

    return best_heldout, best_info


In [ ]:
base_df = X_final_df.copy()
base_df['drug_row_id'] = base_df['drug_row_id'].astype(str)
base_df['drug_col_id'] = base_df['drug_col_id'].astype(str)
base_df['synergy_loewe_bin'] = base_df['synergy_loewe_bin'].replace(-1, 0).astype(int)

heldout_drugs, holdout_info = build_lodo_holdout_random_search(
    base_df,
    drug_test_frac=0.15,
    target_test_frac=0.2,
    n_iter=10,
    random_state=42
)

print(holdout_info)
print(sorted(heldout_drugs))


In [ ]:
mask_test = (
    base_df['drug_row_id'].isin(heldout_drugs)
    | base_df['drug_col_id'].isin(heldout_drugs)
)

df_test_raw = base_df[mask_test].reset_index(drop=True)
df_train_raw = base_df[~mask_test].reset_index(drop=True)

train_drugs = set(pd.concat([
    df_train_raw['drug_row_id'],
    df_train_raw['drug_col_id']
]).astype(str))

assert len(train_drugs & heldout_drugs) == 0



In [ ]:
import os
import json
import shutil
import joblib
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from torch.utils.tensorboard import SummaryWriter


# ============================================================
# FUNCIONES HOLDOUT LODO
# ============================================================

def evaluate_lodo_holdout(df, heldout_drugs, target_pos_rate=None, test_size_target=0.2):
    if target_pos_rate is None:
        target_pos_rate = df['synergy_loewe_bin'].mean()

    mask_test = (
        df['drug_row_id'].astype(str).isin(heldout_drugs)
        | df['drug_col_id'].astype(str).isin(heldout_drugs)
    )

    df_test = df[mask_test]
    df_train = df[~mask_test]

    n_total = len(df)
    n_test = len(df_test)
    n_train = len(df_train)

    test_frac = n_test / n_total if n_total > 0 else 0.0
    pos_rate = df_test['synergy_loewe_bin'].mean() if n_test > 0 else 0.0

    size_error = abs(test_frac - test_size_target)
    pos_error = abs(pos_rate - target_pos_rate)

    score = size_error + 2 * pos_error

    return score, {
        'n_train': int(n_train),
        'n_test': int(n_test),
        'test_frac': float(test_frac),
        'positive_pct': float(100 * pos_rate),
        'baseline_auprc_test': float(pos_rate),
        'n_heldout_drugs': int(len(heldout_drugs)),
        'score': float(score),
    }


def build_lodo_holdout_random_search(
    df,
    drug_test_frac=0.15,
    target_test_frac=0.2,
    n_iter=5000,
    random_state=42
):
    rng = np.random.default_rng(random_state)

    drugs = pd.Index(
        pd.concat([df['drug_row_id'], df['drug_col_id']], ignore_index=True)
        .astype(str)
        .unique()
    ).to_numpy()

    n_test_drugs = max(1, int(round(len(drugs) * drug_test_frac)))

    best_score = np.inf
    best_heldout = None
    best_info = None

    target_pos_rate = df['synergy_loewe_bin'].mean()

    for _ in range(n_iter):
        heldout = set(rng.choice(drugs, size=n_test_drugs, replace=False).tolist())

        score, info = evaluate_lodo_holdout(
            df,
            heldout,
            target_pos_rate=target_pos_rate,
            test_size_target=target_test_frac
        )

        if score < best_score:
            best_score = score
            best_heldout = heldout
            best_info = info

    return best_heldout, best_info


In [ ]:
# ============================================================
# CONFIGURACION
# ============================================================

SAVE_MODELS = False

output_dir = 'results/random_forest_holdout'
tb_dir = os.path.join(output_dir, 'tensorboard')

if os.path.exists(output_dir):
    shutil.rmtree(output_dir)

os.makedirs(output_dir, exist_ok=True)
os.makedirs(os.path.join(output_dir, 'predictions'), exist_ok=True)
os.makedirs(os.path.join(output_dir, 'figures'), exist_ok=True)
os.makedirs(tb_dir, exist_ok=True)

if SAVE_MODELS:
    os.makedirs(os.path.join(output_dir, 'models'), exist_ok=True)

rf_config = {
    'n_estimators': 600,
    'max_depth': None,
    'min_samples_split': 2,
    'min_samples_leaf': 1,
    'max_features': 'sqrt',
    'class_weight': 'balanced_subsample',
    'random_state': 42,
    'n_jobs': 4,
}

experiment_config = {
    'model': 'RandomForestClassifier',
    'split_type': 'single_lodo_holdout',
    'n_genes': 500,
    'random_state': 42,
    'drug_test_frac': 0.15,
    'target_test_frac': 0.2,
    'holdout_search_iter': 10,
    'threshold_grid_min': 0.05,
    'threshold_grid_max': 0.95,
    'threshold_grid_step': 0.01,
    'rf_config': rf_config,
    'save_models': SAVE_MODELS,
    'definition': (
        'Single partial Leave-Drug-Out holdout: test contains all samples with at least '
        'one held-out drug; held-out drugs are excluded from train.'
    )
}

save_experiment_config(experiment_config, output_dir)


In [ ]:
# ============================================================
# DATASET BASE Y HOLDOUT
# ============================================================

base_df = X_final_df.copy()
base_df['drug_row_id'] = base_df['drug_row_id'].astype(str)
base_df['drug_col_id'] = base_df['drug_col_id'].astype(str)
base_df['synergy_loewe_bin'] = base_df['synergy_loewe_bin'].replace(-1, 0).astype(int)

thresholds = np.round(np.arange(0.05, 0.951, 0.01), 2)

heldout_drugs, holdout_info = build_lodo_holdout_random_search(
    base_df,
    drug_test_frac=experiment_config['drug_test_frac'],
    target_test_frac=experiment_config['target_test_frac'],
    n_iter=experiment_config['holdout_search_iter'],
    random_state=experiment_config['random_state']
)

print("Holdout info:")
print(holdout_info)

with open(os.path.join(output_dir, 'heldout_drugs.json'), 'w') as f:
    json.dump(sorted(list(heldout_drugs)), f, indent=4)

with open(os.path.join(output_dir, 'holdout_info.json'), 'w') as f:
    json.dump(holdout_info, f, indent=4)

mask_test = (
    base_df['drug_row_id'].isin(heldout_drugs)
    | base_df['drug_col_id'].isin(heldout_drugs)
)

df_test_raw = base_df[mask_test].reset_index(drop=True)
df_train_raw = base_df[~mask_test].reset_index(drop=True)

train_drugs = set(pd.concat([
    df_train_raw['drug_row_id'],
    df_train_raw['drug_col_id']
]).astype(str))

test_drugs = set(pd.concat([
    df_test_raw['drug_row_id'],
    df_test_raw['drug_col_id']
]).astype(str))

heldout_overlap_train = len(train_drugs & heldout_drugs)

assert heldout_overlap_train == 0, 'Hay fármacos held-out en train.'
assert (
    df_test_raw['drug_row_id'].isin(heldout_drugs)
    | df_test_raw['drug_col_id'].isin(heldout_drugs)
).all(), 'Hay muestras test sin fármaco held-out.'

print("Train samples:", len(df_train_raw))
print("Test samples:", len(df_test_raw))
print("Train positive %:", 100 * df_train_raw['synergy_loewe_bin'].mean())
print("Test positive %:", 100 * df_test_raw['synergy_loewe_bin'].mean())
print("Held-out drugs:", len(heldout_drugs))
print("Held-out overlap train:", heldout_overlap_train)


In [ ]:
# ============================================================
# PREPARACION TRAIN/VAL/TEST
# ============================================================

writer = SummaryWriter(log_dir=os.path.join(tb_dir, 'holdout'))

df_train, df_val, df_test, top_genes = prepare_fold_data_no_leakage(
    df_train_raw,
    df_test_raw,
    data_expr,
    n_genes=experiment_config['n_genes'],
    random_state=experiment_config['random_state']
)

X_train, y_train, X_val, y_val, X_test, y_test, feature_cols = build_tabular_matrices(
    df_train,
    df_val,
    df_test
)

print("Train/Val/Test after gene merge:")
print(len(y_train), len(y_val), len(y_test))
print("Pos %:", 100*np.mean(y_train), 100*np.mean(y_val), 100*np.mean(y_test))


In [ ]:
from PIL import Image

if not hasattr(Image, "Resampling"):
    class Resampling:
        LANCZOS = Image.ANTIALIAS

    Image.Resampling = Resampling

# ============================================================
# ENTRENAMIENTO RF
# ============================================================

rf = RandomForestClassifier(**rf_config)

print("Entrenando RandomForest holdout LODO...")
rf.fit(X_train, y_train)
print("Entrenamiento terminado.")

y_prob_val = rf.predict_proba(X_val)[:, 1]
y_prob_test = rf.predict_proba(X_test)[:, 1]

df_val_thr, best_thr_f1, best_thr_precision = find_best_thresholds(
    y_val,
    y_prob_val,
    thresholds
)

test_thr_rows = []

for thr in thresholds:
    m_test_thr = compute_metrics_at_threshold(y_test, y_prob_test, thr)
    m_test_thr.update({
        'split': 'test',
        'threshold_type': 'grid',
    })
    test_thr_rows.append(m_test_thr)

df_test_thr = pd.DataFrame(test_thr_rows)

val_metrics_f1 = compute_metrics_at_threshold(y_val, y_prob_val, best_thr_f1)
val_metrics_precision = compute_metrics_at_threshold(y_val, y_prob_val, best_thr_precision)

test_metrics_f1 = compute_metrics_at_threshold(y_test, y_prob_test, best_thr_f1)
test_metrics_precision = compute_metrics_at_threshold(y_test, y_prob_test, best_thr_precision)

val_metrics_f1.update({'split': 'val', 'threshold_type': 'best_f1'})
val_metrics_precision.update({'split': 'val', 'threshold_type': 'best_precision'})
test_metrics_f1.update({'split': 'test', 'threshold_type': 'best_f1'})
test_metrics_precision.update({'split': 'test', 'threshold_type': 'best_precision'})

log_curves_and_metrics(y_val, y_prob_val, val_metrics_f1, 1, writer, tag='Val_BestF1')
log_curves_and_metrics(y_val, y_prob_val, val_metrics_precision, 1, writer, tag='Val_BestPrecision')
log_curves_and_metrics(y_test, y_prob_test, test_metrics_f1, 1, writer, tag='Test_BestF1')
log_curves_and_metrics(y_test, y_prob_test, test_metrics_precision, 1, writer, tag='Test_BestPrecision')


In [ ]:
# ============================================================
# GUARDADO
# ============================================================

pd.DataFrame([val_metrics_f1]).to_csv(
    os.path.join(output_dir, 'val_metrics_best_f1.csv'),
    index=False
)

pd.DataFrame([val_metrics_precision]).to_csv(
    os.path.join(output_dir, 'val_metrics_best_precision.csv'),
    index=False
)

pd.DataFrame([test_metrics_f1]).to_csv(
    os.path.join(output_dir, 'test_metrics_best_f1.csv'),
    index=False
)

pd.DataFrame([test_metrics_precision]).to_csv(
    os.path.join(output_dir, 'test_metrics_best_precision.csv'),
    index=False
)

df_val_thr.to_csv(os.path.join(output_dir, 'val_metrics_by_threshold.csv'), index=False)
df_test_thr.to_csv(os.path.join(output_dir, 'test_metrics_by_threshold.csv'), index=False)

split_info = {
    **holdout_info,
    'n_train_after_gene_merge': int(len(y_train)),
    'n_val_after_gene_merge': int(len(y_val)),
    'n_test_after_gene_merge': int(len(y_test)),
    'train_pos_pct_after_gene_merge': float(100 * np.mean(y_train)),
    'val_pos_pct_after_gene_merge': float(100 * np.mean(y_val)),
    'test_pos_pct_after_gene_merge': float(100 * np.mean(y_test)),
    'heldout_overlap_train': int(heldout_overlap_train),
    'best_threshold_f1': float(best_thr_f1),
    'best_threshold_precision': float(best_thr_precision),
    'n_features': int(len(feature_cols)),
    'n_top_genes': int(len(top_genes)),
}

pd.DataFrame([split_info]).to_csv(
    os.path.join(output_dir, 'split_info.csv'),
    index=False
)

predictions = df_test[
    [
        'drug_row_id',
        'drug_col_id',
        'cell_line_name',
        'study_name',
        'tissue',
        'synergy_loewe',
        'synergy_loewe_bin'
    ]
].copy()

predictions['y_true'] = y_test
predictions['y_prob'] = y_prob_test
predictions['y_pred_best_f1'] = (y_prob_test >= best_thr_f1).astype(int)
predictions['y_pred_best_precision'] = (y_prob_test >= best_thr_precision).astype(int)
predictions['threshold_best_f1'] = best_thr_f1
predictions['threshold_best_precision'] = best_thr_precision

predictions.to_csv(
    os.path.join(output_dir, 'predictions', 'predictions_holdout.csv'),
    index=False
)

if SAVE_MODELS:
    joblib.dump(
        rf,
        os.path.join(output_dir, 'models', 'random_forest_lodo_holdout.joblib')
    )

writer.close()

print('\nRESULTADOS RF - SINGLE LODO HOLDOUT')
print('\n[Test - Best F1 threshold]')
print(pd.DataFrame([test_metrics_f1]).T)

print('\n[Test - Best Precision threshold]')
print(pd.DataFrame([test_metrics_precision]).T)

print('\nArchivos guardados en:', output_dir)


## REPEATED HOLD OUT

In [ ]:
import json
import shutil
import joblib

from sklearn.ensemble import RandomForestClassifier
from torch.utils.tensorboard import SummaryWriter


# ============================================================
# CONFIGURACION
# ============================================================

SAVE_MODELS = False

output_dir = 'results/random_forest_repeated_lodo_holdout'
tb_dir = os.path.join(output_dir, 'tensorboard')

if os.path.exists(output_dir):
    shutil.rmtree(output_dir)

os.makedirs(output_dir, exist_ok=True)
os.makedirs(os.path.join(output_dir, 'predictions'), exist_ok=True)
os.makedirs(os.path.join(output_dir, 'figures'), exist_ok=True)
os.makedirs(tb_dir, exist_ok=True)

if SAVE_MODELS:
    os.makedirs(os.path.join(output_dir, 'models'), exist_ok=True)

rf_config = {
    'n_estimators': 600,
    'max_depth': None,
    'min_samples_split': 2,
    'min_samples_leaf': 1,
    'max_features': 'sqrt',
    'class_weight': 'balanced_subsample',
    'random_state': 42,
    'n_jobs': 4,
}

experiment_config = {
    'model': 'RandomForestClassifier',
    'split_type': 'repeated_lodo_holdout',
    'n_repeats': 10,
    'n_genes': 500,
    'random_state': 42,
    'drug_test_frac': 0.15,
    'target_test_frac': 0.2,
    'holdout_search_iter': 20,
    'threshold_grid_min': 0.05,
    'threshold_grid_max': 0.95,
    'threshold_grid_step': 0.01,
    'rf_config': rf_config,
    'save_models': SAVE_MODELS,
    'definition': (
        'Repeated partial Leave-Drug-Out holdout. In each repeat, a subset of held-out drugs '
        'is selected; test contains all samples with at least one held-out drug; held-out drugs '
        'are excluded from train.'
    )
}

save_experiment_config(experiment_config, output_dir)


In [ ]:
# ============================================================
# DATASET BASE
# ============================================================

base_df = X_final_df.copy()
base_df['drug_row_id'] = base_df['drug_row_id'].astype(str)
base_df['drug_col_id'] = base_df['drug_col_id'].astype(str)
base_df['synergy_loewe_bin'] = base_df['synergy_loewe_bin'].replace(-1, 0).astype(int)

thresholds = np.round(np.arange(0.05, 0.951, 0.01), 2)


In [ ]:
# ============================================================
# BUCLE REPEATED HOLDOUT
# ============================================================

repeat_val_results_f1 = []
repeat_val_results_precision = []
repeat_test_results_f1 = []
repeat_test_results_precision = []
repeat_split_info = []
repeat_threshold_metrics_val = []
repeat_threshold_metrics_test = []
repeat_best_thresholds = []

for repeat_idx in range(1, experiment_config['n_repeats'] + 1):
    writer = SummaryWriter(log_dir=os.path.join(tb_dir, f'Repeat_{repeat_idx:02d}'))

    repeat_seed = experiment_config['random_state'] + repeat_idx

    print(f'\nPROCESANDO REPEAT {repeat_idx}/{experiment_config["n_repeats"]} - RF REPEATED LODO HOLDOUT')

    heldout_drugs, holdout_info = build_lodo_holdout_random_search(
        base_df,
        drug_test_frac=experiment_config['drug_test_frac'],
        target_test_frac=experiment_config['target_test_frac'],
        n_iter=experiment_config['holdout_search_iter'],
        random_state=repeat_seed
    )

    mask_test = (
        base_df['drug_row_id'].isin(heldout_drugs)
        | base_df['drug_col_id'].isin(heldout_drugs)
    )

    df_test_raw = base_df[mask_test].reset_index(drop=True)
    df_train_raw = base_df[~mask_test].reset_index(drop=True)

    train_drugs = set(pd.concat([
        df_train_raw['drug_row_id'],
        df_train_raw['drug_col_id']
    ]).astype(str))

    heldout_overlap_train = len(train_drugs & heldout_drugs)

    assert heldout_overlap_train == 0, f'Repeat {repeat_idx}: held-out drugs appear in train.'
    assert (
        df_test_raw['drug_row_id'].isin(heldout_drugs)
        | df_test_raw['drug_col_id'].isin(heldout_drugs)
    ).all(), f'Repeat {repeat_idx}: test sample without held-out drug.'

    df_train, df_val, df_test, top_genes = prepare_fold_data_no_leakage(
        df_train_raw,
        df_test_raw,
        data_expr,
        n_genes=experiment_config['n_genes'],
        random_state=repeat_seed
    )

    X_train, y_train, X_val, y_val, X_test, y_test, feature_cols = build_tabular_matrices(
        df_train,
        df_val,
        df_test
    )

    if len(np.unique(y_train)) < 2:
        print(f'Repeat {repeat_idx}: train con una sola clase, se omite.')
        writer.close()
        continue

    rf_config_repeat = rf_config.copy()
    rf_config_repeat['random_state'] = repeat_seed

    rf = RandomForestClassifier(**rf_config_repeat)

    print('Entrenando RandomForest...')
    rf.fit(X_train, y_train)
    print('Entrenamiento terminado.')

    y_prob_val = rf.predict_proba(X_val)[:, 1]
    y_prob_test = rf.predict_proba(X_test)[:, 1]

    df_val_thr, best_thr_f1, best_thr_precision = find_best_thresholds(
        y_val,
        y_prob_val,
        thresholds
    )

    df_val_thr.insert(0, 'repeat', repeat_idx)
    df_val_thr.insert(1, 'split', 'val')
    df_val_thr.insert(2, 'threshold_type', 'grid')
    repeat_threshold_metrics_val.append(df_val_thr)

    test_thr_rows = []
    for thr in thresholds:
        m_test_thr = compute_metrics_at_threshold(y_test, y_prob_test, thr)
        m_test_thr.update({
            'repeat': repeat_idx,
            'split': 'test',
            'threshold_type': 'grid',
        })
        test_thr_rows.append(m_test_thr)

    df_test_thr = pd.DataFrame(test_thr_rows)
    repeat_threshold_metrics_test.append(df_test_thr)

    val_metrics_f1 = compute_metrics_at_threshold(y_val, y_prob_val, best_thr_f1)
    val_metrics_precision = compute_metrics_at_threshold(y_val, y_prob_val, best_thr_precision)

    test_metrics_f1 = compute_metrics_at_threshold(y_test, y_prob_test, best_thr_f1)
    test_metrics_precision = compute_metrics_at_threshold(y_test, y_prob_test, best_thr_precision)

    val_metrics_f1.update({'repeat': repeat_idx, 'split': 'val', 'threshold_type': 'best_f1'})
    val_metrics_precision.update({'repeat': repeat_idx, 'split': 'val', 'threshold_type': 'best_precision'})
    test_metrics_f1.update({'repeat': repeat_idx, 'split': 'test', 'threshold_type': 'best_f1'})
    test_metrics_precision.update({'repeat': repeat_idx, 'split': 'test', 'threshold_type': 'best_precision'})

    repeat_val_results_f1.append(val_metrics_f1)
    repeat_val_results_precision.append(val_metrics_precision)
    repeat_test_results_f1.append(test_metrics_f1)
    repeat_test_results_precision.append(test_metrics_precision)

    repeat_best_thresholds.append({
        'repeat': repeat_idx,
        'best_threshold_f1': best_thr_f1,
        'best_threshold_precision': best_thr_precision,
        'val_f1_at_best_f1': val_metrics_f1['f1'],
        'val_precision_at_best_f1': val_metrics_f1['precision'],
        'val_recall_at_best_f1': val_metrics_f1['recall'],
        'val_precision_at_best_precision': val_metrics_precision['precision'],
        'val_recall_at_best_precision': val_metrics_precision['recall'],
        'val_f1_at_best_precision': val_metrics_precision['f1'],
    })

    log_curves_and_metrics(y_val, y_prob_val, val_metrics_f1, repeat_idx, writer, tag='Val_BestF1')
    log_curves_and_metrics(y_val, y_prob_val, val_metrics_precision, repeat_idx, writer, tag='Val_BestPrecision')
    log_curves_and_metrics(y_test, y_prob_test, test_metrics_f1, repeat_idx, writer, tag='Test_BestF1')
    log_curves_and_metrics(y_test, y_prob_test, test_metrics_precision, repeat_idx, writer, tag='Test_BestPrecision')

    predictions = df_test[
        [
            'drug_row_id',
            'drug_col_id',
            'cell_line_name',
            'study_name',
            'tissue',
            'synergy_loewe',
            'synergy_loewe_bin'
        ]
    ].copy()

    predictions['y_true'] = y_test
    predictions['y_prob'] = y_prob_test
    predictions['y_pred_best_f1'] = (y_prob_test >= best_thr_f1).astype(int)
    predictions['y_pred_best_precision'] = (y_prob_test >= best_thr_precision).astype(int)
    predictions['threshold_best_f1'] = best_thr_f1
    predictions['threshold_best_precision'] = best_thr_precision

    predictions.to_csv(
        os.path.join(output_dir, 'predictions', f'predictions_repeat_{repeat_idx:02d}.csv'),
        index=False
    )

    with open(os.path.join(output_dir, f'heldout_drugs_repeat_{repeat_idx:02d}.json'), 'w') as f:
        json.dump(sorted(list(heldout_drugs)), f, indent=4)

    split_info = {
        'repeat': repeat_idx,
        **holdout_info,
        'n_train_after_gene_merge': int(len(y_train)),
        'n_val_after_gene_merge': int(len(y_val)),
        'n_test_after_gene_merge': int(len(y_test)),
        'train_pos_pct_after_gene_merge': float(100 * np.mean(y_train)),
        'val_pos_pct_after_gene_merge': float(100 * np.mean(y_val)),
        'test_pos_pct_after_gene_merge': float(100 * np.mean(y_test)),
        'heldout_overlap_train': int(heldout_overlap_train),
        'best_threshold_f1': float(best_thr_f1),
        'best_threshold_precision': float(best_thr_precision),
        'n_features': int(len(feature_cols)),
        'n_top_genes': int(len(top_genes)),
        'n_heldout_drugs': len(heldout_drugs),
        'heldout_drugs': ';'.join(sorted(heldout_drugs)),
    }

    repeat_split_info.append(split_info)

    writer.add_text('Split/Info', json.dumps(split_info, indent=2))

    if SAVE_MODELS:
        joblib.dump(
            rf,
            os.path.join(output_dir, 'models', f'random_forest_repeat_{repeat_idx:02d}.joblib')
        )

    print(
        f"Repeat {repeat_idx} | "
        f"Train={len(y_train)} ({100*np.mean(y_train):.2f}% pos), "
        f"Val={len(y_val)} ({100*np.mean(y_val):.2f}% pos), "
        f"Test={len(y_test)} ({100*np.mean(y_test):.2f}% pos) | "
        f"AUPRC={test_metrics_f1['auprc']:.4f} "
        f"(base={test_metrics_f1['baseline_auprc']:.4f}) | "
        f"F1={test_metrics_f1['f1']:.4f} | "
        f"Prec={test_metrics_f1['precision']:.4f} | "
        f"Rec={test_metrics_f1['recall']:.4f}"
    )

    writer.close()


In [ ]:
# ============================================================
# GUARDADO FINAL
# ============================================================

df_val_f1 = pd.DataFrame(repeat_val_results_f1)
df_val_precision = pd.DataFrame(repeat_val_results_precision)
df_test_f1 = pd.DataFrame(repeat_test_results_f1)
df_test_precision = pd.DataFrame(repeat_test_results_precision)
df_split_info = pd.DataFrame(repeat_split_info)
df_best_thresholds = pd.DataFrame(repeat_best_thresholds)

df_val_thresholds = pd.concat(repeat_threshold_metrics_val, ignore_index=True)
df_test_thresholds = pd.concat(repeat_threshold_metrics_test, ignore_index=True)

df_val_f1.to_csv(os.path.join(output_dir, 'val_metrics_best_f1.csv'), index=False)
df_val_precision.to_csv(os.path.join(output_dir, 'val_metrics_best_precision.csv'), index=False)
df_test_f1.to_csv(os.path.join(output_dir, 'test_metrics_best_f1.csv'), index=False)
df_test_precision.to_csv(os.path.join(output_dir, 'test_metrics_best_precision.csv'), index=False)
df_split_info.to_csv(os.path.join(output_dir, 'split_info.csv'), index=False)
df_best_thresholds.to_csv(os.path.join(output_dir, 'best_thresholds.csv'), index=False)
df_val_thresholds.to_csv(os.path.join(output_dir, 'val_metrics_by_threshold.csv'), index=False)
df_test_thresholds.to_csv(os.path.join(output_dir, 'test_metrics_by_threshold.csv'), index=False)

metric_cols = [
    'auc',
    'auprc',
    'baseline_auprc',
    'f1',
    'precision',
    'recall',
    'specificity',
    'balanced_accuracy',
    'mcc',
    'log_loss',
    'brier',
    'tp',
    'tn',
    'fp',
    'fn',
]

summary_f1 = (
    df_test_f1[metric_cols]
    .agg(['mean', 'std'])
    .T
    .reset_index()
    .rename(columns={'index': 'metric'})
)

summary_precision = (
    df_test_precision[metric_cols]
    .agg(['mean', 'std'])
    .T
    .reset_index()
    .rename(columns={'index': 'metric'})
)

summary_f1.to_csv(os.path.join(output_dir, 'summary_test_best_f1.csv'), index=False)
summary_precision.to_csv(os.path.join(output_dir, 'summary_test_best_precision.csv'), index=False)

print('\n' + '=' * 70)
print('RESULTADOS RF - REPEATED LODO HOLDOUT')
print('=' * 70)

print('\n[TEST - threshold elegido por mejor F1 en validacion]')
print(summary_f1)

print('\n[TEST - threshold conservador por mejor precision en validacion]')
print(summary_precision)

print('\nArchivos guardados en:', output_dir)


Ventaja
Cada repetición puede tener:

- test suficientemente grande
- % positivo razonable
- sin fármacos held-out en train


Desventaja
Las repeticiones no son particiones disjuntas, y puede haber muestras repetidas entre tests de distintas repeticiones.

Por tanto, se debe presentar como:
- Repeated Leave-Drug-Out holdout, no como CV estricta.

En cada repetición se construye un holdout Leave-Drug-Out independiente. Las muestras no se excluyen de futuras repeticiones si ya aparecieron en test, por lo que el método evalúa la variabilidad frente a diferentes conjuntos de fármacos held-out, no una partición disjunta del dataset.


### ¿Por que no utilizar la estrategia de no repetir las entradas ya usadas en test?
## COMPROBACION


In [ ]:
def build_random_drug_folds(df, n_splits=10, random_state=42):
    rng = np.random.default_rng(random_state)

    drugs = pd.Index(
        pd.concat([df['drug_row_id'], df['drug_col_id']], ignore_index=True)
        .astype(str)
        .unique()
    ).to_numpy()

    rng.shuffle(drugs)

    fold_arrays = np.array_split(drugs, n_splits)
    fold_sets = [set(arr.tolist()) for arr in fold_arrays]

    return fold_sets


def assign_samples_to_unique_lodo_fold(df, fold_sets, random_state=42):
    df = df.copy()
    df['drug_row_id'] = df['drug_row_id'].astype(str)
    df['drug_col_id'] = df['drug_col_id'].astype(str)

    drug_to_fold = {
        drug: fold_id
        for fold_id, fold_set in enumerate(fold_sets)
        for drug in fold_set
    }

    rng = np.random.default_rng(random_state)
    row_order = np.arange(len(df))
    rng.shuffle(row_order)

    assigned_fold = np.full(len(df), -1, dtype=int)

    fold_sizes = np.zeros(len(fold_sets), dtype=int)

    for idx in row_order:
        row = df.iloc[idx]

        f1 = drug_to_fold[row['drug_row_id']]
        f2 = drug_to_fold[row['drug_col_id']]

        candidate_folds = list(set([f1, f2]))

        chosen_fold = min(candidate_folds, key=lambda f: fold_sizes[f])

        assigned_fold[idx] = chosen_fold
        fold_sizes[chosen_fold] += 1

    df['unique_lodo_fold'] = assigned_fold

    return df


def summarize_unique_lodo_folds(df, fold_col='unique_lodo_fold'):
    rows = []

    for fold_id in sorted(df[fold_col].unique()):
        fold_df = df[df[fold_col] == fold_id]

        n_examples = len(fold_df)
        n_pos = int(fold_df['synergy_loewe_bin'].sum())
        pos_pct = 100 * n_pos / n_examples if n_examples > 0 else 0.0

        rows.append({
            'fold': fold_id + 1,
            'n_examples': n_examples,
            'n_positive': n_pos,
            'positive_pct': pos_pct
        })

    return pd.DataFrame(rows)


In [ ]:
base_df = X_final_df.copy()
base_df['drug_row_id'] = base_df['drug_row_id'].astype(str)
base_df['drug_col_id'] = base_df['drug_col_id'].astype(str)
base_df['synergy_loewe_bin'] = base_df['synergy_loewe_bin'].replace(-1, 0).astype(int)

global_pos_pct = 100 * base_df['synergy_loewe_bin'].mean()
print(f"% positivo global: {global_pos_pct:.2f}%")

fold_sets_unique = build_random_drug_folds(
    base_df,
    n_splits=10,
    random_state=42
)

df_unique_lodo = assign_samples_to_unique_lodo_fold(
    base_df,
    fold_sets_unique,
    random_state=42
)

summary_unique = summarize_unique_lodo_folds(df_unique_lodo)

print(summary_unique)

print("\nDescribe n_examples:")
print(summary_unique['n_examples'].describe())

print("\nDescribe positive_pct:")
print(summary_unique['positive_pct'].describe())

print("\nFolds con menos de 5% positivos:")
print((summary_unique['positive_pct'] < 5).sum())

print("\nFolds con menos de 5000 ejemplos:")
print((summary_unique['n_examples'] < 5000).sum())


In [ ]:
x = summary_unique['fold'].to_numpy()
n_examples = summary_unique['n_examples'].to_numpy()
positive_pct = summary_unique['positive_pct'].to_numpy()

fig, ax1 = plt.subplots(figsize=(9, 5))

ax1.bar(x, n_examples, alpha=0.75)
ax1.set_xlabel('Fold')
ax1.set_ylabel('Numero de ejemplos')
ax1.grid(axis='y', alpha=0.3)

ax2 = ax1.twinx()
ax2.plot(
    x,
    positive_pct,
    marker='o',
    color='red'
)
ax2.axhline(global_pos_pct, linestyle='--', color='gray')
ax2.set_ylabel('% clase positiva')

plt.title('LODO sin solapamiento: tamano y % positivos por fold')
plt.tight_layout()
plt.show()


Para evaluar la generalización a fármacos no vistos evitando solapamiento entre conjuntos de test, se construyó una variante Leave-Drug-Out con folds disjuntos a nivel de muestra. Los fármacos se asignaron a folds disjuntos y cada muestra se asignó a un único fold compatible con alguno de sus fármacos. De este modo, cada muestra de test contiene al menos un fármaco held-out y no aparece en otros folds de test.


In [ ]:
# Farmaco con mas apariciones en todo el dataset
drug_counts = pd.concat([
    df_unique_lodo['drug_row_id'].astype(str),
    df_unique_lodo['drug_col_id'].astype(str)
]).value_counts()

top_drug = drug_counts.index[0]
top_drug_count = drug_counts.iloc[0]

print("Farmaco con mas apariciones:", top_drug)
print("Numero total de apariciones:", top_drug_count)

# Muestras donde aparece ese farmaco
mask_top_drug = (
    df_unique_lodo['drug_row_id'].astype(str).eq(top_drug)
    | df_unique_lodo['drug_col_id'].astype(str).eq(top_drug)
)

top_drug_df = df_unique_lodo[mask_top_drug].copy()

# Distribucion de esas muestras por fold asignado
fold_distribution = (
    top_drug_df['unique_lodo_fold']
    .value_counts()
    .sort_index()
    .reset_index()
)

fold_distribution.columns = ['fold_zero_based', 'n_samples']
fold_distribution['fold'] = fold_distribution['fold_zero_based'] + 1

fold_distribution = fold_distribution[['fold', 'n_samples']]

print("\nDistribucion de las muestras del farmaco por fold:")
print(fold_distribution)

print("\nSuma:", fold_distribution['n_samples'].sum())


In [ ]:
# Fold al que pertenece el farmaco mas frecuente como held-out
drug_to_fold = {
    drug: fold_id
    for fold_id, fold_set in enumerate(fold_sets_unique)
    for drug in fold_set
}

top_drug_fold = drug_to_fold[top_drug] + 1

print("\nFold held-out asignado al farmaco:", top_drug_fold)

print("\nMuestras del farmaco que NO quedaron en su propio fold:")
print(
    top_drug_df[top_drug_df['unique_lodo_fold'] != (top_drug_fold - 1)]
    .shape[0]
)


## LDO DISJUNTO A NIVEL DE MUESTRA:


In [ ]:
shared_splits_dir = 'results/shared_splits'
os.makedirs(shared_splits_dir, exist_ok=True)

unique_lodo_path = os.path.join(shared_splits_dir, 'unique_sample_lodo_indices.json')

base_df = X_final_df.copy()
base_df['drug_row_id'] = base_df['drug_row_id'].astype(str)
base_df['drug_col_id'] = base_df['drug_col_id'].astype(str)
base_df['synergy_loewe_bin'] = base_df['synergy_loewe_bin'].replace(-1, 0).astype(int)

if os.path.exists(unique_lodo_path):
    with open(unique_lodo_path, 'r') as f:
        unique_lodo_splits = json.load(f)

    print('Splits LODO disjuntos cargados desde:', unique_lodo_path)

else:
    fold_sets_unique = build_random_drug_folds(
        base_df,
        n_splits=10,
        random_state=42
    )

    df_unique_lodo = assign_samples_to_unique_lodo_fold(
        base_df,
        fold_sets_unique,
        random_state=42
    )

    unique_lodo_splits = []

    for fold_id in range(10):
        test_idx = df_unique_lodo.index[
            df_unique_lodo['unique_lodo_fold'] == fold_id
        ].tolist()

        test_set = fold_sets_unique[fold_id]

        # Train excluye cualquier muestra con fármaco held-out de este fold
        heldout_mask = (
            base_df['drug_row_id'].isin(test_set)
            | base_df['drug_col_id'].isin(test_set)
        )

        train_idx = base_df.index[~heldout_mask].tolist()

        unique_lodo_splits.append({
            'fold': fold_id + 1,
            'train_idx': train_idx,
            'test_idx': test_idx,
            'heldout_drugs': sorted(list(test_set))
        })

    with open(unique_lodo_path, 'w') as f:
        json.dump(unique_lodo_splits, f, indent=4)

    print('Splits LODO disjuntos creados y guardados en:', unique_lodo_path)


## XGBOOST LDO DISJUNTO A NIVEL DE MUESTRA

In [ ]:
import json
import shutil
import joblib

from sklearn.ensemble import RandomForestClassifier
from torch.utils.tensorboard import SummaryWriter
from PIL import Image
from xgboost import XGBClassifier

if not hasattr(Image, "Resampling"):
    class Resampling:
        LANCZOS = Image.ANTIALIAS

    Image.Resampling = Resampling

# ============================================================
# CONFIGURACION
# ============================================================

SAVE_MODELS = False

output_dir = 'results/zzz_disjoint_XGBOOST'
tb_dir = os.path.join(output_dir, 'tensorboard')

if os.path.exists(output_dir):
    shutil.rmtree(output_dir)

os.makedirs(output_dir, exist_ok=True)
os.makedirs(os.path.join(output_dir, 'predictions'), exist_ok=True)
os.makedirs(os.path.join(output_dir, 'figures'), exist_ok=True)
os.makedirs(tb_dir, exist_ok=True)

if SAVE_MODELS:
    os.makedirs(os.path.join(output_dir, 'models'), exist_ok=True)

xgb_config = {
    'n_estimators': 600,
    'max_depth': 8,
    'learning_rate': 0.03,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'objective': 'binary:logistic',
    'eval_metric': 'aucpr',
    'tree_method': 'hist',
    'random_state': 42,
    'n_jobs': 4,
}


experiment_config = {
    'model': 'XGBClassifier',
    'split_type': 'unique_sample_lodo',
    'n_splits': 10,
    'n_genes': 500,
    'random_state': 42,
    'threshold_grid_min': 0.05,
    'threshold_grid_max': 0.95,
    'threshold_grid_step': 0.01,
    'xgb_config': xgb_config,
    'save_models': SAVE_MODELS,
    'lodo_definition': (
        'Partial Leave-Drug-Out: each test sample contains at least one held-out drug. '
        'The second drug may appear in train.'
    ),
    'external_lodo_splits_path': 'results/shared_splits/unique_sample_lodo_indices.json'
}

save_experiment_config(experiment_config, output_dir)


# ============================================================
# DATASET BASE
# ============================================================

base_df = X_final_df.copy()
base_df['drug_row_id'] = base_df['drug_row_id'].astype(str)
base_df['drug_col_id'] = base_df['drug_col_id'].astype(str)
base_df['synergy_loewe_bin'] = base_df['synergy_loewe_bin'].replace(-1, 0).astype(int)

thresholds = np.round(np.arange(0.05, 0.951, 0.01), 2)

# ============================================================
# CARGAR SPLITS UNIQUE-SAMPLE LODO COMPARTIDOS
# ============================================================

unique_lodo_path = experiment_config['external_lodo_splits_path']

with open(unique_lodo_path, 'r') as f:
    split_indices = json.load(f)

assert len(split_indices) == experiment_config['n_splits'], (
    f"Se esperaban {experiment_config['n_splits']} folds, "
    f"pero se cargaron {len(split_indices)}."
)

partition_rows = []

for split in split_indices:
    fold_idx = split['fold']
    test_idx = np.array(split['test_idx'])
    test_set = set(split['heldout_drugs'])

    fold_df = base_df.iloc[test_idx]

    n_examples = len(fold_df)
    pos_rate = fold_df['synergy_loewe_bin'].mean() if n_examples > 0 else np.nan

    partition_rows.append({
        'fold': fold_idx,
        'n_examples_test': n_examples,
        'positive_pct_test': pos_rate * 100,
        'baseline_auprc_test': pos_rate,
        'n_heldout_drugs': len(test_set)
    })

fold_partition_summary = pd.DataFrame(partition_rows)
fold_partition_summary.to_csv(
    os.path.join(output_dir, 'unique_sample_lodo_partition_summary.csv'),
    index=False
)

print('\nSplits Unique-Sample LODO cargados desde:', unique_lodo_path)
print('\nResumen particion Unique-Sample LODO:')
print(fold_partition_summary)

all_test_indices = []

for split in split_indices:
    all_test_indices.extend(split['test_idx'])

n_total_test_appearances = len(all_test_indices)
n_unique_test_samples = len(set(all_test_indices))
n_repeated = n_total_test_appearances - n_unique_test_samples
overlap_pct = (
    100 * n_repeated / n_total_test_appearances
    if n_total_test_appearances > 0 else 0.0
)

test_overlap_info = {
    'total_test_appearances': int(n_total_test_appearances),
    'unique_test_samples': int(n_unique_test_samples),
    'repeated_test_samples': int(n_repeated),
    'overlap_pct': float(overlap_pct),
}

with open(os.path.join(output_dir, 'test_overlap_summary.json'), 'w') as f:
    json.dump(test_overlap_info, f, indent=4)

print('\nSolapamiento real entre tests:')
print(test_overlap_info)


# ============================================================
# BUCLE CV
# ============================================================

cv_val_results_f1 = []
cv_val_results_precision = []
cv_test_results_f1 = []
cv_test_results_precision = []
cv_split_info = []
cv_val_threshold_metrics = []
cv_test_threshold_metrics = []
cv_best_thresholds = []


for split in split_indices:
    fold_idx = split['fold']
    train_idx = np.array(split['train_idx'])
    test_idx = np.array(split['test_idx'])
    test_set = set(split['heldout_drugs'])
    writer = SummaryWriter(log_dir=os.path.join(tb_dir, f'Fold_{fold_idx}'))

    print(f'\nPROCESANDO FOLD {fold_idx}/{experiment_config["n_splits"]} - XGBOOST UNIQUE-SAMPLE LODO')
    df_train_raw = base_df.iloc[train_idx].reset_index(drop=True)
    df_test_raw = base_df.iloc[test_idx].reset_index(drop=True)

    if len(df_train_raw) == 0 or len(df_test_raw) == 0:
        print(f'Fold {fold_idx}: sin datos suficientes, se omite.')
        writer.close()
        continue

    assert (
        df_test_raw['drug_row_id'].isin(test_set)
        | df_test_raw['drug_col_id'].isin(test_set)
    ).all(), f'Fold {fold_idx}: hay muestras test sin droga held-out.'

    train_drugs = set(pd.concat([
        df_train_raw['drug_row_id'],
        df_train_raw['drug_col_id']
    ]).astype(str))

    heldout_overlap_train = len(train_drugs & test_set)
    assert heldout_overlap_train == 0, f'Fold {fold_idx}: drogas held-out aparecen en train.'

    df_train, df_val, df_test, top_genes = prepare_fold_data_no_leakage(
        df_train_raw,
        df_test_raw,
        data_expr,
        n_genes=experiment_config['n_genes'],
        random_state=experiment_config['random_state']
    )

    X_train, y_train, X_val, y_val, X_test, y_test, feature_cols = build_tabular_matrices(
        df_train,
        df_val,
        df_test
    )

    if len(np.unique(y_train)) < 2:
        print(f'Fold {fold_idx}: train con una sola clase, se omite.')
        writer.close()
        continue

    n_pos = np.sum(y_train)
    n_neg = len(y_train) - n_pos

    xgb_config_fold = xgb_config.copy()
    xgb_config_fold['scale_pos_weight'] = n_neg / n_pos

    xgb = XGBClassifier(**xgb_config_fold)

    print(f'Entrenando XGBoost fold {fold_idx}...')
    xgb.fit(X_train, y_train)
    print(f'Entrenamiento terminado fold {fold_idx}.')

    y_prob_val = xgb.predict_proba(X_val)[:, 1]
    y_prob_test = xgb.predict_proba(X_test)[:, 1]

    df_val_thr, best_thr_f1, best_thr_precision = find_best_thresholds(
        y_val,
        y_prob_val,
        thresholds
    )

    df_val_thr.insert(0, 'fold', fold_idx)
    df_val_thr.insert(1, 'split', 'val')
    df_val_thr.insert(2, 'threshold_type', 'grid')
    cv_val_threshold_metrics.append(df_val_thr)

    test_thr_rows = []
    for thr in thresholds:
        m_test_thr = compute_metrics_at_threshold(y_test, y_prob_test, thr)
        m_test_thr.update({
            'fold': fold_idx,
            'split': 'test',
            'threshold_type': 'grid',
        })
        test_thr_rows.append(m_test_thr)

    df_test_thr = pd.DataFrame(test_thr_rows)
    cv_test_threshold_metrics.append(df_test_thr)

    val_metrics_f1 = compute_metrics_at_threshold(y_val, y_prob_val, best_thr_f1)
    val_metrics_precision = compute_metrics_at_threshold(y_val, y_prob_val, best_thr_precision)

    test_metrics_f1 = compute_metrics_at_threshold(y_test, y_prob_test, best_thr_f1)
    test_metrics_precision = compute_metrics_at_threshold(y_test, y_prob_test, best_thr_precision)

    val_metrics_f1.update({'fold': fold_idx, 'split': 'val', 'threshold_type': 'best_f1'})
    val_metrics_precision.update({'fold': fold_idx, 'split': 'val', 'threshold_type': 'best_precision'})
    test_metrics_f1.update({'fold': fold_idx, 'split': 'test', 'threshold_type': 'best_f1'})
    test_metrics_precision.update({'fold': fold_idx, 'split': 'test', 'threshold_type': 'best_precision'})

    cv_val_results_f1.append(val_metrics_f1)
    cv_val_results_precision.append(val_metrics_precision)
    cv_test_results_f1.append(test_metrics_f1)
    cv_test_results_precision.append(test_metrics_precision)

    cv_best_thresholds.append({
        'fold': fold_idx,
        'best_threshold_f1': best_thr_f1,
        'best_threshold_precision': best_thr_precision,
        'val_f1_at_best_f1': val_metrics_f1['f1'],
        'val_precision_at_best_f1': val_metrics_f1['precision'],
        'val_recall_at_best_f1': val_metrics_f1['recall'],
        'val_precision_at_best_precision': val_metrics_precision['precision'],
        'val_recall_at_best_precision': val_metrics_precision['recall'],
        'val_f1_at_best_precision': val_metrics_precision['f1'],
    })

    log_curves_and_metrics(y_val, y_prob_val, val_metrics_f1, fold_idx, writer, tag='Val_BestF1')
    log_curves_and_metrics(y_val, y_prob_val, val_metrics_precision, fold_idx, writer, tag='Val_BestPrecision')
    log_curves_and_metrics(y_test, y_prob_test, test_metrics_f1, fold_idx, writer, tag='Test_BestF1')
    log_curves_and_metrics(y_test, y_prob_test, test_metrics_precision, fold_idx, writer, tag='Test_BestPrecision')

    predictions = df_test[
        [
            'drug_row_id',
            'drug_col_id',
            'cell_line_name',
            'study_name',
            'tissue',
            'synergy_loewe',
            'synergy_loewe_bin'
        ]
    ].copy()

    predictions['y_true'] = y_test
    predictions['y_prob'] = y_prob_test
    predictions['y_pred_best_f1'] = (y_prob_test >= best_thr_f1).astype(int)
    predictions['y_pred_best_precision'] = (y_prob_test >= best_thr_precision).astype(int)
    predictions['threshold_best_f1'] = best_thr_f1
    predictions['threshold_best_precision'] = best_thr_precision

    predictions.to_csv(
        os.path.join(output_dir, 'predictions', f'predictions_fold_{fold_idx:02d}.csv'),
        index=False
    )

    if SAVE_MODELS:
        joblib.dump(
            xgb,
            os.path.join(output_dir, 'models', f'xgboost_fold_{fold_idx:02d}.joblib')
        )

    n_train = len(y_train)
    n_val = len(y_val)
    n_test = len(y_test)

    split_info = {
        'fold': fold_idx,
        'n_train': n_train,
        'n_val': n_val,
        'n_test': n_test,
        'train_pos': int(np.sum(y_train)),
        'val_pos': int(np.sum(y_val)),
        'test_pos': int(np.sum(y_test)),
        'train_pos_pct': 100 * np.mean(y_train),
        'val_pos_pct': 100 * np.mean(y_val),
        'test_pos_pct': 100 * np.mean(y_test),
        'baseline_auprc_test': float(np.mean(y_test)),
        'n_heldout_drugs': len(test_set),
        'heldout_drugs': ';'.join(sorted(test_set)),
        'heldout_overlap_train': heldout_overlap_train,
        'best_threshold_f1': best_thr_f1,
        'best_threshold_precision': best_thr_precision,
        'n_features': len(feature_cols),
        'n_top_genes': len(top_genes),
    }

    cv_split_info.append(split_info)

    writer.add_text('Split/Info', json.dumps(split_info, indent=2))
    writer.add_text('Genes/TopGenes_First100', ', '.join(top_genes[:100]))

    importances = pd.Series(xgb.feature_importances_, index=feature_cols)
    top_imp = importances.sort_values(ascending=False).head(30)
    writer.add_text('Model/Top30_Feature_Importance', top_imp.to_string())

    print(
        f"Fold {fold_idx} | "
        f"Train={n_train} ({100*np.mean(y_train):.2f}% pos), "
        f"Val={n_val} ({100*np.mean(y_val):.2f}% pos), "
        f"Test={n_test} ({100*np.mean(y_test):.2f}% pos) | "
        f"ThrF1={best_thr_f1:.2f}, ThrPrec={best_thr_precision:.2f} | "
        f"AUPRC={test_metrics_f1['auprc']:.4f} "
        f"(base={test_metrics_f1['baseline_auprc']:.4f}) | "
        f"F1={test_metrics_f1['f1']:.4f} | "
        f"Prec={test_metrics_f1['precision']:.4f} | "
        f"Rec={test_metrics_f1['recall']:.4f} | "
        f"MCC={test_metrics_f1['mcc']:.4f}"
    )

    writer.close()


# ============================================================
# GUARDADO DE RESULTADOS
# ============================================================

df_val_f1 = pd.DataFrame(cv_val_results_f1)
df_val_precision = pd.DataFrame(cv_val_results_precision)
df_test_f1 = pd.DataFrame(cv_test_results_f1)
df_test_precision = pd.DataFrame(cv_test_results_precision)
df_split_info = pd.DataFrame(cv_split_info)
df_best_thresholds = pd.DataFrame(cv_best_thresholds)

df_val_thresholds = pd.concat(cv_val_threshold_metrics, ignore_index=True)
df_test_thresholds = pd.concat(cv_test_threshold_metrics, ignore_index=True)

df_val_f1.to_csv(os.path.join(output_dir, 'val_metrics_best_f1.csv'), index=False)
df_val_precision.to_csv(os.path.join(output_dir, 'val_metrics_best_precision.csv'), index=False)
df_test_f1.to_csv(os.path.join(output_dir, 'test_metrics_best_f1.csv'), index=False)
df_test_precision.to_csv(os.path.join(output_dir, 'test_metrics_best_precision.csv'), index=False)
df_split_info.to_csv(os.path.join(output_dir, 'split_info.csv'), index=False)
df_best_thresholds.to_csv(os.path.join(output_dir, 'best_thresholds.csv'), index=False)
df_val_thresholds.to_csv(os.path.join(output_dir, 'val_metrics_by_threshold.csv'), index=False)
df_test_thresholds.to_csv(os.path.join(output_dir, 'test_metrics_by_threshold.csv'), index=False)


# ============================================================
# RESUMEN FINAL
# ============================================================

metric_cols = [
    'auc',
    'auprc',
    'baseline_auprc',
    'f1',
    'precision',
    'recall',
    'specificity',
    'balanced_accuracy',
    'mcc',
    'log_loss',
    'brier',
    'tp',
    'tn',
    'fp',
    'fn',
]

summary_f1 = (
    df_test_f1[metric_cols]
    .agg(['mean', 'std'])
    .T
    .reset_index()
    .rename(columns={'index': 'metric'})
)

summary_precision = (
    df_test_precision[metric_cols]
    .agg(['mean', 'std'])
    .T
    .reset_index()
    .rename(columns={'index': 'metric'})
)

summary_f1.to_csv(os.path.join(output_dir, 'summary_test_best_f1.csv'), index=False)
summary_precision.to_csv(os.path.join(output_dir, 'summary_test_best_precision.csv'), index=False)

print('\n' + '=' * 70)
print('RESULTADOS XGBOOST - UNIQUE-SAMPLE LODO CV=10')
print('=' * 70)

print('\n[TEST - threshold elegido por mejor F1 en validacion]')
print(summary_f1)

print('\n[TEST - threshold conservador por mejor precision en validacion]')
print(summary_precision)

print('\nArchivos guardados en:', output_dir)



## TRIPLEMLP LDO DISJUNTO A NIVEL DE MUESTRA

In [ ]:
import os
import json
import copy
import shutil
import numpy as np
import pandas as pd
import torch

from torch import nn
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
from PIL import Image

if not hasattr(Image, "Resampling"):
    class Resampling:
        LANCZOS = Image.ANTIALIAS

    Image.Resampling = Resampling


# ============================================================
# CONFIGURACION
# ============================================================

SAVE_MODELS = False

output_dir = 'results/zzz_disjoint_TripleBranchMLP'
tb_dir = os.path.join(output_dir, 'tensorboard')

if os.path.exists(output_dir):
    shutil.rmtree(output_dir)

os.makedirs(output_dir, exist_ok=True)
os.makedirs(os.path.join(output_dir, 'predictions'), exist_ok=True)
os.makedirs(os.path.join(output_dir, 'figures'), exist_ok=True)
os.makedirs(tb_dir, exist_ok=True)

if SAVE_MODELS:
    os.makedirs(os.path.join(output_dir, 'models'), exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

mlp_config = {
    'lr': 1e-3,
    'batch_size': 256,
    'epochs': 80,
    'patience': 12,
    'weight_decay': 1e-4,
    'drug_dim': 1024,
    'n_genes': 500,
    'random_state': 42,
    'use_pos_weight': True,
}

experiment_config = {
    'model': 'TripleBranchMLP',
    'split_type': 'unique_sample_lodo',
    'n_splits': 10,
    'n_genes': mlp_config['n_genes'],
    'random_state': mlp_config['random_state'],
    'threshold_grid_min': 0.05,
    'threshold_grid_max': 0.95,
    'threshold_grid_step': 0.01,
    'mlp_config': mlp_config,
    'save_models': SAVE_MODELS,
    'device': str(device),
    'external_lodo_splits_path': 'results/shared_splits/unique_sample_lodo_indices.json',
    'lodo_definition': (
        'Unique-sample partial Leave-Drug-Out: each drug is assigned to one fold, '
        'and each sample is assigned to a single compatible test fold to avoid test overlap.'
    )
}

save_experiment_config(experiment_config, output_dir)


# ============================================================
# DATASET BASE
# ============================================================

base_df = X_final_df.copy()
base_df['drug_row_id'] = base_df['drug_row_id'].astype(str)
base_df['drug_col_id'] = base_df['drug_col_id'].astype(str)
base_df['synergy_loewe_bin'] = base_df['synergy_loewe_bin'].replace(-1, 0).astype(int)

thresholds = np.round(np.arange(0.05, 0.951, 0.01), 2)


# ============================================================
# CARGAR SPLITS UNIQUE-SAMPLE LODO COMPARTIDOS
# ============================================================

unique_lodo_path = experiment_config['external_lodo_splits_path']

with open(unique_lodo_path, 'r') as f:
    split_indices = json.load(f)

assert len(split_indices) == experiment_config['n_splits'], (
    f"Se esperaban {experiment_config['n_splits']} folds, "
    f"pero se cargaron {len(split_indices)}."
)

partition_rows = []

for split in split_indices:
    fold_idx = split['fold']
    test_idx = np.array(split['test_idx'])
    test_set = set(split['heldout_drugs'])

    fold_df = base_df.iloc[test_idx]

    n_examples = len(fold_df)
    pos_rate = fold_df['synergy_loewe_bin'].mean() if n_examples > 0 else np.nan

    partition_rows.append({
        'fold': fold_idx,
        'n_examples_test': n_examples,
        'positive_pct_test': pos_rate * 100,
        'baseline_auprc_test': pos_rate,
        'n_heldout_drugs': len(test_set)
    })

fold_partition_summary = pd.DataFrame(partition_rows)
fold_partition_summary.to_csv(
    os.path.join(output_dir, 'unique_sample_lodo_partition_summary.csv'),
    index=False
)

print('\nSplits Unique-Sample LODO cargados desde:', unique_lodo_path)
print('\nResumen particion Unique-Sample LODO:')
print(fold_partition_summary)

all_test_indices = []

for split in split_indices:
    all_test_indices.extend(split['test_idx'])

n_total_test_appearances = len(all_test_indices)
n_unique_test_samples = len(set(all_test_indices))
n_repeated = n_total_test_appearances - n_unique_test_samples
overlap_pct = (
    100 * n_repeated / n_total_test_appearances
    if n_total_test_appearances > 0 else 0.0
)

test_overlap_info = {
    'total_test_appearances': int(n_total_test_appearances),
    'unique_test_samples': int(n_unique_test_samples),
    'repeated_test_samples': int(n_repeated),
    'overlap_pct': float(overlap_pct),
}

with open(os.path.join(output_dir, 'test_overlap_summary.json'), 'w') as f:
    json.dump(test_overlap_info, f, indent=4)

print('\nSolapamiento real entre tests:')
print(test_overlap_info)


# ============================================================
# BUCLE CV
# ============================================================

cv_val_results_f1 = []
cv_val_results_precision = []
cv_test_results_f1 = []
cv_test_results_precision = []
cv_split_info = []
cv_val_threshold_metrics = []
cv_test_threshold_metrics = []
cv_best_thresholds = []
cv_histories = []



for split in split_indices:
    fold_idx = split['fold']
    train_idx = np.array(split['train_idx'])
    test_idx = np.array(split['test_idx'])
    test_set = set(split['heldout_drugs'])

    writer = SummaryWriter(log_dir=os.path.join(tb_dir, f'Fold_{fold_idx}'))
    print(f'\nPROCESANDO FOLD {fold_idx}/{experiment_config["n_splits"]} -TRIPLE BRANCH MLP UNIQUE-SAMPLE LODO')
    df_train_raw = base_df.iloc[train_idx].reset_index(drop=True)
    df_test_raw = base_df.iloc[test_idx].reset_index(drop=True)

    if len(df_train_raw) == 0 or len(df_test_raw) == 0:
        print(f'Fold {fold_idx}: sin datos suficientes, se omite.')
        writer.close()
        continue

    assert (
        df_test_raw['drug_row_id'].isin(test_set)
        | df_test_raw['drug_col_id'].isin(test_set)
    ).all(), f'Fold {fold_idx}: hay muestras test sin droga held-out.'

    train_drugs = set(pd.concat([
        df_train_raw['drug_row_id'],
        df_train_raw['drug_col_id']
    ]).astype(str))

    heldout_overlap_train = len(train_drugs & test_set)
    assert heldout_overlap_train == 0, f'Fold {fold_idx}: drogas held-out aparecen en train.'

    df_train, df_val, df_test, top_genes = prepare_fold_data_no_leakage(
        df_train_raw,
        df_test_raw,
        data_expr,
        n_genes=experiment_config['n_genes'],
        random_state=experiment_config['random_state']
    )

    y_train_np = df_train['synergy_loewe_bin'].astype(int).to_numpy()

    if len(np.unique(y_train_np)) < 2:
        print(f'Fold {fold_idx}: train con una sola clase, se omite.')
        writer.close()
        continue

    train_loader = DataLoader(
        SynergyDataset(df_train, top_genes),
        batch_size=mlp_config['batch_size'],
        shuffle=True,

    )

    val_loader = DataLoader(
        SynergyDataset(df_val, top_genes),
        batch_size=mlp_config['batch_size'],
        shuffle=False
    )

    test_loader = DataLoader(
        SynergyDataset(df_test, top_genes),
        batch_size=mlp_config['batch_size'],
        shuffle=False
    )

    model = TripleBranchMLP(
        drug_dim=mlp_config['drug_dim'],
        gene_dim=experiment_config['n_genes']
    ).to(device)

    model, history_df, best_epoch, best_val_loss = train_torch_model_fold(
        model,
        train_loader,
        val_loader,
        y_train_np,
        mlp_config,
        device,
        writer,
        fold_idx
    )

    history_df.to_csv(
        os.path.join(output_dir, f'training_history_fold_{fold_idx:02d}.csv'),
        index=False
    )
    cv_histories.append(history_df)

    y_val, y_prob_val = predict_torch_model(model, val_loader, device)
    y_test, y_prob_test = predict_torch_model(model, test_loader, device)

    df_val_thr, best_thr_f1, best_thr_precision = find_best_thresholds(
        y_val,
        y_prob_val,
        thresholds
    )

    df_val_thr.insert(0, 'fold', fold_idx)
    df_val_thr.insert(1, 'split', 'val')
    df_val_thr.insert(2, 'threshold_type', 'grid')
    cv_val_threshold_metrics.append(df_val_thr)

    test_thr_rows = []
    for thr in thresholds:
        m_test_thr = compute_metrics_at_threshold(y_test, y_prob_test, thr)
        m_test_thr.update({
            'fold': fold_idx,
            'split': 'test',
            'threshold_type': 'grid',
        })
        test_thr_rows.append(m_test_thr)

    df_test_thr = pd.DataFrame(test_thr_rows)
    cv_test_threshold_metrics.append(df_test_thr)

    val_metrics_f1 = compute_metrics_at_threshold(y_val, y_prob_val, best_thr_f1)
    val_metrics_precision = compute_metrics_at_threshold(y_val, y_prob_val, best_thr_precision)

    test_metrics_f1 = compute_metrics_at_threshold(y_test, y_prob_test, best_thr_f1)
    test_metrics_precision = compute_metrics_at_threshold(y_test, y_prob_test, best_thr_precision)

    val_metrics_f1.update({'fold': fold_idx, 'split': 'val', 'threshold_type': 'best_f1'})
    val_metrics_precision.update({'fold': fold_idx, 'split': 'val', 'threshold_type': 'best_precision'})
    test_metrics_f1.update({'fold': fold_idx, 'split': 'test', 'threshold_type': 'best_f1'})
    test_metrics_precision.update({'fold': fold_idx, 'split': 'test', 'threshold_type': 'best_precision'})

    cv_val_results_f1.append(val_metrics_f1)
    cv_val_results_precision.append(val_metrics_precision)
    cv_test_results_f1.append(test_metrics_f1)
    cv_test_results_precision.append(test_metrics_precision)

    cv_best_thresholds.append({
        'fold': fold_idx,
        'best_threshold_f1': best_thr_f1,
        'best_threshold_precision': best_thr_precision,
        'val_f1_at_best_f1': val_metrics_f1['f1'],
        'val_precision_at_best_f1': val_metrics_f1['precision'],
        'val_recall_at_best_f1': val_metrics_f1['recall'],
        'val_precision_at_best_precision': val_metrics_precision['precision'],
        'val_recall_at_best_precision': val_metrics_precision['recall'],
        'val_f1_at_best_precision': val_metrics_precision['f1'],
        'best_epoch': best_epoch,
        'best_val_loss': best_val_loss,
    })

    log_curves_and_metrics(y_val, y_prob_val, val_metrics_f1, fold_idx, writer, tag='Val_BestF1')
    log_curves_and_metrics(y_val, y_prob_val, val_metrics_precision, fold_idx, writer, tag='Val_BestPrecision')
    log_curves_and_metrics(y_test, y_prob_test, test_metrics_f1, fold_idx, writer, tag='Test_BestF1')
    log_curves_and_metrics(y_test, y_prob_test, test_metrics_precision, fold_idx, writer, tag='Test_BestPrecision')

    predictions = df_test[
        [
            'drug_row_id',
            'drug_col_id',
            'cell_line_name',
            'study_name',
            'tissue',
            'synergy_loewe',
            'synergy_loewe_bin'
        ]
    ].copy()

    predictions['y_true'] = y_test
    predictions['y_prob'] = y_prob_test
    predictions['y_pred_best_f1'] = (y_prob_test >= best_thr_f1).astype(int)
    predictions['y_pred_best_precision'] = (y_prob_test >= best_thr_precision).astype(int)
    predictions['threshold_best_f1'] = best_thr_f1
    predictions['threshold_best_precision'] = best_thr_precision

    predictions.to_csv(
        os.path.join(output_dir, 'predictions', f'predictions_fold_{fold_idx:02d}.csv'),
        index=False
    )

    if SAVE_MODELS:
        torch.save(
            model.state_dict(),
            os.path.join(output_dir, 'models', f'triple_branch_mlp_fold_{fold_idx:02d}.pt')
        )

    n_train = len(y_train_np)
    n_val = len(y_val)
    n_test = len(y_test)

    split_info = {
        'fold': fold_idx,
        'n_train': n_train,
        'n_val': n_val,
        'n_test': n_test,
        'train_pos': int(np.sum(y_train_np)),
        'val_pos': int(np.sum(y_val)),
        'test_pos': int(np.sum(y_test)),
        'train_pos_pct': 100 * np.mean(y_train_np),
        'val_pos_pct': 100 * np.mean(y_val),
        'test_pos_pct': 100 * np.mean(y_test),
        'baseline_auprc_test': float(np.mean(y_test)),
        'n_heldout_drugs': len(test_set),
        'heldout_drugs': ';'.join(sorted(test_set)),
        'heldout_overlap_train': heldout_overlap_train,
        'best_threshold_f1': best_thr_f1,
        'best_threshold_precision': best_thr_precision,
        'best_epoch': best_epoch,
        'best_val_loss': best_val_loss,
        'n_top_genes': len(top_genes),
    }

    cv_split_info.append(split_info)

    writer.add_text('Split/Info', json.dumps(split_info, indent=2))
    writer.add_text('Genes/TopGenes_First100', ', '.join(top_genes[:100]))

    print(
        f"Fold {fold_idx} | "
        f"Train={n_train} ({100*np.mean(y_train_np):.2f}% pos), "
        f"Val={n_val} ({100*np.mean(y_val):.2f}% pos), "
        f"Test={n_test} ({100*np.mean(y_test):.2f}% pos) | "
        f"BestEpoch={best_epoch} | "
        f"ThrF1={best_thr_f1:.2f}, ThrPrec={best_thr_precision:.2f} | "
        f"AUPRC={test_metrics_f1['auprc']:.4f} "
        f"(base={test_metrics_f1['baseline_auprc']:.4f}) | "
        f"F1={test_metrics_f1['f1']:.4f} | "
        f"Prec={test_metrics_f1['precision']:.4f} | "
        f"Rec={test_metrics_f1['recall']:.4f} | "
        f"MCC={test_metrics_f1['mcc']:.4f}"
    )

    writer.close()
    torch.cuda.empty_cache()

# ============================================================
# GUARDADO DE RESULTADOS
# ============================================================

df_val_f1 = pd.DataFrame(cv_val_results_f1)
df_val_precision = pd.DataFrame(cv_val_results_precision)
df_test_f1 = pd.DataFrame(cv_test_results_f1)
df_test_precision = pd.DataFrame(cv_test_results_precision)
df_split_info = pd.DataFrame(cv_split_info)
df_best_thresholds = pd.DataFrame(cv_best_thresholds)

df_val_thresholds = pd.concat(cv_val_threshold_metrics, ignore_index=True)
df_test_thresholds = pd.concat(cv_test_threshold_metrics, ignore_index=True)

df_val_f1.to_csv(os.path.join(output_dir, 'val_metrics_best_f1.csv'), index=False)
df_val_precision.to_csv(os.path.join(output_dir, 'val_metrics_best_precision.csv'), index=False)
df_test_f1.to_csv(os.path.join(output_dir, 'test_metrics_best_f1.csv'), index=False)
df_test_precision.to_csv(os.path.join(output_dir, 'test_metrics_best_precision.csv'), index=False)
df_split_info.to_csv(os.path.join(output_dir, 'split_info.csv'), index=False)
df_best_thresholds.to_csv(os.path.join(output_dir, 'best_thresholds.csv'), index=False)
df_val_thresholds.to_csv(os.path.join(output_dir, 'val_metrics_by_threshold.csv'), index=False)
df_test_thresholds.to_csv(os.path.join(output_dir, 'test_metrics_by_threshold.csv'), index=False)

if len(cv_histories) > 0:
    df_all_history = pd.concat(cv_histories, ignore_index=True)
    df_all_history.to_csv(os.path.join(output_dir, 'training_history_all_folds.csv'), index=False)


# ============================================================
# RESUMEN FINAL
# ============================================================

metric_cols = [
    'auc',
    'auprc',
    'baseline_auprc',
    'f1',
    'precision',
    'recall',
    'specificity',
    'balanced_accuracy',
    'mcc',
    'log_loss',
    'brier',
    'tp',
    'tn',
    'fp',
    'fn',
]

summary_f1 = (
    df_test_f1[metric_cols]
    .agg(['mean', 'std'])
    .T
    .reset_index()
    .rename(columns={'index': 'metric'})
)

summary_precision = (
    df_test_precision[metric_cols]
    .agg(['mean', 'std'])
    .T
    .reset_index()
    .rename(columns={'index': 'metric'})
)

summary_f1.to_csv(os.path.join(output_dir, 'summary_test_best_f1.csv'), index=False)
summary_precision.to_csv(os.path.join(output_dir, 'summary_test_best_precision.csv'), index=False)

print('\n' + '=' * 70)
print('RESULTADOS TRIPLE BRANCH MLP - UNIQUE-SAMPLE LODO CV=10')
print('=' * 70)

print('\n[TEST - threshold elegido por mejor F1 en validacion]')
print(summary_f1)

print('\n[TEST - threshold conservador por mejor precision en validacion]')
print(summary_precision)

print('\nArchivos guardados en:', output_dir)


## MLP LDO DISJUNTO A NIVEL DE MUESTRA

In [ ]:
import os
import json
import copy
import shutil


from torch import nn
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
from PIL import Image

if not hasattr(Image, "Resampling"):
    class Resampling:
        LANCZOS = Image.ANTIALIAS

    Image.Resampling = Resampling


# ============================================================
# CONFIGURACION
# ============================================================

SAVE_MODELS = False

output_dir = 'results/zzz_disjoint_mlp'
tb_dir = os.path.join(output_dir, 'tensorboard')

if os.path.exists(output_dir):
    shutil.rmtree(output_dir)

os.makedirs(output_dir, exist_ok=True)
os.makedirs(os.path.join(output_dir, 'predictions'), exist_ok=True)
os.makedirs(os.path.join(output_dir, 'figures'), exist_ok=True)
os.makedirs(tb_dir, exist_ok=True)

if SAVE_MODELS:
    os.makedirs(os.path.join(output_dir, 'models'), exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

mlp_config = {
    'lr': 1e-3,
    'batch_size': 256,
    'epochs': 80,
    'patience': 12,
    'weight_decay': 1e-4,
    'drug_dim': 1024,
    'n_genes': 500,
    'random_state': 42,
    'use_pos_weight': True,
}

experiment_config = {
    'model': 'EarlyFusionMLP',
    'split_type': 'unique_sample_lodo',
    'n_splits': 10,
    'n_genes': mlp_config['n_genes'],
    'random_state': mlp_config['random_state'],
    'threshold_grid_min': 0.05,
    'threshold_grid_max': 0.95,
    'threshold_grid_step': 0.01,
    'mlp_config': mlp_config,
    'save_models': SAVE_MODELS,
    'device': str(device),
    'external_lodo_splits_path': 'results/shared_splits/unique_sample_lodo_indices.json',
    'lodo_definition': (
        'Unique-sample partial Leave-Drug-Out: each drug is assigned to one fold, '
        'and each sample is assigned to a single compatible test fold to avoid test overlap.'
    ),
    'input_features': 'drug1_fp + drug2_fp + selected_gene_expression'
}

save_experiment_config(experiment_config, output_dir)

# ============================================================
# DATASET BASE
# ============================================================

base_df = X_final_df.copy()
base_df['drug_row_id'] = base_df['drug_row_id'].astype(str)
base_df['drug_col_id'] = base_df['drug_col_id'].astype(str)
base_df['synergy_loewe_bin'] = base_df['synergy_loewe_bin'].replace(-1, 0).astype(int)

thresholds = np.round(np.arange(0.05, 0.951, 0.01), 2)


# ============================================================
# CARGAR SPLITS UNIQUE-SAMPLE LODO COMPARTIDOS
# ============================================================

unique_lodo_path = experiment_config['external_lodo_splits_path']

with open(unique_lodo_path, 'r') as f:
    split_indices = json.load(f)

assert len(split_indices) == experiment_config['n_splits'], (
    f"Se esperaban {experiment_config['n_splits']} folds, "
    f"pero se cargaron {len(split_indices)}."
)

partition_rows = []

for split in split_indices:
    fold_idx = split['fold']
    test_idx = np.array(split['test_idx'])
    test_set = set(split['heldout_drugs'])

    fold_df = base_df.iloc[test_idx]

    n_examples = len(fold_df)
    pos_rate = fold_df['synergy_loewe_bin'].mean() if n_examples > 0 else np.nan

    partition_rows.append({
        'fold': fold_idx,
        'n_examples_test': n_examples,
        'positive_pct_test': pos_rate * 100,
        'baseline_auprc_test': pos_rate,
        'n_heldout_drugs': len(test_set)
    })

fold_partition_summary = pd.DataFrame(partition_rows)
fold_partition_summary.to_csv(
    os.path.join(output_dir, 'unique_sample_lodo_partition_summary.csv'),
    index=False
)

print('\nSplits Unique-Sample LODO cargados desde:', unique_lodo_path)
print('\nResumen particion Unique-Sample LODO:')
print(fold_partition_summary)

all_test_indices = []

for split in split_indices:
    all_test_indices.extend(split['test_idx'])

n_total_test_appearances = len(all_test_indices)
n_unique_test_samples = len(set(all_test_indices))
n_repeated = n_total_test_appearances - n_unique_test_samples
overlap_pct = (
    100 * n_repeated / n_total_test_appearances
    if n_total_test_appearances > 0 else 0.0
)

test_overlap_info = {
    'total_test_appearances': int(n_total_test_appearances),
    'unique_test_samples': int(n_unique_test_samples),
    'repeated_test_samples': int(n_repeated),
    'overlap_pct': float(overlap_pct),
}

with open(os.path.join(output_dir, 'test_overlap_summary.json'), 'w') as f:
    json.dump(test_overlap_info, f, indent=4)

print('\nSolapamiento real entre tests:')
print(test_overlap_info)

# ============================================================
# BUCLE CV
# ============================================================

cv_val_results_f1 = []
cv_val_results_precision = []
cv_test_results_f1 = []
cv_test_results_precision = []
cv_split_info = []
cv_val_threshold_metrics = []
cv_test_threshold_metrics = []
cv_best_thresholds = []
cv_histories = []



for split in split_indices:
    fold_idx = split['fold']
    train_idx = np.array(split['train_idx'])
    test_idx = np.array(split['test_idx'])
    test_set = set(split['heldout_drugs'])


    writer = SummaryWriter(log_dir=os.path.join(tb_dir, f'Fold_{fold_idx}'))
    print(f'\nPROCESANDO FOLD {fold_idx}/{experiment_config["n_splits"]} - BASIC MLP UNIQUE-SAMPLE LODO')

    df_train_raw = base_df.iloc[train_idx].reset_index(drop=True)
    df_test_raw = base_df.iloc[test_idx].reset_index(drop=True)

    if len(df_train_raw) == 0 or len(df_test_raw) == 0:
        print(f'Fold {fold_idx}: sin datos suficientes, se omite.')
        writer.close()
        continue

    assert (
        df_test_raw['drug_row_id'].isin(test_set)
        | df_test_raw['drug_col_id'].isin(test_set)
    ).all(), f'Fold {fold_idx}: hay muestras test sin droga held-out.'

    train_drugs = set(pd.concat([
        df_train_raw['drug_row_id'],
        df_train_raw['drug_col_id']
    ]).astype(str))

    heldout_overlap_train = len(train_drugs & test_set)
    assert heldout_overlap_train == 0, f'Fold {fold_idx}: drogas held-out aparecen en train.'

    df_train, df_val, df_test, top_genes = prepare_fold_data_no_leakage(
        df_train_raw,
        df_test_raw,
        data_expr,
        n_genes=experiment_config['n_genes'],
        random_state=experiment_config['random_state']
    )

    y_train_np = df_train['synergy_loewe_bin'].astype(int).to_numpy()

    if len(np.unique(y_train_np)) < 2:
        print(f'Fold {fold_idx}: train con una sola clase, se omite.')
        writer.close()
        continue

    train_loader = DataLoader(
        TabularSynergyDataset(df_train, top_genes),
        batch_size=mlp_config['batch_size'],
        shuffle=True,

    )

    val_loader = DataLoader(
        TabularSynergyDataset(df_val, top_genes),
        batch_size=mlp_config['batch_size'],
        shuffle=False
    )

    test_loader = DataLoader(
        TabularSynergyDataset(df_test, top_genes),
        batch_size=mlp_config['batch_size'],
        shuffle=False
    )

    input_dim = 1024 + 1024 + experiment_config['n_genes']

    model = DrugSynergyMLP(
        input_dim=input_dim
    ).to(device)

    model, history_df, best_epoch, best_val_loss = train_torch_tabular_model_fold(
        model,
        train_loader,
        val_loader,
        y_train_np,
        mlp_config,
        device,
        writer,
        fold_idx
    )

    history_df.to_csv(
        os.path.join(output_dir, f'training_history_fold_{fold_idx:02d}.csv'),
        index=False
    )
    cv_histories.append(history_df)

    y_val, y_prob_val = predict_torch_tabular_model(model, val_loader, device)
    y_test, y_prob_test = predict_torch_tabular_model(model, test_loader, device)

    df_val_thr, best_thr_f1, best_thr_precision = find_best_thresholds(
        y_val,
        y_prob_val,
        thresholds
    )

    df_val_thr.insert(0, 'fold', fold_idx)
    df_val_thr.insert(1, 'split', 'val')
    df_val_thr.insert(2, 'threshold_type', 'grid')
    cv_val_threshold_metrics.append(df_val_thr)

    test_thr_rows = []
    for thr in thresholds:
        m_test_thr = compute_metrics_at_threshold(y_test, y_prob_test, thr)
        m_test_thr.update({
            'fold': fold_idx,
            'split': 'test',
            'threshold_type': 'grid',
        })
        test_thr_rows.append(m_test_thr)

    df_test_thr = pd.DataFrame(test_thr_rows)
    cv_test_threshold_metrics.append(df_test_thr)

    val_metrics_f1 = compute_metrics_at_threshold(y_val, y_prob_val, best_thr_f1)
    val_metrics_precision = compute_metrics_at_threshold(y_val, y_prob_val, best_thr_precision)

    test_metrics_f1 = compute_metrics_at_threshold(y_test, y_prob_test, best_thr_f1)
    test_metrics_precision = compute_metrics_at_threshold(y_test, y_prob_test, best_thr_precision)

    val_metrics_f1.update({'fold': fold_idx, 'split': 'val', 'threshold_type': 'best_f1'})
    val_metrics_precision.update({'fold': fold_idx, 'split': 'val', 'threshold_type': 'best_precision'})
    test_metrics_f1.update({'fold': fold_idx, 'split': 'test', 'threshold_type': 'best_f1'})
    test_metrics_precision.update({'fold': fold_idx, 'split': 'test', 'threshold_type': 'best_precision'})

    cv_val_results_f1.append(val_metrics_f1)
    cv_val_results_precision.append(val_metrics_precision)
    cv_test_results_f1.append(test_metrics_f1)
    cv_test_results_precision.append(test_metrics_precision)

    cv_best_thresholds.append({
        'fold': fold_idx,
        'best_threshold_f1': best_thr_f1,
        'best_threshold_precision': best_thr_precision,
        'val_f1_at_best_f1': val_metrics_f1['f1'],
        'val_precision_at_best_f1': val_metrics_f1['precision'],
        'val_recall_at_best_f1': val_metrics_f1['recall'],
        'val_precision_at_best_precision': val_metrics_precision['precision'],
        'val_recall_at_best_precision': val_metrics_precision['recall'],
        'val_f1_at_best_precision': val_metrics_precision['f1'],
        'best_epoch': best_epoch,
        'best_val_loss': best_val_loss,
    })

    log_curves_and_metrics(y_val, y_prob_val, val_metrics_f1, fold_idx, writer, tag='Val_BestF1')
    log_curves_and_metrics(y_val, y_prob_val, val_metrics_precision, fold_idx, writer, tag='Val_BestPrecision')
    log_curves_and_metrics(y_test, y_prob_test, test_metrics_f1, fold_idx, writer, tag='Test_BestF1')
    log_curves_and_metrics(y_test, y_prob_test, test_metrics_precision, fold_idx, writer, tag='Test_BestPrecision')

    predictions = df_test[
        [
            'drug_row_id',
            'drug_col_id',
            'cell_line_name',
            'study_name',
            'tissue',
            'synergy_loewe',
            'synergy_loewe_bin'
        ]
    ].copy()

    predictions['y_true'] = y_test
    predictions['y_prob'] = y_prob_test
    predictions['y_pred_best_f1'] = (y_prob_test >= best_thr_f1).astype(int)
    predictions['y_pred_best_precision'] = (y_prob_test >= best_thr_precision).astype(int)
    predictions['threshold_best_f1'] = best_thr_f1
    predictions['threshold_best_precision'] = best_thr_precision

    predictions.to_csv(
        os.path.join(output_dir, 'predictions', f'predictions_fold_{fold_idx:02d}.csv'),
        index=False
    )

    if SAVE_MODELS:
        torch.save(
            model.state_dict(),
            os.path.join(output_dir, 'models', f'triple_branch_mlp_fold_{fold_idx:02d}.pt')
        )

    n_train = len(y_train_np)
    n_val = len(y_val)
    n_test = len(y_test)

    split_info = {
        'fold': fold_idx,
        'n_train': n_train,
        'n_val': n_val,
        'n_test': n_test,
        'train_pos': int(np.sum(y_train_np)),
        'val_pos': int(np.sum(y_val)),
        'test_pos': int(np.sum(y_test)),
        'train_pos_pct': 100 * np.mean(y_train_np),
        'val_pos_pct': 100 * np.mean(y_val),
        'test_pos_pct': 100 * np.mean(y_test),
        'baseline_auprc_test': float(np.mean(y_test)),
        'n_heldout_drugs': len(test_set),
        'heldout_drugs': ';'.join(sorted(test_set)),
        'heldout_overlap_train': heldout_overlap_train,
        'best_threshold_f1': best_thr_f1,
        'best_threshold_precision': best_thr_precision,
        'best_epoch': best_epoch,
        'best_val_loss': best_val_loss,
        'n_top_genes': len(top_genes),
    }

    cv_split_info.append(split_info)

    writer.add_text('Split/Info', json.dumps(split_info, indent=2))
    writer.add_text('Genes/TopGenes_First100', ', '.join(top_genes[:100]))

    print(
        f"Fold {fold_idx} | "
        f"Train={n_train} ({100*np.mean(y_train_np):.2f}% pos), "
        f"Val={n_val} ({100*np.mean(y_val):.2f}% pos), "
        f"Test={n_test} ({100*np.mean(y_test):.2f}% pos) | "
        f"BestEpoch={best_epoch} | "
        f"ThrF1={best_thr_f1:.2f}, ThrPrec={best_thr_precision:.2f} | "
        f"AUPRC={test_metrics_f1['auprc']:.4f} "
        f"(base={test_metrics_f1['baseline_auprc']:.4f}) | "
        f"F1={test_metrics_f1['f1']:.4f} | "
        f"Prec={test_metrics_f1['precision']:.4f} | "
        f"Rec={test_metrics_f1['recall']:.4f} | "
        f"MCC={test_metrics_f1['mcc']:.4f}"
    )

    writer.close()
    torch.cuda.empty_cache()
# ============================================================
# GUARDADO DE RESULTADOS
# ============================================================

df_val_f1 = pd.DataFrame(cv_val_results_f1)
df_val_precision = pd.DataFrame(cv_val_results_precision)
df_test_f1 = pd.DataFrame(cv_test_results_f1)
df_test_precision = pd.DataFrame(cv_test_results_precision)
df_split_info = pd.DataFrame(cv_split_info)
df_best_thresholds = pd.DataFrame(cv_best_thresholds)

df_val_thresholds = pd.concat(cv_val_threshold_metrics, ignore_index=True)
df_test_thresholds = pd.concat(cv_test_threshold_metrics, ignore_index=True)

df_val_f1.to_csv(os.path.join(output_dir, 'val_metrics_best_f1.csv'), index=False)
df_val_precision.to_csv(os.path.join(output_dir, 'val_metrics_best_precision.csv'), index=False)
df_test_f1.to_csv(os.path.join(output_dir, 'test_metrics_best_f1.csv'), index=False)
df_test_precision.to_csv(os.path.join(output_dir, 'test_metrics_best_precision.csv'), index=False)
df_split_info.to_csv(os.path.join(output_dir, 'split_info.csv'), index=False)
df_best_thresholds.to_csv(os.path.join(output_dir, 'best_thresholds.csv'), index=False)
df_val_thresholds.to_csv(os.path.join(output_dir, 'val_metrics_by_threshold.csv'), index=False)
df_test_thresholds.to_csv(os.path.join(output_dir, 'test_metrics_by_threshold.csv'), index=False)

if len(cv_histories) > 0:
    df_all_history = pd.concat(cv_histories, ignore_index=True)
    df_all_history.to_csv(os.path.join(output_dir, 'training_history_all_folds.csv'), index=False)


# ============================================================
# RESUMEN FINAL
# ============================================================

metric_cols = [
    'auc',
    'auprc',
    'baseline_auprc',
    'f1',
    'precision',
    'recall',
    'specificity',
    'balanced_accuracy',
    'mcc',
    'log_loss',
    'brier',
    'tp',
    'tn',
    'fp',
    'fn',
]

summary_f1 = (
    df_test_f1[metric_cols]
    .agg(['mean', 'std'])
    .T
    .reset_index()
    .rename(columns={'index': 'metric'})
)

summary_precision = (
    df_test_precision[metric_cols]
    .agg(['mean', 'std'])
    .T
    .reset_index()
    .rename(columns={'index': 'metric'})
)

summary_f1.to_csv(os.path.join(output_dir, 'summary_test_best_f1.csv'), index=False)
summary_precision.to_csv(os.path.join(output_dir, 'summary_test_best_precision.csv'), index=False)

print('\n' + '=' * 70)
print('RESULTADOS MLP TABULAR - LODO PARCIAL CV=10')
print('=' * 70)

print('\n[TEST - threshold elegido por mejor F1 en validacion]')
print(summary_f1)

print('\n[TEST - threshold conservador por mejor precision en validacion]')
print(summary_precision)

print('\nArchivos guardados en:', output_dir)


## TRANSFORMER LDO DISJUNTO A NIVEL DE MUESTRA

In [ ]:
import json
import copy
import shutil

from torch import nn
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
from PIL import Image

if not hasattr(Image, "Resampling"):
    class Resampling:
        LANCZOS = Image.ANTIALIAS

    Image.Resampling = Resampling


# ============================================================
# CONFIGURACION
# ============================================================

SAVE_MODELS = False

output_dir = 'results/zzz_disjoint_transformer'
tb_dir = os.path.join(output_dir, 'tensorboard')

if os.path.exists(output_dir):
    shutil.rmtree(output_dir)

os.makedirs(output_dir, exist_ok=True)
os.makedirs(os.path.join(output_dir, 'predictions'), exist_ok=True)
os.makedirs(os.path.join(output_dir, 'figures'), exist_ok=True)
os.makedirs(tb_dir, exist_ok=True)

if SAVE_MODELS:
    os.makedirs(os.path.join(output_dir, 'models'), exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

mlp_config = {
    'lr': 1e-3,
    'batch_size': 256,
    'epochs': 80,
    'patience': 12,
    'weight_decay': 1e-4,
    'drug_dim': 1024,
    'n_genes': 500,
    'random_state': 42,
    'use_pos_weight': True,
    'hidden_dim': 256,
    'embed_dim': 128,
    'n_heads': 4,
    'n_layers': 2,
}


experiment_config = {
    'model': 'DrugPairTransformerMLP',
    'architecture': (
        'Transformer encoder over [CLS, drug1, drug2] tokens fused with gene expression encoder.'
    ),
    'split_type': 'unique_sample_lodo',
    'n_splits': 10,
    'n_genes': mlp_config['n_genes'],
    'random_state': mlp_config['random_state'],
    'threshold_grid_min': 0.05,
    'threshold_grid_max': 0.95,
    'threshold_grid_step': 0.01,
    'mlp_config': mlp_config,
    'save_models': SAVE_MODELS,
    'device': str(device),
    'external_lodo_splits_path': 'results/shared_splits/unique_sample_lodo_indices.json',
    'lodo_definition': (
        'Unique-sample partial Leave-Drug-Out: each drug is assigned to one fold, '
        'and each sample is assigned to a single compatible test fold to avoid test overlap.'
    )
}


save_experiment_config(experiment_config, output_dir)




# ============================================================
# DATASET BASE
# ============================================================

base_df = X_final_df.copy()
base_df['drug_row_id'] = base_df['drug_row_id'].astype(str)
base_df['drug_col_id'] = base_df['drug_col_id'].astype(str)
base_df['synergy_loewe_bin'] = base_df['synergy_loewe_bin'].replace(-1, 0).astype(int)

thresholds = np.round(np.arange(0.05, 0.951, 0.01), 2)


# ============================================================
# CARGAR SPLITS UNIQUE-SAMPLE LODO COMPARTIDOS
# ============================================================

unique_lodo_path = experiment_config['external_lodo_splits_path']

with open(unique_lodo_path, 'r') as f:
    split_indices = json.load(f)

assert len(split_indices) == experiment_config['n_splits'], (
    f"Se esperaban {experiment_config['n_splits']} folds, "
    f"pero se cargaron {len(split_indices)}."
)

partition_rows = []

for split in split_indices:
    fold_idx = split['fold']
    test_idx = np.array(split['test_idx'])
    test_set = set(split['heldout_drugs'])

    fold_df = base_df.iloc[test_idx]

    n_examples = len(fold_df)
    pos_rate = fold_df['synergy_loewe_bin'].mean() if n_examples > 0 else np.nan

    partition_rows.append({
        'fold': fold_idx,
        'n_examples_test': n_examples,
        'positive_pct_test': pos_rate * 100,
        'baseline_auprc_test': pos_rate,
        'n_heldout_drugs': len(test_set)
    })

fold_partition_summary = pd.DataFrame(partition_rows)
fold_partition_summary.to_csv(
    os.path.join(output_dir, 'unique_sample_lodo_partition_summary.csv'),
    index=False
)

print('\nSplits Unique-Sample LODO cargados desde:', unique_lodo_path)
print('\nResumen particion Unique-Sample LODO:')
print(fold_partition_summary)

all_test_indices = []

for split in split_indices:
    all_test_indices.extend(split['test_idx'])

n_total_test_appearances = len(all_test_indices)
n_unique_test_samples = len(set(all_test_indices))
n_repeated = n_total_test_appearances - n_unique_test_samples
overlap_pct = (
    100 * n_repeated / n_total_test_appearances
    if n_total_test_appearances > 0 else 0.0
)

test_overlap_info = {
    'total_test_appearances': int(n_total_test_appearances),
    'unique_test_samples': int(n_unique_test_samples),
    'repeated_test_samples': int(n_repeated),
    'overlap_pct': float(overlap_pct),
}

with open(os.path.join(output_dir, 'test_overlap_summary.json'), 'w') as f:
    json.dump(test_overlap_info, f, indent=4)

print('\nSolapamiento real entre tests:')
print(test_overlap_info)

# ============================================================
# BUCLE CV
# ============================================================

cv_val_results_f1 = []
cv_val_results_precision = []
cv_test_results_f1 = []
cv_test_results_precision = []
cv_split_info = []
cv_val_threshold_metrics = []
cv_test_threshold_metrics = []
cv_best_thresholds = []
cv_histories = []



for split in split_indices:
    fold_idx = split['fold']
    train_idx = np.array(split['train_idx'])
    test_idx = np.array(split['test_idx'])
    test_set = set(split['heldout_drugs'])

    writer = SummaryWriter(log_dir=os.path.join(tb_dir, f'Fold_{fold_idx}'))
    print(f'\nPROCESANDO FOLD {fold_idx}/{experiment_config["n_splits"]} - TRANSFORMER + MLP UNIQUE-SAMPLE LODO')

    df_train_raw = base_df.iloc[train_idx].reset_index(drop=True)
    df_test_raw = base_df.iloc[test_idx].reset_index(drop=True)

    if len(df_train_raw) == 0 or len(df_test_raw) == 0:
        print(f'Fold {fold_idx}: sin datos suficientes, se omite.')
        writer.close()
        continue

    assert (
        df_test_raw['drug_row_id'].isin(test_set)
        | df_test_raw['drug_col_id'].isin(test_set)
    ).all(), f'Fold {fold_idx}: hay muestras test sin droga held-out.'

    train_drugs = set(pd.concat([
        df_train_raw['drug_row_id'],
        df_train_raw['drug_col_id']
    ]).astype(str))

    heldout_overlap_train = len(train_drugs & test_set)
    assert heldout_overlap_train == 0, f'Fold {fold_idx}: drogas held-out aparecen en train.'

    df_train, df_val, df_test, top_genes = prepare_fold_data_no_leakage(
        df_train_raw,
        df_test_raw,
        data_expr,
        n_genes=experiment_config['n_genes'],
        random_state=experiment_config['random_state']
    )

    y_train_np = df_train['synergy_loewe_bin'].astype(int).to_numpy()

    if len(np.unique(y_train_np)) < 2:
        print(f'Fold {fold_idx}: train con una sola clase, se omite.')
        writer.close()
        continue

    train_loader = DataLoader(
        SynergyDataset(df_train, top_genes),
        batch_size=mlp_config['batch_size'],
        shuffle=True,
    )

    val_loader = DataLoader(
        SynergyDataset(df_val, top_genes),
        batch_size=mlp_config['batch_size'],
        shuffle=False
    )

    test_loader = DataLoader(
        SynergyDataset(df_test, top_genes),
        batch_size=mlp_config['batch_size'],
        shuffle=False
    )

    model = DrugPairTransformerMLP(
        hiddim=mlp_config['hidden_dim'],
        n_genes=experiment_config['n_genes'],
        drug_dim=mlp_config['drug_dim'],
        embed_dim=mlp_config['embed_dim'],
        n_heads=mlp_config['n_heads'],
        n_layers=mlp_config['n_layers']
    ).to(device)


    model, history_df, best_epoch, best_val_loss = train_torch_model_fold(
        model,
        train_loader,
        val_loader,
        y_train_np,
        mlp_config,
        device,
        writer,
        fold_idx
    )

    history_df.to_csv(
        os.path.join(output_dir, f'training_history_fold_{fold_idx:02d}.csv'),
        index=False
    )
    cv_histories.append(history_df)

    y_val, y_prob_val = predict_torch_model(model, val_loader, device)
    y_test, y_prob_test = predict_torch_model(model, test_loader, device)

    df_val_thr, best_thr_f1, best_thr_precision = find_best_thresholds(
        y_val,
        y_prob_val,
        thresholds
    )

    df_val_thr.insert(0, 'fold', fold_idx)
    df_val_thr.insert(1, 'split', 'val')
    df_val_thr.insert(2, 'threshold_type', 'grid')
    cv_val_threshold_metrics.append(df_val_thr)

    test_thr_rows = []
    for thr in thresholds:
        m_test_thr = compute_metrics_at_threshold(y_test, y_prob_test, thr)
        m_test_thr.update({
            'fold': fold_idx,
            'split': 'test',
            'threshold_type': 'grid',
        })
        test_thr_rows.append(m_test_thr)

    df_test_thr = pd.DataFrame(test_thr_rows)
    cv_test_threshold_metrics.append(df_test_thr)

    val_metrics_f1 = compute_metrics_at_threshold(y_val, y_prob_val, best_thr_f1)
    val_metrics_precision = compute_metrics_at_threshold(y_val, y_prob_val, best_thr_precision)

    test_metrics_f1 = compute_metrics_at_threshold(y_test, y_prob_test, best_thr_f1)
    test_metrics_precision = compute_metrics_at_threshold(y_test, y_prob_test, best_thr_precision)

    val_metrics_f1.update({'fold': fold_idx, 'split': 'val', 'threshold_type': 'best_f1'})
    val_metrics_precision.update({'fold': fold_idx, 'split': 'val', 'threshold_type': 'best_precision'})
    test_metrics_f1.update({'fold': fold_idx, 'split': 'test', 'threshold_type': 'best_f1'})
    test_metrics_precision.update({'fold': fold_idx, 'split': 'test', 'threshold_type': 'best_precision'})

    cv_val_results_f1.append(val_metrics_f1)
    cv_val_results_precision.append(val_metrics_precision)
    cv_test_results_f1.append(test_metrics_f1)
    cv_test_results_precision.append(test_metrics_precision)

    cv_best_thresholds.append({
        'fold': fold_idx,
        'best_threshold_f1': best_thr_f1,
        'best_threshold_precision': best_thr_precision,
        'val_f1_at_best_f1': val_metrics_f1['f1'],
        'val_precision_at_best_f1': val_metrics_f1['precision'],
        'val_recall_at_best_f1': val_metrics_f1['recall'],
        'val_precision_at_best_precision': val_metrics_precision['precision'],
        'val_recall_at_best_precision': val_metrics_precision['recall'],
        'val_f1_at_best_precision': val_metrics_precision['f1'],
        'best_epoch': best_epoch,
        'best_val_loss': best_val_loss,
    })

    log_curves_and_metrics(y_val, y_prob_val, val_metrics_f1, fold_idx, writer, tag='Val_BestF1')
    log_curves_and_metrics(y_val, y_prob_val, val_metrics_precision, fold_idx, writer, tag='Val_BestPrecision')
    log_curves_and_metrics(y_test, y_prob_test, test_metrics_f1, fold_idx, writer, tag='Test_BestF1')
    log_curves_and_metrics(y_test, y_prob_test, test_metrics_precision, fold_idx, writer, tag='Test_BestPrecision')

    predictions = df_test[
        [
            'drug_row_id',
            'drug_col_id',
            'cell_line_name',
            'study_name',
            'tissue',
            'synergy_loewe',
            'synergy_loewe_bin'
        ]
    ].copy()

    predictions['y_true'] = y_test
    predictions['y_prob'] = y_prob_test
    predictions['y_pred_best_f1'] = (y_prob_test >= best_thr_f1).astype(int)
    predictions['y_pred_best_precision'] = (y_prob_test >= best_thr_precision).astype(int)
    predictions['threshold_best_f1'] = best_thr_f1
    predictions['threshold_best_precision'] = best_thr_precision

    predictions.to_csv(
        os.path.join(output_dir, 'predictions', f'predictions_fold_{fold_idx:02d}.csv'),
        index=False
    )

    if SAVE_MODELS:
        torch.save(
            model.state_dict(),
            os.path.join(output_dir, 'models', f'triple_branch_mlp_fold_{fold_idx:02d}.pt')
        )

    n_train = len(y_train_np)
    n_val = len(y_val)
    n_test = len(y_test)

    split_info = {
        'fold': fold_idx,
        'n_train': n_train,
        'n_val': n_val,
        'n_test': n_test,
        'train_pos': int(np.sum(y_train_np)),
        'val_pos': int(np.sum(y_val)),
        'test_pos': int(np.sum(y_test)),
        'train_pos_pct': 100 * np.mean(y_train_np),
        'val_pos_pct': 100 * np.mean(y_val),
        'test_pos_pct': 100 * np.mean(y_test),
        'baseline_auprc_test': float(np.mean(y_test)),
        'n_heldout_drugs': len(test_set),
        'heldout_drugs': ';'.join(sorted(test_set)),
        'heldout_overlap_train': heldout_overlap_train,
        'best_threshold_f1': best_thr_f1,
        'best_threshold_precision': best_thr_precision,
        'best_epoch': best_epoch,
        'best_val_loss': best_val_loss,
        'n_top_genes': len(top_genes),
    }

    cv_split_info.append(split_info)

    writer.add_text('Split/Info', json.dumps(split_info, indent=2))
    writer.add_text('Genes/TopGenes_First100', ', '.join(top_genes[:100]))

    print(
        f"Fold {fold_idx} | "
        f"Train={n_train} ({100*np.mean(y_train_np):.2f}% pos), "
        f"Val={n_val} ({100*np.mean(y_val):.2f}% pos), "
        f"Test={n_test} ({100*np.mean(y_test):.2f}% pos) | "
        f"BestEpoch={best_epoch} | "
        f"ThrF1={best_thr_f1:.2f}, ThrPrec={best_thr_precision:.2f} | "
        f"AUPRC={test_metrics_f1['auprc']:.4f} "
        f"(base={test_metrics_f1['baseline_auprc']:.4f}) | "
        f"F1={test_metrics_f1['f1']:.4f} | "
        f"Prec={test_metrics_f1['precision']:.4f} | "
        f"Rec={test_metrics_f1['recall']:.4f} | "
        f"MCC={test_metrics_f1['mcc']:.4f}"
    )

    writer.close()
    torch.cuda.empty_cache()

    # ============================================================
# GUARDADO DE RESULTADOS
# ============================================================

df_val_f1 = pd.DataFrame(cv_val_results_f1)
df_val_precision = pd.DataFrame(cv_val_results_precision)
df_test_f1 = pd.DataFrame(cv_test_results_f1)
df_test_precision = pd.DataFrame(cv_test_results_precision)
df_split_info = pd.DataFrame(cv_split_info)
df_best_thresholds = pd.DataFrame(cv_best_thresholds)

df_val_thresholds = pd.concat(cv_val_threshold_metrics, ignore_index=True)
df_test_thresholds = pd.concat(cv_test_threshold_metrics, ignore_index=True)

df_val_f1.to_csv(os.path.join(output_dir, 'val_metrics_best_f1.csv'), index=False)
df_val_precision.to_csv(os.path.join(output_dir, 'val_metrics_best_precision.csv'), index=False)
df_test_f1.to_csv(os.path.join(output_dir, 'test_metrics_best_f1.csv'), index=False)
df_test_precision.to_csv(os.path.join(output_dir, 'test_metrics_best_precision.csv'), index=False)
df_split_info.to_csv(os.path.join(output_dir, 'split_info.csv'), index=False)
df_best_thresholds.to_csv(os.path.join(output_dir, 'best_thresholds.csv'), index=False)
df_val_thresholds.to_csv(os.path.join(output_dir, 'val_metrics_by_threshold.csv'), index=False)
df_test_thresholds.to_csv(os.path.join(output_dir, 'test_metrics_by_threshold.csv'), index=False)

if len(cv_histories) > 0:
    df_all_history = pd.concat(cv_histories, ignore_index=True)
    df_all_history.to_csv(os.path.join(output_dir, 'training_history_all_folds.csv'), index=False)




# ============================================================
# RESUMEN FINAL
# ============================================================

metric_cols = [
    'auc',
    'auprc',
    'baseline_auprc',
    'f1',
    'precision',
    'recall',
    'specificity',
    'balanced_accuracy',
    'mcc',
    'log_loss',
    'brier',
    'tp',
    'tn',
    'fp',
    'fn',
]

summary_f1 = (
    df_test_f1[metric_cols]
    .agg(['mean', 'std'])
    .T
    .reset_index()
    .rename(columns={'index': 'metric'})
)

summary_precision = (
    df_test_precision[metric_cols]
    .agg(['mean', 'std'])
    .T
    .reset_index()
    .rename(columns={'index': 'metric'})
)

summary_f1.to_csv(os.path.join(output_dir, 'summary_test_best_f1.csv'), index=False)
summary_precision.to_csv(os.path.join(output_dir, 'summary_test_best_precision.csv'), index=False)

print('\n' + '=' * 70)
print('RESULTADOS DRUG PAIR TRANSFORMER MLP - UNIQUE-SAMPLE LODO CV=10')
print('=' * 70)

print('\n[TEST - threshold elegido por mejor F1 en validacion]')
print(summary_f1)

print('\n[TEST - threshold conservador por mejor precision en validacion]')
print(summary_precision)

print('\nArchivos guardados en:', output_dir)


## GNN LDO DISJUNTO A NIVEL DE MUESTRA

In [ ]:
print(len(X_final_df), len(X_graph_df))
print((X_final_df[['drug_row_id', 'drug_col_id', 'cell_line_name']].astype(str).reset_index(drop=True)
       == X_graph_df[['drug_row_id', 'drug_col_id', 'cell_line_name']].astype(str).reset_index(drop=True)).all())


In [ ]:
import json
import copy


from torch import nn
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
from PIL import Image

if not hasattr(Image, "Resampling"):
    class Resampling:
        LANCZOS = Image.ANTIALIAS

    Image.Resampling = Resampling


# ============================================================
# CONFIGURACION
# ============================================================

SAVE_MODELS = False

output_dir = 'results/zzz_disjoint_gnn'
tb_dir = os.path.join(output_dir, 'tensorboard')

if os.path.exists(output_dir):
    shutil.rmtree(output_dir)

os.makedirs(output_dir, exist_ok=True)
os.makedirs(os.path.join(output_dir, 'predictions'), exist_ok=True)
os.makedirs(os.path.join(output_dir, 'figures'), exist_ok=True)
os.makedirs(tb_dir, exist_ok=True)

if SAVE_MODELS:
    os.makedirs(os.path.join(output_dir, 'models'), exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

mlp_config = {
    'lr': 1e-3,
    'batch_size': 256,
    'epochs': 80,
    'patience': 12,
    'weight_decay': 1e-4,
    'n_genes': 500,
    'random_state': 42,
    'use_pos_weight': True,
    'hidden_dim': 256
}


experiment_config = {
    'model': 'GraphDrugSynergyModel',
    'architecture': (
        'GIN-based molecular graph encoder for each drug, fused with selected gene expression.'
    ),
    'split_type': 'unique_sample_lodo',
    'n_splits': 10,
    'n_genes': mlp_config['n_genes'],
    'random_state': mlp_config['random_state'],
    'threshold_grid_min': 0.05,
    'threshold_grid_max': 0.95,
    'threshold_grid_step': 0.01,
    'mlp_config': mlp_config,
    'save_models': SAVE_MODELS,
    'device': str(device),
    'external_lodo_splits_path': 'results/shared_splits/unique_sample_lodo_indices.json'
}



save_experiment_config(experiment_config, output_dir)



# ============================================================
# DATASET BASE
# ============================================================

base_df = X_graph_df.copy()
base_df['drug_row_id'] = base_df['drug_row_id'].astype(str)
base_df['drug_col_id'] = base_df['drug_col_id'].astype(str)
base_df['synergy_loewe_bin'] = base_df['synergy_loewe_bin'].replace(-1, 0).astype(int)

print(base_df[['smiles_row', 'smiles_col']].isna().sum())
unique_smiles = pd.concat([
    base_df['smiles_row'],
    base_df['smiles_col']
]).dropna().unique()

graph_cache = {
    smiles: smiles_to_graph(smiles)
    for smiles in unique_smiles
}

graph_cache = {
    smiles: graph
    for smiles, graph in graph_cache.items()
    if graph is not None
}

print("Grafos validos:", len(graph_cache), "de", len(unique_smiles))

valid_smiles = set(graph_cache.keys())

before = len(base_df)

base_df = base_df[
    base_df['smiles_row'].isin(valid_smiles)
    & base_df['smiles_col'].isin(valid_smiles)
].reset_index(drop=True)

print("Filas eliminadas por grafos invalidos:", before - len(base_df))

thresholds = np.round(np.arange(0.05, 0.951, 0.01), 2)

unique_lodo_path = experiment_config['external_lodo_splits_path']

with open(unique_lodo_path, 'r') as f:
    split_indices = json.load(f)

assert len(split_indices) == experiment_config['n_splits'], (
    f"Se esperaban {experiment_config['n_splits']} folds, "
    f"pero se cargaron {len(split_indices)}."
)

partition_rows = []

for split in split_indices:
    fold_idx = split['fold']
    test_idx = np.array(split['test_idx'])
    test_set = set(split['heldout_drugs'])

    fold_df = base_df.iloc[test_idx]

    n_examples = len(fold_df)
    pos_rate = fold_df['synergy_loewe_bin'].mean() if n_examples > 0 else np.nan

    partition_rows.append({
        'fold': fold_idx,
        'n_examples_test': n_examples,
        'positive_pct_test': pos_rate * 100,
        'baseline_auprc_test': pos_rate,
        'n_heldout_drugs': len(test_set)
    })

fold_partition_summary = pd.DataFrame(partition_rows)
fold_partition_summary.to_csv(
    os.path.join(output_dir, 'unique_sample_lodo_partition_summary.csv'),
    index=False
)

print('\nSplits Unique-Sample LODO cargados desde:', unique_lodo_path)
print('\nResumen particion Unique-Sample LODO:')
print(fold_partition_summary)

all_test_indices = []

for split in split_indices:
    all_test_indices.extend(split['test_idx'])

n_total_test_appearances = len(all_test_indices)
n_unique_test_samples = len(set(all_test_indices))
n_repeated = n_total_test_appearances - n_unique_test_samples
overlap_pct = (
    100 * n_repeated / n_total_test_appearances
    if n_total_test_appearances > 0 else 0.0
)

test_overlap_info = {
    'total_test_appearances': int(n_total_test_appearances),
    'unique_test_samples': int(n_unique_test_samples),
    'repeated_test_samples': int(n_repeated),
    'overlap_pct': float(overlap_pct),
}

with open(os.path.join(output_dir, 'test_overlap_summary.json'), 'w') as f:
    json.dump(test_overlap_info, f, indent=4)

print('\nSolapamiento real entre tests:')
print(test_overlap_info)


import warnings

warnings.filterwarnings(
    "ignore",
    message=".*torch-scatter.*",
    category=UserWarning
)
# ============================================================
# BUCLE CV
# ============================================================

cv_val_results_f1 = []
cv_val_results_precision = []
cv_test_results_f1 = []
cv_test_results_precision = []
cv_split_info = []
cv_val_threshold_metrics = []
cv_test_threshold_metrics = []
cv_best_thresholds = []
cv_histories = []


unique_smiles = pd.concat([
    base_df['smiles_row'],
    base_df['smiles_col']
]).dropna().unique()


print(base_df[['smiles_row', 'smiles_col']].isna().sum())

graph_cache = {
    smiles: smiles_to_graph(smiles)
    for smiles in unique_smiles
}

graph_cache = {
    smiles: graph
    for smiles, graph in graph_cache.items()
    if graph is not None
}
print("Grafos validos:", len(graph_cache), "de", len(unique_smiles))

valid_smiles = set(graph_cache.keys())

before = len(base_df)

base_df = base_df[
    base_df['smiles_row'].isin(valid_smiles)
    & base_df['smiles_col'].isin(valid_smiles)
].reset_index(drop=True)

print("Filas eliminadas por grafos invalidos:", before - len(base_df))


for split in split_indices:
    fold_idx = split['fold']
    train_idx = np.array(split['train_idx'])
    test_idx = np.array(split['test_idx'])
    test_set = set(split['heldout_drugs'])
    writer = SummaryWriter(log_dir=os.path.join(tb_dir, f'Fold_{fold_idx}'))

    print(f'\nPROCESANDO FOLD {fold_idx}/{experiment_config["n_splits"]} - GNN DRUG SYNERGY UNIQUE-SAMPLE LODO')


    df_train_raw = base_df.iloc[train_idx].reset_index(drop=True)
    df_test_raw = base_df.iloc[test_idx].reset_index(drop=True)

    if len(df_train_raw) == 0 or len(df_test_raw) == 0:
        print(f'Fold {fold_idx}: sin datos suficientes, se omite.')
        writer.close()
        continue

    assert (
        df_test_raw['drug_row_id'].isin(test_set)
        | df_test_raw['drug_col_id'].isin(test_set)
    ).all(), f'Fold {fold_idx}: hay muestras test sin droga held-out.'

    train_drugs = set(pd.concat([
        df_train_raw['drug_row_id'],
        df_train_raw['drug_col_id']
    ]).astype(str))

    heldout_overlap_train = len(train_drugs & test_set)
    assert heldout_overlap_train == 0, f'Fold {fold_idx}: drogas held-out aparecen en train.'

    df_train, df_val, df_test, top_genes = prepare_fold_data_no_leakage(
        df_train_raw,
        df_test_raw,
        data_expr,
        n_genes=experiment_config['n_genes'],
        random_state=experiment_config['random_state']
    )

    y_train_np = df_train['synergy_loewe_bin'].astype(int).to_numpy()

    if len(np.unique(y_train_np)) < 2:
        print(f'Fold {fold_idx}: train con una sola clase, se omite.')
        writer.close()
        continue

    train_loader = DataLoader(
        GraphSynergyDataset(df_train, top_genes, graph_cache),
        batch_size=mlp_config['batch_size'],
        shuffle=True,
        collate_fn=graph_collate_fn
    )

    val_loader = DataLoader(
        GraphSynergyDataset(df_val, top_genes, graph_cache),
        batch_size=mlp_config['batch_size'],
        shuffle=False,
        collate_fn=graph_collate_fn
    )

    test_loader = DataLoader(
        GraphSynergyDataset(df_test, top_genes, graph_cache),
        batch_size=mlp_config['batch_size'],
        shuffle=False,
        collate_fn=graph_collate_fn
    )

    node_dim = next(iter(graph_cache.values())).x.shape[1]

    model = GraphDrugSynergyModel(
        node_dim=node_dim,
        gene_dim=experiment_config['n_genes'],
        hidden_dim=mlp_config['hidden_dim']
    ).to(device)
    
    model, history_df, best_epoch, best_val_loss = train_torch_graph_model_fold(
        model,
        train_loader,
        val_loader,
        y_train_np,
        mlp_config,
        device,
        writer,
        fold_idx
    )


    history_df.to_csv(
        os.path.join(output_dir, f'training_history_fold_{fold_idx:02d}.csv'),
        index=False
    )
    cv_histories.append(history_df)

    y_val, y_prob_val = predict_torch_graph_model(model, val_loader, device)
    y_test, y_prob_test = predict_torch_graph_model(model, test_loader, device)

    df_val_thr, best_thr_f1, best_thr_precision = find_best_thresholds(
        y_val,
        y_prob_val,
        thresholds
    )

    df_val_thr.insert(0, 'fold', fold_idx)
    df_val_thr.insert(1, 'split', 'val')
    df_val_thr.insert(2, 'threshold_type', 'grid')
    cv_val_threshold_metrics.append(df_val_thr)

    test_thr_rows = []
    for thr in thresholds:
        m_test_thr = compute_metrics_at_threshold(y_test, y_prob_test, thr)
        m_test_thr.update({
            'fold': fold_idx,
            'split': 'test',
            'threshold_type': 'grid',
        })
        test_thr_rows.append(m_test_thr)

    df_test_thr = pd.DataFrame(test_thr_rows)
    cv_test_threshold_metrics.append(df_test_thr)

    val_metrics_f1 = compute_metrics_at_threshold(y_val, y_prob_val, best_thr_f1)
    val_metrics_precision = compute_metrics_at_threshold(y_val, y_prob_val, best_thr_precision)

    test_metrics_f1 = compute_metrics_at_threshold(y_test, y_prob_test, best_thr_f1)
    test_metrics_precision = compute_metrics_at_threshold(y_test, y_prob_test, best_thr_precision)

    val_metrics_f1.update({'fold': fold_idx, 'split': 'val', 'threshold_type': 'best_f1'})
    val_metrics_precision.update({'fold': fold_idx, 'split': 'val', 'threshold_type': 'best_precision'})
    test_metrics_f1.update({'fold': fold_idx, 'split': 'test', 'threshold_type': 'best_f1'})
    test_metrics_precision.update({'fold': fold_idx, 'split': 'test', 'threshold_type': 'best_precision'})

    cv_val_results_f1.append(val_metrics_f1)
    cv_val_results_precision.append(val_metrics_precision)
    cv_test_results_f1.append(test_metrics_f1)
    cv_test_results_precision.append(test_metrics_precision)

    cv_best_thresholds.append({
        'fold': fold_idx,
        'best_threshold_f1': best_thr_f1,
        'best_threshold_precision': best_thr_precision,
        'val_f1_at_best_f1': val_metrics_f1['f1'],
        'val_precision_at_best_f1': val_metrics_f1['precision'],
        'val_recall_at_best_f1': val_metrics_f1['recall'],
        'val_precision_at_best_precision': val_metrics_precision['precision'],
        'val_recall_at_best_precision': val_metrics_precision['recall'],
        'val_f1_at_best_precision': val_metrics_precision['f1'],
        'best_epoch': best_epoch,
        'best_val_loss': best_val_loss,
    })

    log_curves_and_metrics(y_val, y_prob_val, val_metrics_f1, fold_idx, writer, tag='Val_BestF1')
    log_curves_and_metrics(y_val, y_prob_val, val_metrics_precision, fold_idx, writer, tag='Val_BestPrecision')
    log_curves_and_metrics(y_test, y_prob_test, test_metrics_f1, fold_idx, writer, tag='Test_BestF1')
    log_curves_and_metrics(y_test, y_prob_test, test_metrics_precision, fold_idx, writer, tag='Test_BestPrecision')

    predictions = df_test[
        [
            'drug_row_id',
            'drug_col_id',
            'cell_line_name',
            'study_name',
            'tissue',
            'synergy_loewe',
            'synergy_loewe_bin'
        ]
    ].copy()

    predictions['y_true'] = y_test
    predictions['y_prob'] = y_prob_test
    predictions['y_pred_best_f1'] = (y_prob_test >= best_thr_f1).astype(int)
    predictions['y_pred_best_precision'] = (y_prob_test >= best_thr_precision).astype(int)
    predictions['threshold_best_f1'] = best_thr_f1
    predictions['threshold_best_precision'] = best_thr_precision

    predictions.to_csv(
        os.path.join(output_dir, 'predictions', f'predictions_fold_{fold_idx:02d}.csv'),
        index=False
    )

    if SAVE_MODELS:
        torch.save(
            model.state_dict(),
            os.path.join(output_dir, 'models', f'triple_branch_mlp_fold_{fold_idx:02d}.pt')
        )

    n_train = len(y_train_np)
    n_val = len(y_val)
    n_test = len(y_test)

    split_info = {
        'fold': fold_idx,
        'n_train': n_train,
        'n_val': n_val,
        'n_test': n_test,
        'train_pos': int(np.sum(y_train_np)),
        'val_pos': int(np.sum(y_val)),
        'test_pos': int(np.sum(y_test)),
        'train_pos_pct': 100 * np.mean(y_train_np),
        'val_pos_pct': 100 * np.mean(y_val),
        'test_pos_pct': 100 * np.mean(y_test),
        'baseline_auprc_test': float(np.mean(y_test)),
        'n_heldout_drugs': len(test_set),
        'heldout_drugs': ';'.join(sorted(test_set)),
        'heldout_overlap_train': heldout_overlap_train,
        'best_threshold_f1': best_thr_f1,
        'best_threshold_precision': best_thr_precision,
        'best_epoch': best_epoch,
        'best_val_loss': best_val_loss,
        'n_top_genes': len(top_genes),
    }

    cv_split_info.append(split_info)

    writer.add_text('Split/Info', json.dumps(split_info, indent=2))
    writer.add_text('Genes/TopGenes_First100', ', '.join(top_genes[:100]))

    print(
        f"Fold {fold_idx} | "
        f"Train={n_train} ({100*np.mean(y_train_np):.2f}% pos), "
        f"Val={n_val} ({100*np.mean(y_val):.2f}% pos), "
        f"Test={n_test} ({100*np.mean(y_test):.2f}% pos) | "
        f"BestEpoch={best_epoch} | "
        f"ThrF1={best_thr_f1:.2f}, ThrPrec={best_thr_precision:.2f} | "
        f"AUPRC={test_metrics_f1['auprc']:.4f} "
        f"(base={test_metrics_f1['baseline_auprc']:.4f}) | "
        f"F1={test_metrics_f1['f1']:.4f} | "
        f"Prec={test_metrics_f1['precision']:.4f} | "
        f"Rec={test_metrics_f1['recall']:.4f} | "
        f"MCC={test_metrics_f1['mcc']:.4f}"
    )

    writer.close()
    torch.cuda.empty_cache()

    
    
# ============================================================
# GUARDADO DE RESULTADOS
# ============================================================

df_val_f1 = pd.DataFrame(cv_val_results_f1)
df_val_precision = pd.DataFrame(cv_val_results_precision)
df_test_f1 = pd.DataFrame(cv_test_results_f1)
df_test_precision = pd.DataFrame(cv_test_results_precision)
df_split_info = pd.DataFrame(cv_split_info)
df_best_thresholds = pd.DataFrame(cv_best_thresholds)

df_val_thresholds = pd.concat(cv_val_threshold_metrics, ignore_index=True)
df_test_thresholds = pd.concat(cv_test_threshold_metrics, ignore_index=True)

df_val_f1.to_csv(os.path.join(output_dir, 'val_metrics_best_f1.csv'), index=False)
df_val_precision.to_csv(os.path.join(output_dir, 'val_metrics_best_precision.csv'), index=False)
df_test_f1.to_csv(os.path.join(output_dir, 'test_metrics_best_f1.csv'), index=False)
df_test_precision.to_csv(os.path.join(output_dir, 'test_metrics_best_precision.csv'), index=False)
df_split_info.to_csv(os.path.join(output_dir, 'split_info.csv'), index=False)
df_best_thresholds.to_csv(os.path.join(output_dir, 'best_thresholds.csv'), index=False)
df_val_thresholds.to_csv(os.path.join(output_dir, 'val_metrics_by_threshold.csv'), index=False)
df_test_thresholds.to_csv(os.path.join(output_dir, 'test_metrics_by_threshold.csv'), index=False)

if len(cv_histories) > 0:
    df_all_history = pd.concat(cv_histories, ignore_index=True)
    df_all_history.to_csv(os.path.join(output_dir, 'training_history_all_folds.csv'), index=False)




# ============================================================
# RESUMEN FINAL
# ============================================================

metric_cols = [
    'auc',
    'auprc',
    'baseline_auprc',
    'f1',
    'precision',
    'recall',
    'specificity',
    'balanced_accuracy',
    'mcc',
    'log_loss',
    'brier',
    'tp',
    'tn',
    'fp',
    'fn',
]

summary_f1 = (
    df_test_f1[metric_cols]
    .agg(['mean', 'std'])
    .T
    .reset_index()
    .rename(columns={'index': 'metric'})
)

summary_precision = (
    df_test_precision[metric_cols]
    .agg(['mean', 'std'])
    .T
    .reset_index()
    .rename(columns={'index': 'metric'})
)

summary_f1.to_csv(os.path.join(output_dir, 'summary_test_best_f1.csv'), index=False)
summary_precision.to_csv(os.path.join(output_dir, 'summary_test_best_precision.csv'), index=False)

print('\n' + '=' * 70)
print('RESULTADOS GNN DRUG SYNERGY MODEL - UNIQUE-SAMPLE LODO CV=10')
print('=' * 70)

print('\n[TEST - threshold elegido por mejor F1 en validacion]')
print(summary_f1)

print('\n[TEST - threshold conservador por mejor precision en validacion]')
print(summary_precision)

print('\nArchivos guardados en:', output_dir)


## Estudios postresultados

In [ ]:
RESULTS_DIR = "results"
analysis_dir = os.path.join(RESULTS_DIR, "analysis")
os.makedirs(analysis_dir, exist_ok=True)


# ============================================================
# LECTURA DE RESULTADOS
# ============================================================

def read_summary_as_row(result_dir, threshold_type):
    summary_path = os.path.join(result_dir, f"summary_test_{threshold_type}.csv")
    config_path = os.path.join(result_dir, "experiment_config.json")

    if not os.path.exists(summary_path):
        return None

    df = pd.read_csv(summary_path)

    row = {}

    if os.path.exists(config_path):
        with open(config_path, "r") as f:
            config = json.load(f)

        row["model"] = config.get("model", os.path.basename(result_dir))
        row["split_type"] = config.get("split_type", "unknown")
        row["n_splits"] = config.get("n_splits", np.nan)
    else:
        row["model"] = os.path.basename(result_dir)
        row["split_type"] = "unknown"
        row["n_splits"] = np.nan

    row["result_folder"] = os.path.basename(result_dir)
    row["threshold_type"] = threshold_type.replace("best_", "")

    for _, r in df.iterrows():
        metric = r["metric"]
        row[f"{metric}_mean"] = r["mean"]
        row[f"{metric}_std"] = r["std"]

    return row


rows = []

for folder in sorted(os.listdir(RESULTS_DIR)):
    result_dir = os.path.join(RESULTS_DIR, folder)

    if not os.path.isdir(result_dir):
        continue

    for threshold_type in ["best_f1", "best_precision"]:
        row = read_summary_as_row(result_dir, threshold_type)
        if row is not None:
            rows.append(row)

global_results = pd.DataFrame(rows)


# ============================================================
# LIMPIEZA: QUITAR MODELOS DE INTERACCION
# ============================================================

global_results = global_results[
    ~global_results["result_folder"].str.contains(
        "interactiondiff|interaction",
        case=False,
        na=False
    )
].copy()


# ============================================================
# ASIGNAR PROTOCOLO Y NOMBRES BONITOS
# ============================================================

def assign_protocol(folder):
    folder = str(folder).lower()

    if "stratified" in folder:
        return "Stratified KFold"

    if folder.startswith("zzz"):
        return "Disjoint LDO"

    return "Partial LDO"


def pretty_model_name(folder, model):
    folder = str(folder).lower()

    if "random_forest" in folder:
        return "Random Forest"
    if "xgboost" in folder:
        return "XGBoost"
    if "basic_mlp" in folder or "disjoint_mlp" in folder:
        return "Early Branch MLP"
    if "triplebranch" in folder or "triple_branch" in folder:
        return "Multimodal MLP"
    if "transformer" in folder or "drug_pair_transformer" in folder:
        return "Drug-Pair Transformer MLP"
    if "gnn" in folder:
        return "Graph Drug Synergy Model"

    return str(model)


global_results["protocol"] = global_results["result_folder"].apply(assign_protocol)

global_results["model_pretty"] = global_results.apply(
    lambda r: pretty_model_name(r["result_folder"], r["model"]),
    axis=1
)

protocol_order = ["Stratified KFold", "Partial LDO", "Disjoint LDO"]

global_results["protocol"] = pd.Categorical(
    global_results["protocol"],
    categories=protocol_order,
    ordered=True
)

global_results = global_results.sort_values(
    ["protocol", "threshold_type", "auprc_mean"],
    ascending=[True, True, False]
).reset_index(drop=True)


# ============================================================
# COLUMNAS PARA TABLAS
# ============================================================

cols_to_show = [
    "model_pretty",
    "model",
    "split_type",
    "protocol",
    "threshold_type",
    "result_folder",
    "auprc_mean",
    "auprc_std",
    "baseline_auprc_mean",
    "auc_mean",
    "f1_mean",
    "precision_mean",
    "recall_mean",
    "specificity_mean",
    "balanced_accuracy_mean",
    "mcc_mean",
]


# ============================================================
# CREAR 6 TABLAS
# ============================================================

def make_protocol_table(df, protocol, threshold):
    table = df[
        (df["protocol"] == protocol)
        & (df["threshold_type"] == threshold)
    ][cols_to_show].copy()

    if threshold == "f1":
        table = table.sort_values(
            ["auprc_mean"],
            ascending=False
        )
    else:
        table = table.sort_values(
            ["precision_mean", "recall_mean", "auprc_mean"],
            ascending=[False, False, False]
        )

    return table.reset_index(drop=True)


table_stratified_f1 = make_protocol_table(global_results, "Stratified KFold", "f1")
table_stratified_precision = make_protocol_table(global_results, "Stratified KFold", "precision")

table_lodo_partial_f1 = make_protocol_table(global_results, "Partial LODO", "f1")
table_lodo_partial_precision = make_protocol_table(global_results, "Partial LODO", "precision")

table_lodo_disjoint_f1 = make_protocol_table(global_results, "Disjoint LODO", "f1")
table_lodo_disjoint_precision = make_protocol_table(global_results, "Disjoint LODO", "precision")


# ============================================================
# GUARDAR 6 TABLAS
# ============================================================

table_stratified_f1.to_csv(
    os.path.join(analysis_dir, "table_stratified_cv_best_f1.csv"),
    index=False
)

table_stratified_precision.to_csv(
    os.path.join(analysis_dir, "table_stratified_cv_best_precision.csv"),
    index=False
)

table_lodo_partial_f1.to_csv(
    os.path.join(analysis_dir, "table_partial_lodo_best_f1.csv"),
    index=False
)

table_lodo_partial_precision.to_csv(
    os.path.join(analysis_dir, "table_partial_lodo_best_precision.csv"),
    index=False
)

table_lodo_disjoint_f1.to_csv(
    os.path.join(analysis_dir, "table_disjoint_lodo_best_f1.csv"),
    index=False
)

table_lodo_disjoint_precision.to_csv(
    os.path.join(analysis_dir, "table_disjoint_lodo_best_precision.csv"),
    index=False
)


# ============================================================
# MOSTRAR TABLAS
# ============================================================

display(table_stratified_f1)
display(table_stratified_precision)

display(table_lodo_partial_f1)
display(table_lodo_partial_precision)

display(table_lodo_disjoint_f1)
display(table_lodo_disjoint_precision)


# ============================================================
# GRAFICA AUPRC POR MODELO Y PROTOCOLO
# ============================================================
# AUPRC no depende del threshold, usamos best_f1 para no duplicar

plot_df = global_results[
    global_results["threshold_type"] == "f1"
].copy()

plot_df = plot_df.sort_values(
    ["protocol", "auprc_mean"],
    ascending=[True, False]
).reset_index(drop=True)

plt.figure(figsize=(11, 7))

ax = sns.barplot(
    data=plot_df,
    y="model_pretty",
    x="auprc_mean",
    hue="protocol",
    hue_order=protocol_order,
    errorbar=None
)

# Barras de desviacion estandar
for container in ax.containers:
    labels = []
    for bar in container:
        width = bar.get_width()
        labels.append(f"{width:.3f}" if width > 0 else "")

    ax.bar_label(
        container,
        labels=labels,
        padding=3,
        fontsize=9
    )

plt.xlabel("AUPRC medio")
plt.ylabel("Modelo")
plt.title("AUPRC por modelo y protocolo de validación")
plt.legend(
    title="Protocolo",
    loc="center left",
    bbox_to_anchor=(1, 0.5)
)
plt.xlim(0, min(1.0, plot_df["auprc_mean"].max() + 0.08))
plt.tight_layout()

plt.savefig(
    os.path.join(analysis_dir, "auprc_by_model_protocol_no_interaction.png"),
    dpi=300,
    bbox_inches="tight"
)

plt.show()


### AUPRC por estudio/tissue

In [ ]:
from sklearn.metrics import average_precision_score

RESULTS_DIR = "results"
analysis_dir = os.path.join(RESULTS_DIR, "analysis")
os.makedirs(analysis_dir, exist_ok=True)

X_final_df["tissue"] = X_final_df["tissue"].replace("haematopoietic_and_lymphoid", "h_lymphoid")
# ============================================================
# FUNCIONES AUXILIARES
# ============================================================

def assign_protocol(folder):
    folder = str(folder).lower()

    if "stratified" in folder:
        return "Stratified KFold"

    if folder.startswith("zzz"):
        return "Disjoint LDO"

    return "Partial LDO"


def pretty_model_name(folder, model=None):
    folder = str(folder).lower()

    if "random_forest" in folder:
        return "Random Forest"
    if "xgboost" in folder:
        return "XGBoost"
    if "basic_mlp" in folder or "disjoint_mlp" in folder:
        return "Early Branch MLP"
    if "triplebranch" in folder or "triple_branch" in folder:
        return "Multimodal MLP"
    if "transformer" in folder or "drug_pair_transformer" in folder:
        return "Drug-Pair Transformer MLP"
    if "gnn" in folder:
        return "Graph Drug Synergy Model"

    return folder


def read_model_name(result_dir):
    config_path = os.path.join(result_dir, "experiment_config.json")

    if os.path.exists(config_path):
        with open(config_path, "r") as f:
            config = json.load(f)
        return config.get("model", os.path.basename(result_dir))

    return os.path.basename(result_dir)


def load_predictions_from_folder(result_dir):
    pred_dir = os.path.join(result_dir, "predictions")

    if not os.path.exists(pred_dir):
        return None

    files = [
        f for f in os.listdir(pred_dir)
        if (
            f.endswith(".csv")
            and (
                f.startswith("predictions_fold_")
                or f.startswith("predicitions_repeat_")
                or f.startswith("predictions_repeat_")
            )
        )
    ]

    if len(files) == 0:
        return None

    dfs = []

    for file in sorted(files):
        path = os.path.join(pred_dir, file)
        df = pd.read_csv(path)

        fold = (
            file
            .replace("predictions_fold_", "")
            .replace("predicitions_repeat_", "")
            .replace("predictions_repeat_", "")
            .replace(".csv", "")
        )

        df["fold"] = int(fold)

        dfs.append(df)

    return pd.concat(dfs, ignore_index=True)

def compute_group_auprc(df, group_col, min_samples=50):
    rows = []

    for group_value, g in df.groupby(group_col):
        y_true = g["y_true"].astype(int).to_numpy()
        y_prob = g["y_prob"].astype(float).to_numpy()

        n_samples = len(g)
        n_pos = int(np.sum(y_true))
        n_neg = int(n_samples - n_pos)
        pos_pct = 100 * np.mean(y_true) if n_samples > 0 else np.nan

        if n_samples < min_samples or n_pos == 0 or n_neg == 0:
            auprc = np.nan
        else:
            auprc = average_precision_score(y_true, y_prob)

        rows.append({
            group_col: group_value,
            "n_samples": n_samples,
            "n_positive": n_pos,
            "n_negative": n_neg,
            "positive_pct": pos_pct,
            "baseline_auprc": np.mean(y_true) if n_samples > 0 else np.nan,
            "auprc": auprc,
        })

    return pd.DataFrame(rows)


# ============================================================
# CARGAR TODAS LAS PREDICCIONES Y CALCULAR AUPRC POR GRUPO
# ============================================================

study_rows = []
tissue_rows = []

for folder in sorted(os.listdir(RESULTS_DIR)):
    result_dir = os.path.join(RESULTS_DIR, folder)

    if not os.path.isdir(result_dir):
        continue

    # Excluir pruebas de interacción
    if "interactiondiff" in folder.lower() or "interaction" in folder.lower():
        continue

    preds = load_predictions_from_folder(result_dir)

    if preds is None:
        continue

    model_raw = read_model_name(result_dir)
    model_pretty = pretty_model_name(folder, model_raw)
    protocol = assign_protocol(folder)

    # AUPRC por estudio
    df_study = compute_group_auprc(preds, "study_name", min_samples=50)
    df_study["model"] = model_pretty
    df_study["protocol"] = protocol
    df_study["result_folder"] = folder
    study_rows.append(df_study)

    # AUPRC por tejido
    df_tissue = compute_group_auprc(preds, "tissue", min_samples=50)
    df_tissue["model"] = model_pretty
    df_tissue["protocol"] = protocol
    df_tissue["result_folder"] = folder
    tissue_rows.append(df_tissue)


auprc_by_study = pd.concat(study_rows, ignore_index=True)
auprc_by_tissue = pd.concat(tissue_rows, ignore_index=True)

auprc_by_study.to_csv(
    os.path.join(analysis_dir, "auprc_by_study.csv"),
    index=False
)

auprc_by_tissue.to_csv(
    os.path.join(analysis_dir, "auprc_by_tissue.csv"),
    index=False
)


study_summary = (
    X_final_df
    .groupby("study_name")["synergy_loewe_bin"]
    .agg(
        n_muestras="size",
        n_positivas="sum"
    )
    .reset_index()
)

study_summary["positive_pct"] = (
    100 * study_summary["n_positivas"] / study_summary["n_muestras"]
)

study_label_map = {
    row["study_name"]: (
        f'{row["study_name"]}\n'
        f'n={int(row["n_muestras"]):,}\n'
        f'pos={row["positive_pct"]:.1f}%'
    )
    for _, row in study_summary.iterrows()
}

In [ ]:
protocol_order = ["Stratified KFold", "Partial LDO", "Disjoint LDO"]

protocol_title_map = {
    "Stratified KFold": "Stratified K-Fold",
    "Partial LDO": "LDOp",
    "Disjoint LDO": "LDOd",
}

for protocol in protocol_order:
    df_p = auprc_by_study[
        auprc_by_study["protocol"] == protocol
    ].copy()

    pivot = df_p.pivot_table(
        index="model",
        columns="study_name",
        values="auprc",
        aggfunc="mean"
    )

    plt.figure(figsize=(9.5, 4.8))

    ax = sns.heatmap(
        pivot,
        annot=True,
        fmt=".3f",
        cmap="viridis",
        linewidths=0.5,
        vmin=0.0,
        vmax=1.0,
        cbar_kws={"label": "AUPRC"}
    )

    ax.set_title(f"AUPRC por estudio - {protocol_title_map[protocol]}")
    ax.set_xlabel("Estudio")
    ax.set_ylabel("Modelo")

    # Etiquetas con n y % positivos
    new_xticklabels = [
        study_label_map.get(study, study)
        for study in pivot.columns
    ]

    ax.set_xticklabels(
        new_xticklabels,
        rotation=0,
        ha="center"
    )

    plt.tight_layout()

    filename = protocol.lower().replace(" ", "_")
    plt.savefig(
        os.path.join(analysis_dir, f"heatmap_auprc_by_study_{filename}_with_counts.png"),
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()

In [ ]:
# ============================================================
# HEATMAP AUPRC POR ESTUDIO
# RANDOM FOREST x PROTOCOLO
# ============================================================

MIN_STUDY_SAMPLES = 50
MODEL_TO_PLOT = "Random Forest"

df_rf_study = auprc_by_study[
    (auprc_by_study["model"] == MODEL_TO_PLOT)
    & (auprc_by_study["n_samples"] >= MIN_STUDY_SAMPLES)
].copy()

protocol_title_map = {
    "Stratified KFold": "Stratified",
    "Partial LDO": "LDOp",
    "Disjoint LDO": "LDOd",
}

df_rf_study["protocol_plot"] = df_rf_study["protocol"].replace(protocol_title_map)

pivot = df_rf_study.pivot_table(
    index="protocol_plot",
    columns="study_name",
    values="auprc",
    aggfunc="mean"
)

# Orden de protocolos
protocol_plot_order = ["Stratified", "LDOp", "LDOd"]
pivot = pivot.reindex(protocol_plot_order)

# Ordenar estudios por AUPRC media
study_order = pivot.mean(axis=0).sort_values(ascending=False).index
pivot = pivot[study_order]

# ============================================================
# ETIQUETAS CON N Y % POSITIVOS
# ============================================================

study_summary = (
    X_final_df
    .groupby("study_name")["synergy_loewe_bin"]
    .agg(
        n_muestras="size",
        n_positivas="sum"
    )
    .reset_index()
)

study_summary["positive_pct"] = (
    100 * study_summary["n_positivas"] / study_summary["n_muestras"]
)

study_label_map = {
    row["study_name"]: (
        f'{row["study_name"]}\n'
        f'n={int(row["n_muestras"]):,}\n'
        f'pos={row["positive_pct"]:.1f}%'
    )
    for _, row in study_summary.iterrows()
}

plt.figure(figsize=(9.5, 4.0))

ax = sns.heatmap(
    pivot,
    annot=True,
    fmt=".3f",
    cmap="viridis",
    linewidths=0.5,
    vmin=0,
    vmax=1,
    cbar_kws={"label": "AUPRC"}
)

ax.set_title("AUPRC por estudio y protocolo - Random Forest")
ax.set_xlabel("Estudio")
ax.set_ylabel("Protocolo")

new_xticklabels = [
    study_label_map.get(study, study)
    for study in pivot.columns
]

ax.set_xticklabels(
    new_xticklabels,
    rotation=0,
    ha="center"
)

plt.tight_layout()


plt.show()

In [ ]:
# ============================================================
# HEATMAP AUPRC POR TEJIDO
# RANDOM FOREST x PROTOCOLO
# ============================================================

tissue_name_map = {
    "haematopoietic_and_lymphoid": "h_lymphoid"
}

MIN_TISSUE_SAMPLES = 100
MODEL_TO_PLOT = "Random Forest"

df_rf = auprc_by_tissue[
    (auprc_by_tissue["model"] == MODEL_TO_PLOT)
    & (auprc_by_tissue["n_samples"] >= MIN_TISSUE_SAMPLES)
].copy()

df_rf["tissue_plot"] = df_rf["tissue"].replace(tissue_name_map)

protocol_title_map = {
    "Stratified KFold": "Stratified",
    "Partial LDO": "LDOp",
    "Disjoint LDO": "LDOd",
}

df_rf["protocol_plot"] = df_rf["protocol"].replace(protocol_title_map)

pivot = df_rf.pivot_table(
    index="protocol_plot",
    columns="tissue_plot",
    values="auprc",
    aggfunc="mean"
)

# Orden de protocolos
protocol_plot_order = ["Stratified", "LDOp", "LDOd"]
pivot = pivot.reindex(protocol_plot_order)

# Ordenar tejidos por AUPRC media
tissue_order = pivot.mean(axis=0).sort_values(ascending=False).index
pivot = pivot[tissue_order]

# Etiquetas con n y % positivos
tissue_summary = (
    X_final_df
    .assign(tissue_plot=lambda x: x["tissue"].replace(tissue_name_map))
    .groupby("tissue_plot")["synergy_loewe_bin"]
    .agg(
        n_muestras="size",
        n_positivas="sum"
    )
    .reset_index()
)

tissue_summary["positive_pct"] = (
    100 * tissue_summary["n_positivas"] / tissue_summary["n_muestras"]
)

tissue_label_map = {
    row["tissue_plot"]: (
        f'{row["tissue_plot"]}\n'
        f'n={int(row["n_muestras"]):,}\n'
        f'pos={row["positive_pct"]:.1f}%'
    )
    for _, row in tissue_summary.iterrows()
}

plt.figure(figsize=(14, 4.2))

ax = sns.heatmap(
    pivot,
    annot=True,
    fmt=".3f",
    cmap="viridis",
    linewidths=0.5,
    vmin=0,
    vmax=1,
    cbar_kws={"label": "AUPRC"}
)

ax.set_title("AUPRC por tejido y protocolo - Random Forest")
ax.set_xlabel("Tejido")
ax.set_ylabel("Protocolo")

new_xticklabels = [
    tissue_label_map.get(tissue, tissue)
    for tissue in pivot.columns
]

ax.set_xticklabels(
    new_xticklabels,
    rotation=35,
    ha="right"
)

plt.tight_layout()


plt.show()

Otra prueba de lo mismo: 


In [ ]:
# ============================================================
# HEATMAPS RF AUPRC POR ESTUDIO: TRAIN Y TEST
# ============================================================

protocol_order = ["Stratified CV", "Partial LODO"]
split_order = ["train", "test"]

for protocol in protocol_order:
    for split_name in split_order:

        df_p = auprc_by_study_rf_train_test[
            (auprc_by_study_rf_train_test["protocol"] == protocol)
            & (auprc_by_study_rf_train_test["split"] == split_name)
        ].copy()

        if len(df_p) == 0:
            continue

        pivot_auprc = df_p.pivot_table(
            index="model",
            columns="study_name",
            values="auprc",
            aggfunc="mean"
        )

        pivot_n = df_p.pivot_table(
            index="model",
            columns="study_name",
            values="n_samples",
            aggfunc="mean"
        )

        pivot_pct = df_p.pivot_table(
            index="model",
            columns="study_name",
            values="sample_pct",
            aggfunc="mean"
        )

        annot = pivot_auprc.copy().astype(object)

        for row in pivot_auprc.index:
            for col in pivot_auprc.columns:
                auprc = pivot_auprc.loc[row, col]
                n = pivot_n.loc[row, col]
                pct = pivot_pct.loc[row, col]

                if pd.isna(auprc):
                    annot.loc[row, col] = ""
                else:
                    annot.loc[row, col] = f"{auprc:.3f}\nn/fold={int(round(n))}\n{pct:.1f}%"

        plt.figure(figsize=(9, 2.2))

        sns.heatmap(
            pivot_auprc,
            annot=annot,
            fmt="",
            cmap="viridis",
            linewidths=0.5,
            vmin=0,
            vmax=1,
            cbar_kws={"label": "AUPRC"}
        )

        split_title = "Train" if split_name == "train" else "Test"

        plt.title(f"Random Forest - AUPRC por estudio - {protocol} - {split_title}")
        plt.xlabel("Estudio")
        plt.ylabel("Modelo")
        plt.tight_layout()

        filename = protocol.lower().replace(" ", "_")
        plt.savefig(
            os.path.join(
                analysis_dir,
                f"rf_heatmap_auprc_by_study_{filename}_{split_name}_with_counts.png"
            ),
            dpi=300,
            bbox_inches="tight"
        )

        plt.show()


In [ ]:
# ============================================================
# CALCULAR AUPRC POR LÍNEA CELULAR
# ============================================================

cell_rows = []

for folder in sorted(os.listdir(RESULTS_DIR)):
    result_dir = os.path.join(RESULTS_DIR, folder)

    if not os.path.isdir(result_dir):
        continue

    # Excluir pruebas de interacción
    if "interactiondiff" in folder.lower() or "interaction" in folder.lower():
        continue

    preds = load_predictions_from_folder(result_dir)

    if preds is None:
        continue

    model_raw = read_model_name(result_dir)
    model_pretty = pretty_model_name(folder, model_raw)
    protocol = assign_protocol(folder)

    df_cell = compute_group_auprc(
        preds,
        "cell_line_name",
        min_samples=50
    )

    df_cell["model"] = model_pretty
    df_cell["protocol"] = protocol
    df_cell["result_folder"] = folder

    cell_rows.append(df_cell)

auprc_by_cell = pd.concat(cell_rows, ignore_index=True)

auprc_by_cell.to_csv(
    os.path.join(analysis_dir, "auprc_by_cell_line.csv"),
    index=False
)

display(auprc_by_cell.head())

In [ ]:
# ============================================================
# AUPRC POR LÍNEA CELULAR
# Random Forest + LDOd
# ============================================================

MIN_CELL_SAMPLES = 500
MODEL_TO_ANALYZE = "Random Forest"
PROTOCOL_TO_ANALYZE = "Disjoint LDO"
auprc_by_cell = pd.concat(cell_rows, ignore_index=True)
df_cell = auprc_by_cell[
    (auprc_by_cell["model"] == MODEL_TO_ANALYZE)
    & (auprc_by_cell["protocol"] == PROTOCOL_TO_ANALYZE)
    & (auprc_by_cell["n_samples"] >= MIN_CELL_SAMPLES)
].copy()

df_cell = df_cell.sort_values("auprc", ascending=False)

top_cells = df_cell.head(5)
bottom_cells = df_cell.tail(5)

cell_summary = pd.concat([
    top_cells.assign(grupo="Mayor AUPRC"),
    bottom_cells.assign(grupo="Menor AUPRC")
])

cell_summary = cell_summary[
    [
        "grupo",
        "cell_line_name",
        "n_samples",
        "n_positive",
        "positive_pct",
        "baseline_auprc",
        "auprc"
    ]
].copy()

cell_summary["positive_pct"] = cell_summary["positive_pct"].round(2)
cell_summary["baseline_auprc"] = cell_summary["baseline_auprc"].round(3)
cell_summary["auprc"] = cell_summary["auprc"].round(3)

display(cell_summary)

cell_rows = []

for folder in sorted(os.listdir(RESULTS_DIR)):
    result_dir = os.path.join(RESULTS_DIR, folder)

    if not os.path.isdir(result_dir):
        continue

    if "interactiondiff" in folder.lower() or "interaction" in folder.lower():
        continue

    preds = load_predictions_from_folder(result_dir)

    if preds is None:
        continue

    model_raw = read_model_name(result_dir)
    model_pretty = pretty_model_name(folder, model_raw)
    protocol = assign_protocol(folder)

    df_cell = compute_group_auprc(preds, "cell_line_name", min_samples=50)
    df_cell["model"] = model_pretty
    df_cell["protocol"] = protocol
    df_cell["result_folder"] = folder

    cell_rows.append(df_cell)





In [ ]:
# ============================================================
# ANÁLISIS RÁPIDO DE ERRORES POR LÍNEA CELULAR
# Random Forest + LDOd
# ============================================================


RF_LDOD_FOLDER = "zzz_disjoint_random_forest"
MIN_CELL_SAMPLES = 100

preds_rf_ldod = load_predictions_from_folder(
    os.path.join(RESULTS_DIR, RF_LDOD_FOLDER)
)

preds_rf_ldod["y_true"] = preds_rf_ldod["y_true"].astype(int)

# Usamos el umbral que maximizó F1 en validación
preds_rf_ldod["y_pred"] = preds_rf_ldod["y_pred_best_f1"].astype(int)

preds_rf_ldod["correct"] = preds_rf_ldod["y_true"] == preds_rf_ldod["y_pred"]
preds_rf_ldod["error"] = preds_rf_ldod["y_true"] != preds_rf_ldod["y_pred"]

preds_rf_ldod["fp"] = (
    (preds_rf_ldod["y_true"] == 0)
    & (preds_rf_ldod["y_pred"] == 1)
)

preds_rf_ldod["fn"] = (
    (preds_rf_ldod["y_true"] == 1)
    & (preds_rf_ldod["y_pred"] == 0)
)

cell_tissue_map = (
    X_final_df[["cell_line_name", "tissue"]]
    .drop_duplicates()
    .set_index("cell_line_name")["tissue"]
)

cell_error_summary = (
    preds_rf_ldod
    .groupby("cell_line_name")
    .agg(
        n_muestras=("y_true", "size"),
        n_positivas=("y_true", "sum"),
        aciertos=("correct", "sum"),
        errores=("error", "sum"),
        fp=("fp", "sum"),
        fn=("fn", "sum")
    )
    .reset_index()
)

cell_error_summary["tissue"] = cell_error_summary["cell_line_name"].map(cell_tissue_map)

cell_error_summary["n_negativas"] = (
    cell_error_summary["n_muestras"] - cell_error_summary["n_positivas"]
)

cell_error_summary["positive_pct"] = (
    100 * cell_error_summary["n_positivas"] / cell_error_summary["n_muestras"]
)

cell_error_summary["error_rate"] = (
    100 * cell_error_summary["errores"] / cell_error_summary["n_muestras"]
)

cell_error_summary["fp_rate"] = (
    100 * cell_error_summary["fp"] / cell_error_summary["n_negativas"]
)

cell_error_summary["fn_rate"] = (
    100 * cell_error_summary["fn"] / cell_error_summary["n_positivas"]
)

cell_error_summary = cell_error_summary.replace([np.inf, -np.inf], np.nan)

cell_error_summary_filtered = (
    cell_error_summary[cell_error_summary["n_muestras"] >= MIN_CELL_SAMPLES]
    .sort_values("error_rate", ascending=False)
)

top_error_cell_lines = cell_error_summary_filtered.head(10).copy()

top_error_cell_lines[
    [
        "cell_line_name",
        "tissue",
        "n_muestras",
        "positive_pct",
        "error_rate",
        "fp",
        "fn",
        "fp_rate",
        "fn_rate"
    ]
]

In [ ]:
# ============================================================
# FP/FN POR ESTUDIO
# Random Forest + LDOd, umbral best F1
# ============================================================


RF_LDOD_FOLDER = "zzz_disjoint_random_forest"

preds_rf_ldod = load_predictions_from_folder(
    os.path.join(RESULTS_DIR, RF_LDOD_FOLDER)
)

preds_rf_ldod["y_true"] = preds_rf_ldod["y_true"].astype(int)
preds_rf_ldod["y_pred"] = preds_rf_ldod["y_pred_best_f1"].astype(int)

preds_rf_ldod["fp"] = (
    (preds_rf_ldod["y_true"] == 0)
    & (preds_rf_ldod["y_pred"] == 1)
)

preds_rf_ldod["fn"] = (
    (preds_rf_ldod["y_true"] == 1)
    & (preds_rf_ldod["y_pred"] == 0)
)

error_by_study = (
    preds_rf_ldod
    .groupby("study_name")
    .agg(
        n_muestras=("y_true", "size"),
        n_positivas=("y_true", "sum"),
        fp=("fp", "sum"),
        fn=("fn", "sum")
    )
    .reset_index()
)

error_by_study["n_negativas"] = (
    error_by_study["n_muestras"] - error_by_study["n_positivas"]
)

error_by_study["positive_pct"] = (
    100 * error_by_study["n_positivas"] / error_by_study["n_muestras"]
)

error_by_study["fp_rate"] = (
    100 * error_by_study["fp"] / error_by_study["n_negativas"]
)

error_by_study["fn_rate"] = (
    100 * error_by_study["fn"] / error_by_study["n_positivas"]
)

error_by_study = error_by_study.replace([np.inf, -np.inf], np.nan)

error_by_study = error_by_study.sort_values("n_muestras", ascending=False)

display(error_by_study)

In [ ]:
plot_df = error_by_study.melt(
    id_vars=["study_name"],
    value_vars=["fp_rate", "fn_rate"],
    var_name="tipo_error",
    value_name="tasa"
)

plot_df["tipo_error"] = plot_df["tipo_error"].replace({
    "fp_rate": "FP rate",
    "fn_rate": "FN rate"
})

plt.figure(figsize=(7.5, 4.8))

ax = sns.barplot(
    data=plot_df,
    x="study_name",
    y="tasa",
    hue="tipo_error",
    palette=["#F58518", "#4C78A8"]
)

ax.set_title("Tasa de falsos positivos y falsos negativos por estudio - Random Forest LDOd")
ax.set_xlabel("Estudio")
ax.set_ylabel("Tasa (%)")

plt.xticks(rotation=0)
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()


plt.show()

In [ ]:
study_tissue_pct_by_study.plot(
    kind="bar",
    stacked=True,
    figsize=(12, 5)
)

plt.ylabel("% dentro de cada estudio")
plt.xlabel("Estudio")
plt.title("Composición de tejidos por estudio")
plt.legend(
    title="Tejido",
    bbox_to_anchor=(1.05, 1),
    loc="upper left"
)
plt.tight_layout()

plt.savefig(
    os.path.join(analysis_dir, "stacked_bar_tissue_distribution_by_study_pct.png"),
    dpi=300,
    bbox_inches="tight"
)

plt.show()


In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

analysis_dir = os.path.join("results", "analysis")
os.makedirs(analysis_dir, exist_ok=True)

sns.set_theme(style="whitegrid")


# ============================================================
# 1. NUMERO DE EJEMPLOS POR TEJIDO EN TODO EL DATASET
# ============================================================

tissue_counts = (
    X_final_df["tissue"]
    .value_counts()
    .reset_index()
)

tissue_counts.columns = ["tissue", "n_examples"]

display(tissue_counts)

tissue_counts.to_csv(
    os.path.join(analysis_dir, "global_tissue_counts.csv"),
    index=False
)

plt.figure(figsize=(10, 5))

ax = sns.barplot(
    data=tissue_counts,
    x="n_examples",
    y="tissue",
    color="#4C78A8"
)

ax.bar_label(ax.containers[0], padding=3, fontsize=9)

plt.xlabel("Número de ejemplos")
plt.ylabel("Tejido")
plt.title("Número de ejemplos por tejido en el dataset global")
plt.tight_layout()

plt.savefig(
    os.path.join(analysis_dir, "global_tissue_counts.png"),
    dpi=300,
    bbox_inches="tight"
)

plt.show()


In [ ]:
# ============================================================
# 3. NUMERO DE EJEMPLOS POR TEJIDO, SEPARADO POR ESTUDIO
# ============================================================
study_tissue_counts_long = (
    X_final_df
    .groupby(["study_name", "tissue"])
    .size()
    .reset_index(name="n_examples")
)

g = sns.catplot(
    data=study_tissue_counts_long,
    kind="bar",
    x="n_examples",
    y="tissue",
    col="study_name",
    col_wrap=2,
    sharex=False,
    height=4,
    aspect=1.4,
    color="#4C78A8"
)

g.set_axis_labels("Número de ejemplos", "Tejido")
g.set_titles("{col_name}")

for ax in g.axes.flat:
    for container in ax.containers:
        ax.bar_label(container, padding=3, fontsize=8)

plt.suptitle("Número de ejemplos por tejido en cada estudio", y=1.03)
plt.tight_layout()

plt.savefig(
    os.path.join(analysis_dir, "tissue_counts_by_study_facets.png"),
    dpi=300,
    bbox_inches="tight"
)

plt.show()


In [ ]:
# ============================================================
# PORCENTAJE DE TEJIDOS DENTRO DE CADA ESTUDIO CON ETIQUETAS
# ============================================================


analysis_dir = os.path.join("results", "analysis")
os.makedirs(analysis_dir, exist_ok=True)

study_tissue_pct = (
    pd.crosstab(
        X_final_df["study_name"],
        X_final_df["tissue"],
        normalize="index"
    ) * 100
)

ax = study_tissue_pct.plot(
    kind="bar",
    stacked=True,
    figsize=(12, 5),
    edgecolor="white",
    linewidth=0.5
)

plt.ylabel("% dentro de cada estudio")
plt.xlabel("Estudio")
plt.title("Composición porcentual de tejidos por estudio")

# Añadir etiquetas porcentuales dentro de cada segmento
for container in ax.containers:
    labels = []

    for bar in container:
        height = bar.get_height()

        if height >= 2:  # evita textos en segmentos muy pequeños
            labels.append(f"{height:.1f}%")
        else:
            labels.append("")

    ax.bar_label(
        container,
        labels=labels,
        label_type="center",
        fontsize=8,
        color="black"
    )

plt.legend(
    title="Tejido",
    bbox_to_anchor=(1.05, 1),
    loc="upper left"
)

plt.xticks(rotation=0)
plt.tight_layout()

plt.savefig(
    os.path.join(analysis_dir, "study_tissue_percentages_stacked_labeled2.png"),
    dpi=300,
    bbox_inches="tight"
)

plt.show()

## Errores FP FN por estudio/tejido/cell line

In [ ]:
import json

RESULTS_DIR = "results"
analysis_dir = os.path.join(RESULTS_DIR, "analysis")
os.makedirs(analysis_dir, exist_ok=True)

ERROR_THRESHOLD = "best_f1"  # "best_f1" o "best_precision"
PRED_COL = f"y_pred_{ERROR_THRESHOLD}"


def assign_protocol(folder):
    folder = str(folder).lower()

    if "stratified" in folder:
        return "Stratified CV"
    if folder.startswith("zzz"):
        return "Disjoint LODO"
    return "Partial LODO"


def pretty_model_name(folder, model=None):
    folder = str(folder).lower()

    if "random_forest" in folder:
        return "Random Forest"
    if "xgboost" in folder:
        return "XGBoost"
    if "basic_mlp" in folder or "disjoint_mlp" in folder:
        return "MLP básica"
    if "triplebranch" in folder or "triple_branch" in folder:
        return "Triple MLP"
    if "transformer" in folder or "drug_pair_transformer" in folder:
        return "Transformer + MLP"
    if "gnn" in folder:
        return "GNN"

    return folder


def read_model_name(result_dir):
    config_path = os.path.join(result_dir, "experiment_config.json")

    if os.path.exists(config_path):
        with open(config_path, "r") as f:
            config = json.load(f)
        return config.get("model", os.path.basename(result_dir))

    return os.path.basename(result_dir)


def load_predictions_from_folder(result_dir):
    pred_dir = os.path.join(result_dir, "predictions")

    if not os.path.exists(pred_dir):
        return None

    files = [
        f for f in os.listdir(pred_dir)
        if f.startswith("predictions_fold_") and f.endswith(".csv")
    ]

    if len(files) == 0:
        return None

    dfs = []

    for file in sorted(files):
        path = os.path.join(pred_dir, file)
        df = pd.read_csv(path)

        fold = file.replace("predictions_fold_", "").replace(".csv", "")
        df["fold"] = int(fold)

        dfs.append(df)

    return pd.concat(dfs, ignore_index=True)


def compute_error_summary(df, group_col, pred_col, min_samples=50):
    rows = []

    for group_value, g in df.groupby(group_col):
        y_true = g["y_true"].astype(int).to_numpy()
        y_pred = g[pred_col].astype(int).to_numpy()

        tp = int(((y_true == 1) & (y_pred == 1)).sum())
        tn = int(((y_true == 0) & (y_pred == 0)).sum())
        fp = int(((y_true == 0) & (y_pred == 1)).sum())
        fn = int(((y_true == 1) & (y_pred == 0)).sum())

        n = len(g)
        n_pos = int((y_true == 1).sum())
        n_neg = int((y_true == 0).sum())

        precision = tp / (tp + fp) if (tp + fp) > 0 else np.nan
        recall = tp / (tp + fn) if (tp + fn) > 0 else np.nan
        f1 = (
            2 * precision * recall / (precision + recall)
            if pd.notna(precision) and pd.notna(recall) and (precision + recall) > 0
            else np.nan
        )

        fp_rate = fp / n_neg if n_neg > 0 else np.nan
        fn_rate = fn / n_pos if n_pos > 0 else np.nan

        rows.append({
            group_col: group_value,
            "n_samples": n,
            "n_positive": n_pos,
            "n_negative": n_neg,
            "positive_pct": 100 * n_pos / n if n > 0 else np.nan,
            "tp": tp,
            "tn": tn,
            "fp": fp,
            "fn": fn,
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "fp_rate": fp_rate,
            "fn_rate": fn_rate,
            "error_rate": (fp + fn) / n if n > 0 else np.nan,
        })

    out = pd.DataFrame(rows)

    out = out[out["n_samples"] >= min_samples].reset_index(drop=True)

    return out


In [ ]:
error_rows_study = []
error_rows_tissue = []
error_rows_cell = []

for folder in sorted(os.listdir(RESULTS_DIR)):
    result_dir = os.path.join(RESULTS_DIR, folder)

    if not os.path.isdir(result_dir):
        continue

    if "interactiondiff" in folder.lower() or "interaction" in folder.lower():
        continue

    preds = load_predictions_from_folder(result_dir)

    if preds is None:
        continue

    if PRED_COL not in preds.columns:
        print(f"Saltando {folder}: no existe columna {PRED_COL}")
        continue

    model_raw = read_model_name(result_dir)
    model_pretty = pretty_model_name(folder, model_raw)
    protocol = assign_protocol(folder)

    df_study = compute_error_summary(preds, "study_name", PRED_COL, min_samples=50)
    df_study["model"] = model_pretty
    df_study["protocol"] = protocol
    df_study["result_folder"] = folder
    error_rows_study.append(df_study)

    df_tissue = compute_error_summary(preds, "tissue", PRED_COL, min_samples=100)
    df_tissue["model"] = model_pretty
    df_tissue["protocol"] = protocol
    df_tissue["result_folder"] = folder
    error_rows_tissue.append(df_tissue)

    df_cell = compute_error_summary(preds, "cell_line_name", PRED_COL, min_samples=200)
    df_cell["model"] = model_pretty
    df_cell["protocol"] = protocol
    df_cell["result_folder"] = folder
    error_rows_cell.append(df_cell)


errors_by_study = pd.concat(error_rows_study, ignore_index=True)
errors_by_tissue = pd.concat(error_rows_tissue, ignore_index=True)
errors_by_cell_line = pd.concat(error_rows_cell, ignore_index=True)

errors_by_study.to_csv(
    os.path.join(analysis_dir, f"errors_by_study_{ERROR_THRESHOLD}.csv"),
    index=False
)

errors_by_tissue.to_csv(
    os.path.join(analysis_dir, f"errors_by_tissue_{ERROR_THRESHOLD}.csv"),
    index=False
)

errors_by_cell_line.to_csv(
    os.path.join(analysis_dir, f"errors_by_cell_line_{ERROR_THRESHOLD}.csv"),
    index=False
)

display(errors_by_study)
display(errors_by_tissue)
display(errors_by_cell_line)


In [ ]:
protocol_order = ["Stratified CV", "Partial LODO", "Disjoint LODO"]

for protocol in protocol_order:
    df_p = errors_by_study[errors_by_study["protocol"] == protocol].copy()

    for metric, title_metric in [
        ("fp_rate", "Tasa de falsos positivos"),
        ("fn_rate", "Tasa de falsos negativos")
    ]:
        pivot = df_p.pivot_table(
            index="model",
            columns="study_name",
            values=metric,
            aggfunc="mean"
        )

        plt.figure(figsize=(8, 4.5))

        sns.heatmap(
            pivot,
            annot=True,
            fmt=".3f",
            cmap="magma",
            linewidths=0.5,
            cbar_kws={"label": title_metric}
        )

        plt.title(f"{title_metric} por estudio - {protocol}")
        plt.xlabel("Estudio")
        plt.ylabel("Modelo")
        plt.tight_layout()

        filename = protocol.lower().replace(" ", "_")
        plt.savefig(
            os.path.join(analysis_dir, f"heatmap_{metric}_by_study_{filename}_{ERROR_THRESHOLD}.png"),
            dpi=300,
            bbox_inches="tight"
        )

        plt.show()


In [ ]:
for protocol in protocol_order:
    df_p = errors_by_tissue[errors_by_tissue["protocol"] == protocol].copy()

    for metric, title_metric in [
        ("fp_rate", "Tasa de falsos positivos"),
        ("fn_rate", "Tasa de falsos negativos")
    ]:
        pivot = df_p.pivot_table(
            index="model",
            columns="tissue",
            values=metric,
            aggfunc="mean"
        )

        tissue_order = pivot.mean(axis=0).sort_values(ascending=False).index
        pivot = pivot[tissue_order]

        plt.figure(figsize=(12, 4.8))

        sns.heatmap(
            pivot,
            annot=True,
            fmt=".3f",
            cmap="magma",
            linewidths=0.5,
            cbar_kws={"label": title_metric}
        )

        plt.title(f"{title_metric} por tejido - {protocol}")
        plt.xlabel("Tejido")
        plt.ylabel("Modelo")
        plt.xticks(rotation=35, ha="right")
        plt.tight_layout()

        filename = protocol.lower().replace(" ", "_")
        plt.savefig(
            os.path.join(analysis_dir, f"heatmap_{metric}_by_tissue_{filename}_{ERROR_THRESHOLD}.png"),
            dpi=300,
            bbox_inches="tight"
        )

        plt.show()


In [ ]:
error_rows_cell_study = []

for folder in sorted(os.listdir(RESULTS_DIR)):
    result_dir = os.path.join(RESULTS_DIR, folder)

    if not os.path.isdir(result_dir):
        continue

    folder_lower = folder.lower()

    if "interactiondiff" in folder_lower or "interaction" in folder_lower:
        continue

    preds = load_predictions_from_folder(result_dir)

    if preds is None:
        continue

    if PRED_COL not in preds.columns:
        continue

    model_raw = read_model_name(result_dir)
    model_pretty = pretty_model_name(folder, model_raw)
    protocol = assign_protocol(folder)

    df_cell_study = compute_error_summary_multi(
        preds,
        group_cols=["study_name", "cell_line_name"],
        pred_col=PRED_COL,
        min_samples=MIN_SAMPLES_CELL_STUDY
    )

    df_cell_study["model"] = model_pretty
    df_cell_study["protocol"] = protocol
    df_cell_study["result_folder"] = folder

    error_rows_cell_study.append(df_cell_study)


errors_by_cell_line_study = pd.concat(error_rows_cell_study, ignore_index=True)


display(errors_by_cell_line_study)


In [ ]:
errors_by_cell_line
X_final_df[["cell_line_name"]].value_counts()

### Fold difficulty vs AUPRC

In [ ]:
fold_difficulty[["result_folder", "model", "protocol"]].drop_duplicates().sort_values(
    ["protocol", "model"]
)


In [ ]:
fold_difficulty_summary = (
    fold_difficulty
    .groupby(["protocol", "model", "result_folder"])
    .agg(
        n_folds=("fold", "nunique"),
        mean_n_test=("n_test", "mean"),
        std_n_test=("n_test", "std"),
        mean_test_pos_pct=("test_pos_pct", "mean"),
        std_test_pos_pct=("test_pos_pct", "std"),
        mean_baseline_auprc=("baseline_auprc_test", "mean"),
        std_baseline_auprc=("baseline_auprc_test", "std"),
        mean_auprc=("auprc", "mean"),
        std_auprc=("auprc", "std"),
        mean_f1=("f1", "mean"),
        mean_precision=("precision", "mean"),
        mean_recall=("recall", "mean"),
        mean_mcc=("mcc", "mean"),
    )
    .reset_index()
    .sort_values(["protocol", "mean_auprc"], ascending=[True, False])
)

fold_difficulty_summary.to_csv(
    os.path.join(analysis_dir, "fold_difficulty_summary_by_model_protocol.csv"),
    index=False
)

display(fold_difficulty_summary)


In [ ]:
plt.figure(figsize=(9, 6))

sns.scatterplot(
    data=fold_difficulty,
    x="baseline_auprc_test",
    y="auprc",
    hue="protocol",
    style="model",
    s=80
)

sns.regplot(
    data=fold_difficulty,
    x="baseline_auprc_test",
    y="auprc",
    scatter=False,
    color="black",
    line_kws={"linestyle": "--", "linewidth": 1}
)

plt.xlabel("Baseline AUPRC del fold")
plt.ylabel("AUPRC del modelo")
plt.title("Relación entre baseline AUPRC y AUPRC por fold")
plt.tight_layout()

plt.savefig(
    os.path.join(analysis_dir, "fold_baseline_auprc_vs_model_auprc.png"),
    dpi=300,
    bbox_inches="tight"
)

plt.show()


In [ ]:
plt.figure(figsize=(9, 6))

sns.scatterplot(
    data=fold_difficulty,
    x="test_pos_pct",
    y="auprc",
    hue="protocol",
    style="model",
    s=80
)

sns.regplot(
    data=fold_difficulty,
    x="test_pos_pct",
    y="auprc",
    scatter=False,
    color="black",
    line_kws={"linestyle": "--", "linewidth": 1}
)

plt.xlabel("% positivos en test")
plt.ylabel("AUPRC")
plt.title("Relación entre porcentaje de positivos y AUPRC por fold")
plt.tight_layout()

plt.savefig(
    os.path.join(analysis_dir, "fold_positive_pct_vs_auprc.png"),
    dpi=300,
    bbox_inches="tight"
)

plt.show()


In [ ]:
plt.figure(figsize=(9, 6))

sns.scatterplot(
    data=fold_difficulty,
    x="n_test",
    y="auprc",
    hue="protocol",
    style="model",
    s=80
)

sns.regplot(
    data=fold_difficulty,
    x="n_test",
    y="auprc",
    scatter=False,
    color="black",
    line_kws={"linestyle": "--", "linewidth": 1}
)

plt.xlabel("Número de muestras en test")
plt.ylabel("AUPRC")
plt.title("Relación entre tamaño del fold y AUPRC")
plt.tight_layout()

plt.savefig(
    os.path.join(analysis_dir, "fold_n_test_vs_auprc.png"),
    dpi=300,
    bbox_inches="tight"
)

plt.show()


In [ ]:
if "n_heldout_drugs" in fold_difficulty.columns:
    df_lodo = fold_difficulty[
        fold_difficulty["n_heldout_drugs"].notna()
    ].copy()

    plt.figure(figsize=(9, 6))

    sns.scatterplot(
        data=df_lodo,
        x="n_heldout_drugs",
        y="auprc",
        hue="protocol",
        style="model",
        s=80
    )

    sns.regplot(
        data=df_lodo,
        x="n_heldout_drugs",
        y="auprc",
        scatter=False,
        color="black",
        line_kws={"linestyle": "--", "linewidth": 1}
    )

    plt.xlabel("Número de fármacos held-out")
    plt.ylabel("AUPRC")
    plt.title("Relación entre fármacos held-out y AUPRC")
    plt.tight_layout()

    plt.savefig(
        os.path.join(analysis_dir, "fold_n_heldout_drugs_vs_auprc.png"),
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()


In [ ]:
difficulty_cols = [
    "n_test",
    "test_pos_pct",
    "baseline_auprc_test",
    "n_heldout_drugs"
]

difficulty_cols = [c for c in difficulty_cols if c in fold_difficulty.columns]

corr_rows = []

for protocol, df_p in fold_difficulty.groupby("protocol"):
    for model, df_m in df_p.groupby("model"):
        for col in difficulty_cols:
            valid = df_m[[col, "auprc"]].dropna()

            if len(valid) >= 3 and valid[col].nunique() > 1:
                corr = valid[[col, "auprc"]].corr(method="spearman").iloc[0, 1]
            else:
                corr = np.nan

            corr_rows.append({
                "protocol": protocol,
                "model": model,
                "difficulty_variable": col,
                "spearman_corr_with_auprc": corr
            })

fold_difficulty_correlations = pd.DataFrame(corr_rows)

fold_difficulty_correlations.to_csv(
    os.path.join(analysis_dir, "fold_difficulty_correlations.csv"),
    index=False
)

display(fold_difficulty_correlations)


### GENERALIZACIÓN DEL MODELO ANTE ESTUDIOS NUEVOS

In [ ]:
# Variable "binaria": sinergia (1) o antagónica (-1), eliminamos los ejemplos con valor de sinergia 0
print("Filtrado antes de eliminar sinergias 0: ", dloewe_extra.shape)
dloewe_extra = dloewe_extra[(dloewe_extra['synergy_loewe_bin'] == 1) | (dloewe_extra['synergy_loewe_bin'] == -1)]
print("Filtrado después de eliminar sinergias 0: ", dloewe_extra.shape)


# Variable tissue (tejido) a partir de cell_line_name
dloewe_extra['tissue'] = dloewe_extra['cell_line_name'].map(cellline2tissue)
# Eliminamos entradas de tejidos que proporcionan ruido en el análisis (stomach)
print("Filtrado antes de eliminar sinergias stomach: ", dloewe_extra.shape)
dloewe_extra = dloewe_extra[dloewe_extra['tissue'] != 'stomach']
print("Filtrado después de eliminar sinergias stomach: ", dloewe_extra.shape)

print(dloewe_extra['synergy_loewe_bin'].value_counts())
print(dloewe_extra['study_name'].value_counts())
print(dloewe_extra['tissue'].value_counts())
print(dloewe_extra['tissue'].isna().sum())

In [ ]:
import copy
from PIL import Image

if not hasattr(Image, "Resampling"):
    class Resampling:
        LANCZOS = Image.ANTIALIAS

    Image.Resampling = Resampling

def normalize_drug_id(x):
    if isinstance(x, np.ndarray):
        if x.size == 1:
            return str(x.item())
        return str(tuple(x.tolist()))
    return str(x)

def smiles_to_fp(smiles):
    if pd.isna(smiles):
        return None

    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        return None

    return np.array(
        AllChem.GetMorganFingerprintAsBitVect(mol, radius=2, nBits=1024)
    )

# ============================================================
# EXTERNAL STUDY GENERALIZATION
# ============================================================
# Paste these cells after the notebook cells where the following objects/functions
# already exist:
# data_loewe_filtered, data_smiles_reduced, cellline2tissue, data_expr,
# normalize_drug_id, smiles_to_fp, prepare_fold_data_no_leakage,
# build_tabular_matrices, compute_metrics_at_threshold, find_best_thresholds.
# For TripleBranchMLP runs, this file reuses the notebook definitions if they
# already exist: SynergyDataset, TripleBranchMLP, train_torch_model_fold,
# predict_torch_model, log_curves_and_metrics.

import os
import json
import shutil
import joblib
import numpy as np
import pandas as pd
import torch

from torch import nn
from torch.utils.data import Dataset, DataLoader
from torch.utils.tensorboard import SummaryWriter

from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier


TRAIN_STUDIES = ['ALMANAC', 'FRIEDMAN', 'ONEIL', 'ASTRAZENECA']
EXTERNAL_OUTPUT_DIR = 'results/external_unseen_studies2'
os.makedirs(EXTERNAL_OUTPUT_DIR, exist_ok=True)
os.makedirs(os.path.join(EXTERNAL_OUTPUT_DIR, 'predictions'), exist_ok=True)

thresholds = np.round(np.arange(0.05, 0.951, 0.01), 2)


def build_fingerprint_dataset_from_loewe(dloewe_input, data_smiles_reduced, cellline2tissue):
    """Builds the same X_final_df-style table, but without filtering to four studies."""
    df0 = dloewe_input.copy()
    df0 = df0[(df0['synergy_loewe_bin'] == 1) | (df0['synergy_loewe_bin'] == -1)].copy()
    df0['tissue'] = df0['cell_line_name'].map(cellline2tissue)
    df0 = df0[df0['tissue'] != 'stomach'].copy()

    df0['drug_row_id'] = df0['drug_row_id'].apply(normalize_drug_id)
    df0['drug_col_id'] = df0['drug_col_id'].apply(normalize_drug_id)

    lookup_smiles = data_smiles_reduced[['drugbank_id', 'cid', 'isomeric_smiles']].copy()
    lookup_smiles['drugbank_id'] = lookup_smiles['drugbank_id'].apply(
        lambda x: normalize_drug_id(x) if pd.notna(x) else np.nan
    )
    lookup_smiles['cid'] = lookup_smiles['cid'].apply(
        lambda x: normalize_drug_id(x) if pd.notna(x) else np.nan
    )

    drugbank_to_smiles = (
        lookup_smiles
        .dropna(subset=['drugbank_id'])
        .drop_duplicates(subset=['drugbank_id'])
        .set_index('drugbank_id')['isomeric_smiles']
    )

    cid_to_smiles = (
        lookup_smiles
        .dropna(subset=['cid'])
        .drop_duplicates(subset=['cid'])
        .set_index('cid')['isomeric_smiles']
    )

    df = df0.copy()
    df['smiles_row'] = df['drug_row_id'].map(drugbank_to_smiles)
    mask = df['smiles_row'].isna()
    df.loc[mask, 'smiles_row'] = df.loc[mask, 'drug_row_id'].map(cid_to_smiles)

    df['smiles_col'] = df['drug_col_id'].map(drugbank_to_smiles)
    mask = df['smiles_col'].isna()
    df.loc[mask, 'smiles_col'] = df.loc[mask, 'drug_col_id'].map(cid_to_smiles)

    before = len(df)
    df = df.dropna(subset=['smiles_row', 'smiles_col']).reset_index(drop=True)
    print('Filas eliminadas por falta de SMILES:', before - len(df))

    swap_mask = df['drug_row_id'] > df['drug_col_id']
    df.loc[swap_mask, ['drug_row_id', 'drug_col_id', 'smiles_row', 'smiles_col']] = (
        df.loc[swap_mask, ['drug_col_id', 'drug_row_id', 'smiles_col', 'smiles_row']].values
    )

    group_cols = ['drug_row_id', 'drug_col_id', 'cell_line_name', 'study_name']
    label_nunique = (
        df.groupby(group_cols)['synergy_loewe_bin']
        .nunique()
        .reset_index(name='n_labels')
    )
    contradictory_groups = label_nunique[label_nunique['n_labels'] > 1][group_cols]

    df = df.merge(
        contradictory_groups.assign(is_contradictory=True),
        on=group_cols,
        how='left'
    )
    print('Filas en grupos contradictorios:', df['is_contradictory'].sum())

    df = (
        df[df['is_contradictory'].isna()]
        .drop(columns='is_contradictory')
        .reset_index(drop=True)
    )

    before = len(df)
    df = (
        df.groupby(group_cols, as_index=False)
        .agg({
            'synergy_loewe': 'mean',
            'synergy_loewe_bin': 'first',
            'tissue': 'first',
            'smiles_row': 'first',
            'smiles_col': 'first'
        })
    )
    print('Filas agregadas por duplicados consistentes:', before - len(df))

    unique_smiles = pd.concat([df['smiles_row'], df['smiles_col']]).unique()
    smiles_dict = {s: smiles_to_fp(s) for s in unique_smiles}

    df['fp_row'] = df['smiles_row'].map(smiles_dict)
    df['fp_col'] = df['smiles_col'].map(smiles_dict)

    before = len(df)
    df = df.dropna(subset=['fp_row', 'fp_col']).reset_index(drop=True)
    print('Filas eliminadas por fingerprints invalidos:', before - len(df))

    fps_row_matrix = np.stack(df['fp_row'].values)
    fps_col_matrix = np.stack(df['fp_col'].values)

    names_drug1 = [f'drug1_bit_{i}' for i in range(1024)]
    names_drug2 = [f'drug2_bit_{i}' for i in range(1024)]

    df_fps_row = pd.DataFrame(fps_row_matrix, columns=names_drug1, index=df.index)
    df_fps_col = pd.DataFrame(fps_col_matrix, columns=names_drug2, index=df.index)

    df_meta = df[
        [
            'drug_row_id',
            'drug_col_id',
            'cell_line_name',
            'study_name',
            'tissue',
            'synergy_loewe',
            'synergy_loewe_bin',
            'smiles_row',
            'smiles_col',
        ]
    ].copy()
    df_meta['synergy_loewe_bin'] = df_meta['synergy_loewe_bin'].replace(-1, 0).astype(int)

    out = pd.concat(
        [df_meta.reset_index(drop=True), df_fps_row.reset_index(drop=True), df_fps_col.reset_index(drop=True)],
        axis=1
    )
    return out


# ============================================================
# BUILD SEEN-STUDY TRAINING SET AND UNSEEN-STUDY EXTERNAL TEST
# ============================================================

X_all_studies_df = build_fingerprint_dataset_from_loewe(
    data_loewe_filtered,
    data_smiles_reduced,
    cellline2tissue
)

seen_studies_df = (
    X_all_studies_df[X_all_studies_df['study_name'].isin(TRAIN_STUDIES)]
    .reset_index(drop=True)
)
seen_studies_df = seen_studies_df[seen_studies_df['tissue'] != 'stomach']

external_studies_df = (
    X_all_studies_df[~X_all_studies_df['study_name'].isin(TRAIN_STUDIES)]
    .reset_index(drop=True)
)

print('Seen studies:', seen_studies_df.shape)
print(seen_studies_df['study_name'].value_counts())
print('\nExternal unseen studies:', external_studies_df.shape)
print(external_studies_df['study_name'].value_counts())
print('\nExternal positive rate:', external_studies_df['synergy_loewe_bin'].mean())

overlap_drugs = set(pd.concat([seen_studies_df['drug_row_id'], seen_studies_df['drug_col_id']]).astype(str)) & set(
    pd.concat([external_studies_df['drug_row_id'], external_studies_df['drug_col_id']]).astype(str)
)
overlap_cells = set(seen_studies_df['cell_line_name'].astype(str)) & set(external_studies_df['cell_line_name'].astype(str))

external_domain_shift_summary = {
    'n_seen': int(len(seen_studies_df)),
    'n_external': int(len(external_studies_df)),
    'seen_positive_rate': float(seen_studies_df['synergy_loewe_bin'].mean()),
    'external_positive_rate': float(external_studies_df['synergy_loewe_bin'].mean()),
    'n_seen_studies': int(seen_studies_df['study_name'].nunique()),
    'n_external_studies': int(external_studies_df['study_name'].nunique()),
    'n_seen_drugs': int(pd.concat([seen_studies_df['drug_row_id'], seen_studies_df['drug_col_id']]).nunique()),
    'n_external_drugs': int(pd.concat([external_studies_df['drug_row_id'], external_studies_df['drug_col_id']]).nunique()),
    'n_overlapping_drugs': int(len(overlap_drugs)),
    'n_seen_cell_lines': int(seen_studies_df['cell_line_name'].nunique()),
    'n_external_cell_lines': int(external_studies_df['cell_line_name'].nunique()),
    'n_overlapping_cell_lines': int(len(overlap_cells)),
}

with open(os.path.join(EXTERNAL_OUTPUT_DIR, 'external_domain_shift_summary.json'), 'w') as f:
    json.dump(external_domain_shift_summary, f, indent=4)

pd.DataFrame([external_domain_shift_summary]).to_csv(
    os.path.join(EXTERNAL_OUTPUT_DIR, 'external_domain_shift_summary.csv'),
    index=False
)


# ============================================================
# MODEL SPECS
# Pick the winner of each internal strategy here.
# Start with XGBoost/RF because they are easiest to rerun and interpret.
# Replace these with the actual winners after reading your summaries.
# ============================================================

rf_config = {
    'n_estimators': 600,
    'max_depth': None,
    'min_samples_split': 2,
    'min_samples_leaf': 1,
    'max_features': 'sqrt',
    'class_weight': 'balanced_subsample',
    'random_state': 42,
    'n_jobs': 4,
}

xgb_config = {
    'n_estimators': 600,
    'max_depth': 8,
    'learning_rate': 0.03,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'objective': 'binary:logistic',
    'eval_metric': 'aucpr',
    'tree_method': 'hist',
    'random_state': 42,
    'n_jobs': 4,
}


mlp_config = {
    'lr': 1e-3,
    'batch_size': 256,
    'epochs': 80,
    'patience': 12,
    'weight_decay': 1e-4,
    'drug_dim': 1024,
    'n_genes': 500,
    'random_state': 42,
    'use_pos_weight': True,
}


BEST_MODEL_BY_STRATEGY = {
    'stratified_kfold': {
        'model_name': 'TripleBranchMLP',
        'model_config': mlp_config,
    },
    'partial_lodo_cv': {
        'model_name': 'RandomForestClassifier',
        'model_config': rf_config,
    },
    'unique_sample_lodo_cv': {
        'model_name': 'RandomForestClassifier',
        'model_config': rf_config,
    },
}


class ExternalSynergyDataset(Dataset):
    def __init__(self, df, gene_cols):
        d1_cols = [f'drug1_bit_{i}' for i in range(1024)]
        d2_cols = [f'drug2_bit_{i}' for i in range(1024)]
        self.d1 = torch.tensor(df[d1_cols].values, dtype=torch.float32)
        self.d2 = torch.tensor(df[d2_cols].values, dtype=torch.float32)
        self.genes = torch.tensor(df[gene_cols].values, dtype=torch.float32)
        self.y = torch.tensor(df['synergy_loewe_bin'].values, dtype=torch.float32).view(-1, 1)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.d1[idx], self.d2[idx], self.genes[idx], self.y[idx]


class ExternalTripleBranchMLP(nn.Module):
    def __init__(self, drug_dim=1024, gene_dim=500):
        super().__init__()
        self.drug_branch = nn.Sequential(
            nn.Linear(drug_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.ReLU()
        )
        self.gene_branch = nn.Sequential(
            nn.Linear(gene_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.ReLU()
        )
        self.classifier = nn.Sequential(
            nn.Linear(256 * 3, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, d1, d2, g):
        z1 = self.drug_branch(d1)
        z2 = self.drug_branch(d2)
        zg = self.gene_branch(g)
        return self.classifier(torch.cat((z1, z2, zg), dim=1))


def train_triple_branch_fold(model, train_loader, val_loader, y_train, config, device):
    if config.get('use_pos_weight', True):
        n_pos = float(np.sum(y_train == 1))
        n_neg = float(np.sum(y_train == 0))
        pos_weight = torch.tensor([n_neg / max(n_pos, 1.0)], dtype=torch.float32, device=device)
        criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    else:
        criterion = nn.BCEWithLogitsLoss()

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config['lr'],
        weight_decay=config['weight_decay']
    )

    best_state = None
    best_val_loss = np.inf
    epochs_no_improve = 0

    for epoch in range(1, config['epochs'] + 1):
        model.train()
        train_losses = []
        for d1, d2, genes, y in train_loader:
            d1 = d1.to(device)
            d2 = d2.to(device)
            genes = genes.to(device)
            y = y.to(device)

            optimizer.zero_grad()
            logits = model(d1, d2, genes)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()
            train_losses.append(loss.item())

        model.eval()
        val_losses = []
        with torch.no_grad():
            for d1, d2, genes, y in val_loader:
                d1 = d1.to(device)
                d2 = d2.to(device)
                genes = genes.to(device)
                y = y.to(device)
                logits = model(d1, d2, genes)
                loss = criterion(logits, y)
                val_losses.append(loss.item())

        val_loss = float(np.mean(val_losses))

        print(
            f'Epoch {epoch:03d}/{config["epochs"]} | '
            f'TrainLoss={np.mean(train_losses):.4f} | ValLoss={val_loss:.4f}',
            end='\r'
        )

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= config['patience']:
                break

    print()
    if best_state is not None:
        model.load_state_dict(best_state)

    return model


def predict_triple_branch(model, loader, device):
    model.eval()
    probs = []
    ys = []

    with torch.no_grad():
        for d1, d2, genes, y in loader:
            d1 = d1.to(device)
            d2 = d2.to(device)
            genes = genes.to(device)
            logits = model(d1, d2, genes)
            prob = torch.sigmoid(logits).detach().cpu().numpy().ravel()
            probs.append(prob)
            ys.append(y.numpy().ravel())

    return np.concatenate(ys), np.concatenate(probs)


def make_sklearn_model(model_name, model_config, y_train, seed):
    cfg = model_config.copy()
    cfg['random_state'] = seed

    if model_name == 'RandomForestClassifier':
        return RandomForestClassifier(**cfg)

    if model_name == 'XGBClassifier':
        n_pos = int(np.sum(y_train == 1))
        n_neg = int(np.sum(y_train == 0))
        cfg['scale_pos_weight'] = n_neg / max(n_pos, 1)
        return XGBClassifier(**cfg)

    raise ValueError(f'Modelo sklearn no soportado aqui: {model_name}')


def set_torch_seed(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def make_internal_splits_for_strategy(strategy_name, base_df, n_splits=10, random_state=42):
    if strategy_name == 'stratified_kfold':
        skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
        return [
            {
                'fold': fold_idx,
                'train_idx': train_idx,
                'test_idx': test_idx,
                'heldout_drugs': [],
            }
            for fold_idx, (train_idx, test_idx) in enumerate(
                skf.split(base_df, base_df['synergy_loewe_bin']),
                start=1
            )
        ]

    if strategy_name == 'partial_lodo_cv':
        fold_sets, _, _, _ = build_lodo_folds_random_search(
            base_df,
            n_splits=n_splits,
            n_iter=1000,
            random_state=random_state
        )

        splits = []
        for fold_idx, fold_set in enumerate(fold_sets, start=1):
            mask_test = (
                base_df['drug_row_id'].astype(str).isin(fold_set)
                | base_df['drug_col_id'].astype(str).isin(fold_set)
            )
            mask_train = ~mask_test
            splits.append({
                'fold': fold_idx,
                'train_idx': np.flatnonzero(mask_train.to_numpy()),
                'test_idx': np.flatnonzero(mask_test.to_numpy()),
                'heldout_drugs': sorted(list(fold_set)),
            })
        return splits

    if strategy_name == 'unique_sample_lodo_cv':
        fold_sets_unique = build_random_drug_folds(
            base_df,
            n_splits=n_splits,
            random_state=random_state
        )
        df_unique_lodo = assign_samples_to_unique_lodo_fold(
            base_df,
            fold_sets_unique,
            random_state=random_state
        )

        splits = []
        for fold_id in range(n_splits):
            test_set = fold_sets_unique[fold_id]
            test_idx = df_unique_lodo.index[
                df_unique_lodo['unique_lodo_fold'] == fold_id
            ].to_numpy()

            heldout_mask = (
                base_df['drug_row_id'].isin(test_set)
                | base_df['drug_col_id'].isin(test_set)
            )
            train_idx = base_df.index[~heldout_mask].to_numpy()

            splits.append({
                'fold': fold_id + 1,
                'train_idx': train_idx,
                'test_idx': test_idx,
                'heldout_drugs': sorted(list(test_set)),
            })
        return splits

    raise ValueError(f'Estrategia no reconocida: {strategy_name}')


def evaluate_external_by_group(predictions, group_col, threshold_col='threshold_best_f1'):
    rows = []
    for group_value, group_df in predictions.groupby(group_col):
        if len(group_df) < 5:
            continue
        threshold = float(group_df[threshold_col].iloc[0])
        metrics = compute_metrics_at_threshold(
            group_df['y_true'].to_numpy(),
            group_df['y_prob'].to_numpy(),
            threshold
        )
        metrics[group_col] = group_value
        metrics['n_samples'] = int(len(group_df))
        metrics['positive_rate'] = float(group_df['y_true'].mean())
        rows.append(metrics)
    return pd.DataFrame(rows)


def train_cv_ensemble_and_predict_external(
    strategy_name,
    model_name,
    model_config,
    seen_df,
    external_df,
    data_expr,
    output_dir,
    n_splits=10,
    n_genes=500,
    random_state=42,
):
    strategy_dir = os.path.join(output_dir, strategy_name)
    if os.path.exists(strategy_dir):
        shutil.rmtree(strategy_dir)
    os.makedirs(strategy_dir, exist_ok=True)
    os.makedirs(os.path.join(strategy_dir, 'predictions'), exist_ok=True)
    os.makedirs(os.path.join(strategy_dir, 'models'), exist_ok=True)
    os.makedirs(os.path.join(strategy_dir, 'figures'), exist_ok=True)
    tb_dir = os.path.join(strategy_dir, 'tensorboard')
    os.makedirs(tb_dir, exist_ok=True)

    experiment_config = {
        'model': model_name,
        'split_type': strategy_name,
        'external_test_definition': 'Studies not included in TRAIN_STUDIES',
        'train_studies': TRAIN_STUDIES,
        'external_studies': sorted(external_df['study_name'].astype(str).unique().tolist()),
        'n_splits': n_splits,
        'n_genes': n_genes,
        'random_state': random_state,
        'threshold_grid_min': 0.05,
        'threshold_grid_max': 0.95,
        'threshold_grid_step': 0.01,
        'model_config': model_config,
    }

    if 'save_experiment_config' in globals():
        save_experiment_config(experiment_config, strategy_dir)
    else:
        with open(os.path.join(strategy_dir, 'config.json'), 'w') as f:
            json.dump(experiment_config, f, indent=4)

    splits = make_internal_splits_for_strategy(
        strategy_name,
        seen_df,
        n_splits=n_splits,
        random_state=random_state
    )

    fold_metrics = []
    fold_thresholds = []
    external_prob_by_fold = []
    external_meta_ref = None
    cv_val_results_f1 = []
    cv_val_results_precision = []
    cv_external_results_f1 = []
    cv_external_results_precision = []
    cv_split_info = []
    cv_val_threshold_metrics = []
    cv_external_threshold_metrics = []
    cv_best_thresholds = []
    cv_histories = []

    for split in splits:
        fold_idx = int(split['fold'])
        seed = random_state + fold_idx

        print(f'\n[{strategy_name}] Fold {fold_idx}/{n_splits} - {model_name}')

        df_train_raw = seen_df.iloc[split['train_idx']].reset_index(drop=True)
        df_internal_test_raw = seen_df.iloc[split['test_idx']].reset_index(drop=True)
        writer = SummaryWriter(log_dir=os.path.join(tb_dir, f'Fold_{fold_idx}'))

        train_drugs = set(pd.concat([
            df_train_raw['drug_row_id'],
            df_train_raw['drug_col_id']
        ]).astype(str))

        internal_test_drugs = set(pd.concat([
            df_internal_test_raw['drug_row_id'],
            df_internal_test_raw['drug_col_id']
        ]).astype(str))

        external_drugs = set(pd.concat([
            external_df['drug_row_id'],
            external_df['drug_col_id']
        ]).astype(str))

        drug_overlap_train_internal_test = len(train_drugs & internal_test_drugs)
        drug_overlap_train_internal_test_pct = (
            100 * drug_overlap_train_internal_test / len(internal_test_drugs)
            if len(internal_test_drugs) > 0 else 0.0
        )

        drug_overlap_train_external = len(train_drugs & external_drugs)
        drug_overlap_train_external_pct = (
            100 * drug_overlap_train_external / len(external_drugs)
            if len(external_drugs) > 0 else 0.0
        )

        df_train, df_val, df_external, top_genes = prepare_fold_data_no_leakage(
            df_train_raw,
            external_df,
            data_expr,
            n_genes=n_genes,
            random_state=seed
        )

        if model_name == 'TripleBranchMLP':
            set_torch_seed(seed)
            device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

            y_train = df_train['synergy_loewe_bin'].astype(int).to_numpy()
            y_val = df_val['synergy_loewe_bin'].astype(int).to_numpy()

            if len(np.unique(y_train)) < 2 or len(np.unique(y_val)) < 2:
                print('Fold omitido por tener una sola clase en train o val.')
                writer.close()
                continue

            dataset_cls = globals().get('SynergyDataset', ExternalSynergyDataset)
            model_cls = globals().get('TripleBranchMLP', ExternalTripleBranchMLP)

            train_ds = dataset_cls(df_train, top_genes)
            val_ds = dataset_cls(df_val, top_genes)
            external_ds = dataset_cls(df_external, top_genes)

            train_loader = DataLoader(
                train_ds,
                batch_size=model_config['batch_size'],
                shuffle=True,
                drop_last=False
            )
            val_loader = DataLoader(
                val_ds,
                batch_size=model_config['batch_size'],
                shuffle=False,
                drop_last=False
            )
            external_loader = DataLoader(
                external_ds,
                batch_size=model_config['batch_size'],
                shuffle=False,
                drop_last=False
            )

            model = model_cls(
                drug_dim=model_config['drug_dim'],
                gene_dim=len(top_genes)
            ).to(device)

            if 'train_torch_model_fold' in globals():
                model, history_df, best_epoch, best_val_loss = train_torch_model_fold(
                    model,
                    train_loader,
                    val_loader,
                    y_train,
                    model_config,
                    device,
                    writer,
                    fold_idx
                )
            else:
                model = train_triple_branch_fold(
                    model,
                    train_loader,
                    val_loader,
                    y_train,
                    model_config,
                    device
                )
                history_df = pd.DataFrame()
                best_epoch = np.nan
                best_val_loss = np.nan

            if len(history_df) > 0:
                history_df.to_csv(
                    os.path.join(strategy_dir, f'training_history_fold_{fold_idx:02d}.csv'),
                    index=False
                )
                cv_histories.append(history_df)

            if 'predict_torch_model' in globals():
                y_val, y_prob_val = predict_torch_model(model, val_loader, device)
                y_external, y_prob_external = predict_torch_model(model, external_loader, device)
            else:
                y_val, y_prob_val = predict_triple_branch(model, val_loader, device)
                y_external, y_prob_external = predict_triple_branch(model, external_loader, device)

            feature_cols = [f'drug1_bit_{i}' for i in range(1024)] + [f'drug2_bit_{i}' for i in range(1024)] + top_genes

        else:
            _, _, X_val, y_val, X_external, y_external, feature_cols = build_tabular_matrices(
                df_train,
                df_val,
                df_external
            )

            X_train = df_train[feature_cols].fillna(0.0).to_numpy(dtype=np.float32)
            y_train = df_train['synergy_loewe_bin'].astype(int).to_numpy()

            if len(np.unique(y_train)) < 2 or len(np.unique(y_val)) < 2:
                print('Fold omitido por tener una sola clase en train o val.')
                writer.close()
                continue

            model = make_sklearn_model(model_name, model_config, y_train, seed)
            model.fit(X_train, y_train)

            y_prob_val = model.predict_proba(X_val)[:, 1]
            y_prob_external = model.predict_proba(X_external)[:, 1]
            best_epoch = np.nan
            best_val_loss = np.nan

        df_val_thr, best_thr_f1, best_thr_precision = find_best_thresholds(
            y_val,
            y_prob_val,
            thresholds
        )

        df_val_thr.insert(0, 'fold', fold_idx)
        df_val_thr.insert(1, 'split', 'val')
        df_val_thr.insert(2, 'threshold_type', 'grid')
        cv_val_threshold_metrics.append(df_val_thr)

        external_thr_rows = []
        for thr in thresholds:
            m_external_thr = compute_metrics_at_threshold(y_external, y_prob_external, thr)
            m_external_thr.update({
                'fold': fold_idx,
                'split': 'external',
                'threshold_type': 'grid',
            })
            external_thr_rows.append(m_external_thr)

        df_external_thr = pd.DataFrame(external_thr_rows)
        cv_external_threshold_metrics.append(df_external_thr)

        val_metrics_f1 = compute_metrics_at_threshold(y_val, y_prob_val, best_thr_f1)
        val_metrics_precision = compute_metrics_at_threshold(y_val, y_prob_val, best_thr_precision)

        external_metrics_f1 = compute_metrics_at_threshold(
            y_external,
            y_prob_external,
            best_thr_f1
        )
        external_metrics_precision = compute_metrics_at_threshold(
            y_external,
            y_prob_external,
            best_thr_precision
        )

        val_metrics_f1.update({'fold': fold_idx, 'split': 'val', 'threshold_type': 'best_f1'})
        val_metrics_precision.update({'fold': fold_idx, 'split': 'val', 'threshold_type': 'best_precision'})
        external_metrics_f1.update({'fold': fold_idx, 'split': 'external', 'threshold_type': 'best_f1'})
        external_metrics_precision.update({'fold': fold_idx, 'split': 'external', 'threshold_type': 'best_precision'})

        cv_val_results_f1.append(val_metrics_f1.copy())
        cv_val_results_precision.append(val_metrics_precision.copy())
        cv_external_results_f1.append(external_metrics_f1.copy())
        cv_external_results_precision.append(external_metrics_precision.copy())

        external_metrics_f1.update({
            'strategy': strategy_name,
            'model': model_name,
            'fold': fold_idx,
            'threshold_type': 'best_f1',
            'n_external_after_gene_merge': int(len(y_external)),
            'n_features': int(len(feature_cols)),
            'n_top_genes': int(len(top_genes)),
        })
        external_metrics_precision.update({
            'strategy': strategy_name,
            'model': model_name,
            'fold': fold_idx,
            'threshold_type': 'best_precision',
            'n_external_after_gene_merge': int(len(y_external)),
            'n_features': int(len(feature_cols)),
            'n_top_genes': int(len(top_genes)),
        })

        fold_metrics.extend([external_metrics_f1, external_metrics_precision])
        fold_thresholds.append({
            'strategy': strategy_name,
            'model': model_name,
            'fold': fold_idx,
            'best_threshold_f1': best_thr_f1,
            'best_threshold_precision': best_thr_precision,
            'val_f1_at_best_f1': val_metrics_f1['f1'],
            'val_precision_at_best_f1': val_metrics_f1['precision'],
            'val_recall_at_best_f1': val_metrics_f1['recall'],
            'val_precision_at_best_precision': val_metrics_precision['precision'],
            'val_recall_at_best_precision': val_metrics_precision['recall'],
            'val_f1_at_best_precision': val_metrics_precision['f1'],
            'best_epoch': best_epoch,
            'best_val_loss': best_val_loss,
        })
        cv_best_thresholds.append(fold_thresholds[-1].copy())

        external_prob_by_fold.append(y_prob_external)

        if external_meta_ref is None:
            external_meta_ref = df_external[
                [
                    'drug_row_id',
                    'drug_col_id',
                    'cell_line_name',
                    'study_name',
                    'tissue',
                    'synergy_loewe',
                    'synergy_loewe_bin',
                ]
            ].copy().reset_index(drop=True)
            external_meta_ref['y_true'] = y_external

        if model_name == 'TripleBranchMLP':
            torch.save(
                {
                    'model_state_dict': model.state_dict(),
                    'top_genes': top_genes,
                    'model_config': model_config,
                },
                os.path.join(strategy_dir, 'models', f'{model_name}_fold_{fold_idx:02d}.pt')
            )
        else:
            joblib.dump(
                model,
                os.path.join(strategy_dir, 'models', f'{model_name}_fold_{fold_idx:02d}.joblib')
            )

        if 'log_curves_and_metrics' in globals():
            log_curves_and_metrics(y_val, y_prob_val, val_metrics_f1, fold_idx, writer, tag='Val_BestF1')
            log_curves_and_metrics(y_val, y_prob_val, val_metrics_precision, fold_idx, writer, tag='Val_BestPrecision')
            log_curves_and_metrics(y_external, y_prob_external, external_metrics_f1, fold_idx, writer, tag='External_BestF1')
            log_curves_and_metrics(y_external, y_prob_external, external_metrics_precision, fold_idx, writer, tag='External_BestPrecision')

        predictions_fold = df_external[
            [
                'drug_row_id',
                'drug_col_id',
                'cell_line_name',
                'study_name',
                'tissue',
                'synergy_loewe',
                'synergy_loewe_bin'
            ]
        ].copy()

        predictions_fold['y_true'] = y_external
        predictions_fold['y_prob'] = y_prob_external
        predictions_fold['y_pred_best_f1'] = (y_prob_external >= best_thr_f1).astype(int)
        predictions_fold['y_pred_best_precision'] = (y_prob_external >= best_thr_precision).astype(int)
        predictions_fold['threshold_best_f1'] = best_thr_f1
        predictions_fold['threshold_best_precision'] = best_thr_precision

        predictions_fold.to_csv(
            os.path.join(strategy_dir, 'predictions', f'predictions_fold_{fold_idx:02d}.csv'),
            index=False
        )

        split_info = {
            'fold': fold_idx,
            'strategy': strategy_name,
            'model': model_name,
            'n_train': int(len(y_train)),
            'n_val': int(len(y_val)),
            'n_external': int(len(y_external)),
            'train_pos': int(np.sum(y_train)),
            'val_pos': int(np.sum(y_val)),
            'external_pos': int(np.sum(y_external)),
            'train_pos_pct': float(100 * np.mean(y_train)),
            'val_pos_pct': float(100 * np.mean(y_val)),
            'external_pos_pct': float(100 * np.mean(y_external)),
            'baseline_auprc_external': float(np.mean(y_external)),
            'n_train_drugs': int(len(train_drugs)),
            'n_internal_test_drugs': int(len(internal_test_drugs)),
            'n_external_drugs': int(len(external_drugs)),
            'drug_overlap_train_internal_test': int(drug_overlap_train_internal_test),
            'drug_overlap_train_internal_test_pct': float(drug_overlap_train_internal_test_pct),
            'drug_overlap_train_external': int(drug_overlap_train_external),
            'drug_overlap_train_external_pct': float(drug_overlap_train_external_pct),
            'best_threshold_f1': float(best_thr_f1),
            'best_threshold_precision': float(best_thr_precision),
            'best_epoch': float(best_epoch) if pd.notna(best_epoch) else np.nan,
            'best_val_loss': float(best_val_loss) if pd.notna(best_val_loss) else np.nan,
            'n_features': int(len(feature_cols)),
            'n_top_genes': int(len(top_genes)),
            'external_studies': ';'.join(sorted(external_df['study_name'].astype(str).unique())),
        }

        if len(split.get('heldout_drugs', [])) > 0:
            split_info['n_heldout_drugs'] = len(split['heldout_drugs'])
            split_info['heldout_drugs'] = ';'.join(split['heldout_drugs'])

        cv_split_info.append(split_info)

        writer.add_text('Split/Info', json.dumps(split_info, indent=2))
        writer.add_text('Genes/TopGenes_First100', ', '.join(top_genes[:100]))

        print(
            f"Fold {fold_idx} | "
            f"Train={len(y_train)} ({100*np.mean(y_train):.2f}% pos), "
            f"Val={len(y_val)} ({100*np.mean(y_val):.2f}% pos), "
            f"External={len(y_external)} ({100*np.mean(y_external):.2f}% pos) | "
            f"BestEpoch={best_epoch} | "
            f"ThrF1={best_thr_f1:.2f}, ThrPrec={best_thr_precision:.2f} | "
            f"External AUPRC={external_metrics_f1['auprc']:.4f} "
            f"(base={external_metrics_f1['baseline_auprc']:.4f}) | "
            f"F1={external_metrics_f1['f1']:.4f} | "
            f"Prec={external_metrics_f1['precision']:.4f} | "
            f"Rec={external_metrics_f1['recall']:.4f} | "
            f"MCC={external_metrics_f1['mcc']:.4f}"
        )

        writer.close()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    fold_metrics_df = pd.DataFrame(fold_metrics)
    fold_thresholds_df = pd.DataFrame(fold_thresholds)

    fold_metrics_df.to_csv(
        os.path.join(strategy_dir, 'external_metrics_by_fold.csv'),
        index=False
    )
    fold_thresholds_df.to_csv(
        os.path.join(strategy_dir, 'external_best_thresholds_by_fold.csv'),
        index=False
    )

    df_val_f1 = pd.DataFrame(cv_val_results_f1)
    df_val_precision = pd.DataFrame(cv_val_results_precision)
    df_external_f1 = pd.DataFrame(cv_external_results_f1)
    df_external_precision = pd.DataFrame(cv_external_results_precision)
    df_split_info = pd.DataFrame(cv_split_info)
    df_best_thresholds = pd.DataFrame(cv_best_thresholds)

    if len(cv_val_threshold_metrics) > 0:
        df_val_thresholds = pd.concat(cv_val_threshold_metrics, ignore_index=True)
    else:
        df_val_thresholds = pd.DataFrame()

    if len(cv_external_threshold_metrics) > 0:
        df_external_thresholds = pd.concat(cv_external_threshold_metrics, ignore_index=True)
    else:
        df_external_thresholds = pd.DataFrame()

    df_val_f1.to_csv(os.path.join(strategy_dir, 'val_metrics_best_f1.csv'), index=False)
    df_val_precision.to_csv(os.path.join(strategy_dir, 'val_metrics_best_precision.csv'), index=False)
    df_external_f1.to_csv(os.path.join(strategy_dir, 'external_metrics_best_f1.csv'), index=False)
    df_external_precision.to_csv(os.path.join(strategy_dir, 'external_metrics_best_precision.csv'), index=False)

    # Aliases with the same naming style as the previous CV experiments.
    df_external_f1.to_csv(os.path.join(strategy_dir, 'test_metrics_best_f1.csv'), index=False)
    df_external_precision.to_csv(os.path.join(strategy_dir, 'test_metrics_best_precision.csv'), index=False)

    df_split_info.to_csv(os.path.join(strategy_dir, 'split_info.csv'), index=False)
    df_best_thresholds.to_csv(os.path.join(strategy_dir, 'best_thresholds.csv'), index=False)
    df_val_thresholds.to_csv(os.path.join(strategy_dir, 'val_metrics_by_threshold.csv'), index=False)
    df_external_thresholds.to_csv(os.path.join(strategy_dir, 'external_metrics_by_threshold.csv'), index=False)
    df_external_thresholds.to_csv(os.path.join(strategy_dir, 'test_metrics_by_threshold.csv'), index=False)

    if len(cv_histories) > 0:
        df_all_history = pd.concat(cv_histories, ignore_index=True)
        df_all_history.to_csv(os.path.join(strategy_dir, 'training_history_all_folds.csv'), index=False)

    metric_cols = [
        'auc',
        'auprc',
        'baseline_auprc',
        'f1',
        'precision',
        'recall',
        'specificity',
        'balanced_accuracy',
        'mcc',
        'log_loss',
        'brier',
        'tp',
        'tn',
        'fp',
        'fn',
    ]

    metric_cols_available_f1 = [c for c in metric_cols if c in df_external_f1.columns]
    metric_cols_available_precision = [c for c in metric_cols if c in df_external_precision.columns]

    summary_f1 = (
        df_external_f1[metric_cols_available_f1]
        .agg(['mean', 'std'])
        .T
        .reset_index()
        .rename(columns={'index': 'metric'})
    )

    summary_precision = (
        df_external_precision[metric_cols_available_precision]
        .agg(['mean', 'std'])
        .T
        .reset_index()
        .rename(columns={'index': 'metric'})
    )

    summary_f1.to_csv(os.path.join(strategy_dir, 'summary_external_best_f1.csv'), index=False)
    summary_precision.to_csv(os.path.join(strategy_dir, 'summary_external_best_precision.csv'), index=False)

    # Aliases with the same naming style as the previous CV experiments.
    summary_f1.to_csv(os.path.join(strategy_dir, 'summary_test_best_f1.csv'), index=False)
    summary_precision.to_csv(os.path.join(strategy_dir, 'summary_test_best_precision.csv'), index=False)

    if external_meta_ref is None:
        raise RuntimeError(f'No se pudo entrenar ningun fold para {strategy_name}.')

    external_prob_matrix = np.vstack(external_prob_by_fold)
    y_prob_ensemble = external_prob_matrix.mean(axis=0)

    ensemble_thr_f1 = float(fold_thresholds_df['best_threshold_f1'].median())
    ensemble_thr_precision = float(fold_thresholds_df['best_threshold_precision'].median())

    y_external = external_meta_ref['y_true'].to_numpy()
    ensemble_metrics_f1 = compute_metrics_at_threshold(
        y_external,
        y_prob_ensemble,
        ensemble_thr_f1
    )
    ensemble_metrics_precision = compute_metrics_at_threshold(
        y_external,
        y_prob_ensemble,
        ensemble_thr_precision
    )

    ensemble_metrics_f1.update({
        'strategy': strategy_name,
        'model': model_name,
        'threshold_type': 'ensemble_median_best_f1',
        'n_folds_in_ensemble': int(len(external_prob_by_fold)),
    })
    ensemble_metrics_precision.update({
        'strategy': strategy_name,
        'model': model_name,
        'threshold_type': 'ensemble_median_best_precision',
        'n_folds_in_ensemble': int(len(external_prob_by_fold)),
    })

    ensemble_metrics_df = pd.DataFrame([ensemble_metrics_f1, ensemble_metrics_precision])
    ensemble_metrics_df.to_csv(
        os.path.join(strategy_dir, 'external_ensemble_metrics.csv'),
        index=False
    )

    predictions = external_meta_ref.copy()
    predictions['y_prob'] = y_prob_ensemble
    predictions['y_pred_best_f1'] = (y_prob_ensemble >= ensemble_thr_f1).astype(int)
    predictions['y_pred_best_precision'] = (y_prob_ensemble >= ensemble_thr_precision).astype(int)
    predictions['threshold_best_f1'] = ensemble_thr_f1
    predictions['threshold_best_precision'] = ensemble_thr_precision

    for i, fold_prob in enumerate(external_prob_by_fold, start=1):
        predictions[f'y_prob_fold_{i:02d}'] = fold_prob

    predictions.to_csv(
        os.path.join(strategy_dir, 'predictions', 'external_unseen_studies_predictions_ensemble.csv'),
        index=False
    )

    by_study = evaluate_external_by_group(predictions, 'study_name')
    by_tissue = evaluate_external_by_group(predictions, 'tissue')
    by_cell = evaluate_external_by_group(predictions, 'cell_line_name')

    by_study.to_csv(os.path.join(strategy_dir, 'external_metrics_by_study.csv'), index=False)
    by_tissue.to_csv(os.path.join(strategy_dir, 'external_metrics_by_tissue.csv'), index=False)
    by_cell.to_csv(os.path.join(strategy_dir, 'external_metrics_by_cell_line.csv'), index=False)

    print('\nExternal ensemble metrics:')
    print(ensemble_metrics_df)
    print('\nExternal metrics by study:')
    print(by_study.sort_values('auprc', ascending=False))

    return {
        'fold_metrics': fold_metrics_df,
        'ensemble_metrics': ensemble_metrics_df,
        'predictions': predictions,
        'by_study': by_study,
        'by_tissue': by_tissue,
        'by_cell_line': by_cell,
    }





In [ ]:
# ============================================================
# RUN EXTERNAL EVALUATION
# ============================================================

all_external_results = {}

for strategy_name, spec in BEST_MODEL_BY_STRATEGY.items():
    all_external_results[strategy_name] = train_cv_ensemble_and_predict_external(
        strategy_name=strategy_name,
        model_name=spec['model_name'],
        model_config=spec['model_config'],
        seen_df=seen_studies_df,
        external_df=external_studies_df,
        data_expr=data_expr,
        output_dir=EXTERNAL_OUTPUT_DIR,
        n_splits=10,
        n_genes=500,
        random_state=42,
    )

summary_external = pd.concat(
    [res['ensemble_metrics'] for res in all_external_results.values()],
    ignore_index=True
)

summary_external.to_csv(
    os.path.join(EXTERNAL_OUTPUT_DIR, 'summary_external_ensemble_metrics.csv'),
    index=False
)

print('\nSUMMARY EXTERNAL UNSEEN STUDIES')
print(summary_external)

In [ ]:
# ============================================================
# CHARACTERIZATION OF SEEN VS UNSEEN STUDIES
# ============================================================
# Expected variables in the notebook:
# dloewe_all   -> dataset from the 4 selected studies
# dloewe_extra -> dataset from the studies left out before
# cellline2tissue
#
# The code saves tables and figures in:
# results/external_unseen_studies/dataset_characterization



CHAR_OUTPUT_DIR = 'results/external_unseen_studies/dataset_characterization'
os.makedirs(CHAR_OUTPUT_DIR, exist_ok=True)

sns.set_theme(style='whitegrid', context='notebook')


def _normalize_binary_label(s):
    return s.replace(-1, 0).astype(int)


def prepare_characterization_df(df, dataset_name, cellline2tissue=None):
    out = df.copy()
    out['dataset'] = dataset_name
    out['drug_row_id'] = out['drug_row_id'].astype(str)
    out['drug_col_id'] = out['drug_col_id'].astype(str)

    if 'synergy_loewe_bin' in out.columns:
        out = out[out['synergy_loewe_bin'].isin([1, -1, 0])].copy()
        # If neutral class is still present, remove it to match the modelling dataset.
        out = out[out['synergy_loewe_bin'] != 0].copy()
        out['synergy_loewe_bin'] = _normalize_binary_label(out['synergy_loewe_bin'])

    if 'tissue' not in out.columns and cellline2tissue is not None:
        out['tissue'] = out['cell_line_name'].map(cellline2tissue)

    out['tissue'] = out['tissue'].fillna('unknown')
    return out


seen_df = prepare_characterization_df(dloewe_all, '4 selected studies', cellline2tissue)
extra_df = prepare_characterization_df(dloewe_extra, 'unseen studies', cellline2tissue)

combined_df = pd.concat([seen_df, extra_df], ignore_index=True)


# ============================================================
# SUMMARY TABLES
# ============================================================

def dataset_summary(df, dataset_name):
    drugs = pd.concat([df['drug_row_id'], df['drug_col_id']], ignore_index=True).astype(str)
    pairs = (
        df[['drug_row_id', 'drug_col_id']]
        .apply(lambda r: tuple(sorted([str(r['drug_row_id']), str(r['drug_col_id'])])), axis=1)
    )

    return {
        'dataset': dataset_name,
        'n_samples': int(len(df)),
        'n_studies': int(df['study_name'].nunique()),
        'n_tissues': int(df['tissue'].nunique()),
        'n_cell_lines': int(df['cell_line_name'].nunique()),
        'n_drugs': int(drugs.nunique()),
        'n_drug_pairs': int(pd.Series(pairs).nunique()),
        'n_positive': int(df['synergy_loewe_bin'].sum()),
        'n_negative': int((df['synergy_loewe_bin'] == 0).sum()),
        'positive_pct': float(100 * df['synergy_loewe_bin'].mean()),
        'negative_pct': float(100 * (1 - df['synergy_loewe_bin'].mean())),
    }


summary_df = pd.DataFrame([
    dataset_summary(seen_df, '4 selected studies'),
    dataset_summary(extra_df, 'unseen studies'),
])

#summary_df.to_csv(os.path.join(CHAR_OUTPUT_DIR, 'dataset_summary_seen_vs_extra.csv'), index=False)
display(summary_df)


seen_drugs = set(pd.concat([seen_df['drug_row_id'], seen_df['drug_col_id']], ignore_index=True).astype(str))
extra_drugs = set(pd.concat([extra_df['drug_row_id'], extra_df['drug_col_id']], ignore_index=True).astype(str))

seen_cells = set(seen_df['cell_line_name'].astype(str))
extra_cells = set(extra_df['cell_line_name'].astype(str))

overlap_summary = pd.DataFrame([{
    'n_seen_drugs': len(seen_drugs),
    'n_extra_drugs': len(extra_drugs),
    'n_shared_drugs': len(seen_drugs & extra_drugs),
    'n_extra_unseen_drugs': len(extra_drugs - seen_drugs),
    'extra_unseen_drug_pct': 100 * len(extra_drugs - seen_drugs) / max(len(extra_drugs), 1),
    'n_seen_cell_lines': len(seen_cells),
    'n_extra_cell_lines': len(extra_cells),
    'n_shared_cell_lines': len(seen_cells & extra_cells),
    'n_extra_unseen_cell_lines': len(extra_cells - seen_cells),
    'extra_unseen_cell_line_pct': 100 * len(extra_cells - seen_cells) / max(len(extra_cells), 1),
}])

#overlap_summary.to_csv(os.path.join(CHAR_OUTPUT_DIR, 'overlap_seen_vs_extra.csv'), index=False)
display(overlap_summary)

pd.Series(sorted(extra_drugs - seen_drugs), name='unseen_drug_in_extra').to_csv(
    os.path.join(CHAR_OUTPUT_DIR, 'extra_unseen_drugs.csv'),
    index=False
)

pd.Series(sorted(extra_cells - seen_cells), name='unseen_cell_line_in_extra').to_csv(
    os.path.join(CHAR_OUTPUT_DIR, 'extra_unseen_cell_lines.csv'),
    index=False
)


# ============================================================
# TISSUE DISTRIBUTION
# ============================================================

tissue_counts = (
    combined_df
    .groupby(['dataset', 'tissue'])
    .size()
    .reset_index(name='n_samples')
)
tissue_counts['pct_within_dataset'] = (
    100
    * tissue_counts['n_samples']
    / tissue_counts.groupby('dataset')['n_samples'].transform('sum')
)

#tissue_counts.to_csv(os.path.join(CHAR_OUTPUT_DIR, 'tissue_distribution_seen_vs_extra.csv'), index=False)

plt.figure(figsize=(12, 6))
plot_df = tissue_counts.sort_values(['dataset', 'pct_within_dataset'], ascending=[True, False])
sns.barplot(
    data=plot_df,
    x='tissue',
    y='pct_within_dataset',
    hue='dataset'
)
plt.xticks(rotation=45, ha='right')
plt.ylabel('% dentro de cada dataset')
plt.xlabel('Tejido')
plt.title('Distribucion de tejidos: estudios usados vs estudios no vistos')
plt.tight_layout()
#plt.savefig(os.path.join(CHAR_OUTPUT_DIR, 'tissue_distribution_pct_seen_vs_extra.png'), dpi=300)
plt.show()


# ============================================================
# CLASS BALANCE
# ============================================================

class_counts = (
    combined_df
    .assign(class_label=lambda x: np.where(x['synergy_loewe_bin'] == 1, 'positive', 'negative'))
    .groupby(['dataset', 'class_label'])
    .size()
    .reset_index(name='n_samples')
)
class_counts['pct_within_dataset'] = (
    100
    * class_counts['n_samples']
    / class_counts.groupby('dataset')['n_samples'].transform('sum')
)

#class_counts.to_csv(os.path.join(CHAR_OUTPUT_DIR, 'class_balance_seen_vs_extra.csv'), index=False)

plt.figure(figsize=(7, 5))
sns.barplot(
    data=class_counts,
    x='dataset',
    y='pct_within_dataset',
    hue='class_label'
)
plt.ylabel('% dentro de cada dataset')
plt.xlabel('')
plt.title('Proporcion de positivos y negativos')
plt.tight_layout()
#plt.savefig(os.path.join(CHAR_OUTPUT_DIR, 'class_balance_seen_vs_extra.png'), dpi=300)
plt.show()


# Extra dataset alone, with absolute counts in labels.
extra_class_counts = class_counts[class_counts['dataset'] == 'unseen studies'].copy()
display(extra_class_counts)


# ============================================================
# CLASS BALANCE BY STUDY AND TISSUE IN EXTRA DATASET
# ============================================================

extra_by_study = (
    extra_df
    .groupby('study_name')
    .agg(
        n_samples=('synergy_loewe_bin', 'size'),
        n_positive=('synergy_loewe_bin', 'sum'),
        positive_pct=('synergy_loewe_bin', lambda x: 100 * x.mean()),
        n_cell_lines=('cell_line_name', 'nunique'),
        n_tissues=('tissue', 'nunique'),
    )
    .reset_index()
    .sort_values('n_samples', ascending=False)
)
extra_by_study['n_negative'] = extra_by_study['n_samples'] - extra_by_study['n_positive']
#extra_by_study.to_csv(os.path.join(CHAR_OUTPUT_DIR, 'extra_summary_by_study.csv'), index=False)
display(extra_by_study)

plt.figure(figsize=(12, 5))
sns.barplot(data=extra_by_study, x='study_name', y='positive_pct')
plt.xticks(rotation=45, ha='right')
plt.ylabel('% positivos')
plt.xlabel('Estudio externo')
plt.title('Proporcion de positivos por estudio no visto')
plt.tight_layout()
#plt.savefig(os.path.join(CHAR_OUTPUT_DIR, 'extra_positive_pct_by_study.png'), dpi=300)
plt.show()

extra_by_tissue = (
    extra_df
    .groupby('tissue')
    .agg(
        n_samples=('synergy_loewe_bin', 'size'),
        n_positive=('synergy_loewe_bin', 'sum'),
        positive_pct=('synergy_loewe_bin', lambda x: 100 * x.mean()),
        n_cell_lines=('cell_line_name', 'nunique'),
    )
    .reset_index()
    .sort_values('n_samples', ascending=False)
)
extra_by_tissue['n_negative'] = extra_by_tissue['n_samples'] - extra_by_tissue['n_positive']
#extra_by_tissue.to_csv(os.path.join(CHAR_OUTPUT_DIR, 'extra_summary_by_tissue.csv'), index=False)
display(extra_by_tissue)


# ============================================================
# DRUG REPETITION ANALYSIS
# ============================================================

def drug_occurrences(df, dataset_name):
    long_drugs = pd.concat([
        df[['drug_row_id', 'synergy_loewe_bin']].rename(columns={'drug_row_id': 'drug_id'}),
        df[['drug_col_id', 'synergy_loewe_bin']].rename(columns={'drug_col_id': 'drug_id'}),
    ], ignore_index=True)

    out = (
        long_drugs
        .groupby('drug_id')
        .agg(
            n_occurrences=('drug_id', 'size'),
            n_positive=('synergy_loewe_bin', 'sum'),
            positive_pct=('synergy_loewe_bin', lambda x: 100 * x.mean()),
        )
        .reset_index()
    )
    out['n_negative'] = out['n_occurrences'] - out['n_positive']
    out['dataset'] = dataset_name
    return out.sort_values('n_occurrences', ascending=False)


seen_drug_counts = drug_occurrences(seen_df, '4 selected studies')
extra_drug_counts = drug_occurrences(extra_df, 'unseen studies')
drug_counts = pd.concat([seen_drug_counts, extra_drug_counts], ignore_index=True)

#seen_drug_counts.to_csv(os.path.join(CHAR_OUTPUT_DIR, 'seen_drug_occurrences.csv'), index=False)
#extra_drug_counts.to_csv(os.path.join(CHAR_OUTPUT_DIR, 'extra_drug_occurrences.csv'), index=False)
#drug_counts.to_csv(os.path.join(CHAR_OUTPUT_DIR, 'drug_occurrences_seen_vs_extra.csv'), index=False)

display(extra_drug_counts.head(30))

plt.figure(figsize=(10, 5))
sns.histplot(
    data=drug_counts,
    x='n_occurrences',
    hue='dataset',
    bins=50,
    element='step',
    stat='count',
    common_norm=False
)
plt.yscale('log')
plt.xlabel('Numero de apariciones por droga')
plt.ylabel('Numero de drogas (escala log)')
plt.title('Distribucion de repeticiones por droga')
plt.tight_layout()
#plt.savefig(os.path.join(CHAR_OUTPUT_DIR, 'drug_occurrence_distribution_seen_vs_extra.png'), dpi=300)
plt.show()

top_n = 30
top_extra_drugs = extra_drug_counts.head(top_n)

plt.figure(figsize=(12, 7))
sns.barplot(
    data=top_extra_drugs,
    y='drug_id',
    x='n_occurrences',
    color='steelblue'
)
plt.xlabel('Numero de apariciones')
plt.ylabel('Droga')
plt.title(f'Top {top_n} drogas mas repetidas en estudios no vistos')
plt.tight_layout()
#plt.savefig(os.path.join(CHAR_OUTPUT_DIR, 'top_extra_drugs_by_occurrence.png'), dpi=300)
plt.show()


# ============================================================
# CELL LINE REPETITION ANALYSIS
# ============================================================

def cell_line_occurrences(df, dataset_name):
    out = (
        df
        .groupby(['cell_line_name', 'tissue'])
        .agg(
            n_samples=('synergy_loewe_bin', 'size'),
            n_positive=('synergy_loewe_bin', 'sum'),
            positive_pct=('synergy_loewe_bin', lambda x: 100 * x.mean()),
        )
        .reset_index()
    )
    out['n_negative'] = out['n_samples'] - out['n_positive']
    out['dataset'] = dataset_name
    return out.sort_values('n_samples', ascending=False)


seen_cell_counts = cell_line_occurrences(seen_df, '4 selected studies')
extra_cell_counts = cell_line_occurrences(extra_df, 'unseen studies')
cell_counts = pd.concat([seen_cell_counts, extra_cell_counts], ignore_index=True)

#seen_cell_counts.to_csv(os.path.join(CHAR_OUTPUT_DIR, 'seen_cell_line_occurrences.csv'), index=False)
#extra_cell_counts.to_csv(os.path.join(CHAR_OUTPUT_DIR, 'extra_cell_line_occurrences.csv'), index=False)
#cell_counts.to_csv(os.path.join(CHAR_OUTPUT_DIR, 'cell_line_occurrences_seen_vs_extra.csv'), index=False)

display(extra_cell_counts.head(30))

plt.figure(figsize=(10, 5))
sns.histplot(
    data=cell_counts,
    x='n_samples',
    hue='dataset',
    bins=50,
    element='step',
    stat='count',
    common_norm=False
)
plt.yscale('log')
plt.xlabel('Numero de muestras por cell line')
plt.ylabel('Numero de cell lines (escala log)')
plt.title('Distribucion de muestras por cell line')
plt.tight_layout()
#plt.savefig(os.path.join(CHAR_OUTPUT_DIR, 'cell_line_sample_distribution_seen_vs_extra.png'), dpi=300)
plt.show()

top_extra_cells = extra_cell_counts.head(top_n)

plt.figure(figsize=(12, 7))
sns.barplot(
    data=top_extra_cells,
    y='cell_line_name',
    x='n_samples',
    hue='tissue',
    dodge=False
)
plt.xlabel('Numero de muestras')
plt.ylabel('Cell line')
plt.title(f'Top {top_n} cell lines mas repetidas en estudios no vistos')
plt.legend(title='Tejido', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
#plt.savefig(os.path.join(CHAR_OUTPUT_DIR, 'top_extra_cell_lines_by_samples.png'), dpi=300)
plt.show()


# ============================================================
# UNSEEN DRUGS / CELL LINES FLAGS IN EXTRA DATASET
# ============================================================

extra_drug_counts['seen_in_training_studies'] = extra_drug_counts['drug_id'].isin(seen_drugs)
extra_cell_counts['seen_in_training_studies'] = extra_cell_counts['cell_line_name'].isin(seen_cells)

extra_drug_seen_flag_summary = (
    extra_drug_counts
    .groupby('seen_in_training_studies')
    .agg(
        n_drugs=('drug_id', 'size'),
        total_occurrences=('n_occurrences', 'sum'),
        mean_occurrences=('n_occurrences', 'mean'),
    )
    .reset_index()
)

extra_cell_seen_flag_summary = (
    extra_cell_counts
    .groupby('seen_in_training_studies')
    .agg(
        n_cell_lines=('cell_line_name', 'size'),
        total_samples=('n_samples', 'sum'),
        mean_samples=('n_samples', 'mean'),
    )
    .reset_index()
)

#extra_drug_seen_flag_summary.to_csv(
#    os.path.join(CHAR_OUTPUT_DIR, 'extra_drugs_seen_flag_summary.csv'),
#    index=False
#)
#extra_cell_seen_flag_summary.to_csv(
#    os.path.join(CHAR_OUTPUT_DIR, 'extra_cell_lines_seen_flag_summary.csv'),
#    index=False
#)

display(extra_drug_seen_flag_summary)
display(extra_cell_seen_flag_summary)

plt.figure(figsize=(7, 5))
sns.countplot(
    data=extra_drug_counts,
    x='seen_in_training_studies'
)
plt.xlabel('Droga vista en los 4 estudios originales')
plt.ylabel('Numero de drogas externas')
plt.title('Drogas externas vistas vs no vistas')
plt.tight_layout()
#plt.savefig(os.path.join(CHAR_OUTPUT_DIR, 'extra_drugs_seen_vs_unseen_count.png'), dpi=300)
plt.show()

plt.figure(figsize=(7, 5))
sns.countplot(
    data=extra_cell_counts,
    x='seen_in_training_studies'
)
plt.xlabel('Cell line vista en los 4 estudios originales')
plt.ylabel('Numero de cell lines externas')
plt.title('Cell lines externas vistas vs no vistas')
plt.tight_layout()
#plt.savefig(os.path.join(CHAR_OUTPUT_DIR, 'extra_cell_lines_seen_vs_unseen_count.png'), dpi=300)
plt.show()


print(f'Analisis guardado en: {CHAR_OUTPUT_DIR}')

